Modulos con los filtros y labels ya bien (ya integra volumne y eso; hay que quitarle "TOP XXX")

In [ ]:
# ================== 1. INICIALIZACIÓN Y FUNCIONES BASE (V1) ==================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import os
import glob
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl 
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from shapely.geometry import Polygon, MultiPoint, Point # Importar Point y MultiPoint
from sklearn.cluster import KMeans 
import shutil # Importar shutil para limpieza
import zipfile # Importar zipfile para Celda 3

# --- Validar rutas locales ---
icon_met_path = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png"
output_dir = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output"

# Crear directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(icon_met_path):
    print(f"⚠️ Advertencia: No se encontró el icono en {icon_met_path}. Se usará icono por defecto.")
else:
    print(f"✅ Icono de MET encontrado: {icon_met_path}")


# --- GLOBALES DEL ESTADO ---
execution_count = 0
df_clientes = pd.DataFrame()
df_cdr = pd.DataFrame()
met_checkboxes = [] 
# --- GLOBALES PARA INTERCAMBIO DE MÓDULOS ---
g_map_path = ""       
g_excel_paths = []    
g_total_clientes_procesados = 0
g_descarga_en_progreso = False # Para el bloqueo de descarga

# --- LISTA DE COLORES GLOBAL ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# =========================================================================
# === FUNCIONES HELPER (LÓGICA) ===========================================
# =========================================================================

# --- HELPER FUNCTION PARA NORMALIZAR IDS ---
def normalizar_id(valor):
    """Convierte un valor a un ID seguro para HTML/JS."""
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    s = s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    s = s.replace('Á','A').replace('É','E').replace('Í','I').replace('Ó','O').replace('Ú','U')
    s = s.replace('ñ','n').replace('Ñ','N')
    return s

# --- FUNCIÓN HELPER PARA VORONOI ---
def _clip_voronoi(vor, bounding_poly):
    """Recorta los polígonos de Voronoi a un polígono de frontera (shapely)."""
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty:
        return clipped_polys
        
    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points): continue
        if not (0 <= region_index < len(vor.regions)): continue
        region_vertices_indices = vor.regions[region_index]
        if -1 in region_vertices_indices: continue
        valid_vertex_indices = [v for v in region_vertices_indices if 0 <= v < len(vor.vertices)]
        if len(valid_vertex_indices) < 3: continue
        region_vertices = [vor.vertices[v] for v in valid_vertex_indices]
        if len(region_vertices) < 3: continue

        try:
            poly = Polygon(region_vertices)
            if not poly.is_valid: poly = poly.buffer(0)
            if not poly.is_valid: continue

            clipped = poly.intersection(bounding_poly)
            
            if clipped.is_empty: continue
            elif clipped.geom_type == 'Polygon' and clipped.exterior is not None:
                if len(clipped.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in clipped.exterior.coords]
                else: continue
            elif clipped.geom_type == 'MultiPolygon':
                if not clipped.geoms: continue
                largest_poly = max(clipped.geoms, key=lambda p: p.area)
                if largest_poly.exterior is not None and len(largest_poly.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in largest_poly.exterior.coords]
                else: continue
            else: continue

            centroid_key = tuple(vor.points[i])
            clipped_polys[centroid_key] = clipped_coords
        except Exception as e:
            pass 
            
    return clipped_polys

# --- FUNCIÓN HELPER PARA TAMAÑO DE MARCADOR (DE CÓDIGO CELAYA) ---
def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    """
    Calcula el tamaño del marcador basado en el volumen.
    Usa escala logarítmica para mejor diferenciación.
    """
    if vol_max == vol_min or volumen <= 0:
        return tamano_min
    
    # Usar escala logarítmica para mejor diferenciación visual
    log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    
    if log_max == log_min:
        return tamano_min
        
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# =========================================================================
# === MÓDULO 1: LÓGICA DE CARGA Y VALIDACIÓN ==============================
# =========================================================================
def handle_upload(change, met_box_widget, output_upload_widget):
    global df_clientes, df_cdr, met_checkboxes 
    
    met_box = met_box_widget
    output_upload = output_upload_widget

    with output_upload:
        clear_output()
        upload_widget = change['owner']
        if not upload_widget.value:
            return

        # Adaptado para manejar diferentes formatos de FileUpload
        try:
            # Intentar obtener el archivo del widget
            if isinstance(upload_widget.value, tuple) and len(upload_widget.value) > 0:
                # Formato: tuple de FileInfo objects
                uploaded_file_info = upload_widget.value[0]
                uploaded_file_name = uploaded_file_info['name']
                uploaded_file_content = uploaded_file_info['content']
            elif isinstance(upload_widget.value, dict):
                # Formato: dict con nombres de archivo como keys
                uploaded_file_details = next(iter(upload_widget.value.values()))
                uploaded_file_name = uploaded_file_details['metadata']['name']
                uploaded_file_content = uploaded_file_details['content']
            else:
                raise ValueError("Formato de archivo no reconocido")
        except Exception as e:
            print(f"❌ Error al procesar el archivo cargado: {e}")
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
            return
        
        try:
            with open(uploaded_file_name, 'wb') as f: f.write(uploaded_file_content)
            df_excel = pd.read_excel(uploaded_file_name, sheet_name=None)
            
            if len(df_excel) < 2:
                raise ValueError("El archivo Excel debe contener al menos dos hojas (Clientes y METs/CDRs).")
                
            sheet1_name, sheet2_name = list(df_excel.keys())[0], list(df_excel.keys())[1]
            df_clientes_raw, df_cdr_raw = df_excel[sheet1_name], df_excel[sheet2_name]

            required_cols_clientes = ['CodCli', 'U_longitud', 'U_latitud', 'm3/día']
            required_cols_cdr = ['CodMET', 'U_longitud', 'U_latitud']
            
            missing_cols_cli = [col for col in required_cols_clientes if col not in df_clientes_raw.columns]
            missing_cols_cdr = [col for col in required_cols_cdr if col not in df_cdr_raw.columns]
            
            if missing_cols_cli:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet1_name}': {', '.join(missing_cols_cli)}")
            if missing_cols_cdr:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet2_name}': {', '.join(missing_cols_cdr)}")

            df_clientes_raw['U_longitud'] = pd.to_numeric(df_clientes_raw['U_longitud'], errors='coerce')
            df_clientes_raw['U_latitud'] = pd.to_numeric(df_clientes_raw['U_latitud'], errors='coerce')
            
            # Convertir m3/día: reemplazar "-" por 0 y convertir a numérico
            df_clientes_raw['m3/día'] = df_clientes_raw['m3/día'].replace('-', 0)
            df_clientes_raw['m3/día'] = pd.to_numeric(df_clientes_raw['m3/día'], errors='coerce').fillna(0)
            df_cdr_raw['U_longitud'] = pd.to_numeric(df_cdr_raw['U_longitud'], errors='coerce')
            df_cdr_raw['U_latitud'] = pd.to_numeric(df_cdr_raw['U_latitud'], errors='coerce')

            df_clientes = df_clientes_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()
            df_cdr = df_cdr_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()

            num_dropped_cli = len(df_clientes_raw) - len(df_clientes)
            num_dropped_cdr = len(df_cdr_raw) - len(df_cdr)
            if num_dropped_cli > 0: print(f"⚠️ Se eliminaron {num_dropped_cli} clientes por coordenadas inválidas.")
            if num_dropped_cdr > 0: print(f"⚠️ Se eliminaron {num_dropped_cdr} METs/CDRs por coordenadas inválidas.")

            print(f"✅ Archivo '{uploaded_file_name}' cargado exitosamente.")
            print(f"📊 Clientes válidos: {df_clientes.shape[0]} filas")
            print(f"📊 METs/CDRs válidos: {df_cdr.shape[0]} filas")

            mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
            met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
            met_box.children = [widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes 

        except ValueError as ve:
            print(f"❌ Error de validación: {ve}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error en formato de archivo.')]
        except Exception as e:
            print(f"❌ Error inesperado al cargar: {e}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
        finally:
            if 'uploaded_file_name' in locals() and os.path.exists(uploaded_file_name):
                os.remove(uploaded_file_name)

# =========================================================================
# === MÓDULO 3: LÓGICA DE CLUSTERING (Silencioso) ==========================
# =========================================================================
# (Todas las funciones de clustering permanecen EXACTAMENTE IGUALES)
def select_customers_for_met(df_all_clients, met_lat, met_lon, n_clients, analyze_all):
    if df_all_clients.empty: return pd.DataFrame()
    if analyze_all:
        return df_all_clients.copy() 
    else:
        distances = np.hypot(df_all_clients['U_latitud'] - met_lat, df_all_clients['U_longitud'] - met_lon)
        df_with_dist = df_all_clients.copy()
        df_with_dist['Distancia_al_MET'] = distances
        n_to_select = min(n_clients, len(df_with_dist))
        if n_to_select <= 0: return pd.DataFrame() 
        return df_with_dist.nsmallest(n_to_select, 'Distancia_al_MET')

def calcular_dispersion_cluster(df_cluster):
    if len(df_cluster) < 2:
        return 0.0
    coords = df_cluster[['U_longitud', 'U_latitud']].values
    distancias_pares = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = np.hypot(coords[i][0] - coords[j][0], coords[i][1] - coords[j][1])
            distancias_pares.append(dist)
    return np.mean(distancias_pares) if distancias_pares else 0.0

def calcular_limites_clientes_dinamico(dispersion, volumen_actual, min_base=20, max_base=40):
    dispersion_normalizada = min(dispersion / 0.5, 1.0) 
    volumen_normalizado = min(volumen_actual / 6.0, 1.0)
    factor_restriccion = (dispersion_normalizada + volumen_normalizado) / 2.0
    rango = max_base - min_base
    ajuste = int(rango * factor_restriccion)
    max_dinamico = max_base - ajuste
    min_dinamico = max(min_base - int(ajuste * 0.5), int(min_base * 0.5)) 
    return min_dinamico, max_dinamico

def find_initial_seeds(coords, k, random_state=42):
    n_puntos = coords.shape[0]
    if k <= 0 or n_puntos == 0:
        print("         ⚠️ No se pueden generar semillas (k<=0 o no hay puntos).")
        return np.array([])
    if n_puntos < k:
        print(f"         ⚠️ No se pueden encontrar {k} semillas con {n_puntos} puntos. Ajustando k a {n_puntos}.")
        k = n_puntos 
    try:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=20, init='k-means++').fit(coords)
        return kmeans.cluster_centers_
    except ValueError as e:
        print(f"         ❌ Error en KMeans para semillas (k={k}, n_puntos={n_puntos}): {e}")
        if n_puntos > 0:
            indices = np.random.choice(n_puntos, min(k, n_puntos), replace=False)
            print(f"         ... Usando {len(indices)} puntos aleatorios como semillas (Fallback).")
            return coords[indices]
        else:
            return np.array([])

def grow_clusters_from_seeds(df_clients_to_cluster, initial_seeds, max_size, min_size=20):
    n_clientes = len(df_clients_to_cluster)
    k = len(initial_seeds)
    if k == 0 or n_clientes == 0:
        print("         ⚠️ No hay semillas o clientes para 'grow_clusters_from_seeds'.")
        df_clients_to_cluster['Ruta_nueva'] = -1
        return df_clients_to_cluster, {}
    df_clients_to_cluster['Ruta_nueva'] = -1
    ruta_volumenes = {i: 0.0 for i in range(k)}
    ruta_counts = {i: 0 for i in range(k)} 
    df_normales = df_clients_to_cluster[df_clients_to_cluster['m3/día'] > 0].copy()
    df_comodines = df_clients_to_cluster[df_clients_to_cluster['m3/día'] == 0].copy()
    coords_normales = df_normales[['U_longitud', 'U_latitud']].values
    coords_comodines = df_comodines[['U_longitud', 'U_latitud']].values
    if len(coords_normales) > 0:
        try:
            dist_matrix = distance_matrix(coords_normales, initial_seeds)
        except ValueError as e:
            print(f"         ❌ Error calculando matriz de distancias: {e}")
            dist_matrix = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_normales])
        asignaciones_pendientes = []
        for idx_cliente, (real_idx, row) in enumerate(df_normales.iterrows()):
            volumen_cliente = row['m3/día']
            for idx_ruta in range(k):
                dist = dist_matrix[idx_cliente, idx_ruta]
                asignaciones_pendientes.append({
                    'cliente_idx': real_idx,
                    'ruta_idx': idx_ruta,
                    'distancia': dist,
                    'volumen': volumen_cliente
                })
        asignaciones_pendientes.sort(key=lambda x: x['distancia'])
        for asignacion in asignaciones_pendientes:
            cliente_idx = asignacion['cliente_idx']
            ruta_idx = asignacion['ruta_idx']
            volumen = asignacion['volumen']
            if df_clients_to_cluster.loc[cliente_idx, 'Ruta_nueva'] != -1:
                continue
            volumen_futuro = ruta_volumenes[ruta_idx] + volumen
            if volumen_futuro > 6.0:
                continue 
            if ruta_counts[ruta_idx] > 0:
                clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)
            else:
                dispersion_actual = 0.0
            min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                dispersion_actual, volumen_futuro, min_base=min_size, max_base=max_size
            )
            if ruta_counts[ruta_idx] >= max_dinamico:
                continue 
            df_clients_to_cluster.loc[cliente_idx, 'Ruta_nueva'] = ruta_idx
            ruta_volumenes[ruta_idx] = volumen_futuro
            ruta_counts[ruta_idx] += 1
    if len(coords_comodines) > 0:
        try:
            dist_matrix_comodines = distance_matrix(coords_comodines, initial_seeds)
        except ValueError as e:
            print(f"         ❌ Error calculando matriz de distancias para comodines: {e}")
            dist_matrix_comodines = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_comodines])
        for idx_comodin, (real_idx, row) in enumerate(df_comodines.iterrows()):
            distancias_rutas = dist_matrix_comodines[idx_comodin]
            rutas_ordenadas = np.argsort(distancias_rutas)
            asignado = False
            for ruta_idx in rutas_ordenadas:
                if ruta_counts[ruta_idx] > 0:
                    clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
                    dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)
                else:
                    dispersion_actual = 0.0
                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, ruta_volumenes[ruta_idx], min_base=min_size, max_base=max_size
                )
                if ruta_counts[ruta_idx] < max_dinamico:
                    df_clients_to_cluster.loc[real_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_counts[ruta_idx] += 1
                    asignado = True
                    break
            if not asignado:
                ruta_mas_cercana = rutas_ordenadas[0]
                df_clients_to_cluster.loc[real_idx, 'Ruta_nueva'] = ruta_mas_cercana
                ruta_counts[ruta_mas_cercana] += 1
    rutas_validas = {}
    for ruta_idx in range(k):
        if ruta_volumenes[ruta_idx] >= 1.0 or ruta_volumenes[ruta_idx] == 0:
            rutas_validas[ruta_idx] = ruta_counts[ruta_idx]
    return df_clients_to_cluster, rutas_validas

def adjust_small_clusters(df_clustered, cluster_counts, min_size, max_size):
    df_adjusted = df_clustered.copy()
    current_cluster_counts = cluster_counts.copy() 
    ruta_volumenes = {}
    for ruta_id in df_adjusted['Ruta_nueva'].unique():
        if ruta_id >= 0:
            clientes_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            volumen_total = clientes_ruta['m3/día'].sum()
            ruta_volumenes[ruta_id] = volumen_total
    attempts = 0
    max_attempts = 5
    while attempts < max_attempts:
        attempts += 1
        final_counts = df_adjusted['Ruta_nueva'][df_adjusted['Ruta_nueva'] != -1].value_counts()
        rutas_pequenas_ids = final_counts[(final_counts < min_size) & (final_counts > 0)].index.tolist()
        rutas_validas_ids = final_counts[final_counts >= min_size].index.tolist()
        if not rutas_pequenas_ids:
            break 
        if not rutas_validas_ids:
            print("         ⚠️ No hay rutas válidas para reasignar las pequeñas. Se mantendrán como están.")
            break 
        clientes_validos = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_validas_ids)]
        clientes_pequenos = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_pequenas_ids)]
        if clientes_validos.empty or clientes_pequenos.empty:
            print("         ⚠️ Datos válidos o pequeños vacíos en el intento de ajuste. Deteniendo.")
            break
        coords_validos = clientes_validos[['U_longitud', 'U_latitud']].values
        coords_pequenos = clientes_pequenos[['U_longitud', 'U_latitud']].values
        dist_pequeno_a_valido = distance_matrix(coords_pequenos, coords_validos)
        df_dist_reassign = pd.DataFrame(dist_pequeno_a_valido, index=clientes_pequenos.index, columns=clientes_validos.index)
        clientes_reasignados_en_intento = 0
        for idx_pequeno in clientes_pequenos.index:
            current_assignment = df_adjusted.loc[idx_pequeno, 'Ruta_nueva']
            if current_assignment in rutas_validas_ids: continue
            volumen_cliente = df_adjusted.loc[idx_pequeno, 'm3/día']
            n_neighbors_to_check = min(10, len(clientes_validos))
            vecinos_cercanos = df_dist_reassign.loc[idx_pequeno].nsmallest(min(n_neighbors_to_check, df_dist_reassign.shape[1]))
            assigned = False
            for idx_vecino, dist in vecinos_cercanos.items():
                if idx_vecino not in df_adjusted.index: continue
                ruta_vecino_id = df_adjusted.loc[idx_vecino, 'Ruta_nueva']
                volumen_vecino_actual = ruta_volumenes.get(ruta_vecino_id, 0)
                volumen_futuro = volumen_vecino_actual + volumen_cliente
                if volumen_futuro > 6.0:
                    continue 
                clientes_ruta_vecino = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_vecino_id]
                dispersion_vecino = calcular_dispersion_cluster(clientes_ruta_vecino)
                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_vecino, volumen_futuro, min_base=min_size, max_base=max_size
                )
                if current_cluster_counts.get(ruta_vecino_id, 0) < max_dinamico:
                    original_ruta = df_adjusted.loc[idx_pequeno, 'Ruta_nueva']
                    df_adjusted.loc[idx_pequeno, 'Ruta_nueva'] = ruta_vecino_id
                    current_cluster_counts[ruta_vecino_id] = current_cluster_counts.get(ruta_vecino_id, 0) + 1
                    if original_ruta in current_cluster_counts:
                        current_cluster_counts[original_ruta] -= 1
                    ruta_volumenes[ruta_vecino_id] = volumen_futuro
                    if original_ruta in ruta_volumenes:
                        ruta_volumenes[original_ruta] = max(0, ruta_volumenes[original_ruta] - volumen_cliente)
                    assigned = True
                    clientes_reasignados_en_intento += 1
                    break
            if not assigned and not vecinos_cercanos.empty:
                try:
                    idx_vecino_mas_cercano = df_dist_reassign.loc[idx_pequeno].idxmin()
                    if idx_vecino_mas_cercano not in df_adjusted.index:
                        continue
                    ruta_mas_cercana_id = df_adjusted.loc[idx_vecino_mas_cercano, 'Ruta_nueva']
                    volumen_mas_cercano_actual = ruta_volumenes.get(ruta_mas_cercana_id, 0)
                    volumen_futuro = volumen_mas_cercano_actual + volumen_cliente
                    if volumen_futuro <= 7.0:
                        original_ruta = df_adjusted.loc[idx_pequeno, 'Ruta_nueva']
                        df_adjusted.loc[idx_pequeno, 'Ruta_nueva'] = ruta_mas_cercana_id
                        current_cluster_counts[ruta_mas_cercana_id] = current_cluster_counts.get(ruta_mas_cercana_id, 0) + 1
                        if original_ruta in current_cluster_counts:
                            current_cluster_counts[original_ruta] -= 1
                        ruta_volumenes[ruta_mas_cercana_id] = volumen_futuro
                        if original_ruta in ruta_volumenes:
                            ruta_volumenes[original_ruta] = max(0, ruta_volumenes[original_ruta] - volumen_cliente)
                        clientes_reasignados_en_intento += 1
                except KeyError: pass
                except Exception as e_force: pass
            elif not assigned: pass
        if clientes_reasignados_en_intento == 0 and rutas_pequenas_ids:
            print("         ... No se pudieron reasignar más clientes pequeños en este intento. Deteniendo ajuste.")
            break 
    if attempts >= max_attempts: print(f"         ⚠️ Se alcanzó el límite de {max_attempts} intentos de ajuste. Pueden quedar rutas pequeñas.")
    return df_adjusted, current_cluster_counts


# =========================================================================
# === MÓDULO 4: CÁLCULO DE POLÍGONOS (Silencioso) ==========================
# =========================================================================
def calculate_cluster_polygons(df_clustered, method='voronoi'):
    polygons = {} 
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])
    if not assigned_ruta_ids: return polygons
    centroides_rutas = [] 
    ruta_id_map = {} 
    for ruta_id in assigned_ruta_ids:
        ruta_clientes_df = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if not ruta_clientes_df.empty:
            lon = pd.to_numeric(ruta_clientes_df['U_longitud'], errors='coerce').mean()
            lat = pd.to_numeric(ruta_clientes_df['U_latitud'], errors='coerce').mean()
            if pd.notna(lon) and pd.notna(lat):
                centroides_rutas.append([lon, lat])
                ruta_id_map[tuple([lon, lat])] = ruta_id
    poligonos_voronoi_recortados = {}
    vor = None 
    if method == 'voronoi' and len(centroides_rutas) >= 4: 
        try:
            todos_los_puntos = df_clustered[df_clustered['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
            if len(todos_los_puntos) >= 3:
                boundary_poly_shapely = MultiPoint(todos_los_puntos).convex_hull.buffer(0).buffer(0.0001) 
                unique_centroids = np.unique(np.array(centroides_rutas), axis=0)
                if len(unique_centroids) >= 4:
                    vor = Voronoi(unique_centroids, qhull_options='Qbb Qc Qz') 
                    poligonos_voronoi_recortados = _clip_voronoi(vor, boundary_poly_shapely)
                else: print("         ... No hay suficientes centroides únicos (>=4) para Voronoi.")
            else: print("         ... No hay suficientes puntos asignados (<3) para definir frontera Voronoi.")
        except Exception as e:
            print(f"         ❌ Error al calcular Voronoi: {e}. Se usará ConvexHull.")
            poligonos_voronoi_recortados = {} 
    elif method == 'voronoi':
        print(f"         ... No hay suficientes rutas (se necesitan >= 4, se encontraron {len(centroides_rutas)}) para Voronoi. Usando ConvexHull.")

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if ruta_clientes.empty: continue
        centroide_lon = ruta_clientes['U_longitud'].mean()
        centroide_lat = ruta_clientes['U_latitud'].mean()
        poly_coords = poligonos_voronoi_recortados.get(tuple([centroide_lon, centroide_lat]))
        if not poly_coords and poligonos_voronoi_recortados:
            min_dist = float('inf')
            best_match = None
            current_centroid = np.array([centroide_lon, centroide_lat])
            for vor_centroid_key, vor_poly in poligonos_voronoi_recortados.items():
                dist = np.linalg.norm(current_centroid - np.array(vor_centroid_key))
                if dist < min_dist:
                    min_dist = dist
                    best_match = vor_poly
            if min_dist < 1e-6: poly_coords = best_match
        if poly_coords:
            polygons[ruta_id] = poly_coords
        else:
            points = ruta_clientes[['U_latitud', 'U_longitud']].values
            if len(points) >= 3:
                try:
                    hull = ConvexHull(points)
                    polygons[ruta_id] = points[hull.vertices].tolist() 
                except Exception:
                    print(f"         ⚠️ No se pudo generar polígono ConvexHull para Ruta {ruta_id}.")
                    polygons[ruta_id] = None
            else: polygons[ruta_id] = None 
    return polygons

# =========================================================================
# === MÓDULO 5: GENERACIÓN DE MAPA (MODIFICADO CON ESTILOS 'CELAYA') =======
# =========================================================================
def create_map_elements(map_instance,df_clustered, cluster_polygons, colores_list, 
                        fg_poligonos_parent, fg_clientes_parent, 
                        vol_min_global, vol_max_global, # Nuevos parámetros para tamaño
                        met_name=None):
    """
    Dibuja los elementos del cluster (estilo 'Celaya') y los AÑADE a las capas de filtrado.
    Usa CircleMarker (tamaño por volumen) y popups HTML estilizados.
    """
    
    # Obtener nombre del MET si no se proporciona
    if met_name is None and 'MET_Asignado' in df_clustered.columns:
        met_name = df_clustered['MET_Asignado'].iloc[0] if not df_clustered.empty else ""
    elif met_name is None:
        met_name = ""
    
    # Limpiar el nombre del MET para evitar duplicación
    met_name_clean = met_name.strip()
    if met_name_clean.upper().startswith('MET'):
        display_name = met_name_clean
    else:
        display_name = f"MET {met_name_clean}" if met_name_clean else "MET"
    
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])

    # Configuración de tamaño de marcadores (del código 'Celaya')
    tamano_min, tamano_max = (6, 25) # Usamos el 'small' por defecto

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id].copy()
        ruta_size = len(ruta_clientes)
        if ruta_size == 0: continue

        color_ruta = colores_list[ruta_id % len(colores_list)] 
        
        # --- 1. Crear el FeatureGroup para el FILTRO INDIVIDUAL ---
        fg_ruta_individual = folium.FeatureGroup(name=f"{display_name} - Ruta {ruta_id}", show=True)
        
        # --- 2. Dibujar Polígono (Voronoi o ConvexHull) ---
        poly_coords = cluster_polygons.get(ruta_id)
        if poly_coords:
            tooltip_text = f"Ruta {ruta_id} - {met_name}"
            
            # Crear el objeto polígono
            poly_obj = folium.Polygon(
                locations=poly_coords,
                color=color_ruta,
                weight=2,
                fill=True,
                fill_color=color_ruta,
                fill_opacity=0.3, 
                tooltip=tooltip_text
            )
            
            # Añadir el polígono a DOS capas:
            #poly_obj.add_to(fg_poligonos_parent)#
            poly_obj_clon = folium.Polygon(
                locations=poly_coords, color=color_ruta, weight=2,
                fill=True, fill_color=color_ruta, fill_opacity=0.3, 
                tooltip=tooltip_text
            )
            poly_obj_clon.add_to(fg_ruta_individual)

        # --- 3. (NUEVO) Dibujar Polígono SOLO DE CLIENTES (de 'Celaya') ---
        clientes_ruta_df = ruta_clientes[ruta_clientes['Ruta_nueva'] == ruta_id]
        if len(clientes_ruta_df) >= 3:
            try:
                puntos_solo_clientes = [Point(row['U_longitud'], row['U_latitud']) for _, row in clientes_ruta_df.iterrows()]
                multipoint_clientes = MultiPoint(puntos_solo_clientes)
                hull_clientes = multipoint_clientes.convex_hull
                
                if hull_clientes.geom_type == 'Polygon':
                    coords_polygon_clientes = [(lat, lon) for lon, lat in hull_clientes.exterior.coords]
                    area_popup_clientes = f'''
                    <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                        <div style="font-size: 14px; font-weight: bold;">👥 Área Solo Clientes</div>
                        <div style="font-size: 12px; margin-top: 4px;">🏠 MET {met_name_clean} - 🛣️ Ruta {ruta_id}</div>
                        <div style="font-size: 11px; margin-top: 2px;">👨‍💼 {len(clientes_ruta_df)} clientes</div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📦 Vol: {clientes_ruta_df['m3/día'].sum():,.2f} m³</div>
                    </div>
                    '''
                    
                    poly_clientes_obj = folium.Polygon(
                        locations=coords_polygon_clientes,
                        color=color_ruta,
                        weight=1,
                        opacity=0.8,
                        fillColor=color_ruta,
                        fillOpacity=0.1, # Más transparente
                        popup=folium.Popup(area_popup_clientes, max_width=250),
                        dash_array='5, 5' # Línea punteada
                    )
                    
                    # Añadir polígono de clientes a AMBAS capas
                    #poly_clientes_obj.add_to(fg_poligonos_parent)#
                    poly_clientes_obj_clon = folium.Polygon(
                        locations=coords_polygon_clientes,
                        color=color_ruta, weight=1, opacity=0.8,
                        fillColor=color_ruta, fillOpacity=0.1,
                        popup=folium.Popup(area_popup_clientes, max_width=250),
                        dash_array='5, 5'
                    )
                    poly_clientes_obj_clon.add_to(fg_ruta_individual)
            except Exception as e:
                print(f"         ⚠️ No se pudo generar polígono 'solo clientes' para Ruta {ruta_id}: {e}")


        # --- 4. Dibujar Marcadores (Círculos estilo 'Celaya') ---
        for _, cliente_row in ruta_clientes.iterrows(): 
            codcli = cliente_row['CodCli']
            nombre = cliente_row.get('Nombre', 'N/A')
            ruta_original_val = cliente_row.get('Ruta', 'N/A') 
            ruta_original = ruta_original_val if pd.notna(ruta_original_val) else 'N/A'
            volumen = cliente_row['m3/día']
            
            # Calcular tamaño basado en volumen
            tamano_marcador = obtener_tamano_marcador(volumen, vol_min_global, vol_max_global, tamano_min, tamano_max)
            
            # Calcular percentil de volumen (para el popup)
            if (vol_max_global - vol_min_global) > 0:
                percentil_vol = ((volumen - vol_min_global) / (vol_max_global - vol_min_global) * 100)
            else:
                percentil_vol = 50 

            # Crear el Popup HTML (estilo 'Celaya', adaptado)
            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                    👨‍💼 {codcli}
                </div>
                <div style="font-size:14px; color:#333; margin-bottom:4px;">
                    {nombre if pd.notna(nombre) else 'N/A'}
                </div>
                <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📦 Volumen: <b>{volumen:,.3f} m³</b> (Top {100-percentil_vol:.0f}°)
                </div>
                
                <div style="background: #eee; height: 6px; border-radius: 3px; margin-bottom: 8px;">
                    <div style="background: {color_ruta}; height: 100%; width: {max(0, min(100, percentil_vol)):.0f}%; border-radius: 3px;"></div>
                </div>

                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                    🗺️ Ruta (Orig): <b>{ruta_original}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                </div>
            </div>
            '''
            
            # Crear el objeto marcador (CircleMarker)
            marker_obj = folium.CircleMarker(
                location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=300),
                radius=tamano_marcador,
                color='#333333', # Borde oscuro (de 'Celaya')
                weight=1,
                fillColor=color_ruta, # Relleno basado en la ruta
                fillOpacity=0.85,
            )

            # Añadir el marcador a DOS capas:
            #marker_obj.add_to(fg_clientes_parent)#
            
            # Clonar para la capa individual
            marker_obj_clon = folium.CircleMarker(
                location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=300),
                radius=tamano_marcador,
                color='#333333', weight=1,
                fillColor=color_ruta, fillOpacity=0.85,
            )
            marker_obj_clon.add_to(fg_ruta_individual)

        # --- 5. Añadir el grupo individual al gran "Filtrar por Ruta" ---
        fg_ruta_individual.add_to(map_instance)

    return

# =========================================================================
# === MÓDULO 6: GENERACIÓN DE EXCEL CONSOLIDADO (Todos los METs) ==========
# =========================================================================
def generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir):
    if not export_data or not summary_data:
        print(f"         ... No hay datos para exportar a Excel.")
        return None
    df_export = pd.DataFrame(export_data)
    df_resumen = pd.DataFrame(summary_data)
    if df_export.empty or df_resumen.empty:
        print(f"         ... DataFrames vacíos, no se puede generar Excel.")
        return None
    excel_filename = f'rutas_consolidadas_{timestamp}.xlsx'
    excel_path = os.path.join(output_dir, excel_filename)
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_resumen_sorted = df_resumen.sort_values(by=['MET', 'Ruta_nueva'])
            df_resumen_sorted.to_excel(writer, sheet_name='Resumen General', index=False)
            df_export_sorted = df_export.sort_values(by=['MET', 'Ruta_nueva', 'Codigo'])
            df_export_sorted.to_excel(writer, sheet_name='Todos los Clientes', index=False)
            for met_name in mets_seleccionados:
                df_met = df_export[df_export['MET'] == met_name].sort_values(by=['Ruta_nueva', 'Codigo'])
                if not df_met.empty:
                    sheet_name = f'MET_{normalizar_id(met_name)}'[:31]
                    df_met.to_excel(writer, sheet_name=sheet_name, index=False)
        format_excel(excel_path)
        return excel_path
    except Exception as e:
        print(f"         ❌ Error al generar Excel consolidado: {e}")
        if os.path.exists(excel_path):
            try: os.remove(excel_path)
            except: pass
        return None

def format_excel(excel_path):
    if not os.path.exists(excel_path):
        print(f"         ⚠️ No se encontró el archivo Excel para formatear: {excel_path}")
        return
    try:
        wb = openpyxl.load_workbook(excel_path)
        header_font = Font(bold=True, color='FFFFFF', size=11)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        summary_fill = PatternFill('solid', fgColor='FFD966')
        border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'), 
                        top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))
        align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
        align_left = Alignment(horizontal='left', vertical='center', wrap_text=False)
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws.max_row <= 1: continue
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align_center
                cell.border = border
                if cell.value: 
                    header_text = str(cell.value)
                    icon = ""
                    if 'Ruta_Original' in header_text: icon = '🗺️ '
                    elif 'Codigo' in header_text or 'CodCli' in header_text: icon = '👨‍💼 ' 
                    elif 'Clientes' in header_text: icon = '👥 ' 
                    elif 'MET' in header_text: icon = '🏠 '
                    elif 'Nombre' in header_text: icon = '📚 '
                    elif 'Ruta_nueva' in header_text or 'Ruta_ID' in header_text: icon = '🚚 '
                    elif 'Volumen' in header_text or 'm3' in header_text: icon = '📦 '
                    elif 'Centroide' in header_text: icon = '📍 '
                    elif 'Longitud' in header_text: icon = '📊 '
                    elif 'Latitud' in header_text: icon = '📊 '
                    cleaned_header = header_text.lstrip('🗺️👨‍💼🏠📚📦📊👥📍 ')
                    cell.value = icon + cleaned_header
            is_summary_sheet = ('Resumen' in sheet_name)
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for col_idx, cell in enumerate(row, start=1):
                    cell.fill = summary_fill if is_summary_sheet else cell_fill
                    cell.border = border
                    header_cell = ws.cell(row=1, column=col_idx)
                    header_val = header_cell.value if header_cell else None
                    if header_val and any(keyword in str(header_val) for keyword in ['Nombre', 'Codigo', 'CodCli']):
                        cell.alignment = align_left
                    else:
                        cell.alignment = align_center
                        if header_val and any(coord in str(header_val) for coord in ['Longitud', 'Latitud']):
                            cell.number_format = '0.000000'
            for col_idx, column_cells in enumerate(ws.columns, 1):
                max_length = 0
                header_text = str(ws.cell(row=1, column=col_idx).value).lstrip('🗺️👨‍💼🏠📚📦📊👥 ')
                max_length = len(header_text)
                for i, cell in enumerate(column_cells):
                    if i == 0: continue
                    if i > 50: break 
                    try:
                        if cell.value is not None:
                            if isinstance(cell.value, (int, float)) and cell.number_format and cell.number_format != 'General':
                                if '0.000000' in cell.number_format: cell_text = f"{cell.value:.6f}"
                                else: cell_text = str(cell.value)
                            else: cell_text = str(cell.value)
                            lines = cell_text.split('\n')
                            max_line_length = max(len(line) for line in lines) if lines else 0
                            max_length = max(max_length, max_line_length)
                    except: pass
                adjusted_width = min(max(12, max_length + 3), 45) 
                ws.column_dimensions[get_column_letter(col_idx)].width = adjusted_width
        wb.save(excel_path)
    except Exception as e:
        print(f"         ⚠️ Error al aplicar formato al Excel '{excel_path}': {e}")
        
print("✅ CELDA 1: Inicialización y funciones (MODIFICADAS con aspecto 'Celaya') cargadas.")
print(f"📁 Directorio de salida: {output_dir}")
print(f"🎯 Icono MET: {'✅ Encontrado' if os.path.exists(icon_met_path) else '⚠️ No encontrado'}")

In [ ]:
# ================== 2. EJECUCIÓN DEL ANÁLISIS (Corregido) ==================

# --- Definición de Widgets ---
upload_widget = widgets.FileUpload(accept='.xlsx', multiple=False, description='Subir Excel')
met_box = widgets.VBox([widgets.Label('⏳ Esperando archivo Excel...')])
output_upload = widgets.Output()
min_clientes_cluster = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Min Clientes:', continuous_update=False)
max_clientes_cluster = widgets.IntSlider(value=40, min=10, max=100, step=1, description='Max Clientes:', continuous_update=False)
total_clientes_analizar = widgets.IntText(value=500, description='Total Clientes:', disabled=False)
analizar_todos_clientes = widgets.Checkbox(value=False, description='Analizar todos los clientes disponibles', indent=False)
param_button = widgets.Button(description='Ejecutar Análisis', button_style='success', disabled=False)
output_param = widgets.Output()

# --- Registrar handler de carga ---
upload_widget.observe(lambda change: handle_upload(change, met_box, output_upload), names='value')

# --- Función Principal de Ejecución ---
def ejecutar_analisis(b):
    global execution_count, df_clientes, df_cdr, met_checkboxes, colores
    global g_map_path, g_excel_paths, g_total_clientes_procesados
    
    execution_count += 1
    
    with output_param:
        clear_output(wait=True)
        
        # Validaciones
        if df_clientes.empty or df_cdr.empty:
            print("❌ Carga un archivo Excel primero.")
            return
        
        mets_seleccionados = [cb.description for cb in met_checkboxes if cb.value]
        if not mets_seleccionados:
            print("❌ Selecciona al menos un MET.")
            return
        
        min_size = min_clientes_cluster.value
        max_size = max_clientes_cluster.value
        n_clientes_param = total_clientes_analizar.value
        analizar_todos = analizar_todos_clientes.value
        
        if min_size >= max_size:
            print("❌ El mínimo debe ser menor que el máximo.")
            return
        
        print(f"\n{'='*70}")
        print(f"🚀 INICIANDO ANÁLISIS DE RUTAS - Ejecución #{execution_count}")
        print(f"{'='*70}")
        print(f"📊 Parámetros:")
        print(f"   • METs seleccionados: {len(mets_seleccionados)} ({', '.join(mets_seleccionados)})")
        print(f"   • Clientes por ruta: {min_size} - {max_size}")
        print(f"   • Modo: {'TODOS los clientes' if analizar_todos else f'{n_clientes_param} clientes más cercanos'}")
        print(f"{'='*70}\n")
        
        g_map_path = ""
        g_excel_paths = []
        g_total_clientes_procesados = 0
        
        # === PROCESAMIENTO POR MET ===
        export_data = []
        summary_data = []
        
        # Crear mapa base
        if df_cdr.empty:
            print("❌ No hay datos de METs.")
            return
        
        centro_lat = df_cdr['U_latitud'].mean()
        centro_lon = df_cdr['U_longitud'].mean()
        mapa_base = folium.Map(location=[centro_lat, centro_lon], zoom_start=6, tiles='OpenStreetMap')
        
        # Preparar colores
        colores_list = colores.copy()
        random.shuffle(colores_list)
        
        
        # --- CAMBIO: Calcular rangos de volumen globales (para tamaño de marcadores) ---
        print("📊 Calculando rangos de volumen para la selección...")
        clientes_seleccionados_global = pd.DataFrame()
        for met_seleccionado in mets_seleccionados:
            met_info_temp = df_cdr[df_cdr['CodMET'] == met_seleccionado]
            if not met_info_temp.empty:
                met_lat_temp = met_info_temp.iloc[0]['U_latitud']
                met_lon_temp = met_info_temp.iloc[0]['U_longitud']
                df_clientes_met_temp = select_customers_for_met(df_clientes, met_lat_temp, met_lon_temp, n_clientes_param, analizar_todos)
                clientes_seleccionados_global = pd.concat([clientes_seleccionados_global, df_clientes_met_temp])

        if not clientes_seleccionados_global.empty:
            # Quitar duplicados por si 2 METs seleccionan al mismo cliente
            clientes_seleccionados_global.drop_duplicates(subset=['CodCli'], inplace=True)
            
            # Usar solo volúmenes mayores a 0 para un rango más realista
            volumenes_positivos = clientes_seleccionados_global[clientes_seleccionados_global['m3/día'] > 0]['m3/día']
            if not volumenes_positivos.empty:
                vol_min_global = volumenes_positivos.min()
                vol_max_global = volumenes_positivos.max()
            else:
                vol_min_global = 0
                vol_max_global = 0
                
            print(f"   • 📦 Rango de Volumen (m³/día) para tamaño: {vol_min_global:,.3f} - {vol_max_global:,.3f}")
            print(f"   • 👥 Clientes únicos en selección: {len(clientes_seleccionados_global)}")
        else:
            vol_min_global = 0
            vol_max_global = 0
            print("⚠️ No se encontraron clientes en la selección actual para calcular rangos.")
        # --- FIN CAMBIO ---


        # === ESTRUCTURA DE CAPAS DEL MAPA ===
        for idx_met, met_seleccionado in enumerate(mets_seleccionados):
            print(f"\n{'─'*70}")
            print(f"🏠 Procesando MET: {met_seleccionado} ({idx_met + 1}/{len(mets_seleccionados)})")
            print(f"{'─'*70}")
            
            # Obtener info del MET
            met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado]
            if met_info.empty:
                print(f"⚠️ No se encontró información para el MET '{met_seleccionado}'. Saltando.")
                continue
            
            met_lat = met_info.iloc[0]['U_latitud']
            met_lon = met_info.iloc[0]['U_longitud']
            
            # Seleccionar clientes
            df_clientes_met = select_customers_for_met(df_clientes, met_lat, met_lon, n_clientes_param, analizar_todos)
            
            if df_clientes_met.empty:
                print(f"⚠️ No hay clientes para el MET '{met_seleccionado}'.")
                continue
            
            print(f"✅ Clientes seleccionados: {len(df_clientes_met)}")
            
            # Clustering (Funcionalidad principal sin cambios)
            coords = df_clientes_met[['U_longitud', 'U_latitud']].values
            k_inicial = max(1, len(df_clientes_met) // max_size)
            
            print(f"🔬 Iniciando clustering (k_inicial = {k_inicial})...")
            initial_seeds = find_initial_seeds(coords, k_inicial)
            
            if len(initial_seeds) == 0:
                print(f"❌ No se pudieron generar semillas para el MET '{met_seleccionado}'.")
                continue
            
            df_clientes_met, cluster_counts = grow_clusters_from_seeds(df_clientes_met, initial_seeds, max_size, min_size)
            df_clientes_met, cluster_counts = adjust_small_clusters(df_clientes_met, cluster_counts, min_size, max_size)
            
            # Calcular polígonos
            cluster_polygons = calculate_cluster_polygons(df_clientes_met, method='voronoi')
            
            # Validar rutas
            final_counts = df_clientes_met['Ruta_nueva'][df_clientes_met['Ruta_nueva'] != -1].value_counts()
            rutas_validas = final_counts[final_counts >= min_size].index.tolist()
            
            print(f"✅ Rutas generadas: {len(rutas_validas)}")
            
            # Filtrar solo clientes en rutas válidas
            df_clientes_met_validos = df_clientes_met[df_clientes_met['Ruta_nueva'].isin(rutas_validas)].copy()
            
            if df_clientes_met_validos.empty:
                print(f"⚠️ No hay rutas válidas para el MET '{met_seleccionado}'.")
                continue
            
            # Preparar nombre limpio del MET (sin duplicar "MET")
            met_name_clean = str(met_seleccionado).strip()
            if met_name_clean.upper().startswith('MET'):
                display_name = met_name_clean
            else:
                display_name = f"MET {met_name_clean}" if met_name_clean else "MET"
            
            # === CREAR ESTRUCTURA DE CAPAS PARA ESTE MET ===
            fg_poligonos_todos = folium.FeatureGroup(name=f"_poligonos_{normalizar_id(met_seleccionado)}", show=True, control=False)
            fg_clientes_todos = folium.FeatureGroup(name=f"_clientes_{normalizar_id(met_seleccionado)}", show=True, control=False)
            
            # --- CAMBIO: Popup de MET con estilo 'Celaya' ---
            
            # Calcular estadísticas agregadas para este MET
            clientes_met_total = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] >= 0]
            volumen_total_met = clientes_met_total['m3/día'].sum()
            total_clientes_met = len(clientes_met_total)

            popup_html_met = f'''
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                    <div style="font-size: 22px; font-weight: bold;">🏠 {met_name_clean}</div>
                </div>
                
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                    <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                        <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                        <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                    </div>
                    <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                        <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                        <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                    </div>
                </div>
                
                <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                    <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                    <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_validas)}</div>
                </div>
            </div>
            '''
            # --- FIN CAMBIO ---
            
            # --- CORRECCIÓN: Capa de ícono de MET ---
            fg_met_icon = folium.FeatureGroup(name=f"{display_name} - 🏠 MET", show=True)
            
            # Añadir marcador del MET a su propia capa filtrable
            if os.path.exists(icon_met_path):
                icon_met_obj = CustomIcon(icon_met_path, icon_size=(40, 40), icon_anchor=(20, 40), popup_anchor=(0, -40))
                folium.Marker(
                    location=[met_lat, met_lon],
                    popup=folium.Popup(popup_html_met, max_width=420),
                    icon=icon_met_obj,
                    tooltip=f"MET: {met_seleccionado}"
                ).add_to(fg_met_icon) # <-- CORREGIDO
            else:
                folium.Marker(
                    location=[met_lat, met_lon],
                    popup=folium.Popup(popup_html_met, max_width=420),
                    icon=folium.Icon(color='red', icon='home', prefix='glyphicon'),
                    tooltip=f"MET: {met_seleccionado}"
                ).add_to(fg_met_icon) # <-- CORREGIDO
            
            # --- CORRECCIÓN: Llamada a create_map_elements con argumentos correctos ---
            create_map_elements(
                mapa_base,                 # <-- 1. mapa_base va primero
                df_clientes_met_validos,
                cluster_polygons,
                colores_list,
                fg_poligonos_todos,
                fg_clientes_todos,
                vol_min_global,
                vol_max_global,
                met_seleccionado
            )
            # --- FIN CORRECCIÓN ---
            
            # Añadir las capas al mapa
            fg_poligonos_todos.add_to(mapa_base)
            fg_clientes_todos.add_to(mapa_base)
            fg_met_icon.add_to(mapa_base) # <-- CORREGIDO

            # Exportar datos
            g_total_clientes_procesados += len(df_clientes_met_validos)
            
            for ruta_id in rutas_validas:
                ruta_clientes = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] == ruta_id]
                n_clientes_ruta = len(ruta_clientes)
                volumen_total = ruta_clientes['m3/día'].sum()
                
                # Datos de resumen por ruta
                summary_data.append({
                    'MET': met_seleccionado,
                    'Ruta_nueva': ruta_id,
                    'Num_Clientes': n_clientes_ruta,
                    'Volumen_Total_m3': round(volumen_total, 2),
                    'Centroide_Lat': ruta_clientes['U_latitud'].mean(),
                    'Centroide_Lon': ruta_clientes['U_longitud'].mean()
                })
                
                # Datos de clientes
                for _, cliente in ruta_clientes.iterrows():
                    export_data.append({
                        'MET': met_seleccionado,
                        'Ruta_nueva': ruta_id,
                        'Codigo': cliente['CodCli'],
                        'Nombre': cliente.get('Nombre', 'N/A'),
                        'Ruta_Original': cliente.get('Ruta', 'N/A'),
                        'm3_dia': cliente['m3/día'],
                        'Latitud': cliente['U_latitud'],
                        'Longitud': cliente['U_longitud']
                    })
            
            print(f"✅ MET '{met_seleccionado}' completado: {len(rutas_validas)} rutas, {len(df_clientes_met_validos)} clientes")
        
        # === GUARDAR MAPA ===
        folium.LayerControl(collapsed=False).add_to(mapa_base)
        
        # --- CAMBIO: Estilos CSS de 'Celaya' para filtros ---
        estilos_css = r'''
        <style>
        /* Panel de Resumen Fijo - Esquina Superior Izquierda (de 'Celaya') */
        #panel-resumen {
            position: fixed;
            top: 80px;
            left: 10px;
            background: white;
            padding: 12px 16px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            z-index: 1000;
            max-width: 280px;
            font-family: Arial, sans-serif;
        }
        #panel-resumen h3 {
            margin: 0 0 10px 0;
            font-size: 16px;
            color: #333;
            border-bottom: 2px solid #007bff;
            padding-bottom: 6px;
        }
        #panel-resumen-content {
            font-size: 13px;
            line-height: 1.6;
            color: #555;
        }
        .resumen-item {
            display: flex;
            justify-content: space-between;
            padding: 4px 0;
            border-bottom: 1px solid #eee;
        }
        .resumen-item:last-child {
            border-bottom: none;
            font-weight: bold;
            margin-top: 6px;
            padding-top: 8px;
            border-top: 2px solid #007bff;
        }
        .resumen-label {
            color: #666;
        }
        .resumen-value {
            color: #007bff;
            font-weight: 600;
        }
        
        /* CSS para scroll y botones de 'Celaya' */
        .leaflet-control-layers-list {
            max-height: 400px;
            overflow-y: auto;
            overflow-x: hidden;
            width: 100%;
            min-width: 200px;
        }
        .layer-control-buttons {
            padding: 8px;
            border-bottom: 1px solid #ddd;
            background: #f8f8f8;
            display: flex;
            gap: 5px;
        }
        .layer-btn {
            padding: 6px 10px;
            font-size: 11px;
            border: 1px solid #ccc;
            background: white;
            cursor: pointer;
            border-radius: 4px;
            flex: 1;
            text-align: center;
            font-weight: 600;
            transition: all 0.2s;
        }
        .layer-btn:hover {
            background: #e6e6e6;
            transform: translateY(-1px);
        }
        .layer-btn.select-all {
            background: #4CAF50;
            color: white;
            border-color: #45a049;
        }
        .layer-btn.select-all:hover {
            background: #45a049;
        }
        .layer-btn.deselect-all {
            background: #f44336;
            color: white;
            border-color: #da190b;
        }
        .layer-btn.deselect-all:hover {
            background: #da190b;
        }
        .met-buttons-row {
            padding: 6px 8px;
            border-bottom: 1px solid #ddd;
            background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
            display: flex;
            gap: 4px;
            flex-wrap: wrap;
        }
        .met-btn {
            padding: 5px 10px;
            font-size: 10px;
            border: 2px solid #2196F3;
            background: white;
            color: #1976D2;
            cursor: pointer;
            border-radius: 6px;
            flex: 1;
            text-align: center;
            min-width: 60px;
            font-weight: 600;
            transition: all 0.2s;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .met-btn:hover {
            background: #2196F3;
            color: white;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0,0,0,0.2);
        }
        .ruta-buttons-row {
            padding: 6px 8px;
            border-bottom: 1px solid #ddd;
            background: linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%);
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(45px, 1fr));
            gap: 4px;
            max-width: 100%;
        }
        .ruta-btn {
            padding: 5px 8px;
            font-size: 10px;
            border: 2px solid #9C27B0;
            background: white;
            color: #7B1FA2;
            cursor: pointer;
            border-radius: 6px;
            text-align: center;
            min-width: 45px;
            font-weight: 700;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
            transition: all 0.2s;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .ruta-btn:hover {
            background: #9C27B0;
            color: white;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0,0,0,0.2);
        }
        /* Re-utilizamos los estilos de búsqueda de 'Celaya' */
        .search-ruta-container {
            display: flex;
            gap: 5px;
            padding: 8px;
            background: #f0f8ff;
            border-bottom: 1px solid #b3d9ff;
            margin-bottom: 8px;
            align-items: center;
        }
        .search-ruta-input {
            flex: 1;
            padding: 6px 10px;
            border: 1px solid #ced4da;
            border-radius: 4px;
            font-size: 12px;
            outline: none;
        }
        .search-ruta-input:focus {
            border-color: #0066cc;
            box-shadow: 0 0 0 2px rgba(0, 102, 204, 0.1);
        }
        .search-ruta-btn {
            padding: 6px 15px;
            border: 1px solid #0066cc;
            border-radius: 4px;
            background: #0066cc;
            color: white;
            cursor: pointer;
            font-size: 12px;
            font-weight: 500;
            transition: all 0.2s;
            white-space: nowrap;
        }
        .search-ruta-btn:hover {
            background: #0052a3;
            border-color: #0052a3;
        }
        .search-ruta-clear {
            padding: 6px 12px;
            border: 1px solid #dc3545;
            border-radius: 4px;
            background: white;
            color: #dc3545;
            cursor: pointer;
            font-size: 12px;
            font-weight: 500;
            transition: all 0.2s;
        }
        .search-ruta-clear:hover {
            background: #dc3545;
            color: white;
        }
        /* Oculta la lista original de checkboxes */
        .leaflet-control-layers-overlays {
            display: none;
        }
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))
        
        # --- JAVASCRIPT PARA PANELES PERSONALIZADOS Y FILTROS ---
        
        paneles_js = r'''
        <script>
        // Crear Panel de Resumen
        function crearPanelResumen() {
            const panel = document.createElement('div');
            panel.id = 'panel-resumen';
            panel.innerHTML = `
                <h3>📊 Resumen de Rutas</h3>
                <div id="panel-resumen-content"></div>
            `;
            document.body.appendChild(panel);
            return panel;
        }
        
        // Inicializar paneles y controles
        function inicializarPaneles(mapInstance) { // <-- Acepta la instancia del mapa
            console.log('=== Inicializando paneles personalizados ===');
            if (document.getElementById('panel-resumen')) {
                console.log('ℹ️ Paneles ya inicializados');
                return true;
            }
            const panelResumen = crearPanelResumen();
            
            // Usamos la instancia del mapa que nos pasó el "polling"
            const map = mapInstance;
            
            if (!map) {
                console.error('❌ Instancia del mapa (mapInstance) es nula.');
                return false;
            }
            
            let totalRutas = 0;
            const metRutas = {};
            map.eachLayer(function(layer) {
                if (layer._url) return;
                if (!layer.options || !layer.options.pane) return;
                const layerName = layer.options.attribution || '';
                const rutaMatch = layerName.match(/^(.+?)\s+-\s+Ruta\s+(\d+)$/);
                if (rutaMatch) {
                    const metNombre = rutaMatch[1];
                    totalRutas++;
                    if (!metRutas[metNombre]) metRutas[metNombre] = 0;
                    metRutas[metNombre]++;
                }
            });
            const resumenHTML = [];
            Object.keys(metRutas).sort().forEach(function(metNombre) {
                resumenHTML.push(`
                    <div class="resumen-item">
                        <span class="resumen-label">${metNombre}:</span>
                        <span class="resumen-value">${metRutas[metNombre]} rutas</span>
                    </div>
                `);
            });
            resumenHTML.push(`
                <div class="resumen-item">
                    <span class="resumen-label">Total:</span>
                    <span class="resumen-value">${totalRutas} rutas</span>
                </div>
            `);
            document.getElementById('panel-resumen-content').innerHTML = resumenHTML.join('');
            console.log('✅ Paneles personalizados inicializados');
            return true;
        }

        // Lógica de RESTRICCIÓN (POLLING) ESTRICTA para PANELES
        let panelPollCount = 0;
        function tryInitPanels() {
            // RESTRICCIÓN MÁS ROBUSTA: Espera a que Leaflet (L) esté listo Y
            // a que el DIV del mapa (.leaflet-container) tenga la instancia del mapa adjunta.
            const mapElement = document.querySelector('.leaflet-container');
            
            let mapInstance = null;
            if (mapElement && mapElement._leaflet_map) { // <-- MÉTODO ROBUSTO
                mapInstance = mapElement._leaflet_map;
            }

            // Comprobar si L y la instancia del mapa están listos
            if (typeof L !== 'undefined' && L.map && mapInstance) {
                console.log('✅ (Poll) Leaflet (L) y MapInstance (_leaflet_map) listos. Iniciando paneles...');
                inicializarPaneles(mapInstance); // <-- ¡Importante! Pasamos la instancia
            } else if (panelPollCount < 40) { // Esperar max 20 segundos
                panelPollCount++;
                console.log('Poll #' + panelPollCount + ': Esperando a L y _leaflet_map...');
                setTimeout(tryInitPanels, 500); // Reintentar cada 500ms
            } else {
                console.error('❌ No se pudo cargar la instancia del mapa (_leaflet_map) después de 20 segundos.');
            }
        }
        
        // Iniciar el sondeo
        tryInitPanels();
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(paneles_js))
        
        # --- JAVASCRIPT PARA FILTROS PERSONALIZADOS ---
        # (Lógica de Polling robusta de 'Celaya')
        filtros_js = r'''
        <script>
        function inicializarFiltros() {
            console.log('=== Inicializando filtros personalizados (Estilo Celaya) ===');
            
            const layerControl = document.querySelector('.leaflet-control-layers');
            if (!layerControl) {
                console.error('❌ No se encontró .leaflet-control-layers');
                return false;
            }
            const layersList = layerControl.querySelector('.leaflet-control-layers-list');
            if (!layersList) {
                console.error('❌ No se encontró .leaflet-control-layers-list');
                return false;
            }
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
            if (!overlaysDiv) {
                console.error('❌ No se encontró .leaflet-control-layers-overlays');
                return false;
            }
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
            
            console.log('✅ Total de checkboxes encontrados:', checkboxes.length);
            
            if (layersList.querySelector('.layer-control-buttons')) {
                console.log('ℹ️ Filtros ya inicializados');
                return true;
            }
            
            console.log('📋 Listando todas las capas:');
            checkboxes.forEach(function(checkbox, index) {
                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                console.log('   ' + (index + 1) + '. "' + label + '"');
            });
            
            // === BOTONES TODO/NADA ===
            const buttonsDiv = document.createElement('div');
            buttonsDiv.className = 'layer-control-buttons';
            const selectAllBtn = document.createElement('button');
            selectAllBtn.textContent = '✓ Todo';
            selectAllBtn.className = 'layer-btn select-all';
            selectAllBtn.title = 'Seleccionar todas las capas';
            const deselectAllBtn = document.createElement('button');
            deselectAllBtn.textContent = '✗ Nada';
            deselectAllBtn.className = 'layer-btn deselect-all';
            deselectAllBtn.title = 'Deseleccionar todas las capas';
            
            selectAllBtn.onclick = function(e) {
                e.preventDefault(); e.stopPropagation();
                checkboxes.forEach(function(cb) { if (!cb.checked) cb.click(); });
            };
            deselectAllBtn.onclick = function(e) {
                e.preventDefault(); e.stopPropagation();
                checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
            };
            
            buttonsDiv.appendChild(selectAllBtn);
            buttonsDiv.appendChild(deselectAllBtn);
            layersList.insertBefore(buttonsDiv, overlaysDiv);
            
            // === FILTRO DE BÚSQUEDA POR RUTA ID (Lógica de Celaya) ===
            const searchContainer = document.createElement('div');
            searchContainer.className = 'search-ruta-container';
            const searchLabel = document.createElement('span');
            searchLabel.textContent = '🔍';
            searchLabel.style.fontSize = '14px';
            const searchInput = document.createElement('input');
            searchInput.type = 'text';
            searchInput.className = 'search-ruta-input';
            searchInput.placeholder = 'Buscar por Ruta ID (ej: 0, 1, 2...)';
            searchInput.title = 'Ingresa el número de ruta para filtrar';
            const searchBtn = document.createElement('button');
            searchBtn.textContent = 'Buscar';
            searchBtn.className = 'search-ruta-btn';
            searchBtn.title = 'Filtrar por Ruta ID';
            const clearBtn = document.createElement('button');
            clearBtn.textContent = 'Limpiar';
            clearBtn.className = 'search-ruta-clear';
            clearBtn.title = 'Limpiar búsqueda';
            
            const filtrarPorRutaId = function() {
                const rutaId = searchInput.value.trim();
                if (rutaId === '') { alert('Por favor ingresa un número de ruta'); return; }
                console.log('🔍 Filtrando capas con Ruta ID:', rutaId);
                checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                setTimeout(function() {
                    let encontradas = 0;
                    // Patrón de 'Celaya' adaptado (termina con "Ruta X")
                    const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$');
                    checkboxes.forEach(function(checkbox) {
                        const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                        if (rutaPattern.test(label)) {
                            console.log('  ✓ Seleccionando:', label);
                            if (!checkbox.checked) { checkbox.click(); encontradas++; }
                        }
                    });
                    if (encontradas === 0) {
                        console.warn('⚠️ No se encontraron capas con Ruta ID ' + rutaId);
                        alert('No se encontraron capas con Ruta ID: ' + rutaId);
                    }
                }, 150);
            };
            
            searchBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); filtrarPorRutaId(); };
            searchInput.addEventListener('keypress', function(e) { if (e.key === 'Enter') { e.preventDefault(); filtrarPorRutaId(); } });
            clearBtn.onclick = function(e) {
                e.preventDefault(); e.stopPropagation();
                searchInput.value = '';
                checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                console.log('🧹 Búsqueda limpiada');
            };
            
            searchContainer.appendChild(searchLabel);
            searchContainer.appendChild(searchInput);
            searchContainer.appendChild(searchBtn);
            searchContainer.appendChild(clearBtn);
            layersList.insertBefore(searchContainer, overlaysDiv);
            
            // === BOTONES POR MET (de 'Celaya') ===
            const metButtonsDiv = document.createElement('div');
            metButtonsDiv.className = 'met-buttons-row';
            const mets = new Set();
            checkboxes.forEach(function(checkbox) {
                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                // Regex de 'Celaya' para encontrar el nombre del MET
                // --- CORRECCIÓN: Ajustamos el Regex para que también capture los íconos de MET ---
                const metMatch = label.match(/^(.*?)\s+-\s+(Ruta|🏠 MET)/);
                if (metMatch) { mets.add(metMatch[1].trim()); }
            });
            
            console.log('🏢 METs disponibles:', Array.from(mets));
            
            Array.from(mets).sort().forEach(function(metNombre) {
                const metBtn = document.createElement('button');
                const nombreCorto = metNombre.replace('MET ', '').trim();
                metBtn.textContent = nombreCorto;
                metBtn.className = 'met-btn';
                metBtn.title = 'Seleccionar solo capas de ' + metNombre;
                
                metBtn.onclick = function(e) {
                    e.preventDefault(); e.stopPropagation();
                    console.log('🔍 Activando filtro para MET:', metNombre);
                    checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                    setTimeout(function() {
                        let seleccionadas = 0;
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (label.startsWith(metNombre + ' -')) { // Esto captura "MET Ceylán - Ruta 5" Y "MET Ceylán - 🏠 MET"
                                console.log('  ✓ Seleccionando:', label);
                                if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                            }
                        });
                        console.log('✅ Seleccionadas ' + seleccionadas + ' capas para ' + metNombre);
                    }, 150);
                };
                metButtonsDiv.appendChild(metBtn);
            });
            
            if (metButtonsDiv.children.length > 0) {
                layersList.insertBefore(metButtonsDiv, overlaysDiv);
                console.log('✅ Botones de MET insertados:', metButtonsDiv.children.length);
            }
            
            // === BOTONES POR RUTA (de 'Celaya') ===
            const rutaButtonsDiv = document.createElement('div');
            rutaButtonsDiv.className = 'ruta-buttons-row';
            const rutas = new Set();
            checkboxes.forEach(function(checkbox) {
                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                // Regex de 'Celaya' para encontrar el número de ruta
                const rutaMatch = label.match(/- Ruta\s+(\d+)/);
                if (rutaMatch) { rutas.add(parseInt(rutaMatch[1])); }
            });
            
            console.log('🚚 Rutas encontradas:', Array.from(rutas).sort(function(a, b) { return a - b; }));
            
            const rutasArray = Array.from(rutas).sort(function(a, b) { return a - b; });
            rutasArray.forEach(function(ruta) {
                const rutaBtn = document.createElement('button');
                rutaBtn.textContent = 'R' + ruta;
                rutaBtn.className = 'ruta-btn';
                rutaBtn.title = 'Seleccionar Ruta ' + ruta + ' de todos los METs';
                
                // --- INICIO CÓDIGO LIMPIO DE 'rutaBtn.onclick' ---
                rutaBtn.onclick = function(e) {
                    e.preventDefault(); e.stopPropagation();
                    
                    const rutaId = ruta; // 'ruta' viene del scope superior
                    console.log('🔍 Activando filtro para Ruta', rutaId);

                    // Patrones para encontrar las capas
                    const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$');
                    const metIconPattern = /- 🏠 MET$/;
                    const metNamePattern = /^(.*?)\s+-\s+(Ruta|🏠 MET)/; // Para extraer el nombre del MET

                    const metsToShow = new Set();

                    // --- 1. PRIMER PASE: Ver qué METs tienen esta ruta ---
                    checkboxes.forEach(function(checkbox) {
                        const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                        if (rutaPattern.test(label)) {
                            // Es una capa de Ruta (ej: "MET Ceylán - Ruta 5")
                            const metMatch = label.match(metNamePattern);
                            if (metMatch) {
                                metsToShow.add(metMatch[1]); // Añade "MET Ceylán" al Set
                            }
                        }
                    });
                    console.log('   ...Mostrando METs que tienen esta ruta:', Array.from(metsToShow));

                    // --- Deseleccionar todo (fuera del timeout) ---
                    checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });

                    // --- 2. SEGUNDO PASE: Activar las capas correctas ---
                    setTimeout(function() {
                        let seleccionadas = 0;
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            
                            // Opción 1: Es la ruta que buscamos (ej: "MET Ceylán - Ruta 5")
                            if (rutaPattern.test(label)) {
                                if (!checkbox.checked) {
                                    checkbox.click();
                                    seleccionadas++;
                                }
                            }
                            // Opción 2: Es un ícono de MET (ej: "MET Ceylán - 🏠 MET")
                            else if (metIconPattern.test(label)) {
                                const metMatch = label.match(metNamePattern);
                                // Y su nombre (ej: "MET Ceylán") está en nuestro Set
                                if (metMatch && metsToShow.has(metMatch[1])) {
                                    if (!checkbox.checked) {
                                        checkbox.click();
                                        seleccionadas++;
                                    }
                                }
                            }
                        });
                        console.log('✅ Seleccionadas ' + seleccionadas + ' capas (Rutas + METs) para Ruta ' + rutaId);
                    }, 150);
                };
                // --- FIN CÓDIGO LIMPIO DE 'rutaBtn.onclick' ---

                rutaButtonsDiv.appendChild(rutaBtn);
            });

            if (rutaButtonsDiv.children.length > 0) {
                layersList.insertBefore(rutaButtonsDiv, overlaysDiv);
                console.log('✅ ' + rutasArray.length + ' botones de ruta insertados');
            }
            
            console.log('=== Filtros inicializados correctamente ===');
            return true;
        }

        
        // Lógica de RESTRICCIÓN (POLLING) robusta para FILTROS
        
        let filterPollCount = 0;
        
        function tryInitFilters() {
            // Espera a que Leaflet (L) esté listo Y
            // a que el panel de control de capas exista en el DOM.
            const layerControlList = document.querySelector('.leaflet-control-layers-list');
            
            // Si ambas cosas existen, ejecuta la inicialización
            if (typeof L !== 'undefined' && layerControlList) {
                console.log('✅ (Poll) Leaflet (L) y .leaflet-control-layers-list listos. Iniciando filtros...');
                try {
                    inicializarFiltros(); // Llama a la función principal de arriba
                } catch (e) {
                    console.error('❌ Error al ejecutar inicializarFiltros:', e);
                }
            } else if (filterPollCount < 40) { // Si no, reintenta (hasta 20 seg)
                filterPollCount++;
                console.log('Poll Filtros #' + filterPollCount + ': Esperando a L y .leaflet-control-layers-list...');
                setTimeout(tryInitFilters, 500); // Reintentar cada 500ms
            } else {
                console.error('❌ No se pudo encontrar .leaflet-control-layers-list después de 20 segundos.');
            }
        }
        
        // Iniciar el sondeo (con un pequeño retraso inicial)
        setTimeout(tryInitFilters, 500); 
        
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(filtros_js))
        
        # --- CAMBIO: Título del mapa estilo 'Celaya', adaptado a nuestro Módulo 2 ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
            <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                🗺️ ANÁLISIS DE CLUSTERS POR VOLUMEN
            </h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas Generadas: <b>{len(summary_data)}</b> | 👥 Clientes Procesados: <b>{g_total_clientes_procesados}</b><br>
                📦 Tamaño de Círculo = Volumen (m³/día) | 🎨 Color de Círculo = Ruta Asignada
            </p>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))
        # --- FIN CAMBIO ---
        
        # Guardar mapa
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        map_filename = f'mapa_rutas_clusters_{timestamp}.html'
        g_map_path = os.path.join(output_dir, map_filename)
        mapa_base.save(g_map_path)
        
        print(f"\n{'='*70}")
        print(f"🗺️ Mapa guardado: {g_map_path}")
        
        # === GENERAR EXCEL CONSOLIDADO ===
        if export_data and summary_data:
            excel_path = generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir)
            if excel_path:
                g_excel_paths = [excel_path]
                print(f"📊 Excel guardado: {excel_path}")
        
        print(f"{'='*70}")
        print(f"✅ ANÁLISIS COMPLETADO")
        print(f"   • Total clientes procesados: {g_total_clientes_procesados}")
        print(f"   • Total rutas generadas: {len(summary_data)}")
        print(f"   • Archivos generados: {len(g_excel_paths) + 1}")
        print(f"{'='*70}\n")

# Registrar el handler
param_button.on_click(ejecutar_analisis)

# === MOSTRAR UI ===
print("================== MÓDULO 1: Análisis de Rutas ==================")
display(widgets.VBox([
    widgets.Label("1. Cargar archivo Excel:", style={'font_weight': 'bold'}),
    upload_widget,
    output_upload,
    widgets.HTML("<hr>"),
    met_box,
    widgets.HTML("<hr>"),
    widgets.Label("2. Configurar parámetros:", style={'font_weight': 'bold'}),
    min_clientes_cluster,
    max_clientes_cluster,
    total_clientes_analizar,
    analizar_todos_clientes,
    param_button,
    output_param
]))

Esta versión está delantada pero aún no me gusta

In [ ]:
# ================== 1. INICIALIZACIÓN Y FUNCIONES BASE (Alfa 5.1) ==================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import os
import glob
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl 
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from shapely.geometry import Polygon, MultiPoint, Point # Importar Point y MultiPoint
from sklearn.cluster import KMeans 
import shutil # Importar shutil para limpieza
import zipfile # Importar zipfile para Celda 3

# --- Validar rutas locales ---
icon_met_path = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png"
output_dir = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output"

# Crear directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(icon_met_path):
    print(f"⚠️ Advertencia: No se encontró el icono en {icon_met_path}. Se usará icono por defecto.")
else:
    print(f"✅ Icono de MET encontrado: {icon_met_path}")


# --- GLOBALES DEL ESTADO ---
execution_count = 0
df_clientes = pd.DataFrame()
df_cdr = pd.DataFrame()
met_checkboxes = [] 
# --- GLOBALES PARA INTERCAMBIO DE MÓDULOS ---
g_map_path = ""       
g_excel_paths = []    
g_total_clientes_procesados = 0
g_descarga_en_progreso = False # Para el bloqueo de descarga

# --- LISTA DE COLORES GLOBAL ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# =========================================================================
# === FUNCIONES HELPER (LÓGICA) ===========================================
# =========================================================================

# --- HELPER FUNCTION PARA NORMALIZAR IDS ---
def normalizar_id(valor):
    """Convierte un valor a un ID seguro para HTML/JS."""
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    s = s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    s = s.replace('Á','A').replace('É','E').replace('Í','I').replace('Ó','O').replace('Ú','U')
    s = s.replace('ñ','n').replace('Ñ','N')
    return s

# --- FUNCIÓN HELPER PARA VORONOI ---
def _clip_voronoi(vor, bounding_poly):
    """Recorta los polígonos de Voronoi a un polígono de frontera (shapely)."""
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty:
        return clipped_polys
        
    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points): continue
        if not (0 <= region_index < len(vor.regions)): continue
        region_vertices_indices = vor.regions[region_index]
        if -1 in region_vertices_indices: continue
        valid_vertex_indices = [v for v in region_vertices_indices if 0 <= v < len(vor.vertices)]
        if len(valid_vertex_indices) < 3: continue
        region_vertices = [vor.vertices[v] for v in valid_vertex_indices]
        if len(region_vertices) < 3: continue

        try:
            poly = Polygon(region_vertices)
            if not poly.is_valid: poly = poly.buffer(0)
            if not poly.is_valid: continue

            clipped = poly.intersection(bounding_poly)
            
            if clipped.is_empty: continue
            elif clipped.geom_type == 'Polygon' and clipped.exterior is not None:
                if len(clipped.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in clipped.exterior.coords]
                else: continue
            elif clipped.geom_type == 'MultiPolygon':
                if not clipped.geoms: continue
                largest_poly = max(clipped.geoms, key=lambda p: p.area)
                if largest_poly.exterior is not None and len(largest_poly.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in largest_poly.exterior.coords]
                else: continue
            else: continue

            centroid_key = tuple(vor.points[i])
            clipped_polys[centroid_key] = clipped_coords
        except Exception as e:
            pass 
            
    return clipped_polys

# --- FUNCIÓN HELPER PARA TAMAÑO DE MARCADOR (DE CÓDIGO CELAYA) ---
def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    """
    Calcula el tamaño del marcador basado en el volumen.
    Usa escala logarítmica para mejor diferenciación.
    """
    if vol_max == vol_min or volumen <= 0:
        return tamano_min
    
    # Usar escala logarítmica para mejor diferenciación visual
    log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    
    if log_max == log_min:
        return tamano_min
        
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# =========================================================================
# === MÓDULO 1: LÓGICA DE CARGA Y VALIDACIÓN ==============================
# =========================================================================
def handle_upload(change, met_box_widget, output_upload_widget):
    global df_clientes, df_cdr, met_checkboxes 
    
    met_box = met_box_widget
    output_upload = output_upload_widget

    with output_upload:
        clear_output()
        upload_widget = change['owner']
        if not upload_widget.value:
            return

        # Adaptado para manejar diferentes formatos de FileUpload
        try:
            # Intentar obtener el archivo del widget
            if isinstance(upload_widget.value, tuple) and len(upload_widget.value) > 0:
                # Formato: tuple de FileInfo objects
                uploaded_file_info = upload_widget.value[0]
                uploaded_file_name = uploaded_file_info['name']
                uploaded_file_content = uploaded_file_info['content']
            elif isinstance(upload_widget.value, dict):
                # Formato: dict con nombres de archivo como keys
                uploaded_file_details = next(iter(upload_widget.value.values()))
                uploaded_file_name = uploaded_file_details['metadata']['name']
                uploaded_file_content = uploaded_file_details['content']
            else:
                raise ValueError("Formato de archivo no reconocido")
        except Exception as e:
            print(f"❌ Error al procesar el archivo cargado: {e}")
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
            return
        
        try:
            with open(uploaded_file_name, 'wb') as f: f.write(uploaded_file_content)
            df_excel = pd.read_excel(uploaded_file_name, sheet_name=None)
            
            if len(df_excel) < 2:
                raise ValueError("El archivo Excel debe contener al menos dos hojas (Clientes y METs/CDRs).")
                
            sheet1_name, sheet2_name = list(df_excel.keys())[0], list(df_excel.keys())[1]
            df_clientes_raw, df_cdr_raw = df_excel[sheet1_name], df_excel[sheet2_name]

            required_cols_clientes = ['CodCli', 'U_longitud', 'U_latitud', 'm3/día']
            required_cols_cdr = ['CodMET', 'U_longitud', 'U_latitud']
            
            missing_cols_cli = [col for col in required_cols_clientes if col not in df_clientes_raw.columns]
            missing_cols_cdr = [col for col in required_cols_cdr if col not in df_cdr_raw.columns]
            
            if missing_cols_cli:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet1_name}': {', '.join(missing_cols_cli)}")
            if missing_cols_cdr:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet2_name}': {', '.join(missing_cols_cdr)}")

            df_clientes_raw['U_longitud'] = pd.to_numeric(df_clientes_raw['U_longitud'], errors='coerce')
            df_clientes_raw['U_latitud'] = pd.to_numeric(df_clientes_raw['U_latitud'], errors='coerce')
            
            # Convertir m3/día: reemplazar "-" por 0 y convertir a numérico
            df_clientes_raw['m3/día'] = df_clientes_raw['m3/día'].replace('-', 0)
            df_clientes_raw['m3/día'] = pd.to_numeric(df_clientes_raw['m3/día'], errors='coerce').fillna(0)
            df_cdr_raw['U_longitud'] = pd.to_numeric(df_cdr_raw['U_longitud'], errors='coerce')
            df_cdr_raw['U_latitud'] = pd.to_numeric(df_cdr_raw['U_latitud'], errors='coerce')

            df_clientes = df_clientes_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()
            df_cdr = df_cdr_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()

            num_dropped_cli = len(df_clientes_raw) - len(df_clientes)
            num_dropped_cdr = len(df_cdr_raw) - len(df_cdr)
            if num_dropped_cli > 0: print(f"⚠️ Se eliminaron {num_dropped_cli} clientes por coordenadas inválidas.")
            if num_dropped_cdr > 0: print(f"⚠️ Se eliminaron {num_dropped_cdr} METs/CDRs por coordenadas inválidas.")

            print(f"✅ Archivo '{uploaded_file_name}' cargado exitosamente.")
            print(f"📊 Clientes válidos: {df_clientes.shape[0]} filas")
            print(f"📊 METs/CDRs válidos: {df_cdr.shape[0]} filas")

            mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
            met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
            met_box.children = [widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes 

        except ValueError as ve:
            print(f"❌ Error de validación: {ve}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error en formato de archivo.')]
        except Exception as e:
            print(f"❌ Error inesperado al cargar: {e}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
        finally:
            if 'uploaded_file_name' in locals() and os.path.exists(uploaded_file_name):
                os.remove(uploaded_file_name)

# =========================================================================
# === MÓDULO 3: LÓGICA DE CLUSTERING (Alfa 5.1) ==========================
# =========================================================================
def select_customers_for_met(df_all_clients, met_lat, met_lon, n_clients, analyze_all):
    # Esta función se mantiene por si se usa en el futuro,
    # pero la lógica de 'Territorios' (Fase 0) en Módulo 2 la reemplaza.
    if df_all_clients.empty: return pd.DataFrame()
    if analyze_all:
        return df_all_clients.copy() 
    else:
        distances = np.hypot(df_all_clients['U_latitud'] - met_lat, df_all_clients['U_longitud'] - met_lon)
        df_with_dist = df_all_clients.copy()
        df_with_dist['Distancia_al_MET'] = distances
        n_to_select = min(n_clients, len(df_with_dist))
        if n_to_select <= 0: return pd.DataFrame() 
        return df_with_dist.nsmallest(n_to_select, 'Distancia_al_MET')

def calcular_dispersion_cluster(df_cluster):
    if len(df_cluster) < 2:
        return 0.0
    coords = df_cluster[['U_longitud', 'U_latitud']].values
    distancias_pares = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = np.hypot(coords[i][0] - coords[j][0], coords[i][1] - coords[j][1])
            distancias_pares.append(dist)
    return np.mean(distancias_pares) if distancias_pares else 0.0

def calcular_limites_clientes_dinamico(dispersion, volumen_actual, min_base=20, max_base=40):
    dispersion_normalizada = min(dispersion / 0.5, 1.0) 
    volumen_normalizado = min(volumen_actual / 6.0, 1.0)
    factor_restriccion = (dispersion_normalizada + volumen_normalizado) / 2.0
    rango = max_base - min_base
    ajuste = int(rango * factor_restriccion)
    max_dinamico = max_base - ajuste
    min_dinamico = max(min_base - int(ajuste * 0.5), int(min_base * 0.5)) 
    return min_dinamico, max_dinamico

def find_initial_seeds(coords, k, random_state=42):
    n_puntos = coords.shape[0]
    if k <= 0 or n_puntos == 0:
        print("       ⚠️ No se pueden generar semillas (k<=0 o no hay puntos).")
        return np.array([])
    if n_puntos < k:
        print(f"       ⚠️ No se pueden encontrar {k} semillas con {n_puntos} puntos. Ajustando k a {n_puntos}.")
        k = n_puntos 
    try:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=20, init='k-means++').fit(coords)
        return kmeans.cluster_centers_
    except ValueError as e:
        print(f"       ❌ Error en KMeans para semillas (k={k}, n_puntos={n_puntos}): {e}")
        if n_puntos > 0:
            indices = np.random.choice(n_puntos, min(k, n_puntos), replace=False)
            print(f"       ... Usando {len(indices)} puntos aleatorios como semillas (Fallback).")
            return coords[indices]
        else:
            return np.array([])

# --- (NUEVO) FUNCIÓN PARA ENCONTRAR ANCLAS DE VOLUMEN ---
def find_volume_anchors(df_clients, percentile=80):
    """Encuentra los clientes 'ancla' (ej. 20% con mayor volumen)"""
    # No tomar en cuenta clientes con 0 m³
    df_with_volume = df_clients[df_clients['m3/día'] > 0.01]
    if df_with_volume.empty:
        print("       ... No se encontraron clientes con volumen para anclaje.")
        return pd.DataFrame() # No hay anclas
    
    # Usar un umbral de percentil para definir "alto volumen"
    min_volume_threshold = np.percentile(df_with_volume['m3/día'], percentile)
    df_anchors = df_with_volume[df_with_volume['m3/día'] >= min_volume_threshold].copy()
    
    print(f"       ... Umbral de anclaje (percentil {percentile}%) = {min_volume_threshold:.2f} m³")
    print(f"       ... {len(df_anchors)} clientes 'ancla' de alto volumen identificados.")
    return df_anchors

# --- (MODIFICADO) LÓGICA DE CRECIMIENTO HÍBRIDA (Alfa 5.1) ---
def grow_clusters_from_seeds(df_clients_to_cluster, initial_seeds, max_size, min_size=20):
    """
    Asigna clientes a rutas usando una estrategia HÍBRIDA (Alfa 5.1):
    1. Fase A: Anclar rutas con clientes de alto volumen (Tu idea).
    2. Fase B: Rellenar rutas con lógica 'greedy' (redonda) (Lógica Beta).
    3. Fase C: Rellenar con comodines (Lógica Alfa).
    """
    k = len(initial_seeds)
    n_clientes_total = len(df_clients_to_cluster)
    if k == 0 or n_clientes_total == 0:
        print("       ⚠️ No hay semillas o clientes para 'grow_clusters_from_seeds'.")
        df_clients_to_cluster['Ruta_nueva'] = -1
        return df_clients_to_cluster, {}

    # --- Inicialización (Lógica Alfa) ---
    df_clients_to_cluster['Ruta_nueva'] = -1
    ruta_volumenes = {i: 0.0 for i in range(k)}
    ruta_counts = {i: 0 for i in range(k)} 
    
    # --- FASE A: ANCLAJE DE VOLUMEN (Tu idea) ---
    print(f"       🔬 Fase A: Anclando {k} rutas con clientes de alto volumen...")
    df_anchors = find_volume_anchors(df_clients_to_cluster)
    n_asignados_anchors = 0
    
    if not df_anchors.empty:
        coords_anchors = df_anchors[['U_longitud', 'U_latitud']].values
        dist_matrix_anchors = distance_matrix(coords_anchors, initial_seeds)
        
        # Iterar por cada ancla
        for i, (anchor_idx, row) in enumerate(df_anchors.iterrows()):
            volumen_ancla = row['m3/día']
            rutas_ordenadas = np.argsort(dist_matrix_anchors[i]) # Rutas más cercanas primero
            
            asignado = False
            for ruta_idx in rutas_ordenadas:
                # Validar (Lógica Alfa)
                volumen_futuro = ruta_volumenes[ruta_idx] + volumen_ancla
                
                # Para la Fase de Anclaje, no calculamos dispersión, usamos un límite estático
                # ya que la ruta está vacía o casi vacía.
                if volumen_futuro <= 6.0 and ruta_counts[ruta_idx] < max_size:
                    df_clients_to_cluster.loc[anchor_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_volumenes[ruta_idx] = volumen_futuro
                    ruta_counts[ruta_idx] += 1
                    n_asignados_anchors += 1
                    asignado = True
                    break # Ancla asignada, pasar a la siguiente ancla
            
            if not asignado:
                print(f"       ⚠️ Ancla {anchor_idx} no pudo ser asignada (todas las rutas cercanas están llenas).")
    
    print(f"       ✅ Fase A (Anclaje) completada: {n_asignados_anchors}/{len(df_anchors)} anclas asignadas.")

    # --- FASE B: RELLENO GEOMÉTRICO (Lógica "Redonda" de Beta) ---
    df_normales_restantes = df_clients_to_cluster[
        (df_clients_to_cluster['m3/día'] > 0) & 
        (df_clients_to_cluster['Ruta_nueva'] == -1)
    ].copy()
    
    n_normales = len(df_normales_restantes)
    n_asignados_normales = 0

    if n_normales > 0:
        print(f"       🔬 Fase B: Rellenando geométricamente {n_normales} clientes restantes...")
        coords_normales = df_normales_restantes[['U_longitud', 'U_latitud']].values
        
        try:
            dist_matrix = distance_matrix(coords_normales, initial_seeds)
        except ValueError as e:
            print(f"       ❌ Error calculando matriz de distancias: {e}")
            dist_matrix = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_normales])
        
        dist_df = pd.DataFrame(dist_matrix, index=df_normales_restantes.index, columns=range(k))
        
        iteracion = 0
        max_iter = n_normales * k # Límite de seguridad
        
        while n_asignados_normales < n_normales and iteracion < max_iter:
            iteracion += 1
            min_dist = dist_df.min().min()
            if min_dist == np.inf:
                break 

            stacked = dist_df.stack()
            best_pair_idx = stacked[stacked == min_dist].index
            client_idx, ruta_idx = best_pair_idx[0]
            
            # --- Validar con Lógica ALFA completa ---
            volumen_cliente = df_normales_restantes.loc[client_idx, 'm3/día']
            volumen_futuro = ruta_volumenes[ruta_idx] + volumen_cliente
            
            clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
            dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

            min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                dispersion_actual, volumen_futuro, min_base=min_size, max_base=max_size
            )

            if (volumen_futuro <= 6.0) and (ruta_counts[ruta_idx] < max_dinamico):
                # Éxito: Asignar
                df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = ruta_idx
                ruta_volumenes[ruta_idx] = volumen_futuro
                ruta_counts[ruta_idx] += 1
                n_asignados_normales += 1
                dist_df.loc[client_idx, :] = np.inf # Cliente asignado
            else:
                # Fallo: Invalidar solo este par
                dist_df.loc[client_idx, ruta_idx] = np.inf

        if iteracion >= max_iter:
            print(f"       ⚠️ Se alcanzó el límite de iteraciones ({max_iter}).")
        
        print(f"       ✅ Fase B completada: {n_asignados_normales}/{n_normales} clientes asignados.")

    # --- FASE C: RELLENO DE COMODINES (Lógica Alfa) ---
    df_comodines = df_clients_to_cluster[
        (df_clients_to_cluster['m3/día'] == 0) & 
        (df_clients_to_cluster['Ruta_nueva'] == -1)
    ].copy()
    
    n_comodines = len(df_comodines)
    if n_comodines > 0:
        print(f"       🔬 Fase C: Asignando {n_comodines} clientes 'comodines' (0 m³)...")
        coords_comodines = df_comodines[['U_longitud', 'U_latitud']].values
        
        try:
            dist_matrix_comodines = distance_matrix(coords_comodines, initial_seeds)
        except ValueError as e:
            print(f"       ❌ Error calculando matriz de distancias para comodines: {e}")
            dist_matrix_comodines = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_comodines])

        for idx_comodin, (real_idx, row) in enumerate(df_comodines.iterrows()):
            distancias_rutas = dist_matrix_comodines[idx_comodin]
            rutas_ordenadas = np.argsort(distancias_rutas)
            
            asignado = False
            for ruta_idx in rutas_ordenadas:
                clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)
                
                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, ruta_volumenes[ruta_idx], min_base=min_size, max_base=max_size
                )
                
                if ruta_counts[ruta_idx] < max_dinamico:
                    df_clients_to_cluster.loc[real_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_counts[ruta_idx] += 1
                    asignado = True
                    break
            
            if not asignado:
                ruta_mas_cercana = rutas_ordenadas[0]
                df_clients_to_cluster.loc[real_idx, 'Ruta_nueva'] = ruta_mas_cercana
                ruta_counts[ruta_mas_cercana] += 1
    
    # --- Devolver resultados (Lógica Alfa Original) ---
    rutas_validas = {}
    for ruta_idx in range(k):
        # Una ruta es válida si tiene volumen O si es una ruta de puros comodines
        if ruta_volumenes[ruta_idx] > 0 or ruta_counts[ruta_idx] > 0:
             rutas_validas[ruta_idx] = ruta_counts[ruta_idx]
            
    return df_clients_to_cluster, rutas_validas

# --- (MODIFICADO) LÓGICA DE AJUSTE CON GEOMETRÍA (Alfa 5.1) ---
def adjust_small_clusters(df_clustered, cluster_counts, min_size, max_size):
    """
    Ajusta rutas pequeñas (< min_size) O no rentables (< 1.0 m³).
    Reasigna clientes al POLÍGONO válido más cercano (no al centroide).
    """
    print("       🔬 Fase D: Ajustando rutas pequeñas y no rentables...")
    df_adjusted = df_clustered.copy()
    
    # Bucle de reintentos
    attempts = 0
    max_attempts = 3 # Limitar los pases de ajuste
    while attempts < max_attempts:
        attempts += 1
        
        # --- 1. Calcular volúmenes y conteos actuales ---
        ruta_volumenes = {}
        ruta_counts_actual = {}
        for ruta_id in df_adjusted['Ruta_nueva'].unique():
            if ruta_id < 0: continue
            clientes_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            ruta_volumenes[ruta_id] = clientes_ruta['m3/día'].sum()
            ruta_counts_actual[ruta_id] = len(clientes_ruta)

        if not ruta_counts_actual:
            print("       ... No hay rutas que ajustar.")
            break # No hay nada que hacer

        # --- 2. Identificar rutas VÁLIDAS vs. RUTAS A MOVER ---
        rutas_validas_ids = set()
        rutas_a_mover_ids = set()
        
        for ruta_id, count in ruta_counts_actual.items():
            volumen = ruta_volumenes.get(ruta_id, 0)
            # Regla de negocio: Pequeña O No Rentable
            if (count < min_size) or (volumen < 1.0):
                rutas_a_mover_ids.add(ruta_id)
            else:
                rutas_validas_ids.add(ruta_id)

        if not rutas_a_mover_ids:
            print(f"       ✅ Fase D completada: No hay rutas pequeñas o no rentables. (Intento {attempts})")
            break # Éxito, no hay nada que ajustar
        
        if not rutas_validas_ids:
            print(f"       ⚠️ Fase D: Hay {len(rutas_a_mover_ids)} rutas pequeñas/no rentables, pero NO hay rutas válidas para moverlas. Deteniendo.")
            break

        # --- 3. Crear Geometrías Válidas (Tu idea) ---
        geometrias_validas = {}
        for ruta_id in rutas_validas_ids:
            clientes_validos = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            points = clientes_validos[['U_longitud', 'U_latitud']].values
            if len(points) >= 3:
                try:
                    # Usamos ConvexHull para obtener el *área* de la ruta
                    hull = ConvexHull(points)
                    geometrias_validas[ruta_id] = Polygon(points[hull.vertices])
                except Exception as e:
                    pass # Ignorar esta ruta si ConvexHull falla
            
        if not geometrias_validas:
            print(f"       ⚠️ Fase D: No se pudieron crear polígonos para las rutas válidas. Deteniendo.")
            break
            
        # --- 4. Iterar y Reasignar Clientes "Huérfanos" ---
        df_clientes_a_mover = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_a_mover_ids)].copy()
        print(f"       ... Intento {attempts}: Moviendo {len(df_clientes_a_mover)} clientes de {len(rutas_a_mover_ids)} rutas...")
        
        clientes_reasignados_en_intento = 0
        
        for idx_cliente, row in df_clientes_a_mover.iterrows():
            punto_cliente = Point(row['U_longitud'], row['U_latitud'])
            volumen_cliente = row['m3/día']
            original_ruta = row['Ruta_nueva']

            # Calcular distancia a TODOS los polígonos válidos
            distancias_a_poligonos = []
            for ruta_id, poly in geometrias_validas.items():
                dist = punto_cliente.distance(poly)
                distancias_a_poligonos.append((dist, ruta_id))
            
            if not distancias_a_poligonos:
                continue # No hay polígonos válidos

            distancias_a_poligonos.sort() # Ordenar por distancia (más cercano primero)

            # Intentar asignar al más cercano que cumpla las reglas
            asignado = False
            for dist, ruta_id_destino in distancias_a_poligonos:
                # Validar con Lógica Alfa
                volumen_actual = ruta_volumenes.get(ruta_id_destino, 0)
                volumen_futuro = volumen_actual + volumen_cliente
                
                if volumen_futuro > 6.0:
                    continue # Excede 6m³, probar siguiente polígono
                
                clientes_ruta_actual = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id_destino]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)
                
                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, volumen_futuro, min_base=min_size, max_base=max_size
                )
                
                if ruta_counts_actual.get(ruta_id_destino, 0) < max_dinamico:
                    # ¡Éxito! Reasignar
                    df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = ruta_id_destino
                    
                    # Actualizar contadores para la siguiente iteración
                    ruta_counts_actual[ruta_id_destino] = ruta_counts_actual.get(ruta_id_destino, 0) + 1
                    ruta_volumenes[ruta_id_destino] = volumen_futuro
                    
                    ruta_counts_actual[original_ruta] -= 1
                    ruta_volumenes[original_ruta] = max(0, ruta_volumenes[original_ruta] - volumen_cliente)
                    
                    clientes_reasignados_en_intento += 1
                    asignado = True
                    break # Cliente asignado, pasar al siguiente cliente
            
            # (No hay 'fallback' forzado. Si no cabe, no cabe)

        if clientes_reasignados_en_intento == 0:
            print("       ... No se pudieron reasignar más clientes en este intento. Deteniendo ajuste.")
            break 

    if attempts >= max_attempts: 
        print(f"       ⚠️ Fase D: Se alcanzó el límite de {max_attempts} intentos. Pueden quedar rutas pequeñas.")
    
    # Devolver el DF ajustado y el último conteo conocido
    return df_adjusted, ruta_counts_actual


# =========================================================================
# === MÓDULO 4: CÁLCULO DE POLÍGONOS (Sin cambios) ========================
# =========================================================================
def calculate_cluster_polygons(df_clustered, method='voronoi'):
    # Esta función se usa solo para la VISUALIZACIÓN FINAL
    polygons = {} 
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])
    if not assigned_ruta_ids: return polygons
    
    centroides_rutas = [] 
    ruta_id_map = {} 
    for ruta_id in assigned_ruta_ids:
        ruta_clientes_df = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if not ruta_clientes_df.empty:
            lon = pd.to_numeric(ruta_clientes_df['U_longitud'], errors='coerce').mean()
            lat = pd.to_numeric(ruta_clientes_df['U_latitud'], errors='coerce').mean()
            if pd.notna(lon) and pd.notna(lat):
                centroides_rutas.append([lon, lat])
                ruta_id_map[tuple([lon, lat])] = ruta_id
    
    poligonos_voronoi_recortados = {}
    vor = None 
    if method == 'voronoi' and len(centroides_rutas) >= 4: 
        try:
            todos_los_puntos = df_clustered[df_clustered['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
            if len(todos_los_puntos) >= 3:
                boundary_poly_shapely = MultiPoint(todos_los_puntos).convex_hull.buffer(0).buffer(0.0001) 
                unique_centroids = np.unique(np.array(centroides_rutas), axis=0)
                if len(unique_centroids) >= 4:
                    vor = Voronoi(unique_centroids, qhull_options='Qbb Qc Qz') 
                    poligonos_voronoi_recortados = _clip_voronoi(vor, boundary_poly_shapely)
                else: print("       ... No hay suficientes centroides únicos (>=4) para Voronoi.")
            else: print("       ... No hay suficientes puntos asignados (<3) para definir frontera Voronoi.")
        except Exception as e:
            print(f"       ❌ Error al calcular Voronoi: {e}. Se usará ConvexHull.")
            poligonos_voronoi_recortados = {} 
    elif method == 'voronoi':
        print(f"       ... No hay suficientes rutas (se necesitan >= 4, se encontraron {len(centroides_rutas)}) para Voronoi. Usando ConvexHull.")

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if ruta_clientes.empty: continue
        
        # Encontrar el polígono de Voronoi que corresponde a este centroide
        centroide_lon = ruta_clientes['U_longitud'].mean()
        centroide_lat = ruta_clientes['U_latitud'].mean()
        poly_coords = poligonos_voronoi_recortados.get(tuple([centroide_lon, centroide_lat]))
        
        if not poly_coords and poligonos_voronoi_recortados:
            # Fallback por si hay errores de precisión de float
            min_dist = float('inf')
            best_match = None
            current_centroid = np.array([centroide_lon, centroide_lat])
            for vor_centroid_key, vor_poly in poligonos_voronoi_recortados.items():
                dist = np.linalg.norm(current_centroid - np.array(vor_centroid_key))
                if dist < min_dist:
                    min_dist = dist
                    best_match = vor_poly
            if min_dist < 1e-6: poly_coords = best_match

        if poly_coords:
            polygons[ruta_id] = poly_coords
        else:
            # Fallback a ConvexHull si Voronoi falló o no se usó
            points = ruta_clientes[['U_latitud', 'U_longitud']].values
            if len(points) >= 3:
                try:
                    hull = ConvexHull(points)
                    polygons[ruta_id] = points[hull.vertices].tolist() 
                except Exception:
                    print(f"       ⚠️ No se pudo generar polígono ConvexHull para Ruta {ruta_id}.")
                    polygons[ruta_id] = None
            else: polygons[ruta_id] = None 
    return polygons

# =========================================================================
# === MÓDULO 5: GENERACIÓN DE MAPA (Alfa 3.1 - Con Rentabilidad) ========
# =========================================================================
def create_map_elements(map_instance, df_clustered, cluster_polygons, colores_list, 
                        vol_min_global, vol_max_global, 
                        met_name=None):
    """
    Dibuja los elementos del cluster y los AÑADE a las capas de filtrado.
    (MODIFICADO): Colorea las rutas no rentables (< 1.0 m³) de rojo.
    """
    
    # --- Color de Advertencia para Rentabilidad ---
    COLOR_NO_RENTABLE = 'red'
    
    if met_name is None and 'MET_Asignado' in df_clustered.columns:
        met_name = df_clustered['MET_Asignado'].iloc[0] if not df_clustered.empty else ""
    elif met_name is None:
        met_name = ""
    
    met_name_clean = met_name.strip()
    display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"
    
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])

    tamano_min, tamano_max = (6, 25)

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id].copy()
        ruta_size = len(ruta_clientes)
        if ruta_size == 0: continue

        # --- FASE 3a: LÓGICA DE RENTABILIDAD ---
        volumen_total_ruta = ruta_clientes['m3/día'].sum()
        is_rentable = volumen_total_ruta >= 1.0
        
        if is_rentable:
            color_ruta = colores_list[ruta_id % len(colores_list)] 
            tooltip_poligono = f"{display_name} - Ruta {ruta_id} ({ruta_size} clientes)<br>Vol: {volumen_total_ruta:,.2f} m³"
            nombre_capa = f"{display_name} - Ruta {ruta_id}"
        else:
            color_ruta = COLOR_NO_RENTABLE
            tooltip_poligono = f"⚠️ RUTA NO RENTABLE ({display_name} - Ruta {ruta_id})<br>{ruta_size} clientes | Vol: {volumen_total_ruta:,.2f} m³"
            nombre_capa = f"⚠️ NO RENTABLE - {display_name} - Ruta {ruta_id}"
        # --- FIN FASE 3a ---

        fg_ruta_individual = folium.FeatureGroup(name=nombre_capa, show=True)
        
        poly_coords = cluster_polygons.get(ruta_id)
        if poly_coords:
            poly_obj_clon = folium.Polygon(
                locations=poly_coords,
                color=color_ruta,
                weight=2,
                fill=True,
                fill_color=color_ruta,
                fill_opacity=0.3 if is_rentable else 0.5,
                tooltip=tooltip_poligono
            )
            poly_obj_clon.add_to(fg_ruta_individual)

        if len(ruta_clientes) >= 3:
            try:
                puntos_solo_clientes = [Point(row['U_longitud'], row['U_latitud']) for _, row in ruta_clientes.iterrows()]
                multipoint_clientes = MultiPoint(puntos_solo_clientes)
                hull_clientes = multipoint_clientes.convex_hull
                
                if hull_clientes.geom_type == 'Polygon':
                    coords_polygon_clientes = [(lat, lon) for lon, lat in hull_clientes.exterior.coords]
                    area_popup_clientes = f'''
                    <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                        <div style="font-size: 14px; font-weight: bold;">👥 Área Solo Clientes</div>
                        <div style="font-size: 12px; margin-top: 4px;">🏠 {met_name_clean} - 🛣️ Ruta {ruta_id}</div>
                        <div style="font-size: 11px; margin-top: 2px;">👨‍💼 {len(ruta_clientes)} clientes</div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📦 Vol: {volumen_total_ruta:,.2f} m³</div>
                    </div>
                    '''
                    
                    poly_clientes_obj_clon = folium.Polygon(
                        locations=coords_polygon_clientes,
                        color=color_ruta,
                        weight=1,
                        opacity=0.8,
                        fillColor=color_ruta,
                        fillOpacity=0.1,
                        popup=folium.Popup(area_popup_clientes, max_width=250),
                        tooltip=tooltip_poligono,
                        dash_array='5, 5'
                    )
                    poly_clientes_obj_clon.add_to(fg_ruta_individual)
            except Exception as e:
                print(f"       ⚠️ No se pudo generar polígono 'solo clientes' para Ruta {ruta_id}: {e}")

        for _, cliente_row in ruta_clientes.iterrows(): 
            volumen = cliente_row['m3/día']
            tamano_marcador = obtener_tamano_marcador(volumen, vol_min_global, vol_max_global, tamano_min, tamano_max)
            
            if (vol_max_global - vol_min_global) > 0:
                percentil_vol = ((volumen - vol_min_global) / (vol_max_global - vol_min_global) * 100)
            else:
                percentil_vol = 50 

            codcli = cliente_row['CodCli']
            nombre = cliente_row.get('Nombre', 'N/A')
            ruta_original_val = cliente_row.get('Ruta', 'N/A') 
            ruta_original = ruta_original_val if pd.notna(ruta_original_val) else 'N/A'

            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                    👨‍💼 {codcli}
                </div>
                <div style="font-size:14px; color:#333; margin-bottom:4px;">
                    {nombre if pd.notna(nombre) else 'N/A'}
                </div>
                <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📦 Volumen: <b>{volumen:,.3f} m³</b> (Top {100-percentil_vol:.0f}°)
                </div>
                
                <div style="background: #eee; height: 6px; border-radius: 3px; margin-bottom: 8px;">
                    <div style="background: {color_ruta}; height: 100%; width: {max(0, min(100, percentil_vol)):.0f}%; border-radius: 3px;"></div>
                </div>

                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                    🗺️ Ruta (Orig): <b>{ruta_original}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                </div>
            </div>
            '''
            
            marker_obj_clon = folium.CircleMarker(
                location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=300),
                radius=tamano_marcador,
                color='#333333',
                weight=1,
                fillColor=color_ruta,
                fillOpacity=0.85,
            )
            marker_obj_clon.add_to(fg_ruta_individual)

        fg_ruta_individual.add_to(map_instance)
    return

# =========================================================================
# === MÓDULO 6: GENERACIÓN DE EXCEL (Alfa 3.1 - Con Rentabilidad) ========
# =========================================================================
def generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir):
    if not export_data or not summary_data:
        print(f"       ... No hay datos para exportar a Excel.")
        return None
    df_export = pd.DataFrame(export_data)
    df_resumen = pd.DataFrame(summary_data)
    if df_export.empty or df_resumen.empty:
        print(f"       ... DataFrames vacíos, no se puede generar Excel.")
        return None
    excel_filename = f'rutas_consolidadas_{timestamp}.xlsx'
    excel_path = os.path.join(output_dir, excel_filename)
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_resumen_sorted = df_resumen.sort_values(by=['MET', 'Ruta_nueva'])
            df_resumen_sorted.to_excel(writer, sheet_name='Resumen General', index=False)
            df_export_sorted = df_export.sort_values(by=['MET', 'Ruta_nueva', 'Codigo'])
            df_export_sorted.to_excel(writer, sheet_name='Todos los Clientes', index=False)
            for met_name in mets_seleccionados:
                df_met = df_export[df_export['MET'] == met_name].sort_values(by=['Ruta_nueva', 'Codigo'])
                if not df_met.empty:
                    sheet_name = f'MET_{normalizar_id(met_name)}'[:31]
                    df_met.to_excel(writer, sheet_name=sheet_name, index=False)
        format_excel(excel_path)
        return excel_path
    except Exception as e:
        print(f"       ❌ Error al generar Excel consolidado: {e}")
        if os.path.exists(excel_path):
            try: os.remove(excel_path)
            except: pass
        return None

def format_excel(excel_path):
    """
    Aplica formato profesional al Excel de salida.
    (MODIFICADO): Resalta las rutas no rentables (< 1 m³) en rojo.
    """
    if not os.path.exists(excel_path):
        print(f"       ⚠️ No se encontró el archivo Excel para formatear: {excel_path}")
        return
    try:
        wb = openpyxl.load_workbook(excel_path)
        
        # --- Definición de Estilos ---
        header_font = Font(bold=True, color='FFFFFF', size=11)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        summary_fill = PatternFill('solid', fgColor='FFD966') # Amarillo para resumen
        
        # --- (NUEVO) Estilo para Rutas No Rentables ---
        not_rentable_fill = PatternFill('solid', fgColor='FFC7CE') # Rojo claro
        not_rentable_font = Font(color='9C0006') # Texto rojo oscuro

        border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'), 
                        top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))
        align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
        align_left = Alignment(horizontal='left', vertical='center', wrap_text=False)
        
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws.max_row <= 1: continue

            # --- (NUEVO) Lógica para encontrar la columna de rentabilidad ---
            rentable_col_idx = None
            if sheet_name == 'Resumen General':
                for col_idx, cell in enumerate(ws[1], 1):
                    if cell.value and "Rentable" in str(cell.value):
                        rentable_col_idx = col_idx
                        break # Encontramos la columna
            
            # Formatear Encabezados (Sin cambios)
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align_center
                cell.border = border
                if cell.value: 
                    header_text = str(cell.value)
                    icon = ""
                    if 'Ruta_Original' in header_text: icon = '🗺️ '
                    elif 'Codigo' in header_text or 'CodCli' in header_text: icon = '👨‍💼 ' 
                    elif 'Clientes' in header_text: icon = '👥 ' 
                    elif 'MET' in header_text: icon = '🏠 '
                    elif 'Nombre' in header_text: icon = '📚 '
                    elif 'Ruta_nueva' in header_text or 'Ruta_ID' in header_text: icon = '🚚 '
                    elif 'Volumen' in header_text or 'm3' in header_text: icon = '📦 '
                    elif 'Centroide' in header_text: icon = '📍 '
                    elif 'Longitud' in header_text: icon = '📊 '
                    elif 'Latitud' in header_text: icon = '📊 '
                    elif 'Rentable' in header_text: icon = '💰 ' # <-- Ícono nuevo
                    cleaned_header = header_text.lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰 ')
                    cell.value = icon + cleaned_header
            
            # Formatear Celdas de Datos
            is_summary_sheet = (sheet_name == 'Resumen General')
            
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                
                # --- (NUEVO) Lógica de coloreado de fila ---
                is_not_rentable_row = False
                if is_summary_sheet and rentable_col_idx:
                    cell_value = ws.cell(row=row_idx, column=rentable_col_idx).value
                    if cell_value == 'NO':
                        is_not_rentable_row = True
                # --- Fin de la lógica de coloreado ---

                for col_idx, cell in enumerate(row, start=1):
                    # Aplicar color de fondo
                    if is_not_rentable_row:
                        cell.fill = not_rentable_fill
                        cell.font = not_rentable_font
                    elif is_summary_sheet:
                        cell.fill = summary_fill
                    else:
                        cell.fill = cell_fill
                    
                    cell.border = border
                    
                    # Lógica de alineación (sin cambios)
                    header_cell = ws.cell(row=1, column=col_idx)
                    header_val = header_cell.value if header_cell else None
                    if header_val and any(keyword in str(header_val) for keyword in ['Nombre', 'Codigo', 'CodCli']):
                        cell.alignment = align_left
                    else:
                        cell.alignment = align_center
                        if header_val and any(coord in str(header_val) for coord in ['Longitud', 'Latitud']):
                            cell.number_format = '0.000000'
            
            # Auto-ajustar columnas (sin cambios)
            for col_idx, column_cells in enumerate(ws.columns, 1):
                max_length = 0
                header_text = str(ws.cell(row=1, column=col_idx).value).lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰 ')
                max_length = len(header_text)
                for i, cell in enumerate(column_cells):
                    if i == 0: continue
                    if i > 50: break 
                    try:
                        if cell.value is not None:
                            if isinstance(cell.value, (int, float)) and cell.number_format and cell.number_format != 'General':
                                if '0.000000' in cell.number_format: cell_text = f"{cell.value:.6f}"
                                else: cell_text = str(cell.value)
                            else: cell_text = str(cell.value)
                            lines = cell_text.split('\n')
                            max_line_length = max(len(line) for line in lines) if lines else 0
                            max_length = max(max_length, max_line_length)
                    except: pass
                adjusted_width = min(max(12, max_length + 3), 45) 
                ws.column_dimensions[get_column_letter(col_idx)].width = adjusted_width
        
        wb.save(excel_path)
    except Exception as e:
        print(f"       ⚠️ Error al aplicar formato al Excel '{excel_path}': {e}")
        
print("✅ CELDA 1: Inicialización y funciones (Alfa 5.1) cargadas.")
print(f"📁 Directorio de salida: {output_dir}")
print(f"🎯 Icono MET: {'✅ Encontrado' if os.path.exists(icon_met_path) else '⚠️ No encontrado'}")

In [ ]:
# ================== 2. EJECUCIÓN DEL ANÁLISIS (Alfa 5.1 - Muestreo y Territorios) ==================

# --- Definición de Widgets ---
upload_widget = widgets.FileUpload(accept='.xlsx', multiple=False, description='Subir Excel')
met_box = widgets.VBox([widgets.Label('⏳ Esperando archivo Excel...')])
output_upload = widgets.Output()
min_clientes_cluster = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Min Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))
max_clientes_cluster = widgets.IntSlider(value=40, min=10, max=100, step=1, description='Max Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))

# --- (NUEVO) WIDGETS DE MUESTREO ---
sample_size_per_met = widgets.IntText(
    value=500, 
    description='Clientes a muestrear por MET:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='48%')
)
run_full_analysis = widgets.Checkbox(
    value=False, 
    description='✅ Analizar TODOS los clientes (Ignora el muestreo)', 
    indent=False, 
    layout=widgets.Layout(width='48%')
)
# --- FIN DE WIDGETS NUEVOS ---

param_button = widgets.Button(description='Ejecutar Análisis de Territorio y Rutas', button_style='success', disabled=False, layout=widgets.Layout(width='98%'))
output_param = widgets.Output()

# --- Registrar handler de carga ---
upload_widget.observe(lambda change: handle_upload(change, met_box, output_upload), names='value')

# --- Función Principal de Ejecución ---
def ejecutar_analisis(b):
    global execution_count, df_clientes, df_cdr, met_checkboxes, colores
    global g_map_path, g_excel_paths, g_total_clientes_procesados
    
    execution_count += 1
    
    with output_param:
        clear_output(wait=True)
        param_button.disabled = True # Bloquear botón
        
        # --- 1. Validaciones y Parámetros ---
        if df_clientes.empty or df_cdr.empty:
            print("❌ Carga un archivo Excel primero.")
            param_button.disabled = False
            return
        
        mets_seleccionados = [cb.description for cb in met_checkboxes if cb.value]
        if not mets_seleccionados or len(mets_seleccionados) < 2:
            print("❌ Selecciona al menos dos METs para optimizar territorios.")
            param_button.disabled = False
            return
        
        min_size = min_clientes_cluster.value
        max_size = max_clientes_cluster.value
        
        # --- (NUEVO) Obtener parámetros de muestreo ---
        n_muestra = sample_size_per_met.value
        analizar_todos = run_full_analysis.value
        
        if min_size >= max_size:
            print("❌ El mínimo debe ser menor que el máximo.")
            param_button.disabled = False
            return
        
        print(f"\n{'='*70}")
        print(f"🚀 INICIANDO ANÁLISIS DE TERRITORIO Y RUTAS - Ejecución #{execution_count}")
        print(f"{'='*70}")
        print(f"📊 Parámetros:")
        print(f"   • METs a optimizar: {len(mets_seleccionados)} ({', '.join(mets_seleccionados)})")
        print(f"   • Clientes por ruta: {min_size} - {max_size}")
        
        # Imprimir modo de ejecución
        if analizar_todos:
            print(f"   • MODO: ✅ COMPLETO (Se analizarán TODOS los clientes)")
        else:
            print(f"   • MODO: ⚠️ MUESTRA (Se analizarán {n_muestra} clientes por MET)")
            
        print(f"{'='*70}\n")
        
        # Limpiar resultados anteriores
        g_map_path = ""
        g_excel_paths = []
        g_total_clientes_procesados = 0
        
        # --- (NUEVO) GUARDAR RESPALDO DE CLIENTES ---
        df_clientes_original = df_clientes.copy()
        
        # --- 2. (NUEVO) FASE 0: ASIGNACIÓN DE TERRITORIOS ---
        try:
            print(f"🗺️ Fase 0: Asignando territorios (cliente al MET más cercano)...")
            
            mets_info = df_cdr[df_cdr['CodMET'].isin(mets_seleccionados)]
            if mets_info.empty:
                print("❌ No se encontraron datos para los METs seleccionados en la hoja de CDRs.")
                param_button.disabled = False
                return

            coords_clientes = df_clientes[['U_latitud', 'U_longitud']].values
            coords_mets = mets_info[['U_latitud', 'U_longitud']].values
            
            dist_matrix_territorio = distance_matrix(coords_clientes, coords_mets)
            indices_met_mas_cercano = np.argmin(dist_matrix_territorio, axis=1)
            nombres_met_mas_cercano = [mets_info['CodMET'].iloc[i] for i in indices_met_mas_cercano]
            
            df_clientes['MET_Asignado'] = nombres_met_mas_cercano
            
            print(f"✅ Territorios asignados. Conteo:")
            print(df_clientes['MET_Asignado'].value_counts())
            
            print("📊 Calculando rangos de volumen global para tamaño de marcadores...")
            volumenes_positivos = df_clientes[df_clientes['m3/día'] > 0]['m3/día']
            if not volumenes_positivos.empty:
                vol_min_global = volumenes_positivos.min()
                vol_max_global = volumenes_positivos.max()
            else:
                vol_min_global = 0
                vol_max_global = 0
            print(f"   • 📦 Rango de Volumen (m³/día): {vol_min_global:,.3f} - {vol_max_global:,.3f}")

            # --- 3. Crear Mapa Base (con zoom automático) ---
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            lats = mets_info['U_latitud']
            lons = mets_info['U_longitud']
            bounds = [[lats.min(), lons.min()], [lats.max(), lons.max()]]
            mapa_base.fit_bounds(bounds, padding=(20, 20))
            
            colores_list = colores.copy()
            random.shuffle(colores_list)
            
            export_data = []
            summary_data = []

            # --- 4. (MODIFICADO) FASE 1: CLUSTERING POR TERRITORIO ---
            print(f"\n🔬 Fase 1: Iniciando clustering por territorio...")
            
            for idx_met, met_seleccionado in enumerate(mets_seleccionados):
                print(f"\n{'─'*70}")
                print(f"🏠 Procesando Territorio: {met_seleccionado} ({idx_met + 1}/{len(mets_seleccionados)})")
                print(f"{'─'*70}")
                
                met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado]
                if met_info.empty:
                    print(f"⚠️ No se encontró información para el MET '{met_seleccionado}'. Saltando.")
                    continue
                met_row = met_info.iloc[0]
                met_lat = met_row['U_latitud']
                met_lon = met_row['U_longitud']
                
                # --- (NUEVO) LÓGICA DE MUESTREO POR TERRITORIO ---
                df_territorio_completo = df_clientes[df_clientes['MET_Asignado'] == met_seleccionado].copy()
                
                if df_territorio_completo.empty:
                    print(f"⚠️ No hay clientes asignados al territorio de '{met_seleccionado}'. Saltando.")
                    continue
                
                df_clientes_met = pd.DataFrame() # DataFrame vacío para el análisis
                if analizar_todos:
                    df_clientes_met = df_territorio_completo
                    print(f"✅ Analizando territorio COMPLETO: {len(df_clientes_met)} clientes")
                else:
                    if len(df_territorio_completo) > n_muestra:
                        print(f"⚠️ MODO MUESTRA: Analizando {n_muestra} de {len(df_territorio_completo)} clientes (aleatorio)")
                        df_clientes_met = df_territorio_completo.sample(n=n_muestra).copy()
                    else:
                        df_clientes_met = df_territorio_completo
                        print(f"✅ Analizando territorio COMPLETO (es menor que la muestra): {len(df_clientes_met)} clientes")
                # --- FIN LÓGICA DE MUESTREO ---

                if df_clientes_met.empty:
                    print(f"⚠️ No hay clientes para procesar en este MET. Saltando.")
                    continue
                
                # --- El resto del script continúa igual, usando df_clientes_met ---
                coords = df_clientes_met[['U_longitud', 'U_latitud']].values
                k_inicial = max(1, len(df_clientes_met) // max_size)
                
                print(f"🔬 Iniciando clustering (k_inicial = {k_inicial})...")
                initial_seeds = find_initial_seeds(coords, k_inicial)
                
                if len(initial_seeds) == 0:
                    print(f"❌ No se pudieron generar semillas para el MET '{met_seleccionado}'.")
                    continue
                
                df_clientes_met, cluster_counts = grow_clusters_from_seeds(df_clientes_met, initial_seeds, max_size, min_size)
                df_clientes_met, cluster_counts = adjust_small_clusters(df_clientes_met, cluster_counts, min_size, max_size)
                
                cluster_polygons = calculate_cluster_polygons(df_clientes_met, method='voronoi')
                
                final_counts = df_clientes_met['Ruta_nueva'][df_clientes_met['Ruta_nueva'] != -1].value_counts()
                rutas_validas = final_counts.index.tolist() # Tomar todas, el ajuste ya limpió
                
                print(f"✅ Rutas generadas: {len(rutas_validas)}")
                
                df_clientes_met_validos = df_clientes_met[df_clientes_met['Ruta_nueva'].isin(rutas_validas)].copy()
                
                if df_clientes_met_validos.empty:
                    print(f"⚠️ No hay rutas válidas para el MET '{met_seleccionado}'.")
                    continue
                
                met_name_clean = str(met_seleccionado).strip()
                display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"
                
                fg_met_icon = folium.FeatureGroup(name=f"{display_name} - 🏠 MET", show=True)
                
                clientes_met_total = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] >= 0]
                volumen_total_met = clientes_met_total['m3/día'].sum()
                total_clientes_met = len(clientes_met_total)
                
                popup_html_met = f'''
                <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                    <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                        <div style="font-size: 22px; font-weight: bold;">🏠 {met_name_clean}</div>
                    </div>
                    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                        <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                            <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                            <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                        </div>
                        <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                            <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                            <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                        </div>
                    </div>
                    <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                        <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                        <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_validas)}</div>
                    </div>
                </div>
                '''

                if os.path.exists(icon_met_path):
                    icon_met_obj = CustomIcon(icon_met_path, icon_size=(40, 40), icon_anchor=(20, 40), popup_anchor=(0, -40))
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=icon_met_obj, tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)
                else:
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=folium.Icon(color='red', icon='home', prefix='glyphicon'),
                        tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)

                create_map_elements(
                    mapa_base,
                    df_clientes_met_validos,
                    cluster_polygons,
                    colores_list,
                    vol_min_global,
                    vol_max_global,
                    met_seleccionado
                )
                
                fg_met_icon.add_to(mapa_base)
                
                g_total_clientes_procesados += len(df_clientes_met_validos)
                
                for ruta_id in rutas_validas:
                    ruta_clientes = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] == ruta_id]
                    n_clientes_ruta = len(ruta_clientes)
                    volumen_total = ruta_clientes['m3/día'].sum()
                    
                    summary_data.append({
                        'MET': met_seleccionado,
                        'Ruta_nueva': ruta_id,
                        'Num_Clientes': n_clientes_ruta,
                        'Volumen_Total_m3': round(volumen_total, 2),
                        'Rentable (>= 1 m³)': 'SÍ' if volumen_total >= 1.0 else 'NO',
                        'Centroide_Lat': ruta_clientes['U_latitud'].mean(),
                        'Centroide_Lon': ruta_clientes['U_longitud'].mean()
                    })
                    
                    for _, cliente in ruta_clientes.iterrows():
                        export_data.append({
                            'MET': met_seleccionado,
                            'Ruta_nueva': ruta_id,
                            'Codigo': cliente['CodCli'],
                            'Nombre': cliente.get('Nombre', 'N/A'),
                            'Ruta_Original': cliente.get('Ruta', 'N/A'),
                            'm3_dia': cliente['m3/día'],
                            'Latitud': cliente['U_latitud'],
                            'Longitud': cliente['U_longitud']
                        })
                
                print(f"✅ Territorio '{met_seleccionado}' completado: {len(rutas_validas)} rutas, {len(df_clientes_met_validos)} clientes")
            
            # --- 5. Guardar Mapa y Excel ---
            folium.LayerControl(collapsed=False).add_to(mapa_base)
            
            estilos_css = r'''
            <style>
            #panel-resumen {
                position: fixed; top: 80px; left: 10px; background: white; padding: 12px 16px;
                border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.2); z-index: 1000;
                max-width: 280px; font-family: Arial, sans-serif;
            }
            #panel-resumen h3 {
                margin: 0 0 10px 0; font-size: 16px; color: #333;
                border-bottom: 2px solid #007bff; padding-bottom: 6px;
            }
            #panel-resumen-content { font-size: 13px; line-height: 1.6; color: #555; }
            .resumen-item { display: flex; justify-content: space-between; padding: 4px 0; border-bottom: 1px solid #eee; }
            .resumen-item:last-child {
                border-bottom: none; font-weight: bold; margin-top: 6px;
                padding-top: 8px; border-top: 2px solid #007bff;
            }
            .resumen-label { color: #666; }
            .resumen-value { color: #007bff; font-weight: 600; }
            .leaflet-control-layers-list {
                max-height: 400px; overflow-y: auto; overflow-x: hidden;
                width: 100%; min-width: 200px;
            }
            .layer-control-buttons {
                padding: 8px; border-bottom: 1px solid #ddd; background: #f8f8f8;
                display: flex; gap: 5px;
            }
            .layer-btn {
                padding: 6px 10px; font-size: 11px; border: 1px solid #ccc; background: white;
                cursor: pointer; border-radius: 4px; flex: 1; text-align: center;
                font-weight: 600; transition: all 0.2s;
            }
            .layer-btn:hover { background: #e6e6e6; transform: translateY(-1px); }
            .layer-btn.select-all { background: #4CAF50; color: white; border-color: #45a049; }
            .layer-btn.select-all:hover { background: #45a049; }
            .layer-btn.deselect-all { background: #f44336; color: white; border-color: #da190b; }
            .layer-btn.deselect-all:hover { background: #da1a0b; }
            .met-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
                display: flex; gap: 4px; flex-wrap: wrap;
            }
            .met-btn {
                padding: 5px 10px; font-size: 10px; border: 2px solid #2196F3; background: white;
                color: #1976D2; cursor: pointer; border-radius: 6px; flex: 1; text-align: center;
                min-width: 60px; font-weight: 600; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .met-btn:hover { background: #2196F3; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .ruta-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%);
                display: grid; grid-template-columns: repeat(auto-fit, minmax(45px, 1fr));
                gap: 4px; max-width: 100%;
            }
            .ruta-btn {
                padding: 5px 8px; font-size: 10px; border: 2px solid #9C27B0; background: white;
                color: #7B1FA2; cursor: pointer; border-radius: 6px; text-align: center;
                min-width: 45px; font-weight: 700; white-space: nowrap; overflow: hidden;
                text-overflow: ellipsis; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .ruta-btn:hover { background: #9C27B0; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .search-ruta-container {
                display: flex; gap: 5px; padding: 8px; background: #f0f8ff;
                border-bottom: 1px solid #b3d9ff; margin-bottom: 8px; align-items: center;
            }
            .search-ruta-input {
                flex: 1; padding: 6px 10px; border: 1px solid #ced4da; border-radius: 4px;
                font-size: 12px; outline: none;
            }
            .search-ruta-input:focus { border-color: #0066cc; box-shadow: 0 0 0 2px rgba(0, 102, 204, 0.1); }
            .search-ruta-btn {
                padding: 6px 15px; border: 1px solid #0066cc; border-radius: 4px; background: #0066cc;
                color: white; cursor: pointer; font-size: 12px; font-weight: 500;
                transition: all 0.2s; white-space: nowrap;
            }
            .search-ruta-btn:hover { background: #0052a3; border-color: #0052a3; }
            .search-ruta-clear {
                padding: 6px 12px; border: 1px solid #dc3545; border-radius: 4px; background: white;
                color: #dc3545; cursor: pointer; font-size: 12px; font-weight: 500; transition: all 0.2s;
            }
            .search-ruta-clear:hover { background: #dc3545; color: white; }
            .leaflet-control-layers-overlays { display: none; }
            </style>
            '''
            mapa_base.get_root().html.add_child(folium.Element(estilos_css))
            
            paneles_js = r'''
            <script>
            function crearPanelResumen() {
                const panel = document.createElement('div');
                panel.id = 'panel-resumen';
                panel.innerHTML = `<h3>📊 Resumen de Rutas</h3><div id="panel-resumen-content"></div>`;
                document.body.appendChild(panel);
                return panel;
            }
            function inicializarPaneles(mapInstance) {
                console.log('=== Inicializando paneles personalizados ===');
                if (document.getElementById('panel-resumen')) {
                    console.log('ℹ️ Paneles ya inicializados');
                    return true;
                }
                const panelResumen = crearPanelResumen();
                const map = mapInstance;
                if (!map) { console.error('❌ Instancia del mapa (mapInstance) es nula.'); return false; }
                let totalRutas = 0;
                const metRutas = {};
                map.eachLayer(function(layer) {
                    if (layer._url || !layer.options || !layer.options.pane) return;
                    const layerName = layer.options.attribution || '';
                    const rutaMatch = layerName.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+Ruta\s+(\d+)$/); // Modificado para ignorar advertencia
                    if (rutaMatch) {
                        const metNombre = rutaMatch[1]; // Captura el nombre limpio
                        totalRutas++;
                        if (!metRutas[metNombre]) metRutas[metNombre] = 0;
                        metRutas[metNombre]++;
                    }
                });
                const resumenHTML = [];
                Object.keys(metRutas).sort().forEach(function(metNombre) {
                    resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">${metNombre}:</span><span class="resumen-value">${metRutas[metNombre]} rutas</span></div>`);
                });
                resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">Total:</span><span class="resumen-value">${totalRutas} rutas</span></div>`);
                document.getElementById('panel-resumen-content').innerHTML = resumenHTML.join('');
                console.log('✅ Paneles personalizados inicializados');
                return true;
            }
            let panelPollCount = 0;
            function tryInitPanels() {
                const mapElement = document.querySelector('.leaflet-container');
                let mapInstance = null;
                if (mapElement && mapElement._leaflet_map) { mapInstance = mapElement._leaflet_map; }
                if (typeof L !== 'undefined' && L.map && mapInstance) {
                    console.log('✅ (Poll) Leaflet (L) y MapInstance (_leaflet_map) listos. Iniciando paneles...');
                    inicializarPaneles(mapInstance);
                } else if (panelPollCount < 40) {
                    panelPollCount++;
                    console.log('Poll #' + panelPollCount + ': Esperando a L y _leaflet_map...');
                    setTimeout(tryInitPanels, 500);
                } else {
                    console.error('❌ No se pudo cargar la instancia del mapa (_leaflet_map) después de 20 segundos.');
                }
            }
            tryInitPanels();
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(paneles_js))
            
            filtros_js = r'''
            <script>
            function inicializarFiltros() {
                console.log('=== Inicializando filtros personalizados (Estilo Celaya) ===');
                const layerControl = document.querySelector('.leaflet-control-layers');
                if (!layerControl) { console.error('❌ No se encontró .leaflet-control-layers'); return false; }
                const layersList = layerControl.querySelector('.leaflet-control-layers-list');
                if (!layersList) { console.error('❌ No se encontró .leaflet-control-layers-list'); return false; }
                const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) { console.error('❌ No se encontró .leaflet-control-layers-overlays'); return false; }
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                console.log('✅ Total de checkboxes encontrados:', checkboxes.length);
                if (layersList.querySelector('.layer-control-buttons')) { console.log('ℹ️ Filtros ya inicializados'); return true; }
                
                console.log('📋 Listando todas las capas:');
                checkboxes.forEach(function(checkbox, index) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    console.log('   ' + (index + 1) + '. "' + label + '"');
                });
                
                const buttonsDiv = document.createElement('div');
                buttonsDiv.className = 'layer-control-buttons';
                const selectAllBtn = document.createElement('button');
                selectAllBtn.textContent = '✓ Todo';
                selectAllBtn.className = 'layer-btn select-all';
                selectAllBtn.title = 'Seleccionar todas las capas';
                const deselectAllBtn = document.createElement('button');
                deselectAllBtn.textContent = '✗ Nada';
                deselectAllBtn.className = 'layer-btn deselect-all';
                deselectAllBtn.title = 'Deseleccionar todas las capas';
                selectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (!cb.checked) cb.click(); }); };
                deselectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); };
                buttonsDiv.appendChild(selectAllBtn);
                buttonsDiv.appendChild(deselectAllBtn);
                layersList.insertBefore(buttonsDiv, overlaysDiv);
                
                const searchContainer = document.createElement('div');
                searchContainer.className = 'search-ruta-container';
                const searchLabel = document.createElement('span');
                searchLabel.textContent = '🔍';
                searchLabel.style.fontSize = '14px';
                const searchInput = document.createElement('input');
                searchInput.type = 'text';
                searchInput.className = 'search-ruta-input';
                searchInput.placeholder = 'Buscar por Ruta ID (ej: 0, 1, 2...)';
                searchInput.title = 'Ingresa el número de ruta para filtrar';
                const searchBtn = document.createElement('button');
                searchBtn.textContent = 'Buscar';
                searchBtn.className = 'search-ruta-btn';
                searchBtn.title = 'Filtrar por Ruta ID';
                const clearBtn = document.createElement('button');
                clearBtn.textContent = 'Limpiar';
                clearBtn.className = 'search-ruta-clear';
                clearBtn.title = 'Limpiar búsqueda';
                const filtrarPorRutaId = function() {
                    const rutaId = searchInput.value.trim();
                    if (rutaId === '') { alert('Por favor ingresa un número de ruta'); return; }
                    console.log('🔍 Filtrando capas con Ruta ID:', rutaId);
                    checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                    setTimeout(function() {
                        let encontradas = 0;
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$'); // Busca el final
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                console.log('  ✓ Seleccionando:', label);
                                if (!checkbox.checked) { checkbox.click(); encontradas++; }
                            }
                        });
                        if (encontradas === 0) { console.warn('⚠️ No se encontraron capas con Ruta ID ' + rutaId); alert('No se encontraron capas con Ruta ID: ' + rutaId); }
                    }, 150);
                };
                searchBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); filtrarPorRutaId(); };
                searchInput.addEventListener('keypress', function(e) { if (e.key === 'Enter') { e.preventDefault(); filtrarPorRutaId(); } });
                clearBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); searchInput.value = ''; checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); console.log('🧹 Búsqueda limpiada'); };
                searchContainer.appendChild(searchLabel);
                searchContainer.appendChild(searchInput);
                searchContainer.appendChild(searchBtn);
                searchContainer.appendChild(clearBtn);
                layersList.insertBefore(searchContainer, overlaysDiv);
                
                const metButtonsDiv = document.createElement('div');
                metButtonsDiv.className = 'met-buttons-row';
                const mets = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const metMatch = label.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/); // Modificado para incluir advertencia
                    if (metMatch) { mets.add(metMatch[1].trim()); } // Limpiar advertencia
                });
                console.log('🏢 METs disponibles:', Array.from(mets));
                Array.from(mets).sort().forEach(function(metNombre) {
                    const metBtn = document.createElement('button');
                    const nombreCorto = metNombre.replace('MET ', '').trim();
                    metBtn.textContent = nombreCorto;
                    metBtn.className = 'met-btn';
                    metBtn.title = 'Seleccionar solo capas de ' + metNombre;
                    metBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        console.log('🔍 Activando filtro para MET:', metNombre);
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                // Modificado para que siempre funcione
                                if (label.includes(metNombre + ' -')) {
                                    console.log('  ✓ Seleccionando:', label);
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas para ' + metNombre);
                        }, 150);
                    };
                    metButtonsDiv.appendChild(metBtn);
                });
                if (metButtonsDiv.children.length > 0) { layersList.insertBefore(metButtonsDiv, overlaysDiv); }
                
                const rutaButtonsDiv = document.createElement('div');
                rutaButtonsDiv.className = 'ruta-buttons-row';
                const rutas = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const rutaMatch = label.match(/- Ruta\s+(\d+)/);
                    if (rutaMatch) { rutas.add(parseInt(rutaMatch[1])); }
                });
                console.log('🚚 Rutas encontradas:', Array.from(rutas).sort(function(a, b) { return a - b; }));
                const rutasArray = Array.from(rutas).sort(function(a, b) { return a - b; });
                rutasArray.forEach(function(ruta) {
                    const rutaBtn = document.createElement('button');
                    rutaBtn.textContent = 'R' + ruta;
                    rutaBtn.className = 'ruta-btn';
                    rutaBtn.title = 'Seleccionar Ruta ' + ruta + ' de todos los METs';
                    rutaBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        const rutaId = ruta;
                        console.log('🔍 Activando filtro para Ruta', rutaId);
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$');
                        const metIconPattern = /- 🏠 MET$/;
                        const metNamePattern = /^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/;
                        const metsToShow = new Set();
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                const metMatch = label.match(metNamePattern);
                                if (metMatch) { metsToShow.add(metMatch[1].trim()); }
                            }
                        });
                        console.log('   ...Mostrando METs que tienen esta ruta:', Array.from(metsToShow));
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                if (rutaPattern.test(label)) {
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                } else if (metIconPattern.test(label)) {
                                    const metMatch = label.match(metNamePattern);
                                    if (metMatch && metsToShow.has(metMatch[1].trim())) {
                                        if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                    }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas (Rutas + METs) para Ruta ' + rutaId);
                        }, 150);
                    };
                    rutaButtonsDiv.appendChild(rutaBtn);
                });
                if (rutaButtonsDiv.children.length > 0) { layersList.insertBefore(rutaButtonsDiv, overlaysDiv); }
                
                console.log('=== Filtros inicializados correctamente ===');
                return true;
            }
            let filterPollCount = 0;
            function tryInitFilters() {
                const layerControlList = document.querySelector('.leaflet-control-layers-list');
                if (typeof L !== 'undefined' && layerControlList) {
                    console.log('✅ (Poll) Leaflet (L) y .leaflet-control-layers-list listos. Iniciando filtros...');
                    try { inicializarFiltros(); } catch (e) { console.error('❌ Error al ejecutar inicializarFiltros:', e); }
                } else if (filterPollCount < 40) {
                    filterPollCount++;
                    console.log('Poll Filtros #' + filterPollCount + ': Esperando a L y .leaflet-control-layers-list...');
                    setTimeout(tryInitFilters, 500);
                } else {
                    console.error('❌ No se pudo encontrar .leaflet-control-layers-list después de 20 segundos.');
                }
            }
            setTimeout(tryInitFilters, 500); 
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(filtros_js))
            
            titulo_html = f'''
            <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
                <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                    🗺️ ANÁLISIS DE CLUSTERS POR VOLUMEN
                </h2>
                <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                    📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas Generadas: <b>{len(summary_data)}</b> | 👥 Clientes Procesados: <b>{g_total_clientes_procesados}</b><br>
                    📦 Tamaño de Círculo = Volumen (m³/día) | 🎨 Color de Círculo = Ruta Asignada
                </p>
            </div>
            '''
            mapa_base.get_root().html.add_child(folium.Element(titulo_html))
            
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            map_filename = f'mapa_rutas_clusters_{timestamp}.html'
            g_map_path = os.path.join(output_dir, map_filename)
            mapa_base.save(g_map_path)
            
            print(f"\n{'='*70}")
            print(f"🗺️ Mapa guardado: {g_map_path}")
            
            if export_data and summary_data:
                excel_path = generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir)
                if excel_path:
                    g_excel_paths = [excel_path]
                    print(f"📊 Excel guardado: {excel_path}")
            
            print(f"{'='*70}")
            print(f"✅ ANÁLISIS COMPLETADO")
            print(f"   • Total clientes procesados: {g_total_clientes_procesados}")
            print(f"   • Total rutas generadas: {len(summary_data)}")
            print(f"   • Archivos generados: {len(g_excel_paths) + 1}")
            print(f"{'='*70}\n")

        except Exception as e:
            import traceback
            print(f"❌ Error Inesperado en ejecución #{execution_count}:")
            traceback.print_exc()
        finally:
            # --- (NUEVO) RESTAURAR DF_CLIENTES ORIGINAL ---
            if 'df_clientes_original' in locals():
                df_clientes = df_clientes_original
                print("ℹ️ DataFrame de clientes restaurado a su estado original.")
            
            param_button.disabled = False # Desbloquear botón

# --- Registrar el handler ---
try:
    param_button.on_click(ejecutar_analisis, remove=True)
except:
    pass
param_button.on_click(ejecutar_analisis)

# === MOSTRAR UI (Modificada) ===
print("================== MÓDULO 2: Análisis de Rutas (Alfa 5.1) ==================")
display(widgets.VBox([
    widgets.Label("1. Cargar archivo Excel:", style={'font_weight': 'bold'}),
    upload_widget,
    output_upload,
    widgets.HTML("<hr>"),
    met_box,
    widgets.HTML("<hr>"),
    widgets.Label("2. Configurar parámetros de Ruta:", style={'font_weight': 'bold'}),
    widgets.HBox([min_clientes_cluster, max_clientes_cluster]),
    
    # --- (NUEVO) WIDGETS DE MUESTREO ---
    widgets.HBox([sample_size_per_met, run_full_analysis]),
    # --- FIN DE WIDGETS NUEVOS ---
    
    param_button,
    output_param
]))

Codigo de una sola celda

In [ ]:
# ================== 1. INICIALIZACIÓN Y FUNCIONES BASE (Motor: Alfa 9.0) ==================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import os
import glob
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl 
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from shapely.geometry import Polygon, MultiPoint, Point
from sklearn.cluster import KMeans 
import shutil
import zipfile
from base64 import b64encode
import traceback # Para ver errores detallados

# --- (NUEVO) Constante de Distancia (Regla 5km) ---
# 1 grado de latitud es ~111.1 km. 5 km son ~0.045 grados.
KM_THRESHOLD_DEG = 5 / 111.1

# --- Rutas Locales (Alfa) ---
# ⚠️ ¡IMPORTANTE! Asegúrate de que esta ruta sea correcta en tu sistema.
icon_met_path = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png"
output_dir = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output"
os.makedirs(output_dir, exist_ok=True)
if not os.path.exists(icon_met_path):
    print(f"⚠️ Advertencia: No se encontró el icono en {icon_met_path}. Se usará icono por defecto.")
else:
    print(f"✅ Icono de MET encontrado: {icon_met_path}")

# --- Globales (Alfa) ---
execution_count = 0
df_clientes = pd.DataFrame()
df_cdr = pd.DataFrame()
met_checkboxes = [] 
g_map_path = ""       
g_excel_paths = []    
g_total_clientes_procesados = 0
g_descarga_en_progreso = False

# --- Colores (Alfa) ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# =========================================================================
# === FUNCIONES HELPER (LÓGICA ALFA) =======================================
# =========================================================================

def normalizar_id(valor):
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    s = s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    s = s.replace('Á','A').replace('É','E').replace('Í','I').replace('Ó','O').replace('Ú','U')
    s = s.replace('ñ','n').replace('Ñ','N')
    return s

def _clip_voronoi(vor, bounding_poly):
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty:
        return clipped_polys
    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points): continue
        if not (0 <= region_index < len(vor.regions)): continue
        region_vertices_indices = vor.regions[region_index]
        if -1 in region_vertices_indices: continue
        valid_vertex_indices = [v for v in region_vertices_indices if 0 <= v < len(vor.vertices)]
        if len(valid_vertex_indices) < 3: continue
        region_vertices = [vor.vertices[v] for v in valid_vertex_indices]
        if len(region_vertices) < 3: continue
        try:
            poly = Polygon(region_vertices)
            if not poly.is_valid: poly = poly.buffer(0)
            if not poly.is_valid: continue
            clipped = poly.intersection(bounding_poly)
            if clipped.is_empty: continue
            elif clipped.geom_type == 'Polygon' and clipped.exterior is not None:
                if len(clipped.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in clipped.exterior.coords]
                else: continue
            elif clipped.geom_type == 'MultiPolygon':
                if not clipped.geoms: continue
                largest_poly = max(clipped.geoms, key=lambda p: p.area)
                if largest_poly.exterior is not None and len(largest_poly.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in largest_poly.exterior.coords]
                else: continue
            else: continue
            centroid_key = tuple(vor.points[i])
            clipped_polys[centroid_key] = clipped_coords
        except Exception as e:
            pass 
    return clipped_polys

def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0:
        return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min:
        return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# =========================================================================
# === MÓDULO 1: LÓGICA DE CARGA (Corregido) ================================
# =========================================================================
def handle_upload(change, met_box_widget, output_upload_widget):
    global df_clientes, df_cdr, met_checkboxes 
    met_box = met_box_widget
    output_upload = output_upload_widget
    with output_upload:
        clear_output()
        upload_widget = change['owner']
        if not upload_widget.value:
            return
        
        uploaded_file_info = {}
        uploaded_file_name = ""
        uploaded_file_content = b""

        try:
            # Lógica para 'multiple=True' (valor es Tupla)
            if isinstance(upload_widget.value, tuple) and len(upload_widget.value) > 0:
                uploaded_file_info = upload_widget.value[0]
                uploaded_file_name = uploaded_file_info['name']
                uploaded_file_content = uploaded_file_info['content']
            # Lógica para 'multiple=False' (valor es Dict)
            elif isinstance(upload_widget.value, dict) and upload_widget.value:
                uploaded_file_name = list(upload_widget.value.keys())[0]
                uploaded_file_info = upload_widget.value[uploaded_file_name]
                uploaded_file_content = uploaded_file_info['content']
            else:
                 raise ValueError("Formato de archivo no reconocido o vacío")
        except Exception as e:
            print(f"❌ Error al procesar el archivo cargado: {e}")
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
            return

        try:
            with open(uploaded_file_name, 'wb') as f: f.write(uploaded_file_content)
            df_excel = pd.read_excel(uploaded_file_name, sheet_name=None)
            if len(df_excel) < 2:
                raise ValueError("El archivo Excel debe contener al menos dos hojas (Clientes y METs/CDRs).")
            sheet1_name, sheet2_name = list(df_excel.keys())[0], list(df_excel.keys())[1]
            df_clientes_raw, df_cdr_raw = df_excel[sheet1_name], df_excel[sheet2_name]
            
            required_cols_clientes = ['CodCli', 'U_longitud', 'U_latitud', 'm3/día', 'entregas/día']
            required_cols_cdr = ['CodMET', 'U_longitud', 'U_latitud']
            
            missing_cols_cli = [col for col in required_cols_clientes if col not in df_clientes_raw.columns]
            missing_cols_cdr = [col for col in required_cols_cdr if col not in df_cdr_raw.columns]
            
            if missing_cols_cli:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet1_name}': {', '.join(missing_cols_cli)}")
            if missing_cols_cdr:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet2_name}': {', '.join(missing_cols_cdr)}")

            df_clientes_raw['U_longitud'] = pd.to_numeric(df_clientes_raw['U_longitud'], errors='coerce')
            df_clientes_raw['U_latitud'] = pd.to_numeric(df_clientes_raw['U_latitud'], errors='coerce')
            
            # REGLA 1: Tratar '-' como 0
            df_clientes_raw['m3/día'] = df_clientes_raw['m3/día'].replace('-', 0)
            df_clientes_raw['m3/día'] = pd.to_numeric(df_clientes_raw['m3/día'], errors='coerce').fillna(0)
            df_clientes_raw['entregas/día'] = df_clientes_raw['entregas/día'].replace('-', 0)
            df_clientes_raw['entregas/día'] = pd.to_numeric(df_clientes_raw['entregas/día'], errors='coerce').fillna(0)
            
            df_cdr_raw['U_longitud'] = pd.to_numeric(df_cdr_raw['U_longitud'], errors='coerce')
            df_cdr_raw['U_latitud'] = pd.to_numeric(df_cdr_raw['U_latitud'], errors='coerce')

            df_clientes = df_clientes_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()
            df_cdr = df_cdr_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()

            num_dropped_cli = len(df_clientes_raw) - len(df_clientes)
            num_dropped_cdr = len(df_cdr_raw) - len(df_cdr)
            if num_dropped_cli > 0: print(f"⚠️ Se eliminaron {num_dropped_cli} clientes por coordenadas inválidas.")
            if num_dropped_cdr > 0: print(f"⚠️ Se eliminaron {num_dropped_cdr} METs/CDRs por coordenadas inválidas.")

            print(f"✅ Archivo '{uploaded_file_name}' cargado exitosamente.")
            print(f"📊 Clientes válidos: {df_clientes.shape[0]} filas")
            print(f"📊 METs/CDRs válidos: {df_cdr.shape[0]} filas")

            mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
            met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
            met_box.children = [widgets.Label('Selecciona los METs para optimizar (se recomienda >= 2):', style={'font_weight': 'bold'})] + met_checkboxes 
        except ValueError as ve:
            print(f"❌ Error de validación: {ve}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error en formato de archivo.')]
        except Exception as e:
            print(f"❌ Error inesperado al cargar: {e}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
        finally:
            if 'uploaded_file_name' in locals() and os.path.exists(uploaded_file_name):
                os.remove(uploaded_file_name)
            
            # --- INICIA CORRECCIÓN ---
            # (CORREGIDO) Resetea el widget de carga
            if isinstance(upload_widget.value, dict):
                upload_widget.value = {}
            else:
                upload_widget.value = ()
            # --- TERMINA CORRECCIÓN ---

# =========================================================================
# === MÓDULO 3: LÓGICA DE CLUSTERING (Alfa 9.0) ==========================
# =========================================================================

# --- (REGLA #2) LÓGICA DE SEMILLAS POR PUNTAJE (Alfa 8.0) ---
def find_initial_seeds(df_clients_met, k, met_lat, met_lon):
    """
    (Alfa 8.0 - Regla #2)
    - FILTRA: Solo considera clientes con entregas > 0.
    - PRIORITIZA: Usa el "Puntaje de Semilla" (Entregas / Distancia).
    """
    
    # REGLA 2a: Filtrar clientes. Semillas SOLO pueden ser clientes con entregas.
    df_seeds = df_clients_met[df_clients_met['entregas/día'] > 0].copy()

    n_puntos = len(df_seeds)
    if k <= 0 or n_puntos == 0:
        print("        ⚠️ No se pueden generar semillas (k<=0 o no hay puntos prioritarios).")
        return [], [] # Devuelve listas vacías
    
    if n_puntos < k:
        print(f"        ⚠️ No se pueden encontrar {k} semillas con {n_puntos} puntos prioritarios. Ajustando k a {n_puntos}.")
        k = n_puntos
        
    df_seeds['distancia_al_MET'] = np.hypot(
        df_seeds['U_latitud'] - met_lat, 
        df_seeds['U_longitud'] - met_lon
    )
    
    # REGLA 2b y 2c: El puntaje balancea Entregas (alto) y Distancia (bajo)
    df_seeds['puntaje_semilla'] = (df_seeds['entregas/día'] + 1) / (df_seeds['distancia_al_MET'] + 0.001)
    
    df_top_seeds = df_seeds.nlargest(k, 'puntaje_semilla')
    
    print(f"        ✅ {k} semillas seleccionadas por 'Puntaje de Semilla' (Prioritarios).")
    
    seed_coords = df_top_seeds[['U_longitud', 'U_latitud']].values
    seed_indices = df_top_seeds.index.tolist()
    
    return seed_coords, seed_indices

# --- (REGLA #3) LÓGICA DE CRECIMIENTO "ANCLA Y RELLENA" (Alfa 9.0) ---
def grow_clusters_from_seeds(df_clients_to_cluster, initial_seeds, seed_indices, max_size, min_size=0, progress_widget=None): # <--- 1. AÑADIR PARÁMETRO
    """
    (Alfa 9.0 - Regla #3 - "Ancla y Rellena por Prioridad")
    - Inicia 'Causa_Huerfano'
    - Asigna 'Sin espacio (Prioritario)' si las rutas se llenan.
    - (NUEVO) Actualiza la barra de progreso.
    """
    n_clientes = len(df_clients_to_cluster)
    k = len(initial_seeds)
    if k == 0 or n_clientes == 0:
        print("        ⚠️ No hay semillas o clientes para 'grow_clusters_from_seeds'.")
        df_clients_to_cluster['Ruta_nueva'] = -1
        return df_clients_to_cluster, {} 

    df_clients_to_cluster['Comodin'] = (df_clients_to_cluster['entregas/día'] <= 0)
    df_clients_to_cluster['Ruta_nueva'] = -1
    df_clients_to_cluster['Causa_Huerfano'] = ''
    cluster_counts_prioritario = {i: 0 for i in range(k)}

    # --- Fase A: Anclaje ---
    for ruta_id, client_idx in enumerate(seed_indices):
        if client_idx not in df_clients_to_cluster.index:
            print(f"        ⚠️ Índice de semilla {client_idx} no encontrado. Saltando anclaje.")
            continue
        df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = ruta_id
        cluster_counts_prioritario[ruta_id] = 1 
    print(f"        🔬 Fase A: {k} clientes-semilla 'anclados' a sus rutas.")

    # --- Fase B: Crecimiento "Redondo" ---
    df_restantes = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == -1].copy()
    n_restantes = len(df_restantes)
    
    if n_restantes == 0:
        print("        ... No hay clientes restantes para rellenar.")
        return df_clients_to_cluster, cluster_counts_prioritario
        
    print(f"        🔬 Fase B: Rellenando con {n_restantes} clientes (Prioritarios y Comodines)...")
    
    coords_restantes = df_restantes[['U_longitud', 'U_latitud']].values
    
    try:
        dist_matrix = distance_matrix(coords_restantes, initial_seeds)
    except ValueError as e:
        print(f"        ❌ Error calculando matriz de distancias: {e}")
        return df_clients_to_cluster, cluster_counts_prioritario

    dist_df = pd.DataFrame(dist_matrix, index=df_restantes.index)
    clientes_asignados = 0
    iteracion = 0
    max_iteraciones = n_restantes * k * 2 

    while clientes_asignados < n_restantes and iteracion < max_iteraciones:
        iteracion += 1
        
        # --- 2. ACTUALIZAR PROGRESO ---
        # (Se actualiza cada 50 iteraciones para no alentar el notebook)
        if progress_widget and iteracion % 50 == 0:
            progress_widget.value = clientes_asignados
        # --- FIN DE ACTUALIZACIÓN ---

        unassigned_indices = df_restantes.index[df_restantes['Ruta_nueva'] == -1]
        if unassigned_indices.empty: break
        
        sub_dist_df = dist_df.loc[unassigned_indices]
        min_dist_val = np.inf
        if not sub_dist_df.empty and np.any(np.isfinite(sub_dist_df.values)):
            min_finite = sub_dist_df[np.isfinite(sub_dist_df)].min().min()
            if pd.notna(min_finite):
                min_dist_val = min_finite
        if min_dist_val == np.inf: break
        
        min_pairs = sub_dist_df.stack()
        best_pairs_series = min_pairs[min_pairs == min_dist_val]
        if best_pairs_series.empty:
            try:
                next_min = min_pairs[min_pairs > min_dist_val].min()
                if pd.notna(next_min): best_pairs_series = min_pairs[min_pairs == next_min]
                if best_pairs_series.empty: break
            except: break
            
        best_pairs = best_pairs_series.index
        assigned_in_step = False
        sorted_best_pairs = sorted(list(best_pairs))
        
        for client_idx, cluster_idx in sorted_best_pairs:
            if df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] != -1: continue
            
            es_comodin = df_clients_to_cluster.loc[client_idx, 'Comodin']
            
            if es_comodin:
                df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = cluster_idx
                df_restantes.loc[client_idx, 'Ruta_nueva'] = cluster_idx
            elif cluster_counts_prioritario.get(cluster_idx, 0) < max_size:
                df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = cluster_idx
                df_restantes.loc[client_idx, 'Ruta_nueva'] = cluster_idx
                cluster_counts_prioritario[cluster_idx] = cluster_counts_prioritario.get(cluster_idx, 0) + 1
            else:
                dist_df.loc[client_idx, cluster_idx] = np.inf 
                continue 

            clientes_asignados += 1
            dist_df.loc[client_idx, :] = np.inf 
            assigned_in_step = True
                
        if not assigned_in_step:
            for client_idx, cluster_idx in best_pairs:
                if client_idx in dist_df.index and df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] == -1:
                    if cluster_idx in dist_df.columns:
                        dist_df.loc[client_idx, cluster_idx] = np.inf

    if iteracion >= max_iteraciones: print(f"        ⚠️ Se alcanzó el límite máximo de iteraciones ({max_iteraciones}) en la asignación.")
    print(f"        ✅ Fase B (Relleno) completada. {clientes_asignados} clientes asignados.")

    # --- 3. FINALIZAR BARRA DE PROGRESO ---
    if progress_widget:
        progress_widget.value = n_restantes # Marcar esta etapa como 100%
    # --- FIN ---

    huerfanos_idx = (df_clients_to_cluster['Ruta_nueva'] == -1) & (df_clients_to_cluster['Comodin'] == False)
    df_clients_to_cluster.loc[huerfanos_idx, 'Causa_Huerfano'] = 'Crecimiento (Sin espacio)'
    
    return df_clients_to_cluster, cluster_counts_prioritario

# --- (REGLAS #3 y #5) LÓGICA DE AJUSTE (Alfa 9.0) ---
def adjust_small_clusters(df_clustered, cluster_counts, min_size, max_size, met_lat, met_lon, progress_widget=None): # <--- 1. AÑADIR PARÁMETRO
    """
    (Alfa 9.0 - Reglas #3, #5 y #6)
    - (NUEVO) Actualiza la barra de progreso en Fases B (Donación) y C (Poda).
    """
    global KM_THRESHOLD_DEG # Necesita la constante global
    
    print(f"        🔬 Fase C: Ajustando clústeres (Min Prioritarios: {min_size}, Regla 5km)...")
    df_adjusted = df_clustered.copy()
    current_cluster_counts = cluster_counts.copy() 
    
    # --- FASE A: Calcular Centroides Iniciales ---
    centroids = {}
    rutas_all_ids = [r_id for r_id, count in current_cluster_counts.items() if count > 0]
    for ruta_id in rutas_all_ids:
        clientes_en_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
        if not clientes_en_ruta.empty:
            centroids[ruta_id] = clientes_en_ruta[['U_longitud', 'U_latitud']].mean().values
        else:
            current_cluster_counts.pop(ruta_id, None)
    if not centroids:
        print("        ... No hay rutas con clientes para ajustar.")
        return df_adjusted, current_cluster_counts

    # --- FASE B: Ajuste de Mínimos (Regla #3 - "No Lleno") ---
    print(f"        ... Sub-fase B: Buscando rutas con < {min_size} clientes PRIORITARIOS...")
    
    final_counts = pd.Series(current_cluster_counts)
    rutas_pequenas_ids = final_counts[(final_counts < min_size) & (final_counts > 0)].index.tolist()
    rutas_validas_ids = final_counts[final_counts >= min_size].index.tolist()

    if rutas_pequenas_ids and rutas_validas_ids:
        rutas_validas_coords = {
            r_id: df_adjusted[df_adjusted['Ruta_nueva'] == r_id][['U_latitud', 'U_longitud']].values
            for r_id in rutas_validas_ids
        }
        
        clientes_a_reasignar = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_pequenas_ids)].copy()
        df_adjusted.loc[clientes_a_reasignar.index, 'Ruta_nueva'] = -1
        df_adjusted.loc[clientes_a_reasignar.index, 'Causa_Huerfano'] = 'Disuelta (Ruta < Min)'
        
        for ruta_id in rutas_pequenas_ids:
            current_cluster_counts[ruta_id] = 0
            centroids.pop(ruta_id, None) 

        centroides_validos = {r_id: centro for r_id, centro in centroids.items() if r_id in rutas_validas_ids}
        
        if centroides_validos:
            print(f"        ... Disolviendo {len(rutas_pequenas_ids)} rutas. Reasignando {len(clientes_a_reasignar)} clientes (Regla 5km)...")
            
            # --- 2. PREPARAR BARRA PARA FASE B ---
            if progress_widget:
                progress_widget.max = len(clientes_a_reasignar)
                progress_widget.value = 0
                progress_widget.description = "Re-donando..."
            # --- FIN ---

            clientes_reasignados_count = 0
            for idx_cliente, row in clientes_a_reasignar.iterrows():
                
                # --- 3. ACTUALIZAR BARRA EN FASE B ---
                if progress_widget:
                    clientes_reasignados_count += 1
                    if clientes_reasignados_count % 20 == 0: # Actualizar cada 20 clientes
                        progress_widget.value = clientes_reasignados_count
                # --- FIN ---

                cliente_coords = row[['U_longitud', 'U_latitud']].values
                es_comodin = row['Comodin']
                candidatas_con_dist = []
                for ruta_id, coords_ruta in rutas_validas_coords.items():
                    if coords_ruta.shape[0] == 0: continue
                    # Usar Coordenadas (Lon, Lat) para distance_matrix
                    coords_ruta_lonlat = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id][['U_longitud', 'U_latitud']].values
                    dist_matrix_cliente = distance_matrix([cliente_coords], coords_ruta_lonlat)
                    min_dist_a_cliente_deg = dist_matrix_cliente.min()
                    
                    if min_dist_a_cliente_deg <= KM_THRESHOLD_DEG:
                        dist_a_centroide = np.linalg.norm(cliente_coords - centroides_validos[ruta_id])
                        candidatas_con_dist.append((dist_a_centroide, ruta_id))
                
                if not candidatas_con_dist:
                    df_adjusted.loc[idx_cliente, 'Causa_Huerfano'] = 'Disuelta (Distancia > 5km)'
                    continue
                
                candidatas_con_dist.sort()
                
                asignado = False
                for dist_centroide, ruta_id_destino in candidatas_con_dist:
                    if es_comodin:
                        df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = ruta_id_destino
                        df_adjusted.loc[idx_cliente, 'Causa_Huerfano'] = ''
                        asignado = True
                        break 
                    elif current_cluster_counts.get(ruta_id_destino, 0) < max_size:
                        df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = ruta_id_destino
                        df_adjusted.loc[idx_cliente, 'Causa_Huerfano'] = ''
                        current_cluster_counts[ruta_id_destino] += 1
                        asignado = True
                        break
                if not asignado and not es_comodin:
                    df_adjusted.loc[idx_cliente, 'Causa_Huerfano'] = 'Disuelta (Rutas Llenas)'

    # --- FASE C (Regla #5): Ajuste de Lejanía (Poda) ---
    print(f"        ... Sub-fase C: Verificando Regla de Lejanía (Poda)...")
    clientes_desasignados = 0
    
    centroids = {}
    rutas_actuales_ids = [r_id for r_id, count in current_cluster_counts.items() if count > 0]
    for ruta_id in rutas_actuales_ids:
        clientes_en_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
        if not clientes_en_ruta.empty:
            centroids[ruta_id] = clientes_en_ruta[['U_longitud', 'U_latitud']].mean().values
        else:
            current_cluster_counts.pop(ruta_id, None)
            
    coords_met = np.array([met_lon, met_lat])
    clientes_a_revisar_idx = df_adjusted[df_adjusted['Ruta_nueva'] != -1].index
    
    # --- 4. PREPARAR BARRA PARA FASE C ---
    if progress_widget:
        progress_widget.max = len(clientes_a_revisar_idx)
        progress_widget.value = 0
        progress_widget.description = "Podando..."
    # --- FIN ---
    
    clientes_revisados_count = 0
    for idx_cliente in clientes_a_revisar_idx:
        
        # --- 5. ACTUALIZAR BARRA EN FASE C ---
        if progress_widget:
            clientes_revisados_count += 1
            if clientes_revisados_count % 50 == 0: # Actualizar cada 50 clientes
                progress_widget.value = clientes_revisados_count
        # --- FIN ---

        row_cliente = df_adjusted.loc[idx_cliente]
        ruta_id = int(row_cliente['Ruta_nueva'])
        
        if ruta_id < 0 or ruta_id not in centroids: continue

        centroide = centroids[ruta_id]
        dist_C_MET = np.linalg.norm(centroide - coords_met)
        limite_distancia = dist_C_MET * 2.0
        
        cliente_coords = row_cliente[['U_longitud', 'U_latitud']].values
        dist_F_C = np.linalg.norm(cliente_coords - centroide)
        
        if dist_F_C > limite_distancia:
            df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = -1
            df_adjusted.loc[idx_cliente, 'Causa_Huerfano'] = 'Poda (Lejano al Centroide)'
            
            if not row_cliente['Comodin']:
                if ruta_id in current_cluster_counts:
                    current_cluster_counts[ruta_id] -= 1
                
            clientes_desasignados += 1

    if clientes_desasignados > 0:
        print(f"        ... {clientes_desasignados} clientes des-asignados por estar demasiado lejos de su ruta.")
    
    print("        ✅ Fase D (Ajuste) completada.")
    return df_adjusted, current_cluster_counts

# =========================================================================
# === MÓDULO 4: CÁLCULO DE POLÍGONOS (Alfa 6.0) ===============
# =========================================================================
def calculate_cluster_polygons(df_clustered, method='voronoi'):
    polygons = {} 
    df_con_ruta = df_clustered[df_clustered['Ruta_nueva'] >= 0]
    assigned_ruta_ids = sorted(df_con_ruta['Ruta_nueva'].unique())
    
    if not assigned_ruta_ids: return polygons
    
    centroides_rutas = [] 
    ruta_id_map = {} 
    for ruta_id in assigned_ruta_ids:
        ruta_clientes_df = df_con_ruta[df_con_ruta['Ruta_nueva'] == ruta_id]
        if not ruta_clientes_df.empty:
            lon = pd.to_numeric(ruta_clientes_df['U_longitud'], errors='coerce').mean()
            lat = pd.to_numeric(ruta_clientes_df['U_latitud'], errors='coerce').mean()
            if pd.notna(lon) and pd.notna(lat):
                centroides_rutas.append([lon, lat])
                ruta_id_map[tuple([lon, lat])] = ruta_id
    
    poligonos_voronoi_recortados = {}
    vor = None 
    
    if method == 'voronoi' and len(centroides_rutas) >= 4: 
        try:
            todos_los_puntos = df_con_ruta[['U_longitud', 'U_latitud']].values
            if len(todos_los_puntos) >= 3:
                boundary_poly_shapely = MultiPoint(todos_los_puntos).convex_hull.buffer(0).buffer(0.0001) 
                unique_centroids = np.unique(np.array(centroides_rutas), axis=0)
                if len(unique_centroids) >= 4:
                    vor = Voronoi(unique_centroids, qhull_options='Qbb Qc Qz') 
                    poligonos_voronoi_recortados = _clip_voronoi(vor, boundary_poly_shapely)
                else: print("        ... No hay suficientes centroides únicos (>=4) para Voronoi. Usando ConvexHull.")
            else: print("        ... No hay suficientes puntos asignados (<3) para definir frontera Voronoi.")
        except Exception as e:
            print(f"        ❌ Error al calcular Voronoi: {e}. Se usará ConvexHull.")
            poligonos_voronoi_recortados = {} 
    elif method == 'voronoi':
        print(f"        ... No hay suficientes rutas (se necesitan >= 4, se encontraron {len(centroides_rutas)}) para Voronoi. Usando ConvexHull.")

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_con_ruta[df_con_ruta['Ruta_nueva'] == ruta_id]
        if ruta_clientes.empty: continue
        
        centroide_lon = ruta_clientes['U_longitud'].mean()
        centroide_lat = ruta_clientes['U_latitud'].mean()
        poly_coords = poligonos_voronoi_recortados.get(tuple([centroide_lon, centroide_lat]))
        
        if not poly_coords and poligonos_voronoi_recortados:
            min_dist = float('inf')
            best_match = None
            current_centroid = np.array([centroide_lon, centroide_lat])
            for vor_centroid_key, vor_poly in poligonos_voronoi_recortados.items():
                dist = np.linalg.norm(current_centroid - np.array(vor_centroid_key))
                if dist < min_dist:
                    min_dist = dist
                    best_match = vor_poly
            if min_dist < 1e-6: poly_coords = best_match
            
        if poly_coords:
            polygons[ruta_id] = poly_coords
        else:
            points = ruta_clientes[['U_latitud', 'U_longitud']].values
            if len(points) >= 3:
                try:
                    hull = ConvexHull(points)
                    polygons[ruta_id] = points[hull.vertices].tolist() 
                except Exception:
                    print(f"        ⚠️ No se pudo generar polígono ConvexHull para Ruta {ruta_id}.")
                    polygons[ruta_id] = None
            else: polygons[ruta_id] = None 
    return polygons

# =========================================================================
# === MÓDULO 5: GENERACIÓN DE MAPA (Alfa 9.0) =============================
# =========================================================================
def create_map_elements(map_instance, df_clustered, cluster_polygons, colores_list, 
                        vol_min_global, vol_max_global, 
                        met_name=None):
    
    if met_name is None and 'MET_Asignado' in df_clustered.columns:
        met_name = df_clustered['MET_Asignado'].iloc[0] if not df_clustered.empty else ""
    elif met_name is None:
        met_name = ""
    
    met_name_clean = met_name.strip()
    display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"
    
    tamano_min, tamano_max = (6, 25)

    # --- 1. DIBUJAR RUTAS ASIGNADAS ---
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])
    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id].copy()
        
        n_prioritarios = (ruta_clientes['Comodin'] == False).sum()
        n_comodines = (ruta_clientes['Comodin'] == True).sum()
        ruta_size = len(ruta_clientes)
        if ruta_size == 0: continue

        color_ruta = colores_list[ruta_id % len(colores_list)] 
        volumen_total_ruta = ruta_clientes['m3/día'].sum()
        
        tooltip_poligono = (
            f"{display_name} - Ruta {ruta_id}<br>"
            f"Clientes: {n_prioritarios} (Prior) + {n_comodines} (Comodín)<br>"
            f"Vol: {volumen_total_ruta:,.2f} m³"
        )
        nombre_capa = f"{display_name} - Ruta {ruta_id}"
        fg_ruta_individual = folium.FeatureGroup(name=nombre_capa, show=True)
        
        poly_coords = cluster_polygons.get(ruta_id)
        if poly_coords:
            folium.Polygon(
                locations=poly_coords, color=color_ruta, weight=2, fill=True,
                fill_color=color_ruta, fill_opacity=0.3, tooltip=tooltip_poligono
            ).add_to(fg_ruta_individual)

        for _, cliente_row in ruta_clientes.iterrows(): 
            volumen = cliente_row['m3/día']
            tamano_marcador = obtener_tamano_marcador(volumen, vol_min_global, vol_max_global, tamano_min, tamano_max)
            
            if (vol_max_global - vol_min_global) > 0:
                percentil_vol = ((volumen - vol_min_global) / (vol_max_global - vol_min_global) * 100)
            else:
                percentil_vol = 50 

            codcli = cliente_row['CodCli']
            nombre = cliente_row.get('Nombre', 'N/A')
            ruta_original_val = cliente_row.get('Ruta', 'N/A')
            ruta_original = ruta_original_val if pd.notna(ruta_original_val) else 'N/A'
            entregas_cliente = cliente_row.get('entregas/día', 0)
            
            es_comodin = cliente_row.get('Comodin', False)
            comodin_label = '<span style="font-size:12px; color:#c7254e; background:#f9f2f4; padding: 2px 5px; border-radius:3px; font-weight:bold; margin-left: 10px;">COMODÍN</span>' if es_comodin else ''
            marker_opacity = 0.4 if es_comodin else 0.85

            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">👨‍💼 {codcli} {comodin_label}</div>
                <div style="font-size:14px; color:#333; margin-bottom:4px;">{nombre if pd.notna(nombre) else 'N/A'}</div>
                <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📦 Volumen: <b>{volumen:,.3f} m³</b> | 📬 Entregas: <b>{int(entregas_cliente)}</b>
                </div>
                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                    🗺️ Ruta (Orig): <b>{ruta_original}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                </div>
            </div>
            '''
            
            folium.CircleMarker(
                location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=350),
                radius=tamano_marcador, color='#333333', weight=1,
                fillColor=color_ruta, fillOpacity=marker_opacity,
            ).add_to(fg_ruta_individual)

        fg_ruta_individual.add_to(map_instance)

    # --- 2. DIBUJAR HUÉRFANOS (Req 1 y 3) ---
    df_huerfanos = df_clustered[df_clustered['Ruta_nueva'] < 0].copy()
    if not df_huerfanos.empty:
        fg_huerfanos = folium.FeatureGroup(name=f"{display_name} - ❌ Huérfanos", show=True)
        
        for _, cliente_row in df_huerfanos.iterrows():
            es_comodin = cliente_row.get('Comodin', False)
            causa_huerfano = cliente_row.get('Causa_Huerfano', 'Desconocida')
            codcli = cliente_row['CodCli']
            entregas_cliente = cliente_row.get('entregas/día', 0)
            
            # REGLA 1: Color Negro, Opacidad variable
            color_ruta = '#000000' # Negro
            marker_opacity = 0.4 if es_comodin else 0.85
            comodin_label = ' (Comodín)' if es_comodin else ' (Prioritario)'
            tamano_marcador = 5 # Tamaño fijo para huérfanos

            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">👨‍💼 {codcli} {comodin_label}</div>
                <div style="font-size:14px; color:#D32F2F; background: #FFEBEE; padding: 8px; border-radius: 4px; border: 1px solid #FFCDD2;">
                    <b>❌ Cliente Huérfano</b><br>
                    Causa: <b>{causa_huerfano}</b>
                </div>
                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px; margin-top: 8px;">
                    📬 Entregas: <b>{int(entregas_cliente)}</b>
                </div>
            </div>
            '''
            
            folium.CircleMarker(
                location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=350),
                radius=tamano_marcador,
                color='#333333', weight=1,
                fillColor=color_ruta,
                fillOpacity=marker_opacity,
            ).add_to(fg_huerfanos)
        
        fg_huerfanos.add_to(map_instance)

    return

# =========================================================================
# === MÓDULO 6: GENERACIÓN DE EXCEL (Alfa 9.0) ============================
# =========================================================================
def generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir):
    if not export_data or not summary_data:
        print(f"        ... No hay datos para exportar a Excel.")
        return None
    df_export = pd.DataFrame(export_data)
    df_resumen = pd.DataFrame(summary_data)
    if df_export.empty or df_resumen.empty:
        print(f"        ... DataFrames vacíos, no se puede generar Excel.")
        return None
    excel_filename = f'rutas_consolidadas_alfa9_{timestamp}.xlsx'
    excel_path = os.path.join(output_dir, excel_filename)
    
    df_resumen_sorted = df_resumen.sort_values(by=['MET', 'Ruta_nueva'])
    df_export_sorted = df_export.sort_values(by=['MET', 'Ruta_nueva', 'Codigo'])

    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_resumen_sorted.to_excel(writer, sheet_name='Resumen General', index=False)
            df_export_sorted.to_excel(writer, sheet_name='Todos los Clientes', index=False)
            for met_name in mets_seleccionados:
                df_met = df_export_sorted[df_export_sorted['MET'] == met_name]
                if not df_met.empty:
                    sheet_name = f'MET_{normalizar_id(met_name)}'[:31]
                    df_met.to_excel(writer, sheet_name=sheet_name, index=False)
        
        format_excel(excel_path) # Llamar a la función de formato
        return excel_path
    
    except Exception as e:
        print(f"        ❌ Error al generar Excel consolidado: {e}")
        if os.path.exists(excel_path):
            try: os.remove(excel_path)
            except: pass
        return None

def format_excel(excel_path):
    if not os.path.exists(excel_path):
        print(f"        ⚠️ No se encontró el archivo Excel para formatear: {excel_path}")
        return
    try:
        wb = openpyxl.load_workbook(excel_path)
        header_font = Font(bold=True, color='FFFFFF', size=11)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        summary_fill = PatternFill('solid', fgColor='FFD966') # Amarillo para resumen
        border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'), 
                        top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))
        align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
        align_left = Alignment(horizontal='left', vertical='center', wrap_text=False)
        
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws.max_row <= 1: continue
            
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align_center
                cell.border = border
                if cell.value: 
                    header_text = str(cell.value)
                    icon = ""
                    # Adaptado para nuevos nombres de columnas
                    if 'Ruta_Original' in header_text: icon = '🗺️ '
                    elif 'Codigo' in header_text or 'CodCli' in header_text: icon = '👨‍💼 ' 
                    elif 'Clientes' in header_text or 'Num_' in header_text: icon = '👥 ' 
                    elif 'MET' in header_text: icon = '🏠 '
                    elif 'Nombre' in header_text: icon = '📚 '
                    elif 'Ruta_nueva' in header_text or 'Ruta_ID' in header_text: icon = '🚚 '
                    elif 'Volumen' in header_text or 'm3' in header_text: icon = '📦 '
                    elif 'Centroide' in header_text: icon = '📍 '
                    elif 'Longitud' in header_text: icon = '📊 '
                    elif 'Latitud' in header_text: icon = '📊 '
                    elif 'entregas' in header_text: icon = '📬 '
                    elif 'Causa' in header_text: icon = '❓ '
                    elif 'Tipo' in header_text: icon = '🏷️ '
                    
                    cleaned_header = header_text.lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰📬❓🏷️ ')
                    cell.value = icon + cleaned_header
            
            is_summary_sheet = (sheet_name == 'Resumen General')
            
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for col_idx, cell in enumerate(row, start=1):
                    if is_summary_sheet:
                        cell.fill = summary_fill
                    else:
                        cell.fill = cell_fill
                    cell.border = border
                    
                    header_cell = ws.cell(row=1, column=col_idx)
                    header_val = header_cell.value if header_cell else None
                    if header_val and any(keyword in str(header_val) for keyword in ['Nombre', 'Codigo', 'CodCli', 'Causa', 'Tipo']):
                        cell.alignment = align_left
                    else:
                        cell.alignment = align_center
                        if header_val and any(coord in str(header_val) for coord in ['Longitud', 'Latitud']):
                            cell.number_format = '0.000000'
                        if header_val and ('Volumen' in str(header_val) or 'm3' in str(header_val)):
                            cell.number_format = '0.00'
                        if header_val and ('entregas' in str(header_val) or 'Num_' in str(header_val)):
                            cell.number_format = '0'
            
            for col_idx, column_cells in enumerate(ws.columns, 1):
                max_length = 0
                header_text = str(ws.cell(row=1, column=col_idx).value).lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰📬❓🏷️ ')
                max_length = len(header_text)
                for i, cell in enumerate(column_cells):
                    if i == 0: continue
                    if i > 50: break 
                    try:
                        if cell.value is not None:
                            if isinstance(cell.value, (int, float)) and cell.number_format and cell.number_format != 'General':
                                if '0.000000' in cell.number_format: cell_text = f"{cell.value:.6f}"
                                elif '0.00' in cell.number_format: cell_text = f"{cell.value:.2f}"
                                else: cell_text = str(cell.value)
                            else: cell_text = str(cell.value)
                            lines = cell_text.split('\n')
                            max_line_length = max(len(line) for line in lines) if lines else 0
                            max_length = max(max_length, max_line_length)
                    except: pass
                adjusted_width = min(max(12, max_length + 3), 45) 
                ws.column_dimensions[get_column_letter(col_idx)].width = adjusted_width
        
        wb.save(excel_path)
    except Exception as e:
        print(f"        ⚠️ Error al aplicar formato al Excel '{excel_path}': {e}")

print("✅ CELDA 1 (Alfa 9.0): Funciones base cargadas.")
print("--- Motor de clustering 'Alfa 9.0' (Híbrido + 5km + Causa Huérfano) implantado ---")
print(f"📁 Directorio de salida: {output_dir}")
print(f"🎯 Icono MET: {'✅ Encontrado' if os.path.exists(icon_met_path) else '⚠️ No encontrado'}")


# ================== 2. INTERFAZ Y EJECUCIÓN (Motor: Alfa 9.0) ==================

# --- Definición de Widgets ---
upload_widget = widgets.FileUpload(accept='.xlsx', multiple=False, description='Subir Excel')
met_box = widgets.VBox([widgets.Label('⏳ Esperando archivo Excel...')])
output_upload = widgets.Output()
min_clientes_cluster = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Min Clientes (Prior):', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))
max_clientes_cluster = widgets.IntSlider(value=40, min=10, max=100, step=1, description='Max Clientes (Prior):', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))

# --- Widgets de Muestreo (Alfa 4.0) ---
sample_size_per_met = widgets.IntText(
    value=500, 
    description='Clientes a muestrear por MET:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='48%')
)
run_full_analysis = widgets.Checkbox(
    value=False, 
    description='✅ Analizar TODOS los clientes (Ignora el muestreo)', 
    indent=False, 
    layout=widgets.Layout(width='48%')
)

param_button = widgets.Button(description='Ejecutar Análisis de Territorio y Rutas (Alfa 9.0)', button_style='success', disabled=False, layout=widgets.Layout(width='98%'))
output_param = widgets.Output()

# --- Registrar handler de carga ---
upload_widget.observe(lambda change: handle_upload(change, met_box, output_upload), names='value')

# --- Función Principal de Ejecución (Alfa 9.0 con Progreso) ---
def ejecutar_analisis(b):
    global execution_count, df_clientes, df_cdr, met_checkboxes, colores
    global g_map_path, g_excel_paths, g_total_clientes_procesados, KM_THRESHOLD_DEG
    global status_label, progress_bar # <--- OBTENER WIDGETS GLOBALES
    
    execution_count += 1
    
    with output_param:
        clear_output(wait=True)
        param_button.disabled = True
        
        # --- 1. Resetear Widgets ---
        status_label.value = "Iniciando..."
        progress_bar.value = 0
        progress_bar.description = "Iniciando..."
        progress_bar.bar_style = 'info' # Azul
        
        # --- 2. Validaciones ---
        if df_clientes.empty or df_cdr.empty:
            print("❌ Carga un archivo Excel primero.")
            status_label.value = "Error: Carga un archivo Excel."
            param_button.disabled = False
            return
        
        mets_seleccionados = [cb.description for cb in met_checkboxes if cb.value]
        if not mets_seleccionados or len(mets_seleccionados) < 2:
            print("❌ Selecciona al menos dos METs para optimizar territorios.")
            status_label.value = "Error: Selecciona al menos dos METs."
            param_button.disabled = False
            return
        
        min_size = min_clientes_cluster.value
        max_size = max_clientes_cluster.value
        n_muestra = sample_size_per_met.value
        analizar_todos = run_full_analysis.value
        
        if min_size >= max_size:
            print("❌ El mínimo debe ser menor que el máximo.")
            status_label.value = "Error: El Mínimo debe ser menor que el Máximo."
            param_button.disabled = False
            return
        
        print(f"\n{'='*70}")
        print(f"🚀 INICIANDO ANÁLISIS - Motor Alfa 9.0 (Regla 5km + Causa Huérfano)")
        print(f"{'='*70}")
        print(f"📊 Parámetros:")
        print(f"    • METs a optimizar: {len(mets_seleccionados)}")
        print(f"    • Clientes Prioritarios por ruta: {min_size} - {max_size}")
        print(f"    • Comodines (Entregas <= 0) se añadirán extra.")
        print(f"    • Límite de donación: {KM_THRESHOLD_DEG*111.1:.2f} km")
        
        if analizar_todos:
            print(f"    • MODO: ✅ COMPLETO (Se analizarán TODOS los clientes)")
        else:
            print(f"    • MODO: ⚠️ MUESTRA (Se analizarán {n_muestra} clientes por MET)")
            
        print(f"{'='*70}\n")
        
        g_map_path = ""
        g_excel_paths = []
        g_total_clientes_procesados = 0
        
        df_clientes_original = df_clientes.copy()
        df_clientes_global_para_crecer = df_clientes.copy()
        
        # --- 3. FASE 0: ASIGNACIÓN DE TERRITORIOS ---
        try:
            status_label.value = "🗺️ Fase 0: Asignando territorios (para semillas)..."
            print(f"🗺️ Fase 0: Asignando territorios (para encontrar semillas)...")
            
            mets_info = df_cdr[df_cdr['CodMET'].isin(mets_seleccionados)]
            if mets_info.empty:
                print("❌ No se encontraron datos para los METs seleccionados.")
                status_label.value = "Error: No se encontraron METs en los datos."
                param_button.disabled = False
                return

            coords_clientes = df_clientes_global_para_crecer[['U_latitud', 'U_longitud']].values
            coords_mets = mets_info[['U_latitud', 'U_longitud']].values
            
            dist_matrix_territorio = distance_matrix(coords_clientes, coords_mets)
            indices_met_mas_cercano = np.argmin(dist_matrix_territorio, axis=1)
            nombres_met_mas_cercano = [mets_info['CodMET'].iloc[i] for i in indices_met_mas_cercano]
            
            df_clientes_con_territorio = df_clientes_global_para_crecer.copy()
            df_clientes_con_territorio['MET_Asignado'] = nombres_met_mas_cercano
            
            print(f"✅ Territorios (para semillas) asignados. Conteo:")
            print(df_clientes_con_territorio['MET_Asignado'].value_counts())
            
            volumenes_positivos = df_clientes_global_para_crecer[df_clientes_global_para_crecer['m3/día'] > 0]['m3/día']
            vol_min_global = volumenes_positivos.min() if not volumenes_positivos.empty else 0
            vol_max_global = volumenes_positivos.max() if not volumenes_positivos.empty else 0

            # --- 3. Crear Mapa Base ---
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            lats = mets_info['U_latitud']
            lons = mets_info['U_longitud']
            bounds = [[lats.min(), lons.min()], [lats.max(), lons.max()]]
            mapa_base.fit_bounds(bounds, padding=(20, 20))
            
            colores_list = colores.copy()
            random.shuffle(colores_list)
            
            export_data = []
            summary_data = []
            df_final_global = pd.DataFrame() # Para acumular resultados

            # --- 4. FASE 1: CLUSTERING POR TERRITORIO ---
            print(f"\n🔬 Fase 1: Iniciando clustering...")
            status_label.value = "🔬 Fase 1: Encontrando semillas..."
            
            all_seed_indices = []
            dfs_para_crecer = []
            
            # --- 4a. Encontrar todas las semillas primero ---
            for idx_met, met_seleccionado in enumerate(mets_seleccionados):
                print(f"\n{'-'*50}")
                print(f"🏠 Preparando Semillas para: {met_seleccionado} ({idx_met + 1}/{len(mets_seleccionados)})")
                status_label.value = f"🏠 Preparando Semillas para: {met_seleccionado}..."
                
                met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado].iloc[0]
                met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
                
                df_territorio_completo = df_clientes_con_territorio[df_clientes_con_territorio['MET_Asignado'] == met_seleccionado].copy()
                
                if df_territorio_completo.empty:
                    print(f"⚠️ No hay clientes en el territorio de '{met_seleccionado}'. Saltando.")
                    continue
                
                df_clientes_para_semillas = pd.DataFrame() 
                if analizar_todos:
                    df_clientes_para_semillas = df_territorio_completo
                    print(f"✅ Buscando semillas en territorio COMPLETO: {len(df_clientes_para_semillas)} clientes")
                else:
                    if len(df_territorio_completo) > n_muestra:
                        print(f"⚠️ MODO MUESTRA: Buscando semillas en {n_muestra} de {len(df_territorio_completo)} clientes")
                        df_clientes_para_semillas = df_territorio_completo.sample(n=n_muestra).copy()
                    else:
                        df_clientes_para_semillas = df_territorio_completo
                        print(f"✅ Buscando semillas en territorio COMPLETO (es menor que la muestra): {len(df_clientes_para_semillas)} clientes")

                if df_clientes_para_semillas.empty:
                    print(f"⚠️ No hay clientes para procesar en este MET. Saltando.")
                    continue
                
                k_inicial = max(1, int(round(len(df_clientes_para_semillas[df_clientes_para_semillas['entregas/día'] > 0]) / ((min_size + max_size) / 2))))
                print(f"🔬 Calculando {k_inicial} semillas (solo de clientes Prioritarios)...")
                
                initial_seeds_coords, seed_indices = find_initial_seeds(df_clientes_para_semillas, k_inicial, met_lat, met_lon)
                
                if len(initial_seeds_coords) == 0:
                    print(f"❌ No se pudieron generar semillas para '{met_seleccionado}'.")
                    continue
                
                all_seed_indices.extend(seed_indices)
                
                # Guardar el trabajo para la Fase 2
                dfs_para_crecer.append({
                    'met_seleccionado': met_seleccionado,
                    'met_lat': met_lat,
                    'met_lon': met_lon,
                    'initial_seeds_coords': initial_seeds_coords,
                    'seed_indices': seed_indices
                })
            
            # --- 4b. Pool Global de Crecimiento ---
            print("\n" + "="*70)
            print(f"🌱 Preparando Pool Global de Crecimiento...")
            
            # Pool Global = Todos los clientes MENOS los que ya son semillas
            df_pool_global_crecimiento = df_clientes_global_para_crecer[~df_clientes_global_para_crecer.index.isin(all_seed_indices)].copy()
            df_semillas = df_clientes_global_para_crecer[df_clientes_global_para_crecer.index.isin(all_seed_indices)].copy()
            
            print(f"    • Total Semillas encontradas: {len(all_seed_indices)}")
            print(f"    • Total Clientes en Pool de Crecimiento: {len(df_pool_global_crecimiento)}")
            print("="*70)

            # --- 4c. Crecer y Ajustar (Iterar sobre el trabajo guardado) ---
            for job in dfs_para_crecer:
                met_seleccionado = job['met_seleccionado']
                met_lat = job['met_lat']
                met_lon = job['met_lon']
                initial_seeds_coords = job['initial_seeds_coords']
                seed_indices = job['seed_indices']
                
                print(f"\n🌱 Creciendo {len(seed_indices)} rutas para {met_seleccionado}...")
                
                # --- 5. CONFIGURAR Y PASAR WIDGETS ---
                n_clientes_a_asignar = len(df_pool_global_crecimiento)
                progress_bar.max = n_clientes_a_asignar
                progress_bar.value = 0
                progress_bar.description = f"Creciendo {met_seleccionado}"
                status_label.value = f"🌱 Creciendo rutas para {met_seleccionado} ({n_clientes_a_asignar} clientes)..."
                
                # REGLA #1 HÍBRIDA: Pasamos el pool GLOBAL + las semillas de este MET
                df_para_cluster = pd.concat([df_pool_global_crecimiento, df_semillas.loc[seed_indices]])
                
                df_clustered, cluster_counts = grow_clusters_from_seeds(
                    df_para_cluster.copy(),
                    initial_seeds_coords, 
                    seed_indices,
                    max_size,
                    min_size,
                    progress_bar # <--- PASAR WIDGET
                )
                
                # Volver a añadir las semillas (que fueron removidas)
                # NOTA: Esto ya no es necesario si las semillas se pasan en el df_para_cluster
                # y se manejan en grow_clusters_from_seeds (Fase A)
                
                status_label.value = f"🔬 Ajustando rutas para {met_seleccionado} (Regla 5km)..."

                df_adjusted, final_counts = adjust_small_clusters(
                    df_clustered, 
                    cluster_counts, # Conteo de Prioritarios
                    min_size, 
                    max_size, 
                    met_lat, 
                    met_lon,
                    progress_bar # <--- PASAR WIDGET
                )
                
                # Acumular resultados
                df_final_met = df_adjusted.copy()
                df_final_met['MET_Asignado'] = met_seleccionado # Marcar todas las rutas con este MET
                df_final_global = pd.concat([df_final_global, df_final_met])

                g_total_clientes_procesados += len(df_final_met) # Este conteo es aproximado

            # --- 4. FASE 2: PROCESAR RESULTADOS GLOBALES ---
            print("\n" + "="*70)
            print("🌎 Procesando resultados globales y dibujando mapa...")
            status_label.value = "🌎 Procesando resultados globales y dibujando mapa..."
            
            # Quitar duplicados
            df_final_global = df_final_global[~df_final_global.index.duplicated(keep='first')]
            
            cluster_polygons_global = calculate_cluster_polygons(df_final_global, method='voronoi')
            
            for met_seleccionado in mets_seleccionados:
                met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado].iloc[0]
                df_rutas_de_este_met = df_final_global[df_final_global['MET_Asignado'] == met_seleccionado]
                
                if df_rutas_de_este_met.empty: 
                    print(f"    ... Sin rutas finales para {met_seleccionado}")
                    continue
                
                # Añadir icono de MET
                folium.Marker(
                    location=[met_info['U_latitud'], met_info['U_longitud']], popup=f"🏠 {met_seleccionado}",
                    icon=folium.Icon(color='red', icon='home'), tooltip=met_seleccionado
                ).add_to(mapa_base)
                
                # Dibujar rutas
                create_map_elements(
                    mapa_base, df_rutas_de_este_met, cluster_polygons_global,
                    colores_list, vol_min_global, vol_max_global, met_seleccionado
                )

                # --- REGLA #3d: Preparar data para Excel ---
                rutas_validas = df_rutas_de_este_met[df_rutas_de_este_met['Ruta_nueva'] >= 0]['Ruta_nueva'].unique()
                for ruta_id in rutas_validas:
                    ruta_clientes = df_rutas_de_este_met[df_rutas_de_este_met['Ruta_nueva'] == ruta_id]
                    
                    n_prioritarios = (ruta_clientes['Comodin'] == False).sum()
                    n_comodines = (ruta_clientes['Comodin'] == True).sum()
                    volumen_total = ruta_clientes['m3/día'].sum()
                    entregas_total = ruta_clientes[ruta_clientes['Comodin'] == False]['entregas/día'].sum()
                    
                    summary_data.append({
                        'MET': met_seleccionado, 'Ruta_nueva': ruta_id,
                        'Num_Prioritarios': n_prioritarios, 'Num_Comodines': n_comodines,
                        'Num_Clientes_Total': n_prioritarios + n_comodines,
                        'Volumen_Total_m3': round(volumen_total, 2),
                        'Entregas_Totales_Prioritarias': int(entregas_total),
                        'Centroide_Lat': ruta_clientes['U_latitud'].mean(),
                        'Centroide_Lon': ruta_clientes['U_longitud'].mean()
                    })
                
                # Añadir todos los clientes (asignados y huérfanos) de este MET
                for _, cliente in df_rutas_de_este_met.iterrows():
                    ruta_val = cliente['Ruta_nueva']
                    export_data.append({
                        'MET': met_seleccionado,
                        'Ruta_nueva': int(ruta_val) if ruta_val >= 0 else 'HUERFANO',
                        'Tipo_Cliente': 'Comodín' if cliente['Comodin'] else 'Prioritario',
                        'Causa_Huerfano': cliente.get('Causa_Huerfano', ''), # REGLA DE CAUSA
                        'Codigo': cliente['CodCli'],
                        'Nombre': cliente.get('Nombre', 'N/A'),
                        'Ruta_Original': cliente.get('Ruta', 'N/A'),
                        'm3_dia': cliente['m3/día'],
                        'entregas_dia': cliente.get('entregas/día', 0),
                        'Latitud': cliente['U_latitud'],
                        'Longitud': cliente['U_longitud']
                    })
            
            # --- 5. Guardar Mapa y Excel ---
            print("\n" + "="*70)
            print("💾 Guardando archivos...")
            status_label.value = "💾 Guardando archivos finales..."
            
            folium.LayerControl(collapsed=False).add_to(mapa_base)
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            map_filename = f'mapa_rutas_alfa9_{timestamp}.html'
            g_map_path = os.path.join(output_dir, map_filename)
            mapa_base.save(g_map_path)
            print(f"🗺️  Mapa guardado: {g_map_path}")

            if export_data and summary_data:
                excel_path = generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir)
                if excel_path:
                    g_excel_paths = [excel_path]
                    print(f"📊 Excel guardado: {excel_path}")
            
            print(f"{'='*70}")
            print(f"✅ ANÁLISIS COMPLETADO (Alfa 9.0)")
            print(f"    • Total rutas generadas: {len(summary_data)}")
            print(f"{'='*70}\n")
            
            # --- 7. ESTADO FINAL ÉXITO ---
            status_label.value = f"✅ ¡Análisis Completado! {len(summary_data)} rutas generadas."
            progress_bar.description = "Completado"
            progress_bar.bar_style = 'success' # Verde
            if progress_bar.max > 0:
                progress_bar.value = progress_bar.max # Llenar la barra

        except Exception as e:
            print(f"❌ Error Inesperado en ejecución #{execution_count}:")
            traceback.print_exc()
            # --- 8. ESTADO FINAL ERROR ---
            status_label.value = f"❌ Error Inesperado. Revisa la consola de salida."
            progress_bar.bar_style = 'danger' # Rojo
            
        finally:
            df_clientes = df_clientes_original
            print("ℹ️ DataFrame de clientes restaurado a su estado original.")
            param_button.disabled = False
            
# --- Registrar el handler ---
try:
    param_button.on_click(ejecutar_analisis, remove=True)
except:
    pass
param_button.on_click(ejecutar_analisis)

# === MOSTRAR UI (Modificada para Alfa 9.0 con Progreso) ===

# --- 1. Definir los widgets de progreso en el alcance global ---
status_label = widgets.Label(value="Estado: Inactivo")
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100, 
    description='Progreso:', 
    bar_style='info', 
    orientation='horizontal',
    layout=widgets.Layout(width='98%')
)

print("================== MÓDULO 2: Análisis de Rutas (Alfa 9.0) ==================")
display(widgets.VBox([
    widgets.Label("1. Cargar archivo Excel:", style={'font_weight': 'bold'}),
    upload_widget,
    output_upload,
    widgets.HTML("<hr>"),
    met_box,
    widgets.HTML("<hr>"),
    widgets.Label("2. Configurar parámetros de Ruta (Prioritarios):", style={'font_weight': 'bold'}),
    widgets.HBox([min_clientes_cluster, max_clientes_cluster]),
    
    # --- Widgets de Muestreo ---
    widgets.HBox([sample_size_per_met, run_full_analysis]),
    # --- Fin ---
    
    param_button,
    
    # --- 2. Añadir los widgets de progreso a la UI ---
    status_label,
    progress_bar,
    # --- Fin ---
    
    output_param
]))

Versión ajustada a frecuencia, con filtros y me gusta 20/11/2025

In [ ]:
#=========== 1. INICIALIZACIÓN Y FUNCIONES BASE (Alfa 5.1) ==================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import os
import glob
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from shapely.geometry import Polygon, MultiPoint, Point # Importar Point y MultiPoint
from sklearn.cluster import KMeans
import shutil # Importar shutil para limpieza
import zipfile # Importar zipfile para Celda 3

# --- Validar rutas locales ---
icon_met_path = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png"
output_dir = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output"

# Crear directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(icon_met_path):
    print(f"⚠️ Advertencia: No se encontró el icono en {icon_met_path}. Se usará icono por defecto.")
else:
    print(f"✅ Icono de MET encontrado: {icon_met_path}")


# --- GLOBALES DEL ESTADO ---
execution_count = 0
df_clientes = pd.DataFrame()
df_cdr = pd.DataFrame()
met_checkboxes = []
# --- GLOBALES PARA INTERCAMBIO DE MÓDULOS ---
g_map_path = ""
g_excel_paths = []
g_total_clientes_procesados = 0
g_descarga_en_progreso = False # Para el bloqueo de descarga

# --- LISTA DE COLORES GLOBAL ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# =========================================================================
# === FUNCIONES HELPER (LÓGICA) ===========================================
# =========================================================================

# --- HELPER FUNCTION PARA NORMALIZAR IDS ---
def normalizar_id(valor):
    """Convierte un valor a un ID seguro para HTML/JS."""
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    s = s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    s = s.replace('Á','A').replace('É','E').replace('Í','I').replace('Ó','O').replace('Ú','U')
    s = s.replace('ñ','n').replace('Ñ','N')
    return s

# --- FUNCIÓN HELPER PARA VORONOI ---
def _clip_voronoi(vor, bounding_poly):
    """Recorta los polígonos de Voronoi a un polígono de frontera (shapely)."""
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty:
        return clipped_polys

    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points): continue
        if not (0 <= region_index < len(vor.regions)): continue
        region_vertices_indices = vor.regions[region_index]
        if -1 in region_vertices_indices: continue
        valid_vertex_indices = [v for v in region_vertices_indices if 0 <= v < len(vor.vertices)]
        if len(valid_vertex_indices) < 3: continue
        region_vertices = [vor.vertices[v] for v in valid_vertex_indices]
        if len(region_vertices) < 3: continue

        try:
            poly = Polygon(region_vertices)
            if not poly.is_valid: poly = poly.buffer(0)
            if not poly.is_valid: continue

            clipped = poly.intersection(bounding_poly)

            if clipped.is_empty: continue
            elif clipped.geom_type == 'Polygon' and clipped.exterior is not None:
                if len(clipped.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in clipped.exterior.coords]
                else: continue
            elif clipped.geom_type == 'MultiPolygon':
                if not clipped.geoms: continue
                largest_poly = max(clipped.geoms, key=lambda p: p.area)
                if largest_poly.exterior is not None and len(largest_poly.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in largest_poly.exterior.coords]
                else: continue
            else: continue

            centroid_key = tuple(vor.points[i])
            clipped_polys[centroid_key] = clipped_coords
        except Exception as e:
            pass

    return clipped_polys

# --- FUNCIÓN HELPER PARA TAMAÑO DE MARCADOR (DE CÓDIGO CELAYA) ---
def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    """
    Calcula el tamaño del marcador basado en el volumen.
    Usa escala logarítmica para mejor diferenciación.
    """
    if vol_max == vol_min or volumen <= 0:
        return tamano_min

    # Usar escala logarítmica para mejor diferenciación visual
    log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)

    if log_max == log_min:
        return tamano_min

    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# =========================================================================
# === MÓDULO 1: LÓGICA DE CARGA Y VALIDACIÓN ==============================
# =========================================================================
def handle_upload(change, met_box_widget, output_upload_widget):
    global df_clientes, df_cdr, met_checkboxes

    # Suppress FutureWarning for pandas replace method downcasting
    pd.set_option('future.no_silent_downcasting', True)

    met_box = met_box_widget
    output_upload = output_upload_widget

    with output_upload:
        clear_output()
        upload_widget = change['owner']
        if not upload_widget.value:
            return

        # Adaptado para manejar diferentes formatos de FileUpload
        try:
            # Intentar obtener el archivo del widget
            if isinstance(upload_widget.value, tuple) and len(upload_widget.value) > 0:
                # Formato: tuple de FileInfo objects
                uploaded_file_info = upload_widget.value[0]
                uploaded_file_name = uploaded_file_info['name']
                uploaded_file_content = uploaded_file_info['content']
            elif isinstance(upload_widget.value, dict):
                # Formato: dict con nombres de archivo como keys
                uploaded_file_details = next(iter(upload_widget.value.values()))
                uploaded_file_name = uploaded_file_details['metadata']['name']
                uploaded_file_content = uploaded_file_details['content']
            else:
                raise ValueError("Formato de archivo no reconocido")
        except Exception as e:
            print(f"❌ Error al procesar el archivo cargado: {e}")
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
            return

        try:
            with open(uploaded_file_name, 'wb') as f: f.write(uploaded_file_content)
            df_excel = pd.read_excel(uploaded_file_name, sheet_name=None)

            if len(df_excel) < 2:
                raise ValueError("El archivo Excel debe contener al menos dos hojas (Clientes y METs/CDRs).")

            sheet1_name, sheet2_name = list(df_excel.keys())[0], list(df_excel.keys())[1]
            df_clientes_raw, df_cdr_raw = df_excel[sheet1_name], df_excel[sheet2_name]

            required_cols_clientes = ['CodCli', 'U_longitud', 'U_latitud', 'm3/día', 'entregas/día']
            required_cols_cdr = ['CodMET', 'U_longitud', 'U_latitud']

            missing_cols_cli = [col for col in required_cols_clientes if col not in df_clientes_raw.columns]
            missing_cols_cdr = [col for col in required_cols_cdr if col not in df_cdr_raw.columns]

            if missing_cols_cli:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet1_name}': {', '.join(missing_cols_cli)}")
            if missing_cols_cdr:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet2_name}': {', '.join(missing_cols_cdr)}")

            df_clientes_raw['U_longitud'] = pd.to_numeric(df_clientes_raw['U_longitud'], errors='coerce')
            df_clientes_raw['U_latitud'] = pd.to_numeric(df_clientes_raw['U_latitud'], errors='coerce')

            # Convertir m3/día: reemplazar "-" por 0 y convertir a numérico
            df_clientes_raw['m3/día'] = df_clientes_raw['m3/día'].replace('-', 0)
            df_clientes_raw['m3/día'] = pd.to_numeric(df_clientes_raw['m3/día'], errors='coerce').fillna(0)

            # Convertir entregas/día: reemplazar "-" por 0 y convertir a numérico (valores entre 0 y 1)
            df_clientes_raw['entregas/día'] = df_clientes_raw['entregas/día'].replace('-', 0)
            df_clientes_raw['entregas/día'] = pd.to_numeric(df_clientes_raw['entregas/día'], errors='coerce').fillna(0)

            df_cdr_raw['U_longitud'] = pd.to_numeric(df_cdr_raw['U_longitud'], errors='coerce')
            df_cdr_raw['U_latitud'] = pd.to_numeric(df_cdr_raw['U_latitud'], errors='coerce')

            df_clientes = df_clientes_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()
            df_cdr = df_cdr_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()

            num_dropped_cli = len(df_clientes_raw) - len(df_clientes)
            num_dropped_cdr = len(df_cdr_raw) - len(df_cdr)
            if num_dropped_cli > 0: print(f"⚠️ Se eliminaron {num_dropped_cli} clientes por coordenadas inválidas.")
            if num_dropped_cdr > 0: print(f"⚠️ Se eliminaron {num_dropped_cdr} METs/CDRs por coordenadas inválidas.")

            print(f"✅ Archivo '{uploaded_file_name}' cargado exitosamente.")
            print(f"📊 Clientes válidos: {df_clientes.shape[0]} filas")
            print(f"📊 METs/CDRs válidos: {df_cdr.shape[0]} filas")

            mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
            met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
            met_box.children = [widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes

        except ValueError as ve:
            print(f"❌ Error de validación: {ve}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error en formato de archivo.')]
        except Exception as e:
            print(f"❌ Error inesperado al cargar: {e}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
        finally:
            if 'uploaded_file_name' in locals() and os.path.exists(uploaded_file_name):
                os.remove(uploaded_file_name)

# =========================================================================
# === MÓDULO 3: LÓGICA DE CLUSTERING (Alfa 5.1) ==========================
# =========================================================================
def select_customers_for_met(df_all_clients, met_lat, met_lon, n_clients, analyze_all):
    # Esta función se mantiene por si se usa en el futuro,
    # pero la lógica de 'Territorios' (Fase 0) en Módulo 2 la reemplaza.
    if df_all_clients.empty: return pd.DataFrame()
    if analyze_all:
        return df_all_clients.copy()
    else:
        distances = np.hypot(df_all_clients['U_latitud'] - met_lat, df_all_clients['U_longitud'] - met_lon)
        df_with_dist = df_all_clients.copy()
        df_with_dist['Distancia_al_MET'] = distances
        n_to_select = min(n_clients, len(df_with_dist))
        if n_to_select <= 0: return pd.DataFrame()
        return df_with_dist.nsmallest(n_to_select, 'Distancia_al_MET')

def calcular_dispersion_cluster(df_cluster):
    if len(df_cluster) < 2:
        return 0.0
    coords = df_cluster[['U_longitud', 'U_latitud']].values
    distancias_pares = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = np.hypot(coords[i][0] - coords[j][0], coords[i][1] - coords[j][1])
            distancias_pares.append(dist)
    return np.mean(distancias_pares) if distancias_pares else 0.0

def calcular_limites_clientes_dinamico(dispersion, frecuencia_visitas_promedio, min_base=20, max_base=40):
    """
    Ajusta dinámicamente el límite de clientes por ruta basándose en:
    - dispersion: Dispersión geográfica de la ruta (distancia promedio entre clientes)
    - frecuencia_visitas_promedio: Frecuencia promedio de entregas/día (0-1, donde 1 = 100%)

    A mayor dispersión geográfica → menos clientes permitidos
    A MAYOR frecuencia de visitas → MÁS clientes permitidos (rutas densas y frecuentes)
    """
    dispersion_normalizada = min(dispersion / 0.5, 1.0)

    # Normalizar frecuencia de visitas (ya viene en escala 0-1, donde 0.9692 = 96.92%)
    # Mayor frecuencia = MENOS restricción (más clientes permitidos para rutas frecuentes)
    frecuencia_normalizada = min(frecuencia_visitas_promedio, 1.0)

    # Factor de restricción: Solo la dispersión restringe, la frecuencia AUMENTA el límite
    # Invertimos el efecto de la frecuencia: alta frecuencia reduce la restricción
    factor_restriccion = dispersion_normalizada * (1.0 - frecuencia_normalizada * 0.5)

    rango = max_base - min_base
    ajuste = int(rango * factor_restriccion)
    max_dinamico = max_base - ajuste
    min_dinamico = max(min_base - int(ajuste * 0.5), int(min_base * 0.5))
    return min_dinamico, max_dinamico

def find_initial_seeds(coords, k, random_state=42):
    n_puntos = coords.shape[0]
    if k <= 0 or n_puntos == 0:
        print("       ⚠️ No se pueden generar semillas (k<=0 o no hay puntos).")
        return np.array([])
    if n_puntos < k:
        print(f"       ⚠️ No se pueden encontrar {k} semillas con {n_puntos} puntos. Ajustando k a {n_puntos}.")
        k = n_puntos
    try:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=20, init='k-means++').fit(coords)
        return kmeans.cluster_centers_
    except ValueError as e:
        print(f"       ❌ Error en KMeans para semillas (k={k}, n_puntos={n_puntos}): {e}")
        if n_puntos > 0:
            indices = np.random.choice(n_puntos, min(k, n_puntos), replace=False)
            print(f"       ... Usando {len(indices)} puntos aleatorios como semillas (Fallback).")
            return coords[indices]
        else:
            return np.array([])

# --- (MODIFICADO) FUNCIÓN PARA ENCONTRAR ANCLAS DE FRECUENCIA ---
def find_frequency_anchors(df_clients, percentile=80):
    """Encuentra los clientes 'ancla' (ej. 20% con mayor frecuencia de entregas/día)"""
    # No tomar en cuenta clientes con frecuencia muy baja o cero
    df_with_frequency = df_clients[df_clients['entregas/día'] > 0.01]
    if df_with_frequency.empty:
        print("       ... No se encontraron clientes con frecuencia significativa para anclaje.")
        return pd.DataFrame() # No hay anclas

    # Usar un umbral de percentil para definir "alta frecuencia"
    min_frequency_threshold = np.percentile(df_with_frequency['entregas/día'], percentile)
    df_anchors = df_with_frequency[df_with_frequency['entregas/día'] >= min_frequency_threshold].copy()

    print(f"       ... Umbral de anclaje (percentil {percentile}%) = {min_frequency_threshold*100:.1f}% entregas/día")
    print(f"       ... {len(df_anchors)} clientes 'ancla' de alta frecuencia identificados.")
    return df_anchors

# --- (MODIFICADO) LÓGICA DE CRECIMIENTO HÍBRIDA (Alfa 5.1) ---
def grow_clusters_from_seeds(df_clients_to_cluster, initial_seeds, max_size, min_size=20):
    """
    Asigna clientes a rutas usando una estrategia HÍBRIDA (Alfa 5.1):
    1. Fase A: Anclar rutas con clientes de alto volumen (Tu idea).
    2. Fase B: Rellenar rutas con lógica 'greedy' (redonda) (Lógica Beta).
    3. Fase C: Rellenar con comodines (Lógica Alfa).
    """
    k = len(initial_seeds)
    n_clientes_total = len(df_clients_to_cluster)
    if k == 0 or n_clientes_total == 0:
        print("       ⚠️ No hay semillas o clientes para 'grow_clusters_from_seeds'.")
        df_clients_to_cluster['Ruta_nueva'] = -1
        return df_clients_to_cluster, {}

    # --- Inicialización (Lógica Alfa) ---
    df_clients_to_cluster['Ruta_nueva'] = -1
    ruta_volumenes = {i: 0.0 for i in range(k)}
    ruta_counts = {i: 0 for i in range(k)}

    # --- FASE A: ANCLAJE POR FRECUENCIA (MODIFICADO) ---
    print(f"       🔬 Fase A: Anclando {k} rutas con clientes de alta frecuencia (entregas/día)...")
    df_anchors = find_frequency_anchors(df_clients_to_cluster)
    n_asignados_anchors = 0

    if not df_anchors.empty:
        coords_anchors = df_anchors[['U_longitud', 'U_latitud']].values
        dist_matrix_anchors = distance_matrix(coords_anchors, initial_seeds)

        # Iterar por cada ancla
        for i, (anchor_idx, row) in enumerate(df_anchors.iterrows()):
            frecuencia_ancla = row['entregas/día']
            volumen_ancla = row['m3/día']
            rutas_ordenadas = np.argsort(dist_matrix_anchors[i]) # Rutas más cercanas primero

            asignado = False
            for ruta_idx in rutas_ordenadas:
                # Validar solo por límite de clientes (sin restricción de volumen)
                # Para la Fase de Anclaje, usamos un límite estático ya que la ruta está vacía
                if ruta_counts[ruta_idx] < max_size:
                    df_clients_to_cluster.loc[anchor_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_volumenes[ruta_idx] += volumen_ancla
                    ruta_counts[ruta_idx] += 1
                    n_asignados_anchors += 1
                    asignado = True
                    break # Ancla asignada, pasar a la siguiente ancla

            if not asignado:
                print(f"       ⚠️ Ancla {anchor_idx} (Frec: {frecuencia_ancla*100:.1f}%) no pudo ser asignada (todas las rutas están llenas).")

    print(f"       ✅ Fase A (Anclaje) completada: {n_asignados_anchors}/{len(df_anchors)} anclas asignadas.")

    # --- FASE B: RELLENO GEOMÉTRICO (Lógica "Redonda" de Beta) ---
    df_normales_restantes = df_clients_to_cluster[
        (df_clients_to_cluster['entregas/día'] > 0.01) &  # Frecuencia > 1%
        (df_clients_to_cluster['Ruta_nueva'] == -1)
    ].copy()

    n_normales = len(df_normales_restantes)
    n_asignados_normales = 0

    if n_normales > 0:
        print(f"       🔬 Fase B: Rellenando geométricamente {n_normales} clientes restantes...")
        coords_normales = df_normales_restantes[['U_longitud', 'U_latitud']].values

        try:
            dist_matrix = distance_matrix(coords_normales, initial_seeds)
        except ValueError as e:
            print(f"       ❌ Error calculando matriz de distancias: {e}")
            dist_matrix = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_normales])

        dist_df = pd.DataFrame(dist_matrix, index=df_normales_restantes.index, columns=range(k))

        iteracion = 0
        max_iter = n_normales * k # Límite de seguridad

        while n_asignados_normales < n_normales and iteracion < max_iter:
            iteracion += 1
            min_dist = dist_df.min().min()
            if min_dist == np.inf:
                break

            stacked = dist_df.stack()
            best_pair_idx = stacked[stacked == min_dist].index
            client_idx, ruta_idx = best_pair_idx[0]

            # --- Validar con límites dinámicos (sin restricción de 6m³) ---
            volumen_cliente = df_normales_restantes.loc[client_idx, 'm3/día']
            # volumen_futuro = ruta_volumenes[ruta_idx] + volumen_cliente # Volumen ya no afecta la asignación

            clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
            dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

            # Calcular frecuencia promedio de visitas de la ruta (incluyendo el nuevo cliente)
            if len(clientes_ruta_actual) > 0:
                frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
            else:
                frecuencia_ruta_actual = 0.0

            # Estimar frecuencia futura (promedio ponderado)
            frecuencia_cliente = df_normales_restantes.loc[client_idx, 'entregas/día']
            if len(clientes_ruta_actual) > 0:
                frecuencia_futura = ((frecuencia_ruta_actual * len(clientes_ruta_actual)) + frecuencia_cliente) / (len(clientes_ruta_actual) + 1)
            else:
                frecuencia_futura = frecuencia_cliente

            min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                dispersion_actual, frecuencia_futura, min_base=min_size, max_base=max_size
            )

            # Solo validar límite de clientes (SIN límite de volumen)
            if ruta_counts[ruta_idx] < max_dinamico:
                # Éxito: Asignar
                df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = ruta_idx
                ruta_volumenes[ruta_idx] += volumen_cliente # Actualizar volumen para resumen, pero no para decisión
                ruta_counts[ruta_idx] += 1
                n_asignados_normales += 1
                dist_df.loc[client_idx, :] = np.inf # Cliente asignado - marca TODAS las distancias como infinitas
            else:
                # Fallo: Invalidar solo este par cliente-ruta
                dist_df.loc[client_idx, ruta_idx] = np.inf

        if iteracion >= max_iter:
            print(f"       ⚠️ Se alcanzó el límite de iteraciones ({max_iter}).")

        print(f"       ✅ Fase B completada: {n_asignados_normales}/{n_normales} clientes asignados.")

    # --- FASE C: RELLENO DE COMODINES (MODIFICADO: entregas/día ≤ 1%) ---
    df_comodines = df_clients_to_cluster[
        (df_clients_to_cluster['entregas/día'] <= 0.01) &
        (df_clients_to_cluster['Ruta_nueva'] == -1)
    ].copy()

    n_comodines = len(df_comodines)
    if n_comodines > 0:
        print(f"       🔬 Fase C: Asignando {n_comodines} clientes 'comodines' (entregas/día ≤ 1%)...")
        coords_comodines = df_comodines[['U_longitud', 'U_latitud']].values

        try:
            dist_matrix_comodines = distance_matrix(coords_comodines, initial_seeds)
        except ValueError as e:
            print(f"       ❌ Error calculando matriz de distancias para comodines: {e}")
            dist_matrix_comodines = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_comodines])

        for idx_comodin, (real_idx, row) in enumerate(df_comodines.iterrows()):
            distancias_rutas = dist_matrix_comodines[idx_comodin]
            rutas_ordenadas = np.argsort(distancias_rutas)

            asignado = False
            for ruta_idx in rutas_ordenadas:
                clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

                # Calcular frecuencia promedio (MODIFICADO)
                if len(clientes_ruta_actual) > 0:
                    frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
                else:
                    frecuencia_ruta_actual = 0.0

                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, frecuencia_ruta_actual, min_base=min_size, max_base=max_size
                )

                if ruta_counts[ruta_idx] < max_dinamico:
                    df_clients_to_cluster.loc[real_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_volumenes[ruta_idx] += row['m3/día'] # Actualizar volumen para resumen, pero no para decisión
                    ruta_counts[ruta_idx] += 1
                    asignado = True
                    break

            if not asignado:
                ruta_mas_cercana = rutas_ordenadas[0]
                df_clients_to_cluster.loc[real_idx, 'Ruta_nueva'] = ruta_mas_cercana
                ruta_volumenes[ruta_mas_cercana] += row['m3/día'] # Actualizar volumen para resumen, pero no para decisión
                ruta_counts[ruta_mas_cercana] += 1

    # --- Devolver resultados (Lógica Alfa Original) ---
    rutas_validas = {}
    for ruta_idx in range(k):
        # Una ruta es válida si tiene volumen O si es una ruta de puros comodines
        if ruta_volumenes[ruta_idx] > 0 or ruta_counts[ruta_idx] > 0:
             rutas_validas[ruta_idx] = ruta_counts[ruta_idx]

    return df_clients_to_cluster, rutas_validas

# --- (MODIFICADO) LÓGICA DE AJUSTE CON GEOMETRÍA (Alfa 5.1) ---
def adjust_small_clusters(df_clustered, cluster_counts, min_size, max_size):
    """
    Ajusta rutas pequeñas (< min_size).
    Reasigna clientes al POLÍGONO válido más cercano (no al centroide).
    (MODIFICADO: Usa frecuencia de visitas en lugar de volumen para dispersión)
    """
    print("       🔬 Fase D: Ajustando rutas pequeñas...")
    df_adjusted = df_clustered.copy()

    # Bucle de reintentos
    attempts = 0
    max_attempts = 3 # Limitar los pases de ajuste
    while attempts < max_attempts:
        attempts += 1

        # --- 1. Calcular volúmenes y conteos actuales ---
        ruta_volumenes = {}
        ruta_counts_actual = {}
        for ruta_id in df_adjusted['Ruta_nueva'].unique():
            if ruta_id < 0: continue
            clientes_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            ruta_volumenes[ruta_id] = clientes_ruta['m3/día'].sum()
            ruta_counts_actual[ruta_id] = len(clientes_ruta)

        if not ruta_counts_actual:
            print("       ... No hay rutas que ajustar.")
            break # No hay nada que hacer

        # --- 2. Identificar rutas VÁLIDAS vs. RUTAS A MOVER ---
        rutas_validas_ids = set()
        rutas_a_mover_ids = set()

        for ruta_id, count in ruta_counts_actual.items():
            # Regla de negocio: Solo Pequeña (sin criterio de rentabilidad)
            if count < min_size:
                rutas_a_mover_ids.add(ruta_id)
            else:
                rutas_validas_ids.add(ruta_id)

        if not rutas_a_mover_ids:
            print(f"       ✅ Fase D completada: No hay rutas pequeñas. (Intento {attempts})")
            break # Éxito, no hay nada que ajustar

        if not rutas_validas_ids:
            print(f"       ⚠️ Fase D: Hay {len(rutas_a_mover_ids)} rutas pequeñas, pero NO hay rutas válidas para moverlas. Deteniendo.")
            break

        # --- 3. Crear Geometrías Válidas (Tu idea) ---
        geometrias_validas = {}
        for ruta_id in rutas_validas_ids:
            clientes_validos = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            points = clientes_validos[['U_longitud', 'U_latitud']].values
            if len(points) >= 3:
                try:
                    # Usamos ConvexHull para obtener el *área* de la ruta
                    hull = ConvexHull(points)
                    geometrias_validas[ruta_id] = Polygon(points[hull.vertices])
                except Exception as e:
                    pass # Ignorar esta ruta si ConvexHull falla

        if not geometrias_validas:
            print(f"       ⚠️ Fase D: No se pudieron crear polígonos para las rutas válidas. Deteniendo.")
            break

        # --- 4. Iterar y Reasignar Clientes "Huérfanos" ---
        df_clientes_a_mover = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_a_mover_ids)].copy()
        print(f"       ... Intento {attempts}: Moviendo {len(df_clientes_a_mover)} clientes de {len(rutas_a_mover_ids)} rutas...")

        clientes_reasignados_en_intento = 0

        for idx_cliente, row in df_clientes_a_mover.iterrows():
            punto_cliente = Point(row['U_longitud'], row['U_latitud'])
            volumen_cliente = row['m3/día']
            original_ruta = row['Ruta_nueva']

            # Calcular distancia a TODOS los polígonos válidos
            distancias_a_poligonos = []
            for ruta_id, poly in geometrias_validas.items():
                dist = punto_cliente.distance(poly)
                distancias_a_poligonos.append((dist, ruta_id))

            if not distancias_a_poligonos:
                continue # No hay polígonos válidos

            distancias_a_poligonos.sort() # Ordenar por distancia (más cercano primero)

            # Intentar asignar al más cercano que cumpla las reglas
            asignado = False
            for dist, ruta_id_destino in distancias_a_poligonos:
                # Validar con Lógica Alfa (MODIFICADO: usa frecuencia)
                # volumen_actual = ruta_volumenes.get(ruta_id_destino, 0)
                # volumen_futuro = volumen_actual + volumen_cliente

                # if volumen_futuro > 6.0:
                #     continue # Excede 6m³, probar siguiente polígono

                clientes_ruta_actual = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id_destino]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

                # Calcular frecuencia promedio futura (MODIFICADO)
                if len(clientes_ruta_actual) > 0:
                    frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
                else:
                    frecuencia_ruta_actual = 0.0

                frecuencia_cliente = row['entregas/día']
                if len(clientes_ruta_actual) > 0:
                    frecuencia_futura = ((frecuencia_ruta_actual * len(clientes_ruta_actual)) + frecuencia_cliente) / (len(clientes_ruta_actual) + 1)
                else:
                    frecuencia_futura = frecuencia_cliente

                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, frecuencia_futura, min_base=min_size, max_base=max_size
                )

                if ruta_counts_actual.get(ruta_id_destino, 0) < max_dinamico:
                    # ¡Éxito! Reasignar
                    df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = ruta_id_destino

                    # Actualizar contadores para la siguiente iteración
                    ruta_counts_actual[ruta_id_destino] = ruta_counts_actual.get(ruta_id_destino, 0) + 1
                    ruta_volumenes[ruta_id_destino] = ruta_volumenes.get(ruta_id_destino, 0) + volumen_cliente # Actualizar volumen para resumen, no para decisión

                    ruta_counts_actual[original_ruta] -= 1
                    ruta_volumenes[original_ruta] = max(0, ruta_volumenes[original_ruta] - volumen_cliente)

                    clientes_reasignados_en_intento += 1
                    asignado = True
                    break # Cliente asignado, pasar al siguiente cliente

            # (No hay 'fallback' forzado. Si no cabe, no cabe)

        if clientes_reasignados_en_intento == 0:
            print("       ... No se pudieron reasignar más clientes en este intento. Deteniendo ajuste.")
            break

    if attempts >= max_attempts:
        print(f"       ⚠️ Fase D: Se alcanzó el límite de {max_attempts} intentos. Pueden quedar rutas pequeñas.")

    # Devolver el DF ajustado y el último conteo conocido
    return df_adjusted, ruta_counts_actual


# =========================================================================
# === MÓDULO 4: CÁLCULO DE POLÍGONOS (Sin cambios) ========================
# =========================================================================
def calculate_cluster_polygons(df_clustered, method='voronoi'):
    # Esta función se usa solo para la VISUALIZACIÓN FINAL
    polygons = {}
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])
    if not assigned_ruta_ids: return polygons

    centroides_rutas = []
    ruta_id_map = {}
    for ruta_id in assigned_ruta_ids:
        ruta_clientes_df = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if not ruta_clientes_df.empty:
            lon = pd.to_numeric(ruta_clientes_df['U_longitud'], errors='coerce').mean()
            lat = pd.to_numeric(ruta_clientes_df['U_latitud'], errors='coerce').mean()
            if pd.notna(lon) and pd.notna(lat):
                centroides_rutas.append([lon, lat])
                ruta_id_map[tuple([lon, lat])] = ruta_id

    poligonos_voronoi_recortados = {}
    vor = None
    if method == 'voronoi' and len(centroides_rutas) >= 4:
        try:
            todos_los_puntos = df_clustered[df_clustered['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
            if len(todos_los_puntos) >= 3:
                boundary_poly_shapely = MultiPoint(todos_los_puntos).convex_hull.buffer(0).buffer(0.0001)
                unique_centroids = np.unique(np.array(centroides_rutas), axis=0)
                if len(unique_centroids) >= 4:
                    vor = Voronoi(unique_centroids, qhull_options='Qbb Qc Qz')
                    poligonos_voronoi_recortados = _clip_voronoi(vor, boundary_poly_shapely)
                else: print("       ... No hay suficientes centroides únicos (>=4) para Voronoi.")
            else: print("       ... No hay suficientes puntos asignados (<3) para definir frontera Voronoi.")
        except Exception as e:
            print(f"       ❌ Error al calcular Voronoi: {e}. Se usará ConvexHull.")
            poligonos_voronoi_recortados = {}
    elif method == 'voronoi':
        print(f"       ... No hay suficientes rutas (se necesitan >= 4, se encontraron {len(centroides_rutas)}) para Voronoi. Usando ConvexHull.")

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if ruta_clientes.empty: continue

        # Encontrar el polígono de Voronoi que corresponde a este centroide
        centroide_lon = ruta_clientes['U_longitud'].mean()
        centroide_lat = ruta_clientes['U_latitud'].mean()
        poly_coords = poligonos_voronoi_recortados.get(tuple([centroide_lon, centroide_lat]))

        if not poly_coords and poligonos_voronoi_recortados:
            # Fallback por si hay errores de precisión de float
            min_dist = float('inf')
            best_match = None
            current_centroid = np.array([centroide_lon, centroide_lat])
            for vor_centroid_key, vor_poly in poligonos_voronoi_recortados.items():
                dist = np.linalg.norm(current_centroid - np.array(vor_centroid_key))
                if dist < min_dist:
                    min_dist = dist
                    best_match = vor_poly
            if min_dist < 1e-6: poly_coords = best_match

        if poly_coords:
            polygons[ruta_id] = poly_coords
        else:
            # Fallback a ConvexHull si Voronoi falló o no se usó
            points = ruta_clientes[['U_latitud', 'U_longitud']].values
            if len(points) >= 3:
                try:
                    hull = ConvexHull(points)
                    polygons[ruta_id] = points[hull.vertices].tolist()
                except Exception:
                    print(f"       ⚠️ No se pudo generar polígono ConvexHull para Ruta {ruta_id}.")
                    polygons[ruta_id] = None
            else: polygons[ruta_id] = None
    return polygons

# =========================================================================
# === MÓDULO 5: GENERACIÓN DE MAPA (Sin Rentabilidad - Solo Frecuencia) ==
# =========================================================================
def create_map_elements(map_instance, df_clustered, cluster_polygons, colores_list,
                        vol_min_global, vol_max_global,
                        met_name=None):
    """
    Dibuja los elementos del cluster y los AÑADE a las capas de filtrado.
    Colores normales para todas las rutas (sin lógica de rentabilidad).
    """

    if met_name is None and 'MET_Asignado' in df_clustered.columns:
        met_name = df_clustered['MET_Asignado'].iloc[0] if not df_clustered.empty else ""
    elif met_name is None:
        met_name = ""

    met_name_clean = met_name.strip()
    display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"

    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])

    tamano_min, tamano_max = (6, 25)

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id].copy()
        ruta_size = len(ruta_clientes)
        if ruta_size == 0: continue

        # --- Asignar color normal a todas las rutas ---
        volumen_total_ruta = ruta_clientes['m3/día'].sum()
        frecuencia_promedio_ruta = ruta_clientes['entregas/día'].mean() * 100  # Convertir a porcentaje

        color_ruta = colores_list[ruta_id % len(colores_list)]
        tooltip_poligono = f"{display_name} - Ruta {ruta_id} ({ruta_size} clientes)<br>Vol: {volumen_total_ruta:,.2f} m³ | Frec: {frecuencia_promedio_ruta:.1f}%"
        nombre_capa = f"{display_name} - Ruta {ruta_id}"

        fg_ruta_individual = folium.FeatureGroup(name=nombre_capa, show=True)

        # Solo se usa el polígono de "solo clientes" (punteado)
        if len(ruta_clientes) >= 3:
            try:
                puntos_solo_clientes = [Point(row['U_longitud'], row['U_latitud']) for _, row in ruta_clientes.iterrows()]
                multipoint_clientes = MultiPoint(puntos_solo_clientes)
                hull_clientes = multipoint_clientes.convex_hull

                if hull_clientes.geom_type == 'Polygon':
                    coords_polygon_clientes = [(lat, lon) for lon, lat in hull_clientes.exterior.coords]
                    area_popup_clientes = f'''
                    <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                        <div style="font-size: 14px; font-weight: bold;">👥 Área Solo Clientes</div>
                        <div style="font-size: 12px; margin-top: 4px;">🏠 {met_name_clean} - 🛣️ Ruta {ruta_id}</div>
                        <div style="font-size: 11px; margin-top: 2px;">👨‍💼 {len(ruta_clientes)} clientes</div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📦 Vol: {volumen_total_ruta:,.2f} m³</div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📊 Frec: {frecuencia_promedio_ruta:.1f}%</div>
                    </div>
                    '''

                    poly_clientes_obj_clon = folium.Polygon(
                        locations=coords_polygon_clientes,
                        color=color_ruta,
                        weight=1,
                        opacity=0.8,
                        fillColor=color_ruta,
                        fillOpacity=0.1,
                        popup=folium.Popup(area_popup_clientes, max_width=250),
                        tooltip=tooltip_poligono,
                        dash_array='5, 5'
                    )
                    poly_clientes_obj_clon.add_to(fg_ruta_individual)
            except Exception as e:
                print(f"       ⚠️ No se pudo generar polígono 'solo clientes' para Ruta {ruta_id}: {e}")

        for _, cliente_row in ruta_clientes.iterrows():
            volumen = cliente_row['m3/día']
            frecuencia = cliente_row['entregas/día']
            frecuencia_pct = frecuencia * 100  # Convertir a porcentaje
            tamano_marcador = obtener_tamano_marcador(volumen, vol_min_global, vol_max_global, tamano_min, tamano_max)

            codcli = cliente_row['CodCli']
            nombre = cliente_row.get('Nombre', 'N/A')
            ruta_original_val = cliente_row.get('Ruta', 'N/A')
            ruta_original = ruta_original_val if pd.notna(ruta_original_val) else 'N/A'

            # Popup sin TOP ni barra de progreso - solo información
            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                    👨‍💼 {codcli}
                </div>
                <div style="font-size:14px; color:#333; margin-bottom:4px;">
                    {nombre if pd.notna(nombre) else 'N/A'}
                </div>
                <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📦 Volumen: <b>{volumen:,.3f} m³</b>
                </div>

                <div style="font-size:14px; color:#1976D2; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📊 Frecuencia: <b>{frecuencia_pct:.1f}%</b> (entregas/día)
                </div>

                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                    🗺️ Ruta (Orig): <b>{ruta_original}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                </div>
            </div>
            '''

            # Tooltip más corto con frecuencia
            tooltip_text = f"{codcli} | Vol: {volumen:.2f} m³ | Frec: {frecuencia_pct:.1f}%"

            marker_obj_clon = folium.CircleMarker(
                location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=tooltip_text,
                radius=tamano_marcador,
                color='#333333',
                weight=1,
                fillColor=color_ruta,
                fillOpacity=0.85,
            )
            marker_obj_clon.add_to(fg_ruta_individual)

        fg_ruta_individual.add_to(map_instance)
    return

# =========================================================================
# === MÓDULO 6: GENERACIÓN DE EXCEL (Sin Rentabilidad) ====================
# =========================================================================
def generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir):
    if not export_data or not summary_data:
        print(f"       ... No hay datos para exportar a Excel.")
        return None
    df_export = pd.DataFrame(export_data)
    df_resumen = pd.DataFrame(summary_data)
    if df_export.empty or df_resumen.empty:
        print(f"       ... DataFrames vacíos, no se puede generar Excel.")
        return None
    excel_filename = f'rutas_consolidadas_{timestamp}.xlsx'
    excel_path = os.path.join(output_dir, excel_filename)
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_resumen_sorted = df_resumen.sort_values(by=['MET', 'Ruta_nueva'])
            df_resumen_sorted.to_excel(writer, sheet_name='Resumen General', index=False)
            df_export_sorted = df_export.sort_values(by=['MET', 'Ruta_nueva', 'Codigo'])
            df_export_sorted.to_excel(writer, sheet_name='Todos los Clientes', index=False)
            for met_name in mets_seleccionados:
                df_met = df_export[df_export['MET'] == met_name].sort_values(by=['Ruta_nueva', 'Codigo'])
                if not df_met.empty:
                    sheet_name = f'MET_{normalizar_id(met_name)}'[:31]
                    df_met.to_excel(writer, sheet_name=sheet_name, index=False)
        format_excel(excel_path)
        return excel_path
    except Exception as e:
        print(f"       ❌ Error al generar Excel consolidado: {e}")
        if os.path.exists(excel_path):
            try: os.remove(excel_path)
            except: pass
        return None

def format_excel(excel_path):
    """
    Aplica formato profesional al Excel de salida.
    (SIN lógica de rentabilidad - todos los formatos normales)
    """
    if not os.path.exists(excel_path):
        print(f"       ⚠️ No se encontró el archivo Excel para formatear: {excel_path}")
        return
    try:
        wb = openpyxl.load_workbook(excel_path)

        # --- Definición de Estilos ---
        header_font = Font(bold=True, color='FFFFFF', size=11)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        summary_fill = PatternFill('solid', fgColor='FFD966') # Amarillo para resumen

        border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'),
                        top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))
        align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
        align_left = Alignment(horizontal='left', vertical='center', wrap_text=False)

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws.max_row <= 1: continue

            # Formatear Encabezados
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align_center
                cell.border = border
                if cell.value:
                    header_text = str(cell.value)
                    icon = ""
                    if 'Ruta_Original' in header_text: icon = '🗺️ '
                    elif 'Codigo' in header_text or 'CodCli' in header_text: icon = '👨‍💼 '
                    elif 'Clientes' in header_text: icon = '👥 '
                    elif 'MET' in header_text: icon = '🏠 '
                    elif 'Nombre' in header_text: icon = '📚 '
                    elif 'Ruta_nueva' in header_text or 'Ruta_ID' in header_text: icon = '🚚 '
                    elif 'Volumen' in header_text or 'm3' in header_text: icon = '📦 '
                    elif 'Centroide' in header_text: icon = '📍 '
                    elif 'Longitud' in header_text: icon = '📊 '
                    elif 'Latitud' in header_text: icon = '📊 '
                    elif 'Rentable' in header_text: icon = '💰 '
                    cleaned_header = header_text.lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰 ')
                    cell.value = icon + cleaned_header

            # Formatear Celdas de Datos
            is_summary_sheet = (sheet_name == 'Resumen General')

            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for col_idx, cell in enumerate(row, start=1):
                    # Aplicar color de fondo normal
                    if is_summary_sheet:
                        cell.fill = summary_fill
                    else:
                        cell.fill = cell_fill

                    cell.border = border

                    # Lógica de alineación
                    header_cell = ws.cell(row=1, column=col_idx)
                    header_val = header_cell.value if header_cell else None
                    if header_val and any(keyword in str(header_val) for keyword in ['Nombre', 'Codigo', 'CodCli']):
                        cell.alignment = align_left
                    else:
                        cell.alignment = align_center
                        if header_val and any(coord in str(header_val) for coord in ['Longitud', 'Latitud']):
                            cell.number_format = '0.000000'

            # Auto-ajustar columnas
            for col_idx, column_cells in enumerate(ws.columns, 1):
                max_length = 0
                header_text = str(ws.cell(row=1, column=col_idx).value).lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰 ')
                max_length = len(header_text)
                for i, cell in enumerate(column_cells):
                    if i == 0: continue
                    if i > 50: break
                    try:
                        if cell.value is not None:
                            if isinstance(cell.value, (int, float)) and cell.number_format and cell.number_format != 'General':
                                if '0.000000' in cell.number_format: cell_text = f"{cell.value:.6f}"
                                else: cell_text = str(cell.value)
                            else: cell_text = str(cell.value)
                            lines = cell_text.split('\n')
                            max_line_length = max(len(line) for line in lines) if lines else 0
                            max_length = max(max_length, max_line_length)
                    except: pass
                adjusted_width = min(max(12, max_length + 3), 45)
                ws.column_dimensions[get_column_letter(col_idx)].width = adjusted_width

        wb.save(excel_path)
    except Exception as e:
        print(f"       ⚠️ Error al aplicar formato al Excel '{excel_path}': {e}")

print("✅ CELDA 1: Inicialización y funciones (Alfa 5.1) cargadas.")
print(f"📁 Directorio de salida: {output_dir}")
print(f"🎯 Icono MET: {'✅ Encontrado' if os.path.exists(icon_met_path) else '⚠️ No encontrado'}")

In [ ]:
# ================== 2. EJECUCIÓN DEL ANÁLISIS (Alfa 5.1 - Muestreo y Territorios) ==================

# --- Definición de Widgets ---
upload_widget = widgets.FileUpload(accept='.xlsx', multiple=False, description='Subir Excel')
met_box = widgets.VBox([widgets.Label('⏳ Esperando archivo Excel...')])
output_upload = widgets.Output()
min_clientes_cluster = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Min Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))
max_clientes_cluster = widgets.IntSlider(value=40, min=10, max=100, step=1, description='Max Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))

# --- (NUEVO) WIDGETS DE MUESTREO ---
sample_size_per_met = widgets.IntText(
    value=500, 
    description='Clientes a muestrear por MET:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='48%')
)
run_full_analysis = widgets.Checkbox(
    value=False, 
    description='✅ Analizar TODOS los clientes (Ignora el muestreo)', 
    indent=False, 
    layout=widgets.Layout(width='48%')
)
# --- FIN DE WIDGETS NUEVOS ---

param_button = widgets.Button(description='Ejecutar Análisis de Territorio y Rutas', button_style='success', disabled=False, layout=widgets.Layout(width='98%'))
output_param = widgets.Output()

# --- Registrar handler de carga ---
upload_widget.observe(lambda change: handle_upload(change, met_box, output_upload), names='value')

# --- Función Principal de Ejecución ---
def ejecutar_analisis(b):
    global execution_count, df_clientes, df_cdr, met_checkboxes, colores
    global g_map_path, g_excel_paths, g_total_clientes_procesados
    
    execution_count += 1
    
    with output_param:
        clear_output(wait=True)
        param_button.disabled = True # Bloquear botón
        
        # --- 1. Validaciones y Parámetros ---
        if df_clientes.empty or df_cdr.empty:
            print("❌ Carga un archivo Excel primero.")
            param_button.disabled = False
            return
        
        mets_seleccionados = [cb.description for cb in met_checkboxes if cb.value]
        if not mets_seleccionados or len(mets_seleccionados) < 2:
            print("❌ Selecciona al menos dos METs para optimizar territorios.")
            param_button.disabled = False
            return
        
        min_size = min_clientes_cluster.value
        max_size = max_clientes_cluster.value
        
        # --- (NUEVO) Obtener parámetros de muestreo ---
        n_muestra = sample_size_per_met.value
        analizar_todos = run_full_analysis.value
        
        if min_size >= max_size:
            print("❌ El mínimo debe ser menor que el máximo.")
            param_button.disabled = False
            return
        
        print(f"\n{'='*70}")
        print(f"🚀 INICIANDO ANÁLISIS DE TERRITORIO Y RUTAS - Ejecución #{execution_count}")
        print(f"{'='*70}")
        print(f"📊 Parámetros:")
        print(f"   • METs a optimizar: {len(mets_seleccionados)} ({', '.join(mets_seleccionados)})")
        print(f"   • Clientes por ruta: {min_size} - {max_size}")
        
        # Imprimir modo de ejecución
        if analizar_todos:
            print(f"   • MODO: ✅ COMPLETO (Se analizarán TODOS los clientes)")
        else:
            print(f"   • MODO: ⚠️ MUESTRA (Se analizarán {n_muestra} clientes por MET)")
            
        print(f"{'='*70}\n")
        
        # Limpiar resultados anteriores
        g_map_path = ""
        g_excel_paths = []
        g_total_clientes_procesados = 0
        
        # --- (NUEVO) GUARDAR RESPALDO DE CLIENTES ---
        df_clientes_original = df_clientes.copy()
        
        # --- 2. (NUEVO) FASE 0: ASIGNACIÓN DE TERRITORIOS ---
        try:
            print(f"🗺️ Fase 0: Asignando territorios (cliente al MET más cercano)...")
            
            mets_info = df_cdr[df_cdr['CodMET'].isin(mets_seleccionados)]
            if mets_info.empty:
                print("❌ No se encontraron datos para los METs seleccionados en la hoja de CDRs.")
                param_button.disabled = False
                return

            coords_clientes = df_clientes[['U_latitud', 'U_longitud']].values
            coords_mets = mets_info[['U_latitud', 'U_longitud']].values
            
            dist_matrix_territorio = distance_matrix(coords_clientes, coords_mets)
            indices_met_mas_cercano = np.argmin(dist_matrix_territorio, axis=1)
            nombres_met_mas_cercano = [mets_info['CodMET'].iloc[i] for i in indices_met_mas_cercano]
            
            df_clientes['MET_Asignado'] = nombres_met_mas_cercano
            
            print(f"✅ Territorios asignados. Conteo:")
            print(df_clientes['MET_Asignado'].value_counts())
            
            print("📊 Calculando rangos de volumen global para tamaño de marcadores...")
            volumenes_positivos = df_clientes[df_clientes['m3/día'] > 0]['m3/día']
            if not volumenes_positivos.empty:
                vol_min_global = volumenes_positivos.min()
                vol_max_global = volumenes_positivos.max()
            else:
                vol_min_global = 0
                vol_max_global = 0
            print(f"   • 📦 Rango de Volumen (m³/día): {vol_min_global:,.3f} - {vol_max_global:,.3f}")

            # --- 3. Crear Mapa Base (con zoom automático) ---
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            lats = mets_info['U_latitud']
            lons = mets_info['U_longitud']
            bounds = [[lats.min(), lons.min()], [lats.max(), lons.max()]]
            mapa_base.fit_bounds(bounds, padding=(20, 20))
            
            colores_list = colores.copy()
            random.shuffle(colores_list)
            
            export_data = []
            summary_data = []

            # --- 4. (MODIFICADO) FASE 1: CLUSTERING POR TERRITORIO ---
            print(f"\n🔬 Fase 1: Iniciando clustering por territorio...")
            
            for idx_met, met_seleccionado in enumerate(mets_seleccionados):
                print(f"\n{'─'*70}")
                print(f"🏠 Procesando Territorio: {met_seleccionado} ({idx_met + 1}/{len(mets_seleccionados)})")
                print(f"{'─'*70}")
                
                met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado]
                if met_info.empty:
                    print(f"⚠️ No se encontró información para el MET '{met_seleccionado}'. Saltando.")
                    continue
                met_row = met_info.iloc[0]
                met_lat = met_row['U_latitud']
                met_lon = met_row['U_longitud']
                
                # --- (NUEVO) LÓGICA DE MUESTREO POR TERRITORIO ---
                df_territorio_completo = df_clientes[df_clientes['MET_Asignado'] == met_seleccionado].copy()
                
                if df_territorio_completo.empty:
                    print(f"⚠️ No hay clientes asignados al territorio de '{met_seleccionado}'. Saltando.")
                    continue
                
                df_clientes_met = pd.DataFrame() # DataFrame vacío para el análisis
                if analizar_todos:
                    df_clientes_met = df_territorio_completo
                    print(f"✅ Analizando territorio COMPLETO: {len(df_clientes_met)} clientes")
                else:
                    if len(df_territorio_completo) > n_muestra:
                        print(f"⚠️ MODO MUESTRA: Analizando {n_muestra} de {len(df_territorio_completo)} clientes (aleatorio)")
                        df_clientes_met = df_territorio_completo.sample(n=n_muestra).copy()
                    else:
                        df_clientes_met = df_territorio_completo
                        print(f"✅ Analizando territorio COMPLETO (es menor que la muestra): {len(df_clientes_met)} clientes")
                # --- FIN LÓGICA DE MUESTREO ---

                if df_clientes_met.empty:
                    print(f"⚠️ No hay clientes para procesar en este MET. Saltando.")
                    continue
                
                # --- El resto del script continúa igual, usando df_clientes_met ---
                coords = df_clientes_met[['U_longitud', 'U_latitud']].values
                k_inicial = max(1, len(df_clientes_met) // max_size)
                
                print(f"🔬 Iniciando clustering (k_inicial = {k_inicial})...")
                initial_seeds = find_initial_seeds(coords, k_inicial)
                
                if len(initial_seeds) == 0:
                    print(f"❌ No se pudieron generar semillas para el MET '{met_seleccionado}'.")
                    continue
                
                df_clientes_met, cluster_counts = grow_clusters_from_seeds(df_clientes_met, initial_seeds, max_size, min_size)
                df_clientes_met, cluster_counts = adjust_small_clusters(df_clientes_met, cluster_counts, min_size, max_size)
                
                cluster_polygons = calculate_cluster_polygons(df_clientes_met, method='voronoi')
                
                final_counts = df_clientes_met['Ruta_nueva'][df_clientes_met['Ruta_nueva'] != -1].value_counts()
                rutas_validas = final_counts.index.tolist() # Tomar todas, el ajuste ya limpió
                
                print(f"✅ Rutas generadas: {len(rutas_validas)}")
                
                df_clientes_met_validos = df_clientes_met[df_clientes_met['Ruta_nueva'].isin(rutas_validas)].copy()
                
                if df_clientes_met_validos.empty:
                    print(f"⚠️ No hay rutas válidas para el MET '{met_seleccionado}'.")
                    continue
                
                met_name_clean = str(met_seleccionado).strip()
                display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"
                
                fg_met_icon = folium.FeatureGroup(name=f"{display_name} - 🏠 MET", show=True)
                
                clientes_met_total = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] >= 0]
                volumen_total_met = clientes_met_total['m3/día'].sum()
                total_clientes_met = len(clientes_met_total)
                
                popup_html_met = f'''
                <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                    <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                        <div style="font-size: 22px; font-weight: bold;">🏠 {met_name_clean}</div>
                    </div>
                    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                        <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                            <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                            <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                        </div>
                        <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                            <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                            <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                        </div>
                    </div>
                    <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                        <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                        <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_validas)}</div>
                    </div>
                </div>
                '''

                if os.path.exists(icon_met_path):
                    icon_met_obj = CustomIcon(icon_met_path, icon_size=(40, 40), icon_anchor=(20, 40), popup_anchor=(0, -40))
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=icon_met_obj, tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)
                else:
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=folium.Icon(color='red', icon='home', prefix='glyphicon'),
                        tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)

                create_map_elements(
                    mapa_base,
                    df_clientes_met_validos,
                    cluster_polygons,
                    colores_list,
                    vol_min_global,
                    vol_max_global,
                    met_seleccionado
                )
                
                fg_met_icon.add_to(mapa_base)
                
                g_total_clientes_procesados += len(df_clientes_met_validos)
                
                for ruta_id in rutas_validas:
                    ruta_clientes = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] == ruta_id]
                    n_clientes_ruta = len(ruta_clientes)
                    volumen_total = ruta_clientes['m3/día'].sum()
                    
                    summary_data.append({
                        'MET': met_seleccionado,
                        'Ruta_nueva': ruta_id,
                        'Num_Clientes': n_clientes_ruta,
                        'Volumen_Total_m3': round(volumen_total, 2),
                        'Rentable (>= 1 m³)': 'SÍ' if volumen_total >= 1.0 else 'NO',
                        'Centroide_Lat': ruta_clientes['U_latitud'].mean(),
                        'Centroide_Lon': ruta_clientes['U_longitud'].mean()
                    })
                    
                    for _, cliente in ruta_clientes.iterrows():
                        export_data.append({
                            'MET': met_seleccionado,
                            'Ruta_nueva': ruta_id,
                            'Codigo': cliente['CodCli'],
                            'Nombre': cliente.get('Nombre', 'N/A'),
                            'Ruta_Original': cliente.get('Ruta', 'N/A'),
                            'm3_dia': cliente['m3/día'],
                            'Latitud': cliente['U_latitud'],
                            'Longitud': cliente['U_longitud']
                        })
                
                print(f"✅ Territorio '{met_seleccionado}' completado: {len(rutas_validas)} rutas, {len(df_clientes_met_validos)} clientes")
            
            # --- 5. Guardar Mapa y Excel ---
            folium.LayerControl(collapsed=False).add_to(mapa_base)
            
            estilos_css = r'''
            <style>
            #panel-resumen {
                position: fixed; top: 80px; left: 10px; background: white; padding: 12px 16px;
                border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.2); z-index: 1000;
                max-width: 280px; font-family: Arial, sans-serif;
            }
            #panel-resumen h3 {
                margin: 0 0 10px 0; font-size: 16px; color: #333;
                border-bottom: 2px solid #007bff; padding-bottom: 6px;
            }
            #panel-resumen-content { font-size: 13px; line-height: 1.6; color: #555; }
            .resumen-item { display: flex; justify-content: space-between; padding: 4px 0; border-bottom: 1px solid #eee; }
            .resumen-item:last-child {
                border-bottom: none; font-weight: bold; margin-top: 6px;
                padding-top: 8px; border-top: 2px solid #007bff;
            }
            .resumen-label { color: #666; }
            .resumen-value { color: #007bff; font-weight: 600; }
            .leaflet-control-layers-list {
                max-height: 400px; overflow-y: auto; overflow-x: hidden;
                width: 100%; min-width: 200px;
            }
            .layer-control-buttons {
                padding: 8px; border-bottom: 1px solid #ddd; background: #f8f8f8;
                display: flex; gap: 5px;
            }
            .layer-btn {
                padding: 6px 10px; font-size: 11px; border: 1px solid #ccc; background: white;
                cursor: pointer; border-radius: 4px; flex: 1; text-align: center;
                font-weight: 600; transition: all 0.2s;
            }
            .layer-btn:hover { background: #e6e6e6; transform: translateY(-1px); }
            .layer-btn.select-all { background: #4CAF50; color: white; border-color: #45a049; }
            .layer-btn.select-all:hover { background: #45a049; }
            .layer-btn.deselect-all { background: #f44336; color: white; border-color: #da190b; }
            .layer-btn.deselect-all:hover { background: #da1a0b; }
            .met-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
                display: flex; gap: 4px; flex-wrap: wrap;
            }
            .met-btn {
                padding: 5px 10px; font-size: 10px; border: 2px solid #2196F3; background: white;
                color: #1976D2; cursor: pointer; border-radius: 6px; flex: 1; text-align: center;
                min-width: 60px; font-weight: 600; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .met-btn:hover { background: #2196F3; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .ruta-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%);
                display: grid; grid-template-columns: repeat(auto-fit, minmax(45px, 1fr));
                gap: 4px; max-width: 100%;
            }
            .ruta-btn {
                padding: 5px 8px; font-size: 10px; border: 2px solid #9C27B0; background: white;
                color: #7B1FA2; cursor: pointer; border-radius: 6px; text-align: center;
                min-width: 45px; font-weight: 700; white-space: nowrap; overflow: hidden;
                text-overflow: ellipsis; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .ruta-btn:hover { background: #9C27B0; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .search-ruta-container {
                display: flex; gap: 5px; padding: 8px; background: #f0f8ff;
                border-bottom: 1px solid #b3d9ff; margin-bottom: 8px; align-items: center;
            }
            .search-ruta-input {
                flex: 1; padding: 6px 10px; border: 1px solid #ced4da; border-radius: 4px;
                font-size: 12px; outline: none;
            }
            .search-ruta-input:focus { border-color: #0066cc; box-shadow: 0 0 0 2px rgba(0, 102, 204, 0.1); }
            .search-ruta-btn {
                padding: 6px 15px; border: 1px solid #0066cc; border-radius: 4px; background: #0066cc;
                color: white; cursor: pointer; font-size: 12px; font-weight: 500;
                transition: all 0.2s; white-space: nowrap;
            }
            .search-ruta-btn:hover { background: #0052a3; border-color: #0052a3; }
            .search-ruta-clear {
                padding: 6px 12px; border: 1px solid #dc3545; border-radius: 4px; background: white;
                color: #dc3545; cursor: pointer; font-size: 12px; font-weight: 500; transition: all 0.2s;
            }
            .search-ruta-clear:hover { background: #dc3545; color: white; }
            .leaflet-control-layers-overlays { display: none; }
            </style>
            '''
            mapa_base.get_root().html.add_child(folium.Element(estilos_css))
            
            paneles_js = r'''
            <script>
            function crearPanelResumen() {
                const panel = document.createElement('div');
                panel.id = 'panel-resumen';
                panel.innerHTML = `<h3>📊 Resumen de Rutas</h3><div id="panel-resumen-content"></div>`;
                document.body.appendChild(panel);
                return panel;
            }
            function inicializarPaneles(mapInstance) {
                console.log('=== Inicializando paneles personalizados ===');
                if (document.getElementById('panel-resumen')) {
                    console.log('ℹ️ Paneles ya inicializados');
                    return true;
                }
                const panelResumen = crearPanelResumen();
                const map = mapInstance;
                if (!map) { console.error('❌ Instancia del mapa (mapInstance) es nula.'); return false; }
                let totalRutas = 0;
                const metRutas = {};
                map.eachLayer(function(layer) {
                    if (layer._url || !layer.options || !layer.options.pane) return;
                    const layerName = layer.options.attribution || '';
                    const rutaMatch = layerName.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+Ruta\s+(\d+)$/); // Modificado para ignorar advertencia
                    if (rutaMatch) {
                        const metNombre = rutaMatch[1]; // Captura el nombre limpio
                        totalRutas++;
                        if (!metRutas[metNombre]) metRutas[metNombre] = 0;
                        metRutas[metNombre]++;
                    }
                });
                const resumenHTML = [];
                Object.keys(metRutas).sort().forEach(function(metNombre) {
                    resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">${metNombre}:</span><span class="resumen-value">${metRutas[metNombre]} rutas</span></div>`);
                });
                resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">Total:</span><span class="resumen-value">${totalRutas} rutas</span></div>`);
                document.getElementById('panel-resumen-content').innerHTML = resumenHTML.join('');
                console.log('✅ Paneles personalizados inicializados');
                return true;
            }
            let panelPollCount = 0;
            function tryInitPanels() {
                const mapElement = document.querySelector('.leaflet-container');
                let mapInstance = null;
                if (mapElement && mapElement._leaflet_map) { mapInstance = mapElement._leaflet_map; }
                if (typeof L !== 'undefined' && L.map && mapInstance) {
                    console.log('✅ (Poll) Leaflet (L) y MapInstance (_leaflet_map) listos. Iniciando paneles...');
                    inicializarPaneles(mapInstance);
                } else if (panelPollCount < 40) {
                    panelPollCount++;
                    console.log('Poll #' + panelPollCount + ': Esperando a L y _leaflet_map...');
                    setTimeout(tryInitPanels, 500);
                } else {
                    console.error('❌ No se pudo cargar la instancia del mapa (_leaflet_map) después de 20 segundos.');
                }
            }
            tryInitPanels();
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(paneles_js))
            
            filtros_js = r'''
            <script>
            function inicializarFiltros() {
                console.log('=== Inicializando filtros personalizados (Estilo Celaya) ===');
                const layerControl = document.querySelector('.leaflet-control-layers');
                if (!layerControl) { console.error('❌ No se encontró .leaflet-control-layers'); return false; }
                const layersList = layerControl.querySelector('.leaflet-control-layers-list');
                if (!layersList) { console.error('❌ No se encontró .leaflet-control-layers-list'); return false; }
                const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) { console.error('❌ No se encontró .leaflet-control-layers-overlays'); return false; }
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                console.log('✅ Total de checkboxes encontrados:', checkboxes.length);
                if (layersList.querySelector('.layer-control-buttons')) { console.log('ℹ️ Filtros ya inicializados'); return true; }
                
                console.log('📋 Listando todas las capas:');
                checkboxes.forEach(function(checkbox, index) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    console.log('   ' + (index + 1) + '. "' + label + '"');
                });
                
                const buttonsDiv = document.createElement('div');
                buttonsDiv.className = 'layer-control-buttons';
                const selectAllBtn = document.createElement('button');
                selectAllBtn.textContent = '✓ Todo';
                selectAllBtn.className = 'layer-btn select-all';
                selectAllBtn.title = 'Seleccionar todas las capas';
                const deselectAllBtn = document.createElement('button');
                deselectAllBtn.textContent = '✗ Nada';
                deselectAllBtn.className = 'layer-btn deselect-all';
                deselectAllBtn.title = 'Deseleccionar todas las capas';
                selectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (!cb.checked) cb.click(); }); };
                deselectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); };
                buttonsDiv.appendChild(selectAllBtn);
                buttonsDiv.appendChild(deselectAllBtn);
                layersList.insertBefore(buttonsDiv, overlaysDiv);
                
                const searchContainer = document.createElement('div');
                searchContainer.className = 'search-ruta-container';
                const searchLabel = document.createElement('span');
                searchLabel.textContent = '🔍';
                searchLabel.style.fontSize = '14px';
                const searchInput = document.createElement('input');
                searchInput.type = 'text';
                searchInput.className = 'search-ruta-input';
                searchInput.placeholder = 'Buscar por Ruta ID (ej: 0, 1, 2...)';
                searchInput.title = 'Ingresa el número de ruta para filtrar';
                const searchBtn = document.createElement('button');
                searchBtn.textContent = 'Buscar';
                searchBtn.className = 'search-ruta-btn';
                searchBtn.title = 'Filtrar por Ruta ID';
                const clearBtn = document.createElement('button');
                clearBtn.textContent = 'Limpiar';
                clearBtn.className = 'search-ruta-clear';
                clearBtn.title = 'Limpiar búsqueda';
                const filtrarPorRutaId = function() {
                    const rutaId = searchInput.value.trim();
                    if (rutaId === '') { alert('Por favor ingresa un número de ruta'); return; }
                    console.log('🔍 Filtrando capas con Ruta ID:', rutaId);
                    checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                    setTimeout(function() {
                        let encontradas = 0;
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$'); // Busca el final
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                console.log('  ✓ Seleccionando:', label);
                                if (!checkbox.checked) { checkbox.click(); encontradas++; }
                            }
                        });
                        if (encontradas === 0) { console.warn('⚠️ No se encontraron capas con Ruta ID ' + rutaId); alert('No se encontraron capas con Ruta ID: ' + rutaId); }
                    }, 150);
                };
                searchBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); filtrarPorRutaId(); };
                searchInput.addEventListener('keypress', function(e) { if (e.key === 'Enter') { e.preventDefault(); filtrarPorRutaId(); } });
                clearBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); searchInput.value = ''; checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); console.log('🧹 Búsqueda limpiada'); };
                searchContainer.appendChild(searchLabel);
                searchContainer.appendChild(searchInput);
                searchContainer.appendChild(searchBtn);
                searchContainer.appendChild(clearBtn);
                layersList.insertBefore(searchContainer, overlaysDiv);
                
                const metButtonsDiv = document.createElement('div');
                metButtonsDiv.className = 'met-buttons-row';
                const mets = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const metMatch = label.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/); // Modificado para incluir advertencia
                    if (metMatch) { mets.add(metMatch[1].trim()); } // Limpiar advertencia
                });
                console.log('🏢 METs disponibles:', Array.from(mets));
                Array.from(mets).sort().forEach(function(metNombre) {
                    const metBtn = document.createElement('button');
                    const nombreCorto = metNombre.replace('MET ', '').trim();
                    metBtn.textContent = nombreCorto;
                    metBtn.className = 'met-btn';
                    metBtn.title = 'Seleccionar solo capas de ' + metNombre;
                    metBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        console.log('🔍 Activando filtro para MET:', metNombre);
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                // Modificado para que siempre funcione
                                if (label.includes(metNombre + ' -')) {
                                    console.log('  ✓ Seleccionando:', label);
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas para ' + metNombre);
                        }, 150);
                    };
                    metButtonsDiv.appendChild(metBtn);
                });
                if (metButtonsDiv.children.length > 0) { layersList.insertBefore(metButtonsDiv, overlaysDiv); }
                
                const rutaButtonsDiv = document.createElement('div');
                rutaButtonsDiv.className = 'ruta-buttons-row';
                const rutas = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const rutaMatch = label.match(/- Ruta\s+(\d+)/);
                    if (rutaMatch) { rutas.add(parseInt(rutaMatch[1])); }
                });
                console.log('🚚 Rutas encontradas:', Array.from(rutas).sort(function(a, b) { return a - b; }));
                const rutasArray = Array.from(rutas).sort(function(a, b) { return a - b; });
                rutasArray.forEach(function(ruta) {
                    const rutaBtn = document.createElement('button');
                    rutaBtn.textContent = 'R' + ruta;
                    rutaBtn.className = 'ruta-btn';
                    rutaBtn.title = 'Seleccionar Ruta ' + ruta + ' de todos los METs';
                    rutaBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        const rutaId = ruta;
                        console.log('🔍 Activando filtro para Ruta', rutaId);
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$');
                        const metIconPattern = /- 🏠 MET$/;
                        const metNamePattern = /^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/;
                        const metsToShow = new Set();
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                const metMatch = label.match(metNamePattern);
                                if (metMatch) { metsToShow.add(metMatch[1].trim()); }
                            }
                        });
                        console.log('   ...Mostrando METs que tienen esta ruta:', Array.from(metsToShow));
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                if (rutaPattern.test(label)) {
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                } else if (metIconPattern.test(label)) {
                                    const metMatch = label.match(metNamePattern);
                                    if (metMatch && metsToShow.has(metMatch[1].trim())) {
                                        if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                    }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas (Rutas + METs) para Ruta ' + rutaId);
                        }, 150);
                    };
                    rutaButtonsDiv.appendChild(rutaBtn);
                });
                if (rutaButtonsDiv.children.length > 0) { layersList.insertBefore(rutaButtonsDiv, overlaysDiv); }
                
                console.log('=== Filtros inicializados correctamente ===');
                return true;
            }
            let filterPollCount = 0;
            function tryInitFilters() {
                const layerControlList = document.querySelector('.leaflet-control-layers-list');
                if (typeof L !== 'undefined' && layerControlList) {
                    console.log('✅ (Poll) Leaflet (L) y .leaflet-control-layers-list listos. Iniciando filtros...');
                    try { inicializarFiltros(); } catch (e) { console.error('❌ Error al ejecutar inicializarFiltros:', e); }
                } else if (filterPollCount < 40) {
                    filterPollCount++;
                    console.log('Poll Filtros #' + filterPollCount + ': Esperando a L y .leaflet-control-layers-list...');
                    setTimeout(tryInitFilters, 500);
                } else {
                    console.error('❌ No se pudo encontrar .leaflet-control-layers-list después de 20 segundos.');
                }
            }
            setTimeout(tryInitFilters, 500); 
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(filtros_js))
            
            titulo_html = f'''
            <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
                <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                    🗺️ ANÁLISIS DE CLUSTERS POR VOLUMEN
                </h2>
                <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                    📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas Generadas: <b>{len(summary_data)}</b> | 👥 Clientes Procesados: <b>{g_total_clientes_procesados}</b><br>
                    📦 Tamaño de Círculo = Volumen (m³/día) | 🎨 Color de Círculo = Ruta Asignada
                </p>
            </div>
            '''
            mapa_base.get_root().html.add_child(folium.Element(titulo_html))
            
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            map_filename = f'mapa_rutas_clusters_{timestamp}.html'
            g_map_path = os.path.join(output_dir, map_filename)
            mapa_base.save(g_map_path)
            
            print(f"\n{'='*70}")
            print(f"🗺️ Mapa guardado: {g_map_path}")
            
            if export_data and summary_data:
                excel_path = generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir)
                if excel_path:
                    g_excel_paths = [excel_path]
                    print(f"📊 Excel guardado: {excel_path}")
            
            print(f"{'='*70}")
            print(f"✅ ANÁLISIS COMPLETADO")
            print(f"   • Total clientes procesados: {g_total_clientes_procesados}")
            print(f"   • Total rutas generadas: {len(summary_data)}")
            print(f"   • Archivos generados: {len(g_excel_paths) + 1}")
            print(f"{'='*70}\n")

        except Exception as e:
            import traceback
            print(f"❌ Error Inesperado en ejecución #{execution_count}:")
            traceback.print_exc()
        finally:
            # --- (NUEVO) RESTAURAR DF_CLIENTES ORIGINAL ---
            if 'df_clientes_original' in locals():
                df_clientes = df_clientes_original
                print("ℹ️ DataFrame de clientes restaurado a su estado original.")
            
            param_button.disabled = False # Desbloquear botón

# --- Registrar el handler ---
try:
    param_button.on_click(ejecutar_analisis, remove=True)
except:
    pass
param_button.on_click(ejecutar_analisis)

# === MOSTRAR UI (Modificada) ===
print("================== MÓDULO 2: Análisis de Rutas (Alfa 5.1) ==================")
display(widgets.VBox([
    widgets.Label("1. Cargar archivo Excel:", style={'font_weight': 'bold'}),
    upload_widget,
    output_upload,
    widgets.HTML("<hr>"),
    met_box,
    widgets.HTML("<hr>"),
    widgets.Label("2. Configurar parámetros de Ruta:", style={'font_weight': 'bold'}),
    widgets.HBox([min_clientes_cluster, max_clientes_cluster]),
    
    # --- (NUEVO) WIDGETS DE MUESTREO ---
    widgets.HBox([sample_size_per_met, run_full_analysis]),
    # --- FIN DE WIDGETS NUEVOS ---
    
    param_button,
    output_param
]))

Versión ajustada a frecuencia, con filtros, manejo e iconos especificos para comodines

In [ ]:
# ================== 2. EJECUCIÓN DEL ANÁLISIS (Alfa 5.1 - Muestreo y Territorios) ==================

# --- Definición de Widgets ---
upload_widget = widgets.FileUpload(accept='.xlsx', multiple=False, description='Subir Excel')
met_box = widgets.VBox([widgets.Label('⏳ Esperando archivo Excel...')])
output_upload = widgets.Output()
min_clientes_cluster = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Min Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))
max_clientes_cluster = widgets.IntSlider(value=40, min=10, max=100, step=1, description='Max Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))

# --- (NUEVO) WIDGETS DE MUESTREO ---
sample_size_per_met = widgets.IntText(
    value=500, 
    description='Clientes a muestrear por MET:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='48%')
)
run_full_analysis = widgets.Checkbox(
    value=False, 
    description='✅ Analizar TODOS los clientes (Ignora el muestreo)', 
    indent=False, 
    layout=widgets.Layout(width='48%')
)
# --- FIN DE WIDGETS NUEVOS ---

param_button = widgets.Button(description='Ejecutar Análisis de Territorio y Rutas', button_style='success', disabled=False, layout=widgets.Layout(width='98%'))
output_param = widgets.Output()

# --- Registrar handler de carga ---
upload_widget.observe(lambda change: handle_upload(change, met_box, output_upload), names='value')

# --- Función Principal de Ejecución ---
def ejecutar_analisis(b):
    global execution_count, df_clientes, df_cdr, met_checkboxes, colores
    global g_map_path, g_excel_paths, g_total_clientes_procesados
    
    execution_count += 1
    
    with output_param:
        clear_output(wait=True)
        param_button.disabled = True # Bloquear botón
        
        # --- 1. Validaciones y Parámetros ---
        if df_clientes.empty or df_cdr.empty:
            print("❌ Carga un archivo Excel primero.")
            param_button.disabled = False
            return
        
        mets_seleccionados = [cb.description for cb in met_checkboxes if cb.value]
        if not mets_seleccionados or len(mets_seleccionados) < 2:
            print("❌ Selecciona al menos dos METs para optimizar territorios.")
            param_button.disabled = False
            return
        
        min_size = min_clientes_cluster.value
        max_size = max_clientes_cluster.value
        
        # --- (NUEVO) Obtener parámetros de muestreo ---
        n_muestra = sample_size_per_met.value
        analizar_todos = run_full_analysis.value
        
        if min_size >= max_size:
            print("❌ El mínimo debe ser menor que el máximo.")
            param_button.disabled = False
            return
        
        print(f"\n{'='*70}")
        print(f"🚀 INICIANDO ANÁLISIS DE TERRITORIO Y RUTAS - Ejecución #{execution_count}")
        print(f"{'='*70}")
        print(f"📊 Parámetros:")
        print(f"   • METs a optimizar: {len(mets_seleccionados)} ({', '.join(mets_seleccionados)})")
        print(f"   • Clientes por ruta: {min_size} - {max_size}")
        
        # Imprimir modo de ejecución
        if analizar_todos:
            print(f"   • MODO: ✅ COMPLETO (Se analizarán TODOS los clientes)")
        else:
            print(f"   • MODO: ⚠️ MUESTRA (Se analizarán {n_muestra} clientes por MET)")
            
        print(f"{'='*70}\n")
        
        # Limpiar resultados anteriores
        g_map_path = ""
        g_excel_paths = []
        g_total_clientes_procesados = 0
        
        # --- (NUEVO) GUARDAR RESPALDO DE CLIENTES ---
        df_clientes_original = df_clientes.copy()
        
        # --- 2. (NUEVO) FASE 0: ASIGNACIÓN DE TERRITORIOS ---
        try:
            print(f"🗺️ Fase 0: Asignando territorios (cliente al MET más cercano)...")
            
            mets_info = df_cdr[df_cdr['CodMET'].isin(mets_seleccionados)]
            if mets_info.empty:
                print("❌ No se encontraron datos para los METs seleccionados en la hoja de CDRs.")
                param_button.disabled = False
                return

            coords_clientes = df_clientes[['U_latitud', 'U_longitud']].values
            coords_mets = mets_info[['U_latitud', 'U_longitud']].values
            
            dist_matrix_territorio = distance_matrix(coords_clientes, coords_mets)
            indices_met_mas_cercano = np.argmin(dist_matrix_territorio, axis=1)
            nombres_met_mas_cercano = [mets_info['CodMET'].iloc[i] for i in indices_met_mas_cercano]
            
            df_clientes['MET_Asignado'] = nombres_met_mas_cercano
            
            print(f"✅ Territorios asignados. Conteo:")
            print(df_clientes['MET_Asignado'].value_counts())
            
            print("📊 Calculando rangos de volumen global para tamaño de marcadores...")
            volumenes_positivos = df_clientes[df_clientes['m3/día'] > 0]['m3/día']
            if not volumenes_positivos.empty:
                vol_min_global = volumenes_positivos.min()
                vol_max_global = volumenes_positivos.max()
            else:
                vol_min_global = 0
                vol_max_global = 0
            print(f"   • 📦 Rango de Volumen (m³/día): {vol_min_global:,.3f} - {vol_max_global:,.3f}")

            # --- 3. Crear Mapa Base (con zoom automático) ---
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            lats = mets_info['U_latitud']
            lons = mets_info['U_longitud']
            bounds = [[lats.min(), lons.min()], [lats.max(), lons.max()]]
            mapa_base.fit_bounds(bounds, padding=(20, 20))
            
            colores_list = colores.copy()
            random.shuffle(colores_list)
            
            export_data = []
            summary_data = []

            # --- 4. (MODIFICADO) FASE 1: CLUSTERING POR TERRITORIO ---
            print(f"\n🔬 Fase 1: Iniciando clustering por territorio...")
            
            for idx_met, met_seleccionado in enumerate(mets_seleccionados):
                print(f"\n{'─'*70}")
                print(f"🏠 Procesando Territorio: {met_seleccionado} ({idx_met + 1}/{len(mets_seleccionados)})")
                print(f"{'─'*70}")
                
                met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado]
                if met_info.empty:
                    print(f"⚠️ No se encontró información para el MET '{met_seleccionado}'. Saltando.")
                    continue
                met_row = met_info.iloc[0]
                met_lat = met_row['U_latitud']
                met_lon = met_row['U_longitud']
                
                # --- (NUEVO) LÓGICA DE MUESTREO POR TERRITORIO ---
                df_territorio_completo = df_clientes[df_clientes['MET_Asignado'] == met_seleccionado].copy()
                
                if df_territorio_completo.empty:
                    print(f"⚠️ No hay clientes asignados al territorio de '{met_seleccionado}'. Saltando.")
                    continue
                
                df_clientes_met = pd.DataFrame() # DataFrame vacío para el análisis
                if analizar_todos:
                    df_clientes_met = df_territorio_completo
                    print(f"✅ Analizando territorio COMPLETO: {len(df_clientes_met)} clientes")
                else:
                    if len(df_territorio_completo) > n_muestra:
                        print(f"⚠️ MODO MUESTRA: Analizando {n_muestra} de {len(df_territorio_completo)} clientes (aleatorio)")
                        df_clientes_met = df_territorio_completo.sample(n=n_muestra).copy()
                    else:
                        df_clientes_met = df_territorio_completo
                        print(f"✅ Analizando territorio COMPLETO (es menor que la muestra): {len(df_clientes_met)} clientes")
                # --- FIN LÓGICA DE MUESTREO ---

                if df_clientes_met.empty:
                    print(f"⚠️ No hay clientes para procesar en este MET. Saltando.")
                    continue
                
                # --- El resto del script continúa igual, usando df_clientes_met ---
                coords = df_clientes_met[['U_longitud', 'U_latitud']].values
                k_inicial = max(1, len(df_clientes_met) // max_size)
                
                print(f"🔬 Iniciando clustering (k_inicial = {k_inicial})...")
                initial_seeds = find_initial_seeds(coords, k_inicial)
                
                if len(initial_seeds) == 0:
                    print(f"❌ No se pudieron generar semillas para el MET '{met_seleccionado}'.")
                    continue
                
                # --- SEGREGAR POR FRECUENCIA (NUEVO) ---
                df_elegibles, df_comodines = segregar_por_frecuencia(df_clientes_met, threshold=0.01)
                
                # --- CLUSTERING SOLO CON ELEGIBLES (MODIFICADO) ---
                df_elegibles, cluster_counts = grow_clusters_from_seeds(df_elegibles, initial_seeds, max_size, min_size)
                df_elegibles, cluster_counts = adjust_small_clusters(df_elegibles, cluster_counts, min_size, max_size)
                
                # --- ASIGNAR COMODINES AL VECINO MÁS CERCANO (NUEVO) ---
                df_comodines = asignar_comodines_post_clustering(df_comodines, df_elegibles)
                
                # --- CONSOLIDAR RESULTADOS ---
                df_clientes_met = pd.concat([df_elegibles, df_comodines], ignore_index=True)
                
                cluster_polygons = calculate_cluster_polygons(df_clientes_met, method='voronoi')
                
                final_counts = df_clientes_met['Ruta_nueva'][df_clientes_met['Ruta_nueva'] != -1].value_counts()
                rutas_validas = final_counts.index.tolist() # Tomar todas, el ajuste ya limpió
                
                print(f"✅ Rutas generadas: {len(rutas_validas)}")
                
                df_clientes_met_validos = df_clientes_met[df_clientes_met['Ruta_nueva'].isin(rutas_validas)].copy()
                
                if df_clientes_met_validos.empty:
                    print(f"⚠️ No hay rutas válidas para el MET '{met_seleccionado}'.")
                    continue
                
                met_name_clean = str(met_seleccionado).strip()
                display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"
                
                fg_met_icon = folium.FeatureGroup(name=f"{display_name} - 🏠 MET", show=True)
                
                clientes_met_total = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] >= 0]
                volumen_total_met = clientes_met_total['m3/día'].sum()
                total_clientes_met = len(clientes_met_total)
                
                popup_html_met = f'''
                <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                    <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                        <div style="font-size: 22px; font-weight: bold;">🏠 {met_name_clean}</div>
                    </div>
                    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                        <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                            <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                            <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                        </div>
                        <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                            <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                            <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                        </div>
                    </div>
                    <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                        <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                        <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_validas)}</div>
                    </div>
                </div>
                '''

                if os.path.exists(icon_met_path):
                    icon_met_obj = CustomIcon(icon_met_path, icon_size=(40, 40), icon_anchor=(20, 40), popup_anchor=(0, -40))
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=icon_met_obj, tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)
                else:
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=folium.Icon(color='red', icon='home', prefix='glyphicon'),
                        tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)

                create_map_elements(
                    mapa_base,
                    df_clientes_met_validos,
                    cluster_polygons,
                    colores_list,
                    vol_min_global,
                    vol_max_global,
                    met_seleccionado
                )
                
                fg_met_icon.add_to(mapa_base)
                
                g_total_clientes_procesados += len(df_clientes_met_validos)
                
                for ruta_id in rutas_validas:
                    ruta_clientes = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] == ruta_id]
                    n_clientes_ruta = len(ruta_clientes)
                    volumen_total = ruta_clientes['m3/día'].sum()
                    
                    # Contar clientes normales vs comodines
                    n_normales = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == False])
                    n_comodines = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == True])
                    
                    summary_data.append({
                        'MET': met_seleccionado,
                        'Ruta_nueva': ruta_id,
                        'Num_Clientes': n_clientes_ruta,
                        'Clientes_Normales': n_normales,
                        'Clientes_Comodines': n_comodines,
                        'Volumen_Total_m3': round(volumen_total, 2),
                        'Rentable (>= 1 m³)': 'SÍ' if volumen_total >= 1.0 else 'NO',
                        'Centroide_Lat': ruta_clientes['U_latitud'].mean(),
                        'Centroide_Lon': ruta_clientes['U_longitud'].mean()
                    })
                    
                    for _, cliente in ruta_clientes.iterrows():
                        es_comodin = cliente.get('es_comodin', False)
                        export_data.append({
                            'MET': met_seleccionado,
                            'Ruta_nueva': ruta_id,
                            'Tipo_Cliente': 'Comodín' if es_comodin else 'Normal',
                            'Codigo': cliente['CodCli'],
                            'Nombre': cliente.get('Nombre', 'N/A'),
                            'Ruta_Original': cliente.get('Ruta', 'N/A'),
                            'm3_dia': cliente['m3/día'],
                            'Frecuencia_%': round(cliente['entregas/día'] * 100, 2),
                            'Latitud': cliente['U_latitud'],
                            'Longitud': cliente['U_longitud']
                        })
                
                print(f"✅ Territorio '{met_seleccionado}' completado: {len(rutas_validas)} rutas, {len(df_clientes_met_validos)} clientes")
            
            # --- 5. Guardar Mapa y Excel ---
            folium.LayerControl(collapsed=False).add_to(mapa_base)
            
            estilos_css = r'''
            <style>
            #panel-resumen {
                position: fixed; top: 80px; left: 10px; background: white; padding: 12px 16px;
                border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.2); z-index: 1000;
                max-width: 280px; font-family: Arial, sans-serif;
            }
            #panel-resumen h3 {
                margin: 0 0 10px 0; font-size: 16px; color: #333;
                border-bottom: 2px solid #007bff; padding-bottom: 6px;
            }
            #panel-resumen-content { font-size: 13px; line-height: 1.6; color: #555; }
            .resumen-item { display: flex; justify-content: space-between; padding: 4px 0; border-bottom: 1px solid #eee; }
            .resumen-item:last-child {
                border-bottom: none; font-weight: bold; margin-top: 6px;
                padding-top: 8px; border-top: 2px solid #007bff;
            }
            .resumen-label { color: #666; }
            .resumen-value { color: #007bff; font-weight: 600; }
            .leaflet-control-layers-list {
                max-height: 400px; overflow-y: auto; overflow-x: hidden;
                width: 100%; min-width: 200px;
            }
            .layer-control-buttons {
                padding: 8px; border-bottom: 1px solid #ddd; background: #f8f8f8;
                display: flex; gap: 5px;
            }
            .layer-btn {
                padding: 6px 10px; font-size: 11px; border: 1px solid #ccc; background: white;
                cursor: pointer; border-radius: 4px; flex: 1; text-align: center;
                font-weight: 600; transition: all 0.2s;
            }
            .layer-btn:hover { background: #e6e6e6; transform: translateY(-1px); }
            .layer-btn.select-all { background: #4CAF50; color: white; border-color: #45a049; }
            .layer-btn.select-all:hover { background: #45a049; }
            .layer-btn.deselect-all { background: #f44336; color: white; border-color: #da190b; }
            .layer-btn.deselect-all:hover { background: #da1a0b; }
            .met-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
                display: flex; gap: 4px; flex-wrap: wrap;
            }
            .met-btn {
                padding: 5px 10px; font-size: 10px; border: 2px solid #2196F3; background: white;
                color: #1976D2; cursor: pointer; border-radius: 6px; flex: 1; text-align: center;
                min-width: 60px; font-weight: 600; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .met-btn:hover { background: #2196F3; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .ruta-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%);
                display: grid; grid-template-columns: repeat(auto-fit, minmax(45px, 1fr));
                gap: 4px; max-width: 100%;
            }
            .ruta-btn {
                padding: 5px 8px; font-size: 10px; border: 2px solid #9C27B0; background: white;
                color: #7B1FA2; cursor: pointer; border-radius: 6px; text-align: center;
                min-width: 45px; font-weight: 700; white-space: nowrap; overflow: hidden;
                text-overflow: ellipsis; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .ruta-btn:hover { background: #9C27B0; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .search-ruta-container {
                display: flex; gap: 5px; padding: 8px; background: #f0f8ff;
                border-bottom: 1px solid #b3d9ff; margin-bottom: 8px; align-items: center;
            }
            .search-ruta-input {
                flex: 1; padding: 6px 10px; border: 1px solid #ced4da; border-radius: 4px;
                font-size: 12px; outline: none;
            }
            .search-ruta-input:focus { border-color: #0066cc; box-shadow: 0 0 0 2px rgba(0, 102, 204, 0.1); }
            .search-ruta-btn {
                padding: 6px 15px; border: 1px solid #0066cc; border-radius: 4px; background: #0066cc;
                color: white; cursor: pointer; font-size: 12px; font-weight: 500;
                transition: all 0.2s; white-space: nowrap;
            }
            .search-ruta-btn:hover { background: #0052a3; border-color: #0052a3; }
            .search-ruta-clear {
                padding: 6px 12px; border: 1px solid #dc3545; border-radius: 4px; background: white;
                color: #dc3545; cursor: pointer; font-size: 12px; font-weight: 500; transition: all 0.2s;
            }
            .search-ruta-clear:hover { background: #dc3545; color: white; }
            .leaflet-control-layers-overlays { display: none; }
            </style>
            '''
            mapa_base.get_root().html.add_child(folium.Element(estilos_css))
            
            paneles_js = r'''
            <script>
            function crearPanelResumen() {
                const panel = document.createElement('div');
                panel.id = 'panel-resumen';
                panel.innerHTML = `<h3>📊 Resumen de Rutas</h3><div id="panel-resumen-content"></div>`;
                document.body.appendChild(panel);
                return panel;
            }
            function inicializarPaneles(mapInstance) {
                console.log('=== Inicializando paneles personalizados ===');
                if (document.getElementById('panel-resumen')) {
                    console.log('ℹ️ Paneles ya inicializados');
                    return true;
                }
                const panelResumen = crearPanelResumen();
                const map = mapInstance;
                if (!map) { console.error('❌ Instancia del mapa (mapInstance) es nula.'); return false; }
                let totalRutas = 0;
                const metRutas = {};
                map.eachLayer(function(layer) {
                    if (layer._url || !layer.options || !layer.options.pane) return;
                    const layerName = layer.options.attribution || '';
                    const rutaMatch = layerName.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+Ruta\s+(\d+)$/); // Modificado para ignorar advertencia
                    if (rutaMatch) {
                        const metNombre = rutaMatch[1]; // Captura el nombre limpio
                        totalRutas++;
                        if (!metRutas[metNombre]) metRutas[metNombre] = 0;
                        metRutas[metNombre]++;
                    }
                });
                const resumenHTML = [];
                Object.keys(metRutas).sort().forEach(function(metNombre) {
                    resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">${metNombre}:</span><span class="resumen-value">${metRutas[metNombre]} rutas</span></div>`);
                });
                resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">Total:</span><span class="resumen-value">${totalRutas} rutas</span></div>`);
                document.getElementById('panel-resumen-content').innerHTML = resumenHTML.join('');
                console.log('✅ Paneles personalizados inicializados');
                return true;
            }
            let panelPollCount = 0;
            function tryInitPanels() {
                const mapElement = document.querySelector('.leaflet-container');
                let mapInstance = null;
                if (mapElement && mapElement._leaflet_map) { mapInstance = mapElement._leaflet_map; }
                if (typeof L !== 'undefined' && L.map && mapInstance) {
                    console.log('✅ (Poll) Leaflet (L) y MapInstance (_leaflet_map) listos. Iniciando paneles...');
                    inicializarPaneles(mapInstance);
                } else if (panelPollCount < 40) {
                    panelPollCount++;
                    console.log('Poll #' + panelPollCount + ': Esperando a L y _leaflet_map...');
                    setTimeout(tryInitPanels, 500);
                } else {
                    console.error('❌ No se pudo cargar la instancia del mapa (_leaflet_map) después de 20 segundos.');
                }
            }
            tryInitPanels();
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(paneles_js))
            
            filtros_js = r'''
            <script>
            function inicializarFiltros() {
                console.log('=== Inicializando filtros personalizados (Estilo Celaya) ===');
                const layerControl = document.querySelector('.leaflet-control-layers');
                if (!layerControl) { console.error('❌ No se encontró .leaflet-control-layers'); return false; }
                const layersList = layerControl.querySelector('.leaflet-control-layers-list');
                if (!layersList) { console.error('❌ No se encontró .leaflet-control-layers-list'); return false; }
                const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) { console.error('❌ No se encontró .leaflet-control-layers-overlays'); return false; }
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                console.log('✅ Total de checkboxes encontrados:', checkboxes.length);
                if (layersList.querySelector('.layer-control-buttons')) { console.log('ℹ️ Filtros ya inicializados'); return true; }
                
                console.log('📋 Listando todas las capas:');
                checkboxes.forEach(function(checkbox, index) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    console.log('   ' + (index + 1) + '. "' + label + '"');
                });
                
                const buttonsDiv = document.createElement('div');
                buttonsDiv.className = 'layer-control-buttons';
                const selectAllBtn = document.createElement('button');
                selectAllBtn.textContent = '✓ Todo';
                selectAllBtn.className = 'layer-btn select-all';
                selectAllBtn.title = 'Seleccionar todas las capas';
                const deselectAllBtn = document.createElement('button');
                deselectAllBtn.textContent = '✗ Nada';
                deselectAllBtn.className = 'layer-btn deselect-all';
                deselectAllBtn.title = 'Deseleccionar todas las capas';
                selectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (!cb.checked) cb.click(); }); };
                deselectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); };
                buttonsDiv.appendChild(selectAllBtn);
                buttonsDiv.appendChild(deselectAllBtn);
                layersList.insertBefore(buttonsDiv, overlaysDiv);
                
                const searchContainer = document.createElement('div');
                searchContainer.className = 'search-ruta-container';
                const searchLabel = document.createElement('span');
                searchLabel.textContent = '🔍';
                searchLabel.style.fontSize = '14px';
                const searchInput = document.createElement('input');
                searchInput.type = 'text';
                searchInput.className = 'search-ruta-input';
                searchInput.placeholder = 'Buscar por Ruta ID (ej: 0, 1, 2...)';
                searchInput.title = 'Ingresa el número de ruta para filtrar';
                const searchBtn = document.createElement('button');
                searchBtn.textContent = 'Buscar';
                searchBtn.className = 'search-ruta-btn';
                searchBtn.title = 'Filtrar por Ruta ID';
                const clearBtn = document.createElement('button');
                clearBtn.textContent = 'Limpiar';
                clearBtn.className = 'search-ruta-clear';
                clearBtn.title = 'Limpiar búsqueda';
                const filtrarPorRutaId = function() {
                    const rutaId = searchInput.value.trim();
                    if (rutaId === '') { alert('Por favor ingresa un número de ruta'); return; }
                    console.log('🔍 Filtrando capas con Ruta ID:', rutaId);
                    checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                    setTimeout(function() {
                        let encontradas = 0;
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$'); // Busca el final
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                console.log('  ✓ Seleccionando:', label);
                                if (!checkbox.checked) { checkbox.click(); encontradas++; }
                            }
                        });
                        if (encontradas === 0) { console.warn('⚠️ No se encontraron capas con Ruta ID ' + rutaId); alert('No se encontraron capas con Ruta ID: ' + rutaId); }
                    }, 150);
                };
                searchBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); filtrarPorRutaId(); };
                searchInput.addEventListener('keypress', function(e) { if (e.key === 'Enter') { e.preventDefault(); filtrarPorRutaId(); } });
                clearBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); searchInput.value = ''; checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); console.log('🧹 Búsqueda limpiada'); };
                searchContainer.appendChild(searchLabel);
                searchContainer.appendChild(searchInput);
                searchContainer.appendChild(searchBtn);
                searchContainer.appendChild(clearBtn);
                layersList.insertBefore(searchContainer, overlaysDiv);
                
                const metButtonsDiv = document.createElement('div');
                metButtonsDiv.className = 'met-buttons-row';
                const mets = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const metMatch = label.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/); // Modificado para incluir advertencia
                    if (metMatch) { mets.add(metMatch[1].trim()); } // Limpiar advertencia
                });
                console.log('🏢 METs disponibles:', Array.from(mets));
                Array.from(mets).sort().forEach(function(metNombre) {
                    const metBtn = document.createElement('button');
                    const nombreCorto = metNombre.replace('MET ', '').trim();
                    metBtn.textContent = nombreCorto;
                    metBtn.className = 'met-btn';
                    metBtn.title = 'Seleccionar solo capas de ' + metNombre;
                    metBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        console.log('🔍 Activando filtro para MET:', metNombre);
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                // Modificado para que siempre funcione
                                if (label.includes(metNombre + ' -')) {
                                    console.log('  ✓ Seleccionando:', label);
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas para ' + metNombre);
                        }, 150);
                    };
                    metButtonsDiv.appendChild(metBtn);
                });
                if (metButtonsDiv.children.length > 0) { layersList.insertBefore(metButtonsDiv, overlaysDiv); }
                
                const rutaButtonsDiv = document.createElement('div');
                rutaButtonsDiv.className = 'ruta-buttons-row';
                const rutas = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const rutaMatch = label.match(/- Ruta\s+(\d+)/);
                    if (rutaMatch) { rutas.add(parseInt(rutaMatch[1])); }
                });
                console.log('🚚 Rutas encontradas:', Array.from(rutas).sort(function(a, b) { return a - b; }));
                const rutasArray = Array.from(rutas).sort(function(a, b) { return a - b; });
                rutasArray.forEach(function(ruta) {
                    const rutaBtn = document.createElement('button');
                    rutaBtn.textContent = 'R' + ruta;
                    rutaBtn.className = 'ruta-btn';
                    rutaBtn.title = 'Seleccionar Ruta ' + ruta + ' de todos los METs';
                    rutaBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        const rutaId = ruta;
                        console.log('🔍 Activando filtro para Ruta', rutaId);
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$');
                        const metIconPattern = /- 🏠 MET$/;
                        const metNamePattern = /^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/;
                        const metsToShow = new Set();
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                const metMatch = label.match(metNamePattern);
                                if (metMatch) { metsToShow.add(metMatch[1].trim()); }
                            }
                        });
                        console.log('   ...Mostrando METs que tienen esta ruta:', Array.from(metsToShow));
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                if (rutaPattern.test(label)) {
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                } else if (metIconPattern.test(label)) {
                                    const metMatch = label.match(metNamePattern);
                                    if (metMatch && metsToShow.has(metMatch[1].trim())) {
                                        if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                    }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas (Rutas + METs) para Ruta ' + rutaId);
                        }, 150);
                    };
                    rutaButtonsDiv.appendChild(rutaBtn);
                });
                if (rutaButtonsDiv.children.length > 0) { layersList.insertBefore(rutaButtonsDiv, overlaysDiv); }
                
                console.log('=== Filtros inicializados correctamente ===');
                return true;
            }
            let filterPollCount = 0;
            function tryInitFilters() {
                const layerControlList = document.querySelector('.leaflet-control-layers-list');
                if (typeof L !== 'undefined' && layerControlList) {
                    console.log('✅ (Poll) Leaflet (L) y .leaflet-control-layers-list listos. Iniciando filtros...');
                    try { inicializarFiltros(); } catch (e) { console.error('❌ Error al ejecutar inicializarFiltros:', e); }
                } else if (filterPollCount < 40) {
                    filterPollCount++;
                    console.log('Poll Filtros #' + filterPollCount + ': Esperando a L y .leaflet-control-layers-list...');
                    setTimeout(tryInitFilters, 500);
                } else {
                    console.error('❌ No se pudo encontrar .leaflet-control-layers-list después de 20 segundos.');
                }
            }
            setTimeout(tryInitFilters, 500); 
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(filtros_js))
            
            titulo_html = f'''
            <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
                <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                    🗺️ ANÁLISIS DE CLUSTERS POR VOLUMEN
                </h2>
                <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                    📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas Generadas: <b>{len(summary_data)}</b> | 👥 Clientes Procesados: <b>{g_total_clientes_procesados}</b><br>
                    📦 Tamaño de Círculo = Volumen (m³/día) | 🎨 Color de Círculo = Ruta Asignada
                </p>
            </div>
            '''
            mapa_base.get_root().html.add_child(folium.Element(titulo_html))
            
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            map_filename = f'mapa_rutas_clusters_{timestamp}.html'
            g_map_path = os.path.join(output_dir, map_filename)
            mapa_base.save(g_map_path)
            
            print(f"\n{'='*70}")
            print(f"🗺️ Mapa guardado: {g_map_path}")
            
            if export_data and summary_data:
                excel_path = generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir)
                if excel_path:
                    g_excel_paths = [excel_path]
                    print(f"📊 Excel guardado: {excel_path}")
            
            print(f"{'='*70}")
            print(f"✅ ANÁLISIS COMPLETADO")
            print(f"   • Total clientes procesados: {g_total_clientes_procesados}")
            print(f"   • Total rutas generadas: {len(summary_data)}")
            print(f"   • Archivos generados: {len(g_excel_paths) + 1}")
            print(f"{'='*70}\n")

        except Exception as e:
            import traceback
            print(f"❌ Error Inesperado en ejecución #{execution_count}:")
            traceback.print_exc()
        finally:
            # --- (NUEVO) RESTAURAR DF_CLIENTES ORIGINAL ---
            if 'df_clientes_original' in locals():
                df_clientes = df_clientes_original
                print("ℹ️ DataFrame de clientes restaurado a su estado original.")
            
            param_button.disabled = False # Desbloquear botón

# --- Registrar el handler ---
try:
    param_button.on_click(ejecutar_analisis, remove=True)
except:
    pass
param_button.on_click(ejecutar_analisis)

# === MOSTRAR UI (Modificada) ===
print("================== MÓDULO 2: Análisis de Rutas (Alfa 5.1) ==================")
display(widgets.VBox([
    widgets.Label("1. Cargar archivo Excel:", style={'font_weight': 'bold'}),
    upload_widget,
    output_upload,
    widgets.HTML("<hr>"),
    met_box,
    widgets.HTML("<hr>"),
    widgets.Label("2. Configurar parámetros de Ruta:", style={'font_weight': 'bold'}),
    widgets.HBox([min_clientes_cluster, max_clientes_cluster]),
    
    # --- (NUEVO) WIDGETS DE MUESTREO ---
    widgets.HBox([sample_size_per_met, run_full_analysis]),
    # --- FIN DE WIDGETS NUEVOS ---
    
    param_button,
    output_param
]))

In [ ]:
#=========== 1. INICIALIZACIÓN Y FUNCIONES BASE (Alfa 5.1) ==================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import os
import glob
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from shapely.geometry import Polygon, MultiPoint, Point # Importar Point y MultiPoint
from sklearn.cluster import KMeans
import shutil # Importar shutil para limpieza
import zipfile # Importar zipfile para Celda 3

# --- Validar rutas locales ---
icon_met_path = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png"
output_dir = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output"

# Crear directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(icon_met_path):
    print(f"⚠️ Advertencia: No se encontró el icono en {icon_met_path}. Se usará icono por defecto.")
else:
    print(f"✅ Icono de MET encontrado: {icon_met_path}")


# --- GLOBALES DEL ESTADO ---
execution_count = 0
df_clientes = pd.DataFrame()
df_cdr = pd.DataFrame()
met_checkboxes = []
# --- GLOBALES PARA INTERCAMBIO DE MÓDULOS ---
g_map_path = ""
g_excel_paths = []
g_total_clientes_procesados = 0
g_descarga_en_progreso = False # Para el bloqueo de descarga

# --- LISTA DE COLORES GLOBAL ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# =========================================================================
# === FUNCIONES HELPER (LÓGICA) ===========================================
# =========================================================================

# --- HELPER FUNCTION PARA NORMALIZAR IDS ---
def normalizar_id(valor):
    """Convierte un valor a un ID seguro para HTML/JS."""
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    s = s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    s = s.replace('Á','A').replace('É','E').replace('Í','I').replace('Ó','O').replace('Ú','U')
    s = s.replace('ñ','n').replace('Ñ','N')
    return s

# --- FUNCIÓN HELPER PARA VORONOI ---
def _clip_voronoi(vor, bounding_poly):
    """Recorta los polígonos de Voronoi a un polígono de frontera (shapely)."""
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty:
        return clipped_polys

    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points): continue
        if not (0 <= region_index < len(vor.regions)): continue
        region_vertices_indices = vor.regions[region_index]
        if -1 in region_vertices_indices: continue
        valid_vertex_indices = [v for v in region_vertices_indices if 0 <= v < len(vor.vertices)]
        if len(valid_vertex_indices) < 3: continue
        region_vertices = [vor.vertices[v] for v in valid_vertex_indices]
        if len(region_vertices) < 3: continue

        try:
            poly = Polygon(region_vertices)
            if not poly.is_valid: poly = poly.buffer(0)
            if not poly.is_valid: continue

            clipped = poly.intersection(bounding_poly)

            if clipped.is_empty: continue
            elif clipped.geom_type == 'Polygon' and clipped.exterior is not None:
                if len(clipped.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in clipped.exterior.coords]
                else: continue
            elif clipped.geom_type == 'MultiPolygon':
                if not clipped.geoms: continue
                largest_poly = max(clipped.geoms, key=lambda p: p.area)
                if largest_poly.exterior is not None and len(largest_poly.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in largest_poly.exterior.coords]
                else: continue
            else: continue

            centroid_key = tuple(vor.points[i])
            clipped_polys[centroid_key] = clipped_coords
        except Exception as e:
            pass

    return clipped_polys

# --- FUNCIÓN HELPER PARA TAMAÑO DE MARCADOR (DE CÓDIGO CELAYA) ---
def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    """
    Calcula el tamaño del marcador basado en el volumen.
    Usa escala logarítmica para mejor diferenciación.
    """
    if vol_max == vol_min or volumen <= 0:
        return tamano_min

    # Usar escala logarítmica para mejor diferenciación visual
    log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)

    if log_max == log_min:
        return tamano_min

    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# =========================================================================
# === MÓDULO 1: LÓGICA DE CARGA Y VALIDACIÓN ==============================
# =========================================================================
def handle_upload(change, met_box_widget, output_upload_widget):
    global df_clientes, df_cdr, met_checkboxes

    # Suppress FutureWarning for pandas replace method downcasting
    pd.set_option('future.no_silent_downcasting', True)

    met_box = met_box_widget
    output_upload = output_upload_widget

    with output_upload:
        clear_output()
        upload_widget = change['owner']
        if not upload_widget.value:
            return

        # Adaptado para manejar diferentes formatos de FileUpload
        try:
            # Intentar obtener el archivo del widget
            if isinstance(upload_widget.value, tuple) and len(upload_widget.value) > 0:
                # Formato: tuple de FileInfo objects
                uploaded_file_info = upload_widget.value[0]
                uploaded_file_name = uploaded_file_info['name']
                uploaded_file_content = uploaded_file_info['content']
            elif isinstance(upload_widget.value, dict):
                # Formato: dict con nombres de archivo como keys
                uploaded_file_details = next(iter(upload_widget.value.values()))
                uploaded_file_name = uploaded_file_details['metadata']['name']
                uploaded_file_content = uploaded_file_details['content']
            else:
                raise ValueError("Formato de archivo no reconocido")
        except Exception as e:
            print(f"❌ Error al procesar el archivo cargado: {e}")
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
            return

        try:
            with open(uploaded_file_name, 'wb') as f: f.write(uploaded_file_content)
            df_excel = pd.read_excel(uploaded_file_name, sheet_name=None)

            if len(df_excel) < 2:
                raise ValueError("El archivo Excel debe contener al menos dos hojas (Clientes y METs/CDRs).")

            sheet1_name, sheet2_name = list(df_excel.keys())[0], list(df_excel.keys())[1]
            df_clientes_raw, df_cdr_raw = df_excel[sheet1_name], df_excel[sheet2_name]

            required_cols_clientes = ['CodCli', 'U_longitud', 'U_latitud', 'm3/día', 'entregas/día']
            required_cols_cdr = ['CodMET', 'U_longitud', 'U_latitud']

            missing_cols_cli = [col for col in required_cols_clientes if col not in df_clientes_raw.columns]
            missing_cols_cdr = [col for col in required_cols_cdr if col not in df_cdr_raw.columns]

            if missing_cols_cli:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet1_name}': {', '.join(missing_cols_cli)}")
            if missing_cols_cdr:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet2_name}': {', '.join(missing_cols_cdr)}")

            df_clientes_raw['U_longitud'] = pd.to_numeric(df_clientes_raw['U_longitud'], errors='coerce')
            df_clientes_raw['U_latitud'] = pd.to_numeric(df_clientes_raw['U_latitud'], errors='coerce')

            # Convertir m3/día: reemplazar "-" por 0 y convertir a numérico
            df_clientes_raw['m3/día'] = df_clientes_raw['m3/día'].replace('-', 0)
            df_clientes_raw['m3/día'] = pd.to_numeric(df_clientes_raw['m3/día'], errors='coerce').fillna(0)

            # Convertir entregas/día: reemplazar "-" por 0 y convertir a numérico (valores entre 0 y 1)
            df_clientes_raw['entregas/día'] = df_clientes_raw['entregas/día'].replace('-', 0)
            df_clientes_raw['entregas/día'] = pd.to_numeric(df_clientes_raw['entregas/día'], errors='coerce').fillna(0)

            df_cdr_raw['U_longitud'] = pd.to_numeric(df_cdr_raw['U_longitud'], errors='coerce')
            df_cdr_raw['U_latitud'] = pd.to_numeric(df_cdr_raw['U_latitud'], errors='coerce')

            df_clientes = df_clientes_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()
            df_cdr = df_cdr_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()

            num_dropped_cli = len(df_clientes_raw) - len(df_clientes)
            num_dropped_cdr = len(df_cdr_raw) - len(df_cdr)
            if num_dropped_cli > 0: print(f"⚠️ Se eliminaron {num_dropped_cli} clientes por coordenadas inválidas.")
            if num_dropped_cdr > 0: print(f"⚠️ Se eliminaron {num_dropped_cdr} METs/CDRs por coordenadas inválidas.")

            print(f"✅ Archivo '{uploaded_file_name}' cargado exitosamente.")
            print(f"📊 Clientes válidos: {df_clientes.shape[0]} filas")
            print(f"📊 METs/CDRs válidos: {df_cdr.shape[0]} filas")

            mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
            met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
            met_box.children = [widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes

        except ValueError as ve:
            print(f"❌ Error de validación: {ve}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error en formato de archivo.')]
        except Exception as e:
            print(f"❌ Error inesperado al cargar: {e}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
        finally:
            if 'uploaded_file_name' in locals() and os.path.exists(uploaded_file_name):
                os.remove(uploaded_file_name)

# =========================================================================
# === MÓDULO 3: LÓGICA DE CLUSTERING (Alfa 5.1) ==========================
# =========================================================================
def select_customers_for_met(df_all_clients, met_lat, met_lon, n_clients, analyze_all):
    # Esta función se mantiene por si se usa en el futuro,
    # pero la lógica de 'Territorios' (Fase 0) en Módulo 2 la reemplaza.
    if df_all_clients.empty: return pd.DataFrame()
    if analyze_all:
        return df_all_clients.copy()
    else:
        distances = np.hypot(df_all_clients['U_latitud'] - met_lat, df_all_clients['U_longitud'] - met_lon)
        df_with_dist = df_all_clients.copy()
        df_with_dist['Distancia_al_MET'] = distances
        n_to_select = min(n_clients, len(df_with_dist))
        if n_to_select <= 0: return pd.DataFrame()
        return df_with_dist.nsmallest(n_to_select, 'Distancia_al_MET')

def calcular_dispersion_cluster(df_cluster):
    if len(df_cluster) < 2:
        return 0.0
    coords = df_cluster[['U_longitud', 'U_latitud']].values
    distancias_pares = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = np.hypot(coords[i][0] - coords[j][0], coords[i][1] - coords[j][1])
            distancias_pares.append(dist)
    return np.mean(distancias_pares) if distancias_pares else 0.0

def calcular_limites_clientes_dinamico(dispersion, frecuencia_visitas_promedio, min_base=20, max_base=40):
    """
    Ajusta dinámicamente el límite de clientes por ruta basándose en:
    - dispersion: Dispersión geográfica de la ruta (distancia promedio entre clientes)
    - frecuencia_visitas_promedio: Frecuencia promedio de entregas/día (0-1, donde 1 = 100%)

    A mayor dispersión geográfica → menos clientes permitidos
    A MAYOR frecuencia de visitas → MÁS clientes permitidos (rutas densas y frecuentes)
    """
    dispersion_normalizada = min(dispersion / 0.5, 1.0)

    # Normalizar frecuencia de visitas (ya viene en escala 0-1, donde 0.9692 = 96.92%)
    # Mayor frecuencia = MENOS restricción (más clientes permitidos para rutas frecuentes)
    frecuencia_normalizada = min(frecuencia_visitas_promedio, 1.0)

    # Factor de restricción: Solo la dispersión restringe, la frecuencia AUMENTA el límite
    # Invertimos el efecto de la frecuencia: alta frecuencia reduce la restricción
    factor_restriccion = dispersion_normalizada * (1.0 - frecuencia_normalizada * 0.5)

    rango = max_base - min_base
    ajuste = int(rango * factor_restriccion)
    max_dinamico = max_base - ajuste
    min_dinamico = max(min_base - int(ajuste * 0.5), int(min_base * 0.5))
    return min_dinamico, max_dinamico

def segregar_por_frecuencia(df_clients, threshold=0.01):
    """
    Separa clientes en elegibles y comodines ANTES del clustering.
    
    Parámetros:
        df_clients: DataFrame con todos los clientes (sin columna Ruta_nueva aún)
        threshold: Umbral de frecuencia (default: 0.01 = 1%)
    
    Retorna:
        df_elegibles: entregas/día > threshold
        df_comodines: entregas/día ≤ threshold
    """
    # Inicializar columna Ruta_nueva si no existe
    if 'Ruta_nueva' not in df_clients.columns:
        df_clients['Ruta_nueva'] = -1
    
    # Filtrar solo clientes no asignados a MET (Ruta_nueva == -1)
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    
    print(f"  📊 Segregación por frecuencia (threshold={threshold*100:.1f}%):")
    print(f"     ✅ Elegibles (>{threshold*100:.1f}%): {len(df_elegibles)} clientes")
    print(f"     🔺 Comodines (≤{threshold*100:.1f}%): {len(df_comodines)} clientes")
    
    return df_elegibles, df_comodines

def asignar_comodines_post_clustering(df_comodines, df_elegibles_asignados):
    """
    Asigna comodines al vecino más cercano SIN afectar límites.
    
    Parámetros:
        df_comodines: Clientes con entregas/día ≤ 1%
        df_elegibles_asignados: Clientes normales ya clusterizados
    
    Retorna:
        df_comodines con columna 'Ruta_nueva' asignada
    """
    from scipy.spatial.distance import cdist
    
    if len(df_comodines) == 0:
        print("  ✅ No hay comodines para asignar")
        return df_comodines
    
    print(f"  🔺 Asignando {len(df_comodines)} comodines (método: vecino más cercano)...")
    
    coords_comodines = df_comodines[['U_longitud', 'U_latitud']].values
    coords_elegibles = df_elegibles_asignados[['U_longitud', 'U_latitud']].values
    
    # Calcular distancias entre comodines y elegibles
    dist_matrix = cdist(coords_comodines, coords_elegibles, metric='euclidean')
    
    # Para cada comodín, encontrar el elegible más cercano
    idx_vecinos = np.argmin(dist_matrix, axis=1)
    rutas_asignadas = df_elegibles_asignados.iloc[idx_vecinos]['Ruta_nueva'].values
    
    df_comodines['Ruta_nueva'] = rutas_asignadas
    df_comodines['es_comodin'] = True  # Bandera para identificación
    
    # Estadísticas
    print(f"     ✅ {len(df_comodines)} comodines asignados a {len(np.unique(rutas_asignadas))} rutas")
    
    return df_comodines

def find_initial_seeds(coords, k, random_state=42):
    n_puntos = coords.shape[0]
    if k <= 0 or n_puntos == 0:
        print("       ⚠️ No se pueden generar semillas (k<=0 o no hay puntos).")
        return np.array([])
    if n_puntos < k:
        print(f"       ⚠️ No se pueden encontrar {k} semillas con {n_puntos} puntos. Ajustando k a {n_puntos}.")
        k = n_puntos
    try:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=20, init='k-means++').fit(coords)
        return kmeans.cluster_centers_
    except ValueError as e:
        print(f"       ❌ Error en KMeans para semillas (k={k}, n_puntos={n_puntos}): {e}")
        if n_puntos > 0:
            indices = np.random.choice(n_puntos, min(k, n_puntos), replace=False)
            print(f"       ... Usando {len(indices)} puntos aleatorios como semillas (Fallback).")
            return coords[indices]
        else:
            return np.array([])

# --- (MODIFICADO) FUNCIÓN PARA ENCONTRAR ANCLAS DE FRECUENCIA ---
def find_frequency_anchors(df_clients, percentile=80):
    """Encuentra los clientes 'ancla' (ej. 20% con mayor frecuencia de entregas/día)"""
    # No tomar en cuenta clientes con frecuencia muy baja o cero
    df_with_frequency = df_clients[df_clients['entregas/día'] > 0.01]
    if df_with_frequency.empty:
        print("       ... No se encontraron clientes con frecuencia significativa para anclaje.")
        return pd.DataFrame() # No hay anclas

    # Usar un umbral de percentil para definir "alta frecuencia"
    min_frequency_threshold = np.percentile(df_with_frequency['entregas/día'], percentile)
    df_anchors = df_with_frequency[df_with_frequency['entregas/día'] >= min_frequency_threshold].copy()

    print(f"       ... Umbral de anclaje (percentil {percentile}%) = {min_frequency_threshold*100:.1f}% entregas/día")
    print(f"       ... {len(df_anchors)} clientes 'ancla' de alta frecuencia identificados.")
    return df_anchors

# --- (MODIFICADO) LÓGICA DE CRECIMIENTO HÍBRIDA (Alfa 5.1) ---
def grow_clusters_from_seeds(df_elegibles_only, initial_seeds, max_size, min_size=20):
    """
    Asigna SOLO clientes elegibles (>1% frecuencia) a rutas.
    Los comodines (≤1%) se asignan DESPUÉS con asignar_comodines_post_clustering().
    
    Estrategia HÍBRIDA (Alfa 5.1):
    1. Fase A: Anclar rutas con clientes de alta frecuencia
    2. Fase B: Rellenar rutas con lógica 'greedy'
    
    Parámetros:
        df_elegibles_only: Solo clientes con entregas/día > 0.01
        initial_seeds: Coordenadas de centroides iniciales
        max_size: Límite máximo de clientes normales por ruta
        min_size: Límite mínimo de clientes normales por ruta
    """
    k = len(initial_seeds)
    n_clientes_total = len(df_elegibles_only)
    if k == 0 or n_clientes_total == 0:
        print("       ⚠️ No hay semillas o clientes elegibles para 'grow_clusters_from_seeds'.")
        df_elegibles_only['Ruta_nueva'] = -1
        return df_elegibles_only, {}

    # --- Inicialización (Lógica Alfa) ---
    df_clients_to_cluster = df_elegibles_only.copy()
    df_clients_to_cluster['Ruta_nueva'] = -1
    df_clients_to_cluster['es_comodin'] = False  # Todos los elegibles NO son comodines
    ruta_volumenes = {i: 0.0 for i in range(k)}
    ruta_counts = {i: 0 for i in range(k)}

    # --- FASE A: ANCLAJE POR FRECUENCIA (MODIFICADO) ---
    print(f"       🔬 Fase A: Anclando {k} rutas con clientes de alta frecuencia (entregas/día)...")
    df_anchors = find_frequency_anchors(df_clients_to_cluster)
    n_asignados_anchors = 0

    if not df_anchors.empty:
        coords_anchors = df_anchors[['U_longitud', 'U_latitud']].values
        dist_matrix_anchors = distance_matrix(coords_anchors, initial_seeds)

        # Iterar por cada ancla
        for i, (anchor_idx, row) in enumerate(df_anchors.iterrows()):
            frecuencia_ancla = row['entregas/día']
            volumen_ancla = row['m3/día']
            rutas_ordenadas = np.argsort(dist_matrix_anchors[i]) # Rutas más cercanas primero

            asignado = False
            for ruta_idx in rutas_ordenadas:
                # Validar solo por límite de clientes (sin restricción de volumen)
                # Para la Fase de Anclaje, usamos un límite estático ya que la ruta está vacía
                if ruta_counts[ruta_idx] < max_size:
                    df_clients_to_cluster.loc[anchor_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_volumenes[ruta_idx] += volumen_ancla
                    ruta_counts[ruta_idx] += 1
                    n_asignados_anchors += 1
                    asignado = True
                    break # Ancla asignada, pasar a la siguiente ancla

            if not asignado:
                print(f"       ⚠️ Ancla {anchor_idx} (Frec: {frecuencia_ancla*100:.1f}%) no pudo ser asignada (todas las rutas están llenas).")

    print(f"       ✅ Fase A (Anclaje) completada: {n_asignados_anchors}/{len(df_anchors)} anclas asignadas.")

    # --- FASE B: RELLENO GEOMÉTRICO (Lógica "Redonda" de Beta) ---
    df_normales_restantes = df_clients_to_cluster[
        (df_clients_to_cluster['entregas/día'] > 0.01) &  # Frecuencia > 1%
        (df_clients_to_cluster['Ruta_nueva'] == -1)
    ].copy()

    n_normales = len(df_normales_restantes)
    n_asignados_normales = 0

    if n_normales > 0:
        print(f"       🔬 Fase B: Rellenando geométricamente {n_normales} clientes restantes...")
        coords_normales = df_normales_restantes[['U_longitud', 'U_latitud']].values

        try:
            dist_matrix = distance_matrix(coords_normales, initial_seeds)
        except ValueError as e:
            print(f"       ❌ Error calculando matriz de distancias: {e}")
            dist_matrix = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_normales])

        dist_df = pd.DataFrame(dist_matrix, index=df_normales_restantes.index, columns=range(k))

        iteracion = 0
        max_iter = n_normales * k # Límite de seguridad

        while n_asignados_normales < n_normales and iteracion < max_iter:
            iteracion += 1
            min_dist = dist_df.min().min()
            if min_dist == np.inf:
                break

            stacked = dist_df.stack()
            best_pair_idx = stacked[stacked == min_dist].index
            client_idx, ruta_idx = best_pair_idx[0]

            # --- Validar con límites dinámicos (sin restricción de 6m³) ---
            volumen_cliente = df_normales_restantes.loc[client_idx, 'm3/día']
            # volumen_futuro = ruta_volumenes[ruta_idx] + volumen_cliente # Volumen ya no afecta la asignación

            clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
            dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

            # Calcular frecuencia promedio de visitas de la ruta (incluyendo el nuevo cliente)
            if len(clientes_ruta_actual) > 0:
                frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
            else:
                frecuencia_ruta_actual = 0.0

            # Estimar frecuencia futura (promedio ponderado)
            frecuencia_cliente = df_normales_restantes.loc[client_idx, 'entregas/día']
            if len(clientes_ruta_actual) > 0:
                frecuencia_futura = ((frecuencia_ruta_actual * len(clientes_ruta_actual)) + frecuencia_cliente) / (len(clientes_ruta_actual) + 1)
            else:
                frecuencia_futura = frecuencia_cliente

            min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                dispersion_actual, frecuencia_futura, min_base=min_size, max_base=max_size
            )

            # Solo validar límite de clientes (SIN límite de volumen)
            if ruta_counts[ruta_idx] < max_dinamico:
                # Éxito: Asignar
                df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = ruta_idx
                ruta_volumenes[ruta_idx] += volumen_cliente # Actualizar volumen para resumen, pero no para decisión
                ruta_counts[ruta_idx] += 1
                n_asignados_normales += 1
                dist_df.loc[client_idx, :] = np.inf # Cliente asignado - marca TODAS las distancias como infinitas
            else:
                # Fallo: Invalidar solo este par cliente-ruta
                dist_df.loc[client_idx, ruta_idx] = np.inf

        if iteracion >= max_iter:
            print(f"       ⚠️ Se alcanzó el límite de iteraciones ({max_iter}).")

        print(f"       ✅ Fase B completada: {n_asignados_normales}/{n_normales} clientes elegibles asignados.")

    # --- FIN: Retornar solo elegibles asignados ---
    # Los comodines se asignan DESPUÉS con asignar_comodines_post_clustering()
    return df_clients_to_cluster, ruta_volumenes

# --- (MODIFICADO) LÓGICA DE AJUSTE CON GEOMETRÍA (Alfa 5.1) ---
def adjust_small_clusters(df_clustered, cluster_counts, min_size, max_size):
    """
    Ajusta rutas pequeñas (< min_size).
    Reasigna clientes al POLÍGONO válido más cercano (no al centroide).
    (MODIFICADO: Usa frecuencia de visitas en lugar de volumen para dispersión)
    """
    print("       🔬 Fase D: Ajustando rutas pequeñas...")
    df_adjusted = df_clustered.copy()

    # Bucle de reintentos
    attempts = 0
    max_attempts = 3 # Limitar los pases de ajuste
    while attempts < max_attempts:
        attempts += 1

        # --- 1. Calcular volúmenes y conteos actuales ---
        ruta_volumenes = {}
        ruta_counts_actual = {}
        for ruta_id in df_adjusted['Ruta_nueva'].unique():
            if ruta_id < 0: continue
            clientes_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            ruta_volumenes[ruta_id] = clientes_ruta['m3/día'].sum()
            ruta_counts_actual[ruta_id] = len(clientes_ruta)

        if not ruta_counts_actual:
            print("       ... No hay rutas que ajustar.")
            break # No hay nada que hacer

        # --- 2. Identificar rutas VÁLIDAS vs. RUTAS A MOVER ---
        rutas_validas_ids = set()
        rutas_a_mover_ids = set()

        for ruta_id, count in ruta_counts_actual.items():
            # Regla de negocio: Solo Pequeña (sin criterio de rentabilidad)
            if count < min_size:
                rutas_a_mover_ids.add(ruta_id)
            else:
                rutas_validas_ids.add(ruta_id)

        if not rutas_a_mover_ids:
            print(f"       ✅ Fase D completada: No hay rutas pequeñas. (Intento {attempts})")
            break # Éxito, no hay nada que ajustar

        if not rutas_validas_ids:
            print(f"       ⚠️ Fase D: Hay {len(rutas_a_mover_ids)} rutas pequeñas, pero NO hay rutas válidas para moverlas. Deteniendo.")
            break

        # --- 3. Crear Geometrías Válidas (Tu idea) ---
        geometrias_validas = {}
        for ruta_id in rutas_validas_ids:
            clientes_validos = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            points = clientes_validos[['U_longitud', 'U_latitud']].values
            if len(points) >= 3:
                try:
                    # Usamos ConvexHull para obtener el *área* de la ruta
                    hull = ConvexHull(points)
                    geometrias_validas[ruta_id] = Polygon(points[hull.vertices])
                except Exception as e:
                    pass # Ignorar esta ruta si ConvexHull falla

        if not geometrias_validas:
            print(f"       ⚠️ Fase D: No se pudieron crear polígonos para las rutas válidas. Deteniendo.")
            break

        # --- 4. Iterar y Reasignar Clientes "Huérfanos" ---
        df_clientes_a_mover = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_a_mover_ids)].copy()
        print(f"       ... Intento {attempts}: Moviendo {len(df_clientes_a_mover)} clientes de {len(rutas_a_mover_ids)} rutas...")

        clientes_reasignados_en_intento = 0

        for idx_cliente, row in df_clientes_a_mover.iterrows():
            punto_cliente = Point(row['U_longitud'], row['U_latitud'])
            volumen_cliente = row['m3/día']
            original_ruta = row['Ruta_nueva']

            # Calcular distancia a TODOS los polígonos válidos
            distancias_a_poligonos = []
            for ruta_id, poly in geometrias_validas.items():
                dist = punto_cliente.distance(poly)
                distancias_a_poligonos.append((dist, ruta_id))

            if not distancias_a_poligonos:
                continue # No hay polígonos válidos

            distancias_a_poligonos.sort() # Ordenar por distancia (más cercano primero)

            # Intentar asignar al más cercano que cumpla las reglas
            asignado = False
            for dist, ruta_id_destino in distancias_a_poligonos:
                # Validar con Lógica Alfa (MODIFICADO: usa frecuencia)
                # volumen_actual = ruta_volumenes.get(ruta_id_destino, 0)
                # volumen_futuro = volumen_actual + volumen_cliente

                # if volumen_futuro > 6.0:
                #     continue # Excede 6m³, probar siguiente polígono

                clientes_ruta_actual = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id_destino]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

                # Calcular frecuencia promedio futura (MODIFICADO)
                if len(clientes_ruta_actual) > 0:
                    frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
                else:
                    frecuencia_ruta_actual = 0.0

                frecuencia_cliente = row['entregas/día']
                if len(clientes_ruta_actual) > 0:
                    frecuencia_futura = ((frecuencia_ruta_actual * len(clientes_ruta_actual)) + frecuencia_cliente) / (len(clientes_ruta_actual) + 1)
                else:
                    frecuencia_futura = frecuencia_cliente

                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, frecuencia_futura, min_base=min_size, max_base=max_size
                )

                if ruta_counts_actual.get(ruta_id_destino, 0) < max_dinamico:
                    # ¡Éxito! Reasignar
                    df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = ruta_id_destino

                    # Actualizar contadores para la siguiente iteración
                    ruta_counts_actual[ruta_id_destino] = ruta_counts_actual.get(ruta_id_destino, 0) + 1
                    ruta_volumenes[ruta_id_destino] = ruta_volumenes.get(ruta_id_destino, 0) + volumen_cliente # Actualizar volumen para resumen, no para decisión

                    ruta_counts_actual[original_ruta] -= 1
                    ruta_volumenes[original_ruta] = max(0, ruta_volumenes[original_ruta] - volumen_cliente)

                    clientes_reasignados_en_intento += 1
                    asignado = True
                    break # Cliente asignado, pasar al siguiente cliente

            # (No hay 'fallback' forzado. Si no cabe, no cabe)

        if clientes_reasignados_en_intento == 0:
            print("       ... No se pudieron reasignar más clientes en este intento. Deteniendo ajuste.")
            break

    if attempts >= max_attempts:
        print(f"       ⚠️ Fase D: Se alcanzó el límite de {max_attempts} intentos. Pueden quedar rutas pequeñas.")

    # Devolver el DF ajustado y el último conteo conocido
    return df_adjusted, ruta_counts_actual


# =========================================================================
# === MÓDULO 4: CÁLCULO DE POLÍGONOS (Sin cambios) ========================
# =========================================================================
def calculate_cluster_polygons(df_clustered, method='voronoi'):
    # Esta función se usa solo para la VISUALIZACIÓN FINAL
    polygons = {}
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])
    if not assigned_ruta_ids: return polygons

    centroides_rutas = []
    ruta_id_map = {}
    for ruta_id in assigned_ruta_ids:
        ruta_clientes_df = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if not ruta_clientes_df.empty:
            lon = pd.to_numeric(ruta_clientes_df['U_longitud'], errors='coerce').mean()
            lat = pd.to_numeric(ruta_clientes_df['U_latitud'], errors='coerce').mean()
            if pd.notna(lon) and pd.notna(lat):
                centroides_rutas.append([lon, lat])
                ruta_id_map[tuple([lon, lat])] = ruta_id

    poligonos_voronoi_recortados = {}
    vor = None
    if method == 'voronoi' and len(centroides_rutas) >= 4:
        try:
            todos_los_puntos = df_clustered[df_clustered['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
            if len(todos_los_puntos) >= 3:
                boundary_poly_shapely = MultiPoint(todos_los_puntos).convex_hull.buffer(0).buffer(0.0001)
                unique_centroids = np.unique(np.array(centroides_rutas), axis=0)
                if len(unique_centroids) >= 4:
                    vor = Voronoi(unique_centroids, qhull_options='Qbb Qc Qz')
                    poligonos_voronoi_recortados = _clip_voronoi(vor, boundary_poly_shapely)
                else: print("       ... No hay suficientes centroides únicos (>=4) para Voronoi.")
            else: print("       ... No hay suficientes puntos asignados (<3) para definir frontera Voronoi.")
        except Exception as e:
            print(f"       ❌ Error al calcular Voronoi: {e}. Se usará ConvexHull.")
            poligonos_voronoi_recortados = {}
    elif method == 'voronoi':
        print(f"       ... No hay suficientes rutas (se necesitan >= 4, se encontraron {len(centroides_rutas)}) para Voronoi. Usando ConvexHull.")

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if ruta_clientes.empty: continue

        # Encontrar el polígono de Voronoi que corresponde a este centroide
        centroide_lon = ruta_clientes['U_longitud'].mean()
        centroide_lat = ruta_clientes['U_latitud'].mean()
        poly_coords = poligonos_voronoi_recortados.get(tuple([centroide_lon, centroide_lat]))

        if not poly_coords and poligonos_voronoi_recortados:
            # Fallback por si hay errores de precisión de float
            min_dist = float('inf')
            best_match = None
            current_centroid = np.array([centroide_lon, centroide_lat])
            for vor_centroid_key, vor_poly in poligonos_voronoi_recortados.items():
                dist = np.linalg.norm(current_centroid - np.array(vor_centroid_key))
                if dist < min_dist:
                    min_dist = dist
                    best_match = vor_poly
            if min_dist < 1e-6: poly_coords = best_match

        if poly_coords:
            polygons[ruta_id] = poly_coords
        else:
            # Fallback a ConvexHull si Voronoi falló o no se usó
            points = ruta_clientes[['U_latitud', 'U_longitud']].values
            if len(points) >= 3:
                try:
                    hull = ConvexHull(points)
                    polygons[ruta_id] = points[hull.vertices].tolist()
                except Exception:
                    print(f"       ⚠️ No se pudo generar polígono ConvexHull para Ruta {ruta_id}.")
                    polygons[ruta_id] = None
            else: polygons[ruta_id] = None
    return polygons

# =========================================================================
# === MÓDULO 5: GENERACIÓN DE MAPA (Sin Rentabilidad - Solo Frecuencia) ==
# =========================================================================
def create_map_elements(map_instance, df_clustered, cluster_polygons, colores_list,
                        vol_min_global, vol_max_global,
                        met_name=None):
    """
    Dibuja los elementos del cluster y los AÑADE a las capas de filtrado.
    Colores normales para todas las rutas (sin lógica de rentabilidad).
    """

    if met_name is None and 'MET_Asignado' in df_clustered.columns:
        met_name = df_clustered['MET_Asignado'].iloc[0] if not df_clustered.empty else ""
    elif met_name is None:
        met_name = ""

    met_name_clean = met_name.strip()
    display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"

    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])

    tamano_min, tamano_max = (6, 25)

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id].copy()
        ruta_size = len(ruta_clientes)
        if ruta_size == 0: continue

        # --- Asignar color normal a todas las rutas ---
        volumen_total_ruta = ruta_clientes['m3/día'].sum()
        frecuencia_promedio_ruta = ruta_clientes['entregas/día'].mean() * 100  # Convertir a porcentaje
        
        # Contar clientes normales vs comodines
        n_normales = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == False])
        n_comodines = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == True])

        color_ruta = colores_list[ruta_id % len(colores_list)]
        tooltip_poligono = f"{display_name} - Ruta {ruta_id} | Normal: {n_normales} | Comodín: {n_comodines} | Vol: {volumen_total_ruta:,.2f} m³ | Frec: {frecuencia_promedio_ruta:.1f}%"
        nombre_capa = f"{display_name} - Ruta {ruta_id}"

        fg_ruta_individual = folium.FeatureGroup(name=nombre_capa, show=True)

        # Solo se usa el polígono de "solo clientes" (punteado)
        if len(ruta_clientes) >= 3:
            try:
                puntos_solo_clientes = [Point(row['U_longitud'], row['U_latitud']) for _, row in ruta_clientes.iterrows()]
                multipoint_clientes = MultiPoint(puntos_solo_clientes)
                hull_clientes = multipoint_clientes.convex_hull

                if hull_clientes.geom_type == 'Polygon':
                    coords_polygon_clientes = [(lat, lon) for lon, lat in hull_clientes.exterior.coords]
                    
                    # Contar clientes normales vs comodines
                    n_normales = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == False])
                    n_comodines = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == True])
                    
                    area_popup_clientes = f'''
                    <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                        <div style="font-size: 14px; font-weight: bold;">👥 Área Solo Clientes</div>
                        <div style="font-size: 12px; margin-top: 4px;">🏠 {met_name_clean} - 🛣️ Ruta {ruta_id}</div>
                        <div style="font-size: 11px; margin-top: 2px;">✅ Normales: {n_normales} | 🔺 Comodines: {n_comodines}</div>
                        <div style="font-size: 11px; margin-top: 2px;"><b>Total: {len(ruta_clientes)} clientes</b></div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📦 Vol: {volumen_total_ruta:,.2f} m³</div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📊 Frec: {frecuencia_promedio_ruta:.1f}%</div>
                    </div>
                    '''

                    poly_clientes_obj_clon = folium.Polygon(
                        locations=coords_polygon_clientes,
                        color=color_ruta,
                        weight=1,
                        opacity=0.8,
                        fillColor=color_ruta,
                        fillOpacity=0.1,
                        popup=folium.Popup(area_popup_clientes, max_width=250),
                        tooltip=tooltip_poligono,
                        dash_array='5, 5'
                    )
                    poly_clientes_obj_clon.add_to(fg_ruta_individual)
            except Exception as e:
                print(f"       ⚠️ No se pudo generar polígono 'solo clientes' para Ruta {ruta_id}: {e}")

        for _, cliente_row in ruta_clientes.iterrows():
            volumen = cliente_row['m3/día']
            frecuencia = cliente_row['entregas/día']
            frecuencia_pct = frecuencia * 100  # Convertir a porcentaje
            tamano_marcador = obtener_tamano_marcador(volumen, vol_min_global, vol_max_global, tamano_min, tamano_max)

            codcli = cliente_row['CodCli']
            nombre = cliente_row.get('Nombre', 'N/A')
            ruta_original_val = cliente_row.get('Ruta', 'N/A')
            ruta_original = ruta_original_val if pd.notna(ruta_original_val) else 'N/A'
            
            # Detectar si es comodín
            es_comodin = cliente_row.get('es_comodin', False)

            # Popup con indicador de tipo de cliente
            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                    {'🔺 COMODÍN' if es_comodin else '✅ NORMAL'} | {codcli}
                </div>
                <div style="font-size:14px; color:#333; margin-bottom:4px;">
                    {nombre if pd.notna(nombre) else 'N/A'}
                </div>
                <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📦 Volumen: <b>{volumen:,.3f} m³</b>
                </div>

                <div style="font-size:14px; color:{'#888' if es_comodin else '#1976D2'}; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📊 Frecuencia: <b>{frecuencia_pct:.1f}%</b> (entregas/día)
                </div>

                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                    🗺️ Ruta (Orig): <b>{ruta_original}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                </div>
            </div>
            '''

            # Tooltip más corto con frecuencia
            tooltip_text = f"{'🔺' if es_comodin else '✅'} {codcli} | Vol: {volumen:.2f} m³ | Frec: {frecuencia_pct:.1f}%"

            # Icono diferenciado: Triángulo rojo pequeño para comodines, CircleMarker normal para elegibles
            if es_comodin:
                # Triángulo rojo pequeño y discreto para comodines
                marker_obj_clon = folium.CircleMarker(
                    location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=tooltip_text,
                    radius=4,  # Pequeño y discreto
                    color='#D32F2F',  # Rojo oscuro
                    weight=1,
                    fillColor='#F44336',  # Rojo brillante
                    fillOpacity=0.7,
                )
            else:
                # CircleMarker normal para elegibles
                marker_obj_clon = folium.CircleMarker(
                    location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=tooltip_text,
                    radius=tamano_marcador,
                    color='#333333',
                    weight=1,
                    fillColor=color_ruta,
                    fillOpacity=0.85,
                )
            marker_obj_clon.add_to(fg_ruta_individual)

        fg_ruta_individual.add_to(map_instance)
    return

# =========================================================================
# === MÓDULO 6: GENERACIÓN DE EXCEL (Sin Rentabilidad) ====================
# =========================================================================
def generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir):
    if not export_data or not summary_data:
        print(f"       ... No hay datos para exportar a Excel.")
        return None
    df_export = pd.DataFrame(export_data)
    df_resumen = pd.DataFrame(summary_data)
    if df_export.empty or df_resumen.empty:
        print(f"       ... DataFrames vacíos, no se puede generar Excel.")
        return None
    excel_filename = f'rutas_consolidadas_{timestamp}.xlsx'
    excel_path = os.path.join(output_dir, excel_filename)
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_resumen_sorted = df_resumen.sort_values(by=['MET', 'Ruta_nueva'])
            df_resumen_sorted.to_excel(writer, sheet_name='Resumen General', index=False)
            df_export_sorted = df_export.sort_values(by=['MET', 'Ruta_nueva', 'Codigo'])
            df_export_sorted.to_excel(writer, sheet_name='Todos los Clientes', index=False)
            for met_name in mets_seleccionados:
                df_met = df_export[df_export['MET'] == met_name].sort_values(by=['Ruta_nueva', 'Codigo'])
                if not df_met.empty:
                    sheet_name = f'MET_{normalizar_id(met_name)}'[:31]
                    df_met.to_excel(writer, sheet_name=sheet_name, index=False)
        format_excel(excel_path)
        return excel_path
    except Exception as e:
        print(f"       ❌ Error al generar Excel consolidado: {e}")
        if os.path.exists(excel_path):
            try: os.remove(excel_path)
            except: pass
        return None

def format_excel(excel_path):
    """
    Aplica formato profesional al Excel de salida.
    (SIN lógica de rentabilidad - todos los formatos normales)
    """
    if not os.path.exists(excel_path):
        print(f"       ⚠️ No se encontró el archivo Excel para formatear: {excel_path}")
        return
    try:
        wb = openpyxl.load_workbook(excel_path)

        # --- Definición de Estilos ---
        header_font = Font(bold=True, color='FFFFFF', size=11)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        summary_fill = PatternFill('solid', fgColor='FFD966') # Amarillo para resumen

        border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'),
                        top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))
        align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
        align_left = Alignment(horizontal='left', vertical='center', wrap_text=False)

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws.max_row <= 1: continue

            # Formatear Encabezados
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align_center
                cell.border = border
                if cell.value:
                    header_text = str(cell.value)
                    icon = ""
                    if 'Ruta_Original' in header_text: icon = '🗺️ '
                    elif 'Codigo' in header_text or 'CodCli' in header_text: icon = '👨‍💼 '
                    elif 'Clientes' in header_text: icon = '👥 '
                    elif 'MET' in header_text: icon = '🏠 '
                    elif 'Nombre' in header_text: icon = '📚 '
                    elif 'Ruta_nueva' in header_text or 'Ruta_ID' in header_text: icon = '🚚 '
                    elif 'Volumen' in header_text or 'm3' in header_text: icon = '📦 '
                    elif 'Centroide' in header_text: icon = '📍 '
                    elif 'Longitud' in header_text: icon = '📊 '
                    elif 'Latitud' in header_text: icon = '📊 '
                    elif 'Frecuencia' in header_text: icon = '📊 '
                    elif 'Tipo' in header_text and 'Cliente' in header_text: icon = '🏷️ '
                    elif 'Rentable' in header_text: icon = '💰 '
                    elif 'Comodines' in header_text or 'Comodín' in header_text: icon = '🔺 '
                    elif 'Normales' in header_text: icon = '✅ '
                    cleaned_header = header_text.lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰🏷️🔺✅ ')
                    cell.value = icon + cleaned_header

            # Formatear Celdas de Datos
            is_summary_sheet = (sheet_name == 'Resumen General')

            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for col_idx, cell in enumerate(row, start=1):
                    # Aplicar color de fondo normal
                    if is_summary_sheet:
                        cell.fill = summary_fill
                    else:
                        cell.fill = cell_fill

                    cell.border = border

                    # Lógica de alineación
                    header_cell = ws.cell(row=1, column=col_idx)
                    header_val = header_cell.value if header_cell else None
                    if header_val and any(keyword in str(header_val) for keyword in ['Nombre', 'Codigo', 'CodCli']):
                        cell.alignment = align_left
                    else:
                        cell.alignment = align_center
                        if header_val and any(coord in str(header_val) for coord in ['Longitud', 'Latitud']):
                            cell.number_format = '0.000000'

            # Auto-ajustar columnas
            for col_idx, column_cells in enumerate(ws.columns, 1):
                max_length = 0
                header_text = str(ws.cell(row=1, column=col_idx).value).lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰 ')
                max_length = len(header_text)
                for i, cell in enumerate(column_cells):
                    if i == 0: continue
                    if i > 50: break
                    try:
                        if cell.value is not None:
                            if isinstance(cell.value, (int, float)) and cell.number_format and cell.number_format != 'General':
                                if '0.000000' in cell.number_format: cell_text = f"{cell.value:.6f}"
                                else: cell_text = str(cell.value)
                            else: cell_text = str(cell.value)
                            lines = cell_text.split('\n')
                            max_line_length = max(len(line) for line in lines) if lines else 0
                            max_length = max(max_length, max_line_length)
                    except: pass
                adjusted_width = min(max(12, max_length + 3), 45)
                ws.column_dimensions[get_column_letter(col_idx)].width = adjusted_width

        wb.save(excel_path)
    except Exception as e:
        print(f"       ⚠️ Error al aplicar formato al Excel '{excel_path}': {e}")

print("✅ CELDA 1: Inicialización y funciones (Alfa 5.1) cargadas.")
print(f"📁 Directorio de salida: {output_dir}")
print(f"🎯 Icono MET: {'✅ Encontrado' if os.path.exists(icon_met_path) else '⚠️ No encontrado'}")

Versión 5.1 26/11/2025

In [ ]:
#=========== 1. INICIALIZACIÓN Y FUNCIONES BASE (Alfa 5.1) ==================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import os
import glob
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from shapely.geometry import Polygon, MultiPoint, Point # Importar Point y MultiPoint
from sklearn.cluster import KMeans
import shutil # Importar shutil para limpieza
import zipfile # Importar zipfile para Celda 3

# --- Validar rutas locales ---
icon_met_path = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png"
output_dir = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output"

# Crear directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(icon_met_path):
    print(f"⚠️ Advertencia: No se encontró el icono en {icon_met_path}. Se usará icono por defecto.")
else:
    print(f"✅ Icono de MET encontrado: {icon_met_path}")


# --- GLOBALES DEL ESTADO ---
execution_count = 0
df_clientes = pd.DataFrame()
df_cdr = pd.DataFrame()
met_checkboxes = []
# --- GLOBALES PARA INTERCAMBIO DE MÓDULOS ---
g_map_path = ""
g_excel_paths = []
g_total_clientes_procesados = 0
g_descarga_en_progreso = False # Para el bloqueo de descarga

# --- LISTA DE COLORES GLOBAL ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# =========================================================================
# === FUNCIONES HELPER (LÓGICA) ===========================================
# =========================================================================

# --- HELPER FUNCTION PARA NORMALIZAR IDS ---
def normalizar_id(valor):
    """Convierte un valor a un ID seguro para HTML/JS."""
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    s = s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    s = s.replace('Á','A').replace('É','E').replace('Í','I').replace('Ó','O').replace('Ú','U')
    s = s.replace('ñ','n').replace('Ñ','N')
    return s

# --- FUNCIÓN HELPER PARA VORONOI ---
def _clip_voronoi(vor, bounding_poly):
    """Recorta los polígonos de Voronoi a un polígono de frontera (shapely)."""
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty:
        return clipped_polys

    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points): continue
        if not (0 <= region_index < len(vor.regions)): continue
        region_vertices_indices = vor.regions[region_index]
        if -1 in region_vertices_indices: continue
        valid_vertex_indices = [v for v in region_vertices_indices if 0 <= v < len(vor.vertices)]
        if len(valid_vertex_indices) < 3: continue
        region_vertices = [vor.vertices[v] for v in valid_vertex_indices]
        if len(region_vertices) < 3: continue

        try:
            poly = Polygon(region_vertices)
            if not poly.is_valid: poly = poly.buffer(0)
            if not poly.is_valid: continue

            clipped = poly.intersection(bounding_poly)

            if clipped.is_empty: continue
            elif clipped.geom_type == 'Polygon' and clipped.exterior is not None:
                if len(clipped.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in clipped.exterior.coords]
                else: continue
            elif clipped.geom_type == 'MultiPolygon':
                if not clipped.geoms: continue
                largest_poly = max(clipped.geoms, key=lambda p: p.area)
                if largest_poly.exterior is not None and len(largest_poly.exterior.coords) >= 4:
                    clipped_coords = [[y, x] for x, y in largest_poly.exterior.coords]
                else: continue
            else: continue

            centroid_key = tuple(vor.points[i])
            clipped_polys[centroid_key] = clipped_coords
        except Exception as e:
            pass

    return clipped_polys

# --- FUNCIÓN HELPER PARA TAMAÑO DE MARCADOR (DE CÓDIGO CELAYA) ---
def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    """
    Calcula el tamaño del marcador basado en el volumen.
    Usa escala logarítmica para mejor diferenciación.
    """
    if vol_max == vol_min or volumen <= 0:
        return tamano_min

    # Usar escala logarítmica para mejor diferenciación visual
    log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)

    if log_max == log_min:
        return tamano_min

    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# =========================================================================
# === MÓDULO 1: LÓGICA DE CARGA Y VALIDACIÓN ==============================
# =========================================================================
def handle_upload(change, met_box_widget, output_upload_widget):
    global df_clientes, df_cdr, met_checkboxes

    # Suppress FutureWarning for pandas replace method downcasting
    pd.set_option('future.no_silent_downcasting', True)

    met_box = met_box_widget
    output_upload = output_upload_widget

    with output_upload:
        clear_output()
        upload_widget = change['owner']
        if not upload_widget.value:
            return

        # Adaptado para manejar diferentes formatos de FileUpload
        try:
            # Intentar obtener el archivo del widget
            if isinstance(upload_widget.value, tuple) and len(upload_widget.value) > 0:
                # Formato: tuple de FileInfo objects
                uploaded_file_info = upload_widget.value[0]
                uploaded_file_name = uploaded_file_info['name']
                uploaded_file_content = uploaded_file_info['content']
            elif isinstance(upload_widget.value, dict):
                # Formato: dict con nombres de archivo como keys
                uploaded_file_details = next(iter(upload_widget.value.values()))
                uploaded_file_name = uploaded_file_details['metadata']['name']
                uploaded_file_content = uploaded_file_details['content']
            else:
                raise ValueError("Formato de archivo no reconocido")
        except Exception as e:
            print(f"❌ Error al procesar el archivo cargado: {e}")
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
            return

        try:
            with open(uploaded_file_name, 'wb') as f: f.write(uploaded_file_content)
            df_excel = pd.read_excel(uploaded_file_name, sheet_name=None)

            if len(df_excel) < 2:
                raise ValueError("El archivo Excel debe contener al menos dos hojas (Clientes y METs/CDRs).")

            sheet1_name, sheet2_name = list(df_excel.keys())[0], list(df_excel.keys())[1]
            df_clientes_raw, df_cdr_raw = df_excel[sheet1_name], df_excel[sheet2_name]

            required_cols_clientes = ['CodCli', 'U_longitud', 'U_latitud', 'm3/día', 'entregas/día']
            required_cols_cdr = ['CodMET', 'U_longitud', 'U_latitud']

            missing_cols_cli = [col for col in required_cols_clientes if col not in df_clientes_raw.columns]
            missing_cols_cdr = [col for col in required_cols_cdr if col not in df_cdr_raw.columns]

            if missing_cols_cli:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet1_name}': {', '.join(missing_cols_cli)}")
            if missing_cols_cdr:
                raise ValueError(f"Faltan columnas requeridas en la hoja '{sheet2_name}': {', '.join(missing_cols_cdr)}")

            df_clientes_raw['U_longitud'] = pd.to_numeric(df_clientes_raw['U_longitud'], errors='coerce')
            df_clientes_raw['U_latitud'] = pd.to_numeric(df_clientes_raw['U_latitud'], errors='coerce')

            # Convertir m3/día: reemplazar "-" por 0 y convertir a numérico
            df_clientes_raw['m3/día'] = df_clientes_raw['m3/día'].replace('-', 0)
            df_clientes_raw['m3/día'] = pd.to_numeric(df_clientes_raw['m3/día'], errors='coerce').fillna(0)

            # Convertir entregas/día: reemplazar "-" por 0 y convertir a numérico (valores entre 0 y 1)
            df_clientes_raw['entregas/día'] = df_clientes_raw['entregas/día'].replace('-', 0)
            df_clientes_raw['entregas/día'] = pd.to_numeric(df_clientes_raw['entregas/día'], errors='coerce').fillna(0)

            df_cdr_raw['U_longitud'] = pd.to_numeric(df_cdr_raw['U_longitud'], errors='coerce')
            df_cdr_raw['U_latitud'] = pd.to_numeric(df_cdr_raw['U_latitud'], errors='coerce')

            df_clientes = df_clientes_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()
            df_cdr = df_cdr_raw.dropna(subset=['U_longitud', 'U_latitud']).copy()

            num_dropped_cli = len(df_clientes_raw) - len(df_clientes)
            num_dropped_cdr = len(df_cdr_raw) - len(df_cdr)
            if num_dropped_cli > 0: print(f"⚠️ Se eliminaron {num_dropped_cli} clientes por coordenadas inválidas.")
            if num_dropped_cdr > 0: print(f"⚠️ Se eliminaron {num_dropped_cdr} METs/CDRs por coordenadas inválidas.")

            print(f"✅ Archivo '{uploaded_file_name}' cargado exitosamente.")
            print(f"📊 Clientes válidos: {df_clientes.shape[0]} filas")
            print(f"📊 METs/CDRs válidos: {df_cdr.shape[0]} filas")

            mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
            met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
            met_box.children = [widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes

        except ValueError as ve:
            print(f"❌ Error de validación: {ve}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error en formato de archivo.')]
        except Exception as e:
            print(f"❌ Error inesperado al cargar: {e}")
            df_clientes, df_cdr, met_checkboxes = pd.DataFrame(), pd.DataFrame(), []
            met_box.children = [widgets.Label('❌ Error al cargar archivo.')]
        finally:
            if 'uploaded_file_name' in locals() and os.path.exists(uploaded_file_name):
                os.remove(uploaded_file_name)

# =========================================================================
# === MÓDULO 3: LÓGICA DE CLUSTERING (Alfa 5.1) ==========================
# =========================================================================
def select_customers_for_met(df_all_clients, met_lat, met_lon, n_clients, analyze_all):
    # Esta función se mantiene por si se usa en el futuro,
    # pero la lógica de 'Territorios' (Fase 0) en Módulo 2 la reemplaza.
    if df_all_clients.empty: return pd.DataFrame()
    if analyze_all:
        return df_all_clients.copy()
    else:
        distances = np.hypot(df_all_clients['U_latitud'] - met_lat, df_all_clients['U_longitud'] - met_lon)
        df_with_dist = df_all_clients.copy()
        df_with_dist['Distancia_al_MET'] = distances
        n_to_select = min(n_clients, len(df_with_dist))
        if n_to_select <= 0: return pd.DataFrame()
        return df_with_dist.nsmallest(n_to_select, 'Distancia_al_MET')

def calcular_dispersion_cluster(df_cluster):
    if len(df_cluster) < 2:
        return 0.0
    coords = df_cluster[['U_longitud', 'U_latitud']].values
    distancias_pares = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = np.hypot(coords[i][0] - coords[j][0], coords[i][1] - coords[j][1])
            distancias_pares.append(dist)
    return np.mean(distancias_pares) if distancias_pares else 0.0

def calcular_limites_clientes_dinamico(dispersion, frecuencia_visitas_promedio, min_base=20, max_base=40):
    """
    Ajusta dinámicamente el límite de clientes por ruta basándose en:
    - dispersion: Dispersión geográfica de la ruta (distancia promedio entre clientes)
    - frecuencia_visitas_promedio: Frecuencia promedio de entregas/día (0-1, donde 1 = 100%)

    A mayor dispersión geográfica → menos clientes permitidos
    A MAYOR frecuencia de visitas → MÁS clientes permitidos (rutas densas y frecuentes)
    """
    dispersion_normalizada = min(dispersion / 0.5, 1.0)

    # Normalizar frecuencia de visitas (ya viene en escala 0-1, donde 0.9692 = 96.92%)
    # Mayor frecuencia = MENOS restricción (más clientes permitidos para rutas frecuentes)
    frecuencia_normalizada = min(frecuencia_visitas_promedio, 1.0)

    # Factor de restricción: Solo la dispersión restringe, la frecuencia AUMENTA el límite
    # Invertimos el efecto de la frecuencia: alta frecuencia reduce la restricción
    factor_restriccion = dispersion_normalizada * (1.0 - frecuencia_normalizada * 0.5)

    rango = max_base - min_base
    ajuste = int(rango * factor_restriccion)
    max_dinamico = max_base - ajuste
    min_dinamico = max(min_base - int(ajuste * 0.5), int(min_base * 0.5))
    return min_dinamico, max_dinamico

def segregar_por_frecuencia(df_clients, threshold=0.01):
    """
    Separa clientes en elegibles y comodines ANTES del clustering.
    
    Parámetros:
        df_clients: DataFrame con todos los clientes (sin columna Ruta_nueva aún)
        threshold: Umbral de frecuencia (default: 0.01 = 1%)
    
    Retorna:
        df_elegibles: entregas/día > threshold
        df_comodines: entregas/día ≤ threshold
    """
    # Inicializar columna Ruta_nueva si no existe
    if 'Ruta_nueva' not in df_clients.columns:
        df_clients['Ruta_nueva'] = -1
    
    # Filtrar solo clientes no asignados a MET (Ruta_nueva == -1)
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    
    print(f"  📊 Segregación por frecuencia (threshold={threshold*100:.1f}%):")
    print(f"     ✅ Elegibles (>{threshold*100:.1f}%): {len(df_elegibles)} clientes")
    print(f"     🔺 Comodines (≤{threshold*100:.1f}%): {len(df_comodines)} clientes")
    
    return df_elegibles, df_comodines

def asignar_comodines_post_clustering(df_comodines, df_elegibles_asignados):
    """
    Asigna comodines al vecino más cercano SIN afectar límites.
    
    Parámetros:
        df_comodines: Clientes con entregas/día ≤ 1%
        df_elegibles_asignados: Clientes normales ya clusterizados
    
    Retorna:
        df_comodines con columna 'Ruta_nueva' asignada
    """
    from scipy.spatial.distance import cdist
    
    if len(df_comodines) == 0:
        print("  ✅ No hay comodines para asignar")
        return df_comodines
    
    print(f"  🔺 Asignando {len(df_comodines)} comodines (método: vecino más cercano)...")
    
    coords_comodines = df_comodines[['U_longitud', 'U_latitud']].values
    coords_elegibles = df_elegibles_asignados[['U_longitud', 'U_latitud']].values
    
    # Calcular distancias entre comodines y elegibles
    dist_matrix = cdist(coords_comodines, coords_elegibles, metric='euclidean')
    
    # Para cada comodín, encontrar el elegible más cercano
    idx_vecinos = np.argmin(dist_matrix, axis=1)
    rutas_asignadas = df_elegibles_asignados.iloc[idx_vecinos]['Ruta_nueva'].values
    
    df_comodines['Ruta_nueva'] = rutas_asignadas
    df_comodines['es_comodin'] = True  # Bandera para identificación
    
    # Estadísticas
    print(f"     ✅ {len(df_comodines)} comodines asignados a {len(np.unique(rutas_asignadas))} rutas")
    
    return df_comodines

def find_initial_seeds(coords, k, random_state=42):
    n_puntos = coords.shape[0]
    if k <= 0 or n_puntos == 0:
        print("       ⚠️ No se pueden generar semillas (k<=0 o no hay puntos).")
        return np.array([])
    if n_puntos < k:
        print(f"       ⚠️ No se pueden encontrar {k} semillas con {n_puntos} puntos. Ajustando k a {n_puntos}.")
        k = n_puntos
    try:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=20, init='k-means++').fit(coords)
        return kmeans.cluster_centers_
    except ValueError as e:
        print(f"       ❌ Error en KMeans para semillas (k={k}, n_puntos={n_puntos}): {e}")
        if n_puntos > 0:
            indices = np.random.choice(n_puntos, min(k, n_puntos), replace=False)
            print(f"       ... Usando {len(indices)} puntos aleatorios como semillas (Fallback).")
            return coords[indices]
        else:
            return np.array([])

# --- (MODIFICADO) FUNCIÓN PARA ENCONTRAR ANCLAS DE FRECUENCIA ---
def find_frequency_anchors(df_clients, percentile=80):
    """Encuentra los clientes 'ancla' (ej. 20% con mayor frecuencia de entregas/día)"""
    # No tomar en cuenta clientes con frecuencia muy baja o cero
    df_with_frequency = df_clients[df_clients['entregas/día'] > 0.01]
    if df_with_frequency.empty:
        print("       ... No se encontraron clientes con frecuencia significativa para anclaje.")
        return pd.DataFrame() # No hay anclas

    # Usar un umbral de percentil para definir "alta frecuencia"
    min_frequency_threshold = np.percentile(df_with_frequency['entregas/día'], percentile)
    df_anchors = df_with_frequency[df_with_frequency['entregas/día'] >= min_frequency_threshold].copy()

    print(f"       ... Umbral de anclaje (percentil {percentile}%) = {min_frequency_threshold*100:.1f}% entregas/día")
    print(f"       ... {len(df_anchors)} clientes 'ancla' de alta frecuencia identificados.")
    return df_anchors

# --- (MODIFICADO) LÓGICA DE CRECIMIENTO HÍBRIDA (Alfa 5.1) ---
def grow_clusters_from_seeds(df_elegibles_only, initial_seeds, max_size, min_size=20):
    """
    Asigna SOLO clientes elegibles (>1% frecuencia) a rutas.
    Los comodines (≤1%) se asignan DESPUÉS con asignar_comodines_post_clustering().
    
    Estrategia HÍBRIDA (Alfa 5.1):
    1. Fase A: Anclar rutas con clientes de alta frecuencia
    2. Fase B: Rellenar rutas con lógica 'greedy'
    
    Parámetros:
        df_elegibles_only: Solo clientes con entregas/día > 0.01
        initial_seeds: Coordenadas de centroides iniciales
        max_size: Límite máximo de clientes normales por ruta
        min_size: Límite mínimo de clientes normales por ruta
    """
    k = len(initial_seeds)
    n_clientes_total = len(df_elegibles_only)
    if k == 0 or n_clientes_total == 0:
        print("       ⚠️ No hay semillas o clientes elegibles para 'grow_clusters_from_seeds'.")
        df_elegibles_only['Ruta_nueva'] = -1
        return df_elegibles_only, {}

    # --- Inicialización (Lógica Alfa) ---
    df_clients_to_cluster = df_elegibles_only.copy()
    df_clients_to_cluster['Ruta_nueva'] = -1
    df_clients_to_cluster['es_comodin'] = False  # Todos los elegibles NO son comodines
    ruta_volumenes = {i: 0.0 for i in range(k)}
    ruta_counts = {i: 0 for i in range(k)}

    # --- FASE A: ANCLAJE POR FRECUENCIA (MODIFICADO) ---
    print(f"       🔬 Fase A: Anclando {k} rutas con clientes de alta frecuencia (entregas/día)...")
    df_anchors = find_frequency_anchors(df_clients_to_cluster)
    n_asignados_anchors = 0

    if not df_anchors.empty:
        coords_anchors = df_anchors[['U_longitud', 'U_latitud']].values
        dist_matrix_anchors = distance_matrix(coords_anchors, initial_seeds)

        # Iterar por cada ancla
        for i, (anchor_idx, row) in enumerate(df_anchors.iterrows()):
            frecuencia_ancla = row['entregas/día']
            volumen_ancla = row['m3/día']
            rutas_ordenadas = np.argsort(dist_matrix_anchors[i]) # Rutas más cercanas primero

            asignado = False
            for ruta_idx in rutas_ordenadas:
                # Validar solo por límite de clientes (sin restricción de volumen)
                # Para la Fase de Anclaje, usamos un límite estático ya que la ruta está vacía
                if ruta_counts[ruta_idx] < max_size:
                    df_clients_to_cluster.loc[anchor_idx, 'Ruta_nueva'] = ruta_idx
                    ruta_volumenes[ruta_idx] += volumen_ancla
                    ruta_counts[ruta_idx] += 1
                    n_asignados_anchors += 1
                    asignado = True
                    break # Ancla asignada, pasar a la siguiente ancla

            if not asignado:
                print(f"       ⚠️ Ancla {anchor_idx} (Frec: {frecuencia_ancla*100:.1f}%) no pudo ser asignada (todas las rutas están llenas).")

    print(f"       ✅ Fase A (Anclaje) completada: {n_asignados_anchors}/{len(df_anchors)} anclas asignadas.")

    # --- FASE B: RELLENO GEOMÉTRICO (Lógica "Redonda" de Beta) ---
    df_normales_restantes = df_clients_to_cluster[
        (df_clients_to_cluster['entregas/día'] > 0.01) &  # Frecuencia > 1%
        (df_clients_to_cluster['Ruta_nueva'] == -1)
    ].copy()

    n_normales = len(df_normales_restantes)
    n_asignados_normales = 0

    if n_normales > 0:
        print(f"       🔬 Fase B: Rellenando geométricamente {n_normales} clientes restantes...")
        coords_normales = df_normales_restantes[['U_longitud', 'U_latitud']].values

        try:
            dist_matrix = distance_matrix(coords_normales, initial_seeds)
        except ValueError as e:
            print(f"       ❌ Error calculando matriz de distancias: {e}")
            dist_matrix = np.array([[np.hypot(c[0]-s[0], c[1]-s[1]) for s in initial_seeds] for c in coords_normales])

        dist_df = pd.DataFrame(dist_matrix, index=df_normales_restantes.index, columns=range(k))

        iteracion = 0
        max_iter = n_normales * k # Límite de seguridad

        while n_asignados_normales < n_normales and iteracion < max_iter:
            iteracion += 1
            min_dist = dist_df.min().min()
            if min_dist == np.inf:
                break

            stacked = dist_df.stack()
            best_pair_idx = stacked[stacked == min_dist].index
            client_idx, ruta_idx = best_pair_idx[0]

            # --- Validar con límites dinámicos (sin restricción de 6m³) ---
            volumen_cliente = df_normales_restantes.loc[client_idx, 'm3/día']
            # volumen_futuro = ruta_volumenes[ruta_idx] + volumen_cliente # Volumen ya no afecta la asignación

            clientes_ruta_actual = df_clients_to_cluster[df_clients_to_cluster['Ruta_nueva'] == ruta_idx]
            dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

            # Calcular frecuencia promedio de visitas de la ruta (incluyendo el nuevo cliente)
            if len(clientes_ruta_actual) > 0:
                frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
            else:
                frecuencia_ruta_actual = 0.0

            # Estimar frecuencia futura (promedio ponderado)
            frecuencia_cliente = df_normales_restantes.loc[client_idx, 'entregas/día']
            if len(clientes_ruta_actual) > 0:
                frecuencia_futura = ((frecuencia_ruta_actual * len(clientes_ruta_actual)) + frecuencia_cliente) / (len(clientes_ruta_actual) + 1)
            else:
                frecuencia_futura = frecuencia_cliente

            min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                dispersion_actual, frecuencia_futura, min_base=min_size, max_base=max_size
            )

            # Solo validar límite de clientes (SIN límite de volumen)
            if ruta_counts[ruta_idx] < max_dinamico:
                # Éxito: Asignar
                df_clients_to_cluster.loc[client_idx, 'Ruta_nueva'] = ruta_idx
                ruta_volumenes[ruta_idx] += volumen_cliente # Actualizar volumen para resumen, pero no para decisión
                ruta_counts[ruta_idx] += 1
                n_asignados_normales += 1
                dist_df.loc[client_idx, :] = np.inf # Cliente asignado - marca TODAS las distancias como infinitas
            else:
                # Fallo: Invalidar solo este par cliente-ruta
                dist_df.loc[client_idx, ruta_idx] = np.inf

        if iteracion >= max_iter:
            print(f"       ⚠️ Se alcanzó el límite de iteraciones ({max_iter}).")

        print(f"       ✅ Fase B completada: {n_asignados_normales}/{n_normales} clientes elegibles asignados.")

    # --- FIN: Retornar solo elegibles asignados ---
    # Los comodines se asignan DESPUÉS con asignar_comodines_post_clustering()
    return df_clients_to_cluster, ruta_volumenes

# --- (MODIFICADO) LÓGICA DE AJUSTE CON GEOMETRÍA (Alfa 5.1) ---
def adjust_small_clusters(df_clustered, cluster_counts, min_size, max_size):
    """
    Ajusta rutas pequeñas (< min_size).
    Reasigna clientes al POLÍGONO válido más cercano (no al centroide).
    (MODIFICADO: Usa frecuencia de visitas en lugar de volumen para dispersión)
    """
    print("       🔬 Fase D: Ajustando rutas pequeñas...")
    df_adjusted = df_clustered.copy()

    # Bucle de reintentos
    attempts = 0
    max_attempts = 3 # Limitar los pases de ajuste
    while attempts < max_attempts:
        attempts += 1

        # --- 1. Calcular volúmenes y conteos actuales ---
        ruta_volumenes = {}
        ruta_counts_actual = {}
        for ruta_id in df_adjusted['Ruta_nueva'].unique():
            if ruta_id < 0: continue
            clientes_ruta = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            ruta_volumenes[ruta_id] = clientes_ruta['m3/día'].sum()
            ruta_counts_actual[ruta_id] = len(clientes_ruta)

        if not ruta_counts_actual:
            print("       ... No hay rutas que ajustar.")
            break # No hay nada que hacer

        # --- 2. Identificar rutas VÁLIDAS vs. RUTAS A MOVER ---
        rutas_validas_ids = set()
        rutas_a_mover_ids = set()

        for ruta_id, count in ruta_counts_actual.items():
            # Regla de negocio: Solo Pequeña (sin criterio de rentabilidad)
            if count < min_size:
                rutas_a_mover_ids.add(ruta_id)
            else:
                rutas_validas_ids.add(ruta_id)

        if not rutas_a_mover_ids:
            print(f"       ✅ Fase D completada: No hay rutas pequeñas. (Intento {attempts})")
            break # Éxito, no hay nada que ajustar

        if not rutas_validas_ids:
            print(f"       ⚠️ Fase D: Hay {len(rutas_a_mover_ids)} rutas pequeñas, pero NO hay rutas válidas para moverlas. Deteniendo.")
            break

        # --- 3. Crear Geometrías Válidas (Tu idea) ---
        geometrias_validas = {}
        for ruta_id in rutas_validas_ids:
            clientes_validos = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id]
            points = clientes_validos[['U_longitud', 'U_latitud']].values
            if len(points) >= 3:
                try:
                    # Usamos ConvexHull para obtener el *área* de la ruta
                    hull = ConvexHull(points)
                    geometrias_validas[ruta_id] = Polygon(points[hull.vertices])
                except Exception as e:
                    pass # Ignorar esta ruta si ConvexHull falla

        if not geometrias_validas:
            print(f"       ⚠️ Fase D: No se pudieron crear polígonos para las rutas válidas. Deteniendo.")
            break

        # --- 4. Iterar y Reasignar Clientes "Huérfanos" ---
        df_clientes_a_mover = df_adjusted[df_adjusted['Ruta_nueva'].isin(rutas_a_mover_ids)].copy()
        print(f"       ... Intento {attempts}: Moviendo {len(df_clientes_a_mover)} clientes de {len(rutas_a_mover_ids)} rutas...")

        clientes_reasignados_en_intento = 0

        for idx_cliente, row in df_clientes_a_mover.iterrows():
            punto_cliente = Point(row['U_longitud'], row['U_latitud'])
            volumen_cliente = row['m3/día']
            original_ruta = row['Ruta_nueva']

            # Calcular distancia a TODOS los polígonos válidos
            distancias_a_poligonos = []
            for ruta_id, poly in geometrias_validas.items():
                dist = punto_cliente.distance(poly)
                distancias_a_poligonos.append((dist, ruta_id))

            if not distancias_a_poligonos:
                continue # No hay polígonos válidos

            distancias_a_poligonos.sort() # Ordenar por distancia (más cercano primero)

            # Intentar asignar al más cercano que cumpla las reglas
            asignado = False
            for dist, ruta_id_destino in distancias_a_poligonos:
                # Validar con Lógica Alfa (MODIFICADO: usa frecuencia)
                # volumen_actual = ruta_volumenes.get(ruta_id_destino, 0)
                # volumen_futuro = volumen_actual + volumen_cliente

                # if volumen_futuro > 6.0:
                #     continue # Excede 6m³, probar siguiente polígono

                clientes_ruta_actual = df_adjusted[df_adjusted['Ruta_nueva'] == ruta_id_destino]
                dispersion_actual = calcular_dispersion_cluster(clientes_ruta_actual)

                # Calcular frecuencia promedio futura (MODIFICADO)
                if len(clientes_ruta_actual) > 0:
                    frecuencia_ruta_actual = clientes_ruta_actual['entregas/día'].mean()
                else:
                    frecuencia_ruta_actual = 0.0

                frecuencia_cliente = row['entregas/día']
                if len(clientes_ruta_actual) > 0:
                    frecuencia_futura = ((frecuencia_ruta_actual * len(clientes_ruta_actual)) + frecuencia_cliente) / (len(clientes_ruta_actual) + 1)
                else:
                    frecuencia_futura = frecuencia_cliente

                min_dinamico, max_dinamico = calcular_limites_clientes_dinamico(
                    dispersion_actual, frecuencia_futura, min_base=min_size, max_base=max_size
                )

                if ruta_counts_actual.get(ruta_id_destino, 0) < max_dinamico:
                    # ¡Éxito! Reasignar
                    df_adjusted.loc[idx_cliente, 'Ruta_nueva'] = ruta_id_destino

                    # Actualizar contadores para la siguiente iteración
                    ruta_counts_actual[ruta_id_destino] = ruta_counts_actual.get(ruta_id_destino, 0) + 1
                    ruta_volumenes[ruta_id_destino] = ruta_volumenes.get(ruta_id_destino, 0) + volumen_cliente # Actualizar volumen para resumen, no para decisión

                    ruta_counts_actual[original_ruta] -= 1
                    ruta_volumenes[original_ruta] = max(0, ruta_volumenes[original_ruta] - volumen_cliente)

                    clientes_reasignados_en_intento += 1
                    asignado = True
                    break # Cliente asignado, pasar al siguiente cliente

            # (No hay 'fallback' forzado. Si no cabe, no cabe)

        if clientes_reasignados_en_intento == 0:
            print("       ... No se pudieron reasignar más clientes en este intento. Deteniendo ajuste.")
            break

    if attempts >= max_attempts:
        print(f"       ⚠️ Fase D: Se alcanzó el límite de {max_attempts} intentos. Pueden quedar rutas pequeñas.")

    # Devolver el DF ajustado y el último conteo conocido
    return df_adjusted, ruta_counts_actual


# =========================================================================
# === MÓDULO 4: CÁLCULO DE POLÍGONOS (Sin cambios) ========================
# =========================================================================
def calculate_cluster_polygons(df_clustered, method='voronoi'):
    # Esta función se usa solo para la VISUALIZACIÓN FINAL
    polygons = {}
    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])
    if not assigned_ruta_ids: return polygons

    centroides_rutas = []
    ruta_id_map = {}
    for ruta_id in assigned_ruta_ids:
        ruta_clientes_df = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if not ruta_clientes_df.empty:
            lon = pd.to_numeric(ruta_clientes_df['U_longitud'], errors='coerce').mean()
            lat = pd.to_numeric(ruta_clientes_df['U_latitud'], errors='coerce').mean()
            if pd.notna(lon) and pd.notna(lat):
                centroides_rutas.append([lon, lat])
                ruta_id_map[tuple([lon, lat])] = ruta_id

    poligonos_voronoi_recortados = {}
    vor = None
    if method == 'voronoi' and len(centroides_rutas) >= 4:
        try:
            todos_los_puntos = df_clustered[df_clustered['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
            if len(todos_los_puntos) >= 3:
                boundary_poly_shapely = MultiPoint(todos_los_puntos).convex_hull.buffer(0).buffer(0.0001)
                unique_centroids = np.unique(np.array(centroides_rutas), axis=0)
                if len(unique_centroids) >= 4:
                    vor = Voronoi(unique_centroids, qhull_options='Qbb Qc Qz')
                    poligonos_voronoi_recortados = _clip_voronoi(vor, boundary_poly_shapely)
                else: print("       ... No hay suficientes centroides únicos (>=4) para Voronoi.")
            else: print("       ... No hay suficientes puntos asignados (<3) para definir frontera Voronoi.")
        except Exception as e:
            print(f"       ❌ Error al calcular Voronoi: {e}. Se usará ConvexHull.")
            poligonos_voronoi_recortados = {}
    elif method == 'voronoi':
        print(f"       ... No hay suficientes rutas (se necesitan >= 4, se encontraron {len(centroides_rutas)}) para Voronoi. Usando ConvexHull.")

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id]
        if ruta_clientes.empty: continue

        # Encontrar el polígono de Voronoi que corresponde a este centroide
        centroide_lon = ruta_clientes['U_longitud'].mean()
        centroide_lat = ruta_clientes['U_latitud'].mean()
        poly_coords = poligonos_voronoi_recortados.get(tuple([centroide_lon, centroide_lat]))

        if not poly_coords and poligonos_voronoi_recortados:
            # Fallback por si hay errores de precisión de float
            min_dist = float('inf')
            best_match = None
            current_centroid = np.array([centroide_lon, centroide_lat])
            for vor_centroid_key, vor_poly in poligonos_voronoi_recortados.items():
                dist = np.linalg.norm(current_centroid - np.array(vor_centroid_key))
                if dist < min_dist:
                    min_dist = dist
                    best_match = vor_poly
            if min_dist < 1e-6: poly_coords = best_match

        if poly_coords:
            polygons[ruta_id] = poly_coords
        else:
            # Fallback a ConvexHull si Voronoi falló o no se usó
            points = ruta_clientes[['U_latitud', 'U_longitud']].values
            if len(points) >= 3:
                try:
                    hull = ConvexHull(points)
                    polygons[ruta_id] = points[hull.vertices].tolist()
                except Exception:
                    print(f"       ⚠️ No se pudo generar polígono ConvexHull para Ruta {ruta_id}.")
                    polygons[ruta_id] = None
            else: polygons[ruta_id] = None
    return polygons

# =========================================================================
# === MÓDULO 5: GENERACIÓN DE MAPA (Sin Rentabilidad - Solo Frecuencia) ==
# =========================================================================
def create_map_elements(map_instance, df_clustered, cluster_polygons, colores_list,
                        vol_min_global, vol_max_global,
                        met_name=None):
    """
    Dibuja los elementos del cluster y los AÑADE a las capas de filtrado.
    Colores normales para todas las rutas (sin lógica de rentabilidad).
    """

    if met_name is None and 'MET_Asignado' in df_clustered.columns:
        met_name = df_clustered['MET_Asignado'].iloc[0] if not df_clustered.empty else ""
    elif met_name is None:
        met_name = ""

    met_name_clean = met_name.strip()
    display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"

    assigned_ruta_ids = sorted([c for c in df_clustered['Ruta_nueva'].unique() if c >= 0])

    tamano_min, tamano_max = (6, 25)

    for ruta_id in assigned_ruta_ids:
        ruta_clientes = df_clustered[df_clustered['Ruta_nueva'] == ruta_id].copy()
        ruta_size = len(ruta_clientes)
        if ruta_size == 0: continue

        # --- Asignar color normal a todas las rutas ---
        volumen_total_ruta = ruta_clientes['m3/día'].sum()
        frecuencia_promedio_ruta = ruta_clientes['entregas/día'].mean() * 100  # Convertir a porcentaje
        
        # Contar clientes normales vs comodines
        n_normales = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == False])
        n_comodines = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == True])

        color_ruta = colores_list[ruta_id % len(colores_list)]
        tooltip_poligono = f"{display_name} - Ruta {ruta_id} | Normal: {n_normales} | Comodín: {n_comodines} | Vol: {volumen_total_ruta:,.2f} m³ | Frec: {frecuencia_promedio_ruta:.1f}%"
        nombre_capa = f"{display_name} - Ruta {ruta_id}"

        fg_ruta_individual = folium.FeatureGroup(name=nombre_capa, show=True)

        # Solo se usa el polígono de "solo clientes" (punteado)
        if len(ruta_clientes) >= 3:
            try:
                puntos_solo_clientes = [Point(row['U_longitud'], row['U_latitud']) for _, row in ruta_clientes.iterrows()]
                multipoint_clientes = MultiPoint(puntos_solo_clientes)
                hull_clientes = multipoint_clientes.convex_hull

                if hull_clientes.geom_type == 'Polygon':
                    coords_polygon_clientes = [(lat, lon) for lon, lat in hull_clientes.exterior.coords]
                    
                    # Contar clientes normales vs comodines
                    n_normales = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == False])
                    n_comodines = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == True])
                    
                    area_popup_clientes = f'''
                    <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                        <div style="font-size: 14px; font-weight: bold;">👥 Área Solo Clientes</div>
                        <div style="font-size: 12px; margin-top: 4px;">🏠 {met_name_clean} - 🛣️ Ruta {ruta_id}</div>
                        <div style="font-size: 11px; margin-top: 2px;">✅ Normales: {n_normales} | 🔺 Comodines: {n_comodines}</div>
                        <div style="font-size: 11px; margin-top: 2px;"><b>Total: {len(ruta_clientes)} clientes</b></div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📦 Vol: {volumen_total_ruta:,.2f} m³</div>
                        <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📊 Frec: {frecuencia_promedio_ruta:.1f}%</div>
                    </div>
                    '''

                    poly_clientes_obj_clon = folium.Polygon(
                        locations=coords_polygon_clientes,
                        color=color_ruta,
                        weight=1,
                        opacity=0.8,
                        fillColor=color_ruta,
                        fillOpacity=0.1,
                        popup=folium.Popup(area_popup_clientes, max_width=250),
                        tooltip=tooltip_poligono,
                        dash_array='5, 5'
                    )
                    poly_clientes_obj_clon.add_to(fg_ruta_individual)
            except Exception as e:
                print(f"       ⚠️ No se pudo generar polígono 'solo clientes' para Ruta {ruta_id}: {e}")

        for _, cliente_row in ruta_clientes.iterrows():
            volumen = cliente_row['m3/día']
            frecuencia = cliente_row['entregas/día']
            frecuencia_pct = frecuencia * 100  # Convertir a porcentaje
            tamano_marcador = obtener_tamano_marcador(volumen, vol_min_global, vol_max_global, tamano_min, tamano_max)

            codcli = cliente_row['CodCli']
            nombre = cliente_row.get('Nombre', 'N/A')
            ruta_original_val = cliente_row.get('Ruta', 'N/A')
            ruta_original = ruta_original_val if pd.notna(ruta_original_val) else 'N/A'
            
            # Detectar si es comodín
            es_comodin = cliente_row.get('es_comodin', False)

            # Popup con indicador de tipo de cliente
            popup_html = f'''
            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                    {'🔺 COMODÍN' if es_comodin else '✅ NORMAL'} | {codcli}
                </div>
                <div style="font-size:14px; color:#333; margin-bottom:4px;">
                    {nombre if pd.notna(nombre) else 'N/A'}
                </div>
                <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📦 Volumen: <b>{volumen:,.3f} m³</b>
                </div>

                <div style="font-size:14px; color:{'#888' if es_comodin else '#1976D2'}; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                    📊 Frecuencia: <b>{frecuencia_pct:.1f}%</b> (entregas/día)
                </div>

                <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                    🗺️ Ruta (Orig): <b>{ruta_original}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                </div>
            </div>
            '''

            # Tooltip más corto con frecuencia
            tooltip_text = f"{'🔺' if es_comodin else '✅'} {codcli} | Vol: {volumen:.2f} m³ | Frec: {frecuencia_pct:.1f}%"

            # Icono diferenciado: Triángulo rojo pequeño para comodines, CircleMarker normal para elegibles
            if es_comodin:
                # Triángulo rojo pequeño y discreto para comodines
                marker_obj_clon = folium.CircleMarker(
                    location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=tooltip_text,
                    radius=4,  # Pequeño y discreto
                    color='#D32F2F',  # Rojo oscuro
                    weight=1,
                    fillColor='#F44336',  # Rojo brillante
                    fillOpacity=0.7,
                )
            else:
                # CircleMarker normal para elegibles
                marker_obj_clon = folium.CircleMarker(
                    location=[cliente_row['U_latitud'], cliente_row['U_longitud']],
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=tooltip_text,
                    radius=tamano_marcador,
                    color='#333333',
                    weight=1,
                    fillColor=color_ruta,
                    fillOpacity=0.85,
                )
            marker_obj_clon.add_to(fg_ruta_individual)

        fg_ruta_individual.add_to(map_instance)
    return

# =========================================================================
# === MÓDULO 6: GENERACIÓN DE EXCEL (Sin Rentabilidad) ====================
# =========================================================================
def generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir):
    if not export_data or not summary_data:
        print(f"       ... No hay datos para exportar a Excel.")
        return None
    df_export = pd.DataFrame(export_data)
    df_resumen = pd.DataFrame(summary_data)
    if df_export.empty or df_resumen.empty:
        print(f"       ... DataFrames vacíos, no se puede generar Excel.")
        return None
    excel_filename = f'rutas_consolidadas_{timestamp}.xlsx'
    excel_path = os.path.join(output_dir, excel_filename)
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df_resumen_sorted = df_resumen.sort_values(by=['MET', 'Ruta_nueva'])
            df_resumen_sorted.to_excel(writer, sheet_name='Resumen General', index=False)
            df_export_sorted = df_export.sort_values(by=['MET', 'Ruta_nueva', 'Codigo'])
            df_export_sorted.to_excel(writer, sheet_name='Todos los Clientes', index=False)
            for met_name in mets_seleccionados:
                df_met = df_export[df_export['MET'] == met_name].sort_values(by=['Ruta_nueva', 'Codigo'])
                if not df_met.empty:
                    sheet_name = f'MET_{normalizar_id(met_name)}'[:31]
                    df_met.to_excel(writer, sheet_name=sheet_name, index=False)
        format_excel(excel_path)
        return excel_path
    except Exception as e:
        print(f"       ❌ Error al generar Excel consolidado: {e}")
        if os.path.exists(excel_path):
            try: os.remove(excel_path)
            except: pass
        return None

def format_excel(excel_path):
    """
    Aplica formato profesional al Excel de salida.
    (SIN lógica de rentabilidad - todos los formatos normales)
    """
    if not os.path.exists(excel_path):
        print(f"       ⚠️ No se encontró el archivo Excel para formatear: {excel_path}")
        return
    try:
        wb = openpyxl.load_workbook(excel_path)

        # --- Definición de Estilos ---
        header_font = Font(bold=True, color='FFFFFF', size=11)
        header_fill = PatternFill('solid', fgColor='4F81BD')
        cell_fill = PatternFill('solid', fgColor='DDEBF7')
        summary_fill = PatternFill('solid', fgColor='FFD966') # Amarillo para resumen

        border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'),
                        top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))
        align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
        align_left = Alignment(horizontal='left', vertical='center', wrap_text=False)

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws.max_row <= 1: continue

            # Formatear Encabezados
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = align_center
                cell.border = border
                if cell.value:
                    header_text = str(cell.value)
                    icon = ""
                    if 'Ruta_Original' in header_text: icon = '🗺️ '
                    elif 'Codigo' in header_text or 'CodCli' in header_text: icon = '👨‍💼 '
                    elif 'Clientes' in header_text: icon = '👥 '
                    elif 'MET' in header_text: icon = '🏠 '
                    elif 'Nombre' in header_text: icon = '📚 '
                    elif 'Ruta_nueva' in header_text or 'Ruta_ID' in header_text: icon = '🚚 '
                    elif 'Volumen' in header_text or 'm3' in header_text: icon = '📦 '
                    elif 'Centroide' in header_text: icon = '📍 '
                    elif 'Longitud' in header_text: icon = '📊 '
                    elif 'Latitud' in header_text: icon = '📊 '
                    elif 'Frecuencia' in header_text: icon = '📊 '
                    elif 'Tipo' in header_text and 'Cliente' in header_text: icon = '🏷️ '
                    elif 'Rentable' in header_text: icon = '💰 '
                    elif 'Comodines' in header_text or 'Comodín' in header_text: icon = '🔺 '
                    elif 'Normales' in header_text: icon = '✅ '
                    cleaned_header = header_text.lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰🏷️🔺✅ ')
                    cell.value = icon + cleaned_header

            # Formatear Celdas de Datos
            is_summary_sheet = (sheet_name == 'Resumen General')

            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for col_idx, cell in enumerate(row, start=1):
                    # Aplicar color de fondo normal
                    if is_summary_sheet:
                        cell.fill = summary_fill
                    else:
                        cell.fill = cell_fill

                    cell.border = border

                    # Lógica de alineación
                    header_cell = ws.cell(row=1, column=col_idx)
                    header_val = header_cell.value if header_cell else None
                    if header_val and any(keyword in str(header_val) for keyword in ['Nombre', 'Codigo', 'CodCli']):
                        cell.alignment = align_left
                    else:
                        cell.alignment = align_center
                        if header_val and any(coord in str(header_val) for coord in ['Longitud', 'Latitud']):
                            cell.number_format = '0.000000'

            # Auto-ajustar columnas
            for col_idx, column_cells in enumerate(ws.columns, 1):
                max_length = 0
                header_text = str(ws.cell(row=1, column=col_idx).value).lstrip('🗺️👨‍💼🏠📚📦📊👥📍💰 ')
                max_length = len(header_text)
                for i, cell in enumerate(column_cells):
                    if i == 0: continue
                    if i > 50: break
                    try:
                        if cell.value is not None:
                            if isinstance(cell.value, (int, float)) and cell.number_format and cell.number_format != 'General':
                                if '0.000000' in cell.number_format: cell_text = f"{cell.value:.6f}"
                                else: cell_text = str(cell.value)
                            else: cell_text = str(cell.value)
                            lines = cell_text.split('\n')
                            max_line_length = max(len(line) for line in lines) if lines else 0
                            max_length = max(max_length, max_line_length)
                    except: pass
                adjusted_width = min(max(12, max_length + 3), 45)
                ws.column_dimensions[get_column_letter(col_idx)].width = adjusted_width

        wb.save(excel_path)
    except Exception as e:
        print(f"       ⚠️ Error al aplicar formato al Excel '{excel_path}': {e}")

print("✅ CELDA 1: Inicialización y funciones (Alfa 5.1) cargadas.")
print(f"📁 Directorio de salida: {output_dir}")
print(f"🎯 Icono MET: {'✅ Encontrado' if os.path.exists(icon_met_path) else '⚠️ No encontrado'}")

In [ ]:
# ================== 2. EJECUCIÓN DEL ANÁLISIS (Alfa 5.1 - Muestreo y Territorios) ==================

# --- Definición de Widgets ---
upload_widget = widgets.FileUpload(accept='.xlsx', multiple=False, description='Subir Excel')
met_box = widgets.VBox([widgets.Label('⏳ Esperando archivo Excel...')])
output_upload = widgets.Output()
min_clientes_cluster = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Min Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))
max_clientes_cluster = widgets.IntSlider(value=40, min=10, max=100, step=1, description='Max Clientes:', continuous_update=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='48%'))

# --- (NUEVO) WIDGETS DE MUESTREO ---
sample_size_per_met = widgets.IntText(
    value=500, 
    description='Clientes a muestrear por MET:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='48%')
)
run_full_analysis = widgets.Checkbox(
    value=False, 
    description='✅ Analizar TODOS los clientes (Ignora el muestreo)', 
    indent=False, 
    layout=widgets.Layout(width='48%')
)
# --- FIN DE WIDGETS NUEVOS ---

param_button = widgets.Button(description='Ejecutar Análisis de Territorio y Rutas', button_style='success', disabled=False, layout=widgets.Layout(width='98%'))
output_param = widgets.Output()

# --- Registrar handler de carga ---
upload_widget.observe(lambda change: handle_upload(change, met_box, output_upload), names='value')

# --- Función Principal de Ejecución ---
def ejecutar_analisis(b):
    global execution_count, df_clientes, df_cdr, met_checkboxes, colores
    global g_map_path, g_excel_paths, g_total_clientes_procesados
    
    execution_count += 1
    
    with output_param:
        clear_output(wait=True)
        param_button.disabled = True # Bloquear botón
        
        # --- 1. Validaciones y Parámetros ---
        if df_clientes.empty or df_cdr.empty:
            print("❌ Carga un archivo Excel primero.")
            param_button.disabled = False
            return
        
        mets_seleccionados = [cb.description for cb in met_checkboxes if cb.value]
        if not mets_seleccionados or len(mets_seleccionados) < 2:
            print("❌ Selecciona al menos dos METs para optimizar territorios.")
            param_button.disabled = False
            return
        
        min_size = min_clientes_cluster.value
        max_size = max_clientes_cluster.value
        
        # --- (NUEVO) Obtener parámetros de muestreo ---
        n_muestra = sample_size_per_met.value
        analizar_todos = run_full_analysis.value
        
        if min_size >= max_size:
            print("❌ El mínimo debe ser menor que el máximo.")
            param_button.disabled = False
            return
        
        print(f"\n{'='*70}")
        print(f"🚀 INICIANDO ANÁLISIS DE TERRITORIO Y RUTAS - Ejecución #{execution_count}")
        print(f"{'='*70}")
        print(f"📊 Parámetros:")
        print(f"   • METs a optimizar: {len(mets_seleccionados)} ({', '.join(mets_seleccionados)})")
        print(f"   • Clientes por ruta: {min_size} - {max_size}")
        
        # Imprimir modo de ejecución
        if analizar_todos:
            print(f"   • MODO: ✅ COMPLETO (Se analizarán TODOS los clientes)")
        else:
            print(f"   • MODO: ⚠️ MUESTRA (Se analizarán {n_muestra} clientes por MET)")
            
        print(f"{'='*70}\n")
        
        # Limpiar resultados anteriores
        g_map_path = ""
        g_excel_paths = []
        g_total_clientes_procesados = 0
        
        # --- (NUEVO) GUARDAR RESPALDO DE CLIENTES ---
        df_clientes_original = df_clientes.copy()
        
        # --- 2. (NUEVO) FASE 0: ASIGNACIÓN DE TERRITORIOS ---
        try:
            print(f"🗺️ Fase 0: Asignando territorios (cliente al MET más cercano)...")
            
            mets_info = df_cdr[df_cdr['CodMET'].isin(mets_seleccionados)]
            if mets_info.empty:
                print("❌ No se encontraron datos para los METs seleccionados en la hoja de CDRs.")
                param_button.disabled = False
                return

            coords_clientes = df_clientes[['U_latitud', 'U_longitud']].values
            coords_mets = mets_info[['U_latitud', 'U_longitud']].values
            
            dist_matrix_territorio = distance_matrix(coords_clientes, coords_mets)
            indices_met_mas_cercano = np.argmin(dist_matrix_territorio, axis=1)
            nombres_met_mas_cercano = [mets_info['CodMET'].iloc[i] for i in indices_met_mas_cercano]
            
            df_clientes['MET_Asignado'] = nombres_met_mas_cercano
            
            print(f"✅ Territorios asignados. Conteo:")
            print(df_clientes['MET_Asignado'].value_counts())
            
            print("📊 Calculando rangos de volumen global para tamaño de marcadores...")
            volumenes_positivos = df_clientes[df_clientes['m3/día'] > 0]['m3/día']
            if not volumenes_positivos.empty:
                vol_min_global = volumenes_positivos.min()
                vol_max_global = volumenes_positivos.max()
            else:
                vol_min_global = 0
                vol_max_global = 0
            print(f"   • 📦 Rango de Volumen (m³/día): {vol_min_global:,.3f} - {vol_max_global:,.3f}")

            # --- 3. Crear Mapa Base (con zoom automático) ---
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            lats = mets_info['U_latitud']
            lons = mets_info['U_longitud']
            bounds = [[lats.min(), lons.min()], [lats.max(), lons.max()]]
            mapa_base.fit_bounds(bounds, padding=(20, 20))
            
            colores_list = colores.copy()
            random.shuffle(colores_list)
            
            export_data = []
            summary_data = []

            # --- 4. (MODIFICADO) FASE 1: CLUSTERING POR TERRITORIO ---
            print(f"\n🔬 Fase 1: Iniciando clustering por territorio...")
            
            for idx_met, met_seleccionado in enumerate(mets_seleccionados):
                print(f"\n{'─'*70}")
                print(f"🏠 Procesando Territorio: {met_seleccionado} ({idx_met + 1}/{len(mets_seleccionados)})")
                print(f"{'─'*70}")
                
                met_info = df_cdr[df_cdr['CodMET'] == met_seleccionado]
                if met_info.empty:
                    print(f"⚠️ No se encontró información para el MET '{met_seleccionado}'. Saltando.")
                    continue
                met_row = met_info.iloc[0]
                met_lat = met_row['U_latitud']
                met_lon = met_row['U_longitud']
                
                # --- (NUEVO) LÓGICA DE MUESTREO POR TERRITORIO ---
                df_territorio_completo = df_clientes[df_clientes['MET_Asignado'] == met_seleccionado].copy()
                
                if df_territorio_completo.empty:
                    print(f"⚠️ No hay clientes asignados al territorio de '{met_seleccionado}'. Saltando.")
                    continue
                
                df_clientes_met = pd.DataFrame() # DataFrame vacío para el análisis
                if analizar_todos:
                    df_clientes_met = df_territorio_completo
                    print(f"✅ Analizando territorio COMPLETO: {len(df_clientes_met)} clientes")
                else:
                    if len(df_territorio_completo) > n_muestra:
                        print(f"⚠️ MODO MUESTRA: Analizando {n_muestra} de {len(df_territorio_completo)} clientes (aleatorio)")
                        df_clientes_met = df_territorio_completo.sample(n=n_muestra).copy()
                    else:
                        df_clientes_met = df_territorio_completo
                        print(f"✅ Analizando territorio COMPLETO (es menor que la muestra): {len(df_clientes_met)} clientes")
                # --- FIN LÓGICA DE MUESTREO ---

                if df_clientes_met.empty:
                    print(f"⚠️ No hay clientes para procesar en este MET. Saltando.")
                    continue
                
                # --- El resto del script continúa igual, usando df_clientes_met ---
                coords = df_clientes_met[['U_longitud', 'U_latitud']].values
                k_inicial = max(1, len(df_clientes_met) // max_size)
                
                print(f"🔬 Iniciando clustering (k_inicial = {k_inicial})...")
                initial_seeds = find_initial_seeds(coords, k_inicial)
                
                if len(initial_seeds) == 0:
                    print(f"❌ No se pudieron generar semillas para el MET '{met_seleccionado}'.")
                    continue
                
                # --- SEGREGAR POR FRECUENCIA (NUEVO) ---
                df_elegibles, df_comodines = segregar_por_frecuencia(df_clientes_met, threshold=0.01)
                
                # --- CLUSTERING SOLO CON ELEGIBLES (MODIFICADO) ---
                df_elegibles, cluster_counts = grow_clusters_from_seeds(df_elegibles, initial_seeds, max_size, min_size)
                df_elegibles, cluster_counts = adjust_small_clusters(df_elegibles, cluster_counts, min_size, max_size)
                
                # --- ASIGNAR COMODINES AL VECINO MÁS CERCANO (NUEVO) ---
                df_comodines = asignar_comodines_post_clustering(df_comodines, df_elegibles)
                
                # --- CONSOLIDAR RESULTADOS ---
                df_clientes_met = pd.concat([df_elegibles, df_comodines], ignore_index=True)
                
                cluster_polygons = calculate_cluster_polygons(df_clientes_met, method='voronoi')
                
                final_counts = df_clientes_met['Ruta_nueva'][df_clientes_met['Ruta_nueva'] != -1].value_counts()
                rutas_validas = final_counts.index.tolist() # Tomar todas, el ajuste ya limpió
                
                print(f"✅ Rutas generadas: {len(rutas_validas)}")
                
                df_clientes_met_validos = df_clientes_met[df_clientes_met['Ruta_nueva'].isin(rutas_validas)].copy()
                
                if df_clientes_met_validos.empty:
                    print(f"⚠️ No hay rutas válidas para el MET '{met_seleccionado}'.")
                    continue
                
                met_name_clean = str(met_seleccionado).strip()
                display_name = met_name_clean if met_name_clean.upper().startswith('MET') else f"MET {met_name_clean}"
                
                fg_met_icon = folium.FeatureGroup(name=f"{display_name} - 🏠 MET", show=True)
                
                clientes_met_total = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] >= 0]
                volumen_total_met = clientes_met_total['m3/día'].sum()
                total_clientes_met = len(clientes_met_total)
                
                popup_html_met = f'''
                <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                    <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                        <div style="font-size: 22px; font-weight: bold;">🏠 {met_name_clean}</div>
                    </div>
                    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                        <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                            <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                            <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                        </div>
                        <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                            <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                            <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                        </div>
                    </div>
                    <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                        <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                        <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_validas)}</div>
                    </div>
                </div>
                '''

                if os.path.exists(icon_met_path):
                    icon_met_obj = CustomIcon(icon_met_path, icon_size=(40, 40), icon_anchor=(20, 40), popup_anchor=(0, -40))
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=icon_met_obj, tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)
                else:
                    folium.Marker(
                        location=[met_lat, met_lon], popup=folium.Popup(popup_html_met, max_width=420),
                        icon=folium.Icon(color='red', icon='home', prefix='glyphicon'),
                        tooltip=f"MET: {met_seleccionado}"
                    ).add_to(fg_met_icon)

                create_map_elements(
                    mapa_base,
                    df_clientes_met_validos,
                    cluster_polygons,
                    colores_list,
                    vol_min_global,
                    vol_max_global,
                    met_seleccionado
                )
                
                fg_met_icon.add_to(mapa_base)
                
                g_total_clientes_procesados += len(df_clientes_met_validos)
                
                for ruta_id in rutas_validas:
                    ruta_clientes = df_clientes_met_validos[df_clientes_met_validos['Ruta_nueva'] == ruta_id]
                    n_clientes_ruta = len(ruta_clientes)
                    volumen_total = ruta_clientes['m3/día'].sum()
                    
                    # Contar clientes normales vs comodines
                    n_normales = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == False])
                    n_comodines = len(ruta_clientes[ruta_clientes.get('es_comodin', False) == True])
                    
                    summary_data.append({
                        'MET': met_seleccionado,
                        'Ruta_nueva': ruta_id,
                        'Num_Clientes': n_clientes_ruta,
                        'Clientes_Normales': n_normales,
                        'Clientes_Comodines': n_comodines,
                        'Volumen_Total_m3': round(volumen_total, 2),
                        'Rentable (>= 1 m³)': 'SÍ' if volumen_total >= 1.0 else 'NO',
                        'Centroide_Lat': ruta_clientes['U_latitud'].mean(),
                        'Centroide_Lon': ruta_clientes['U_longitud'].mean()
                    })
                    
                    for _, cliente in ruta_clientes.iterrows():
                        es_comodin = cliente.get('es_comodin', False)
                        export_data.append({
                            'MET': met_seleccionado,
                            'Ruta_nueva': ruta_id,
                            'Tipo_Cliente': 'Comodín' if es_comodin else 'Normal',
                            'Codigo': cliente['CodCli'],
                            'Nombre': cliente.get('Nombre', 'N/A'),
                            'Ruta_Original': cliente.get('Ruta', 'N/A'),
                            'm3_dia': cliente['m3/día'],
                            'Frecuencia_%': round(cliente['entregas/día'] * 100, 2),
                            'Latitud': cliente['U_latitud'],
                            'Longitud': cliente['U_longitud']
                        })
                
                print(f"✅ Territorio '{met_seleccionado}' completado: {len(rutas_validas)} rutas, {len(df_clientes_met_validos)} clientes")
            
            # --- 5. Guardar Mapa y Excel ---
            folium.LayerControl(collapsed=False).add_to(mapa_base)
            
            estilos_css = r'''
            <style>
            #panel-resumen {
                position: fixed; top: 80px; left: 10px; background: white; padding: 12px 16px;
                border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.2); z-index: 1000;
                max-width: 280px; font-family: Arial, sans-serif;
            }
            #panel-resumen h3 {
                margin: 0 0 10px 0; font-size: 16px; color: #333;
                border-bottom: 2px solid #007bff; padding-bottom: 6px;
            }
            #panel-resumen-content { font-size: 13px; line-height: 1.6; color: #555; }
            .resumen-item { display: flex; justify-content: space-between; padding: 4px 0; border-bottom: 1px solid #eee; }
            .resumen-item:last-child {
                border-bottom: none; font-weight: bold; margin-top: 6px;
                padding-top: 8px; border-top: 2px solid #007bff;
            }
            .resumen-label { color: #666; }
            .resumen-value { color: #007bff; font-weight: 600; }
            .leaflet-control-layers-list {
                max-height: 400px; overflow-y: auto; overflow-x: hidden;
                width: 100%; min-width: 200px;
            }
            .layer-control-buttons {
                padding: 8px; border-bottom: 1px solid #ddd; background: #f8f8f8;
                display: flex; gap: 5px;
            }
            .layer-btn {
                padding: 6px 10px; font-size: 11px; border: 1px solid #ccc; background: white;
                cursor: pointer; border-radius: 4px; flex: 1; text-align: center;
                font-weight: 600; transition: all 0.2s;
            }
            .layer-btn:hover { background: #e6e6e6; transform: translateY(-1px); }
            .layer-btn.select-all { background: #4CAF50; color: white; border-color: #45a049; }
            .layer-btn.select-all:hover { background: #45a049; }
            .layer-btn.deselect-all { background: #f44336; color: white; border-color: #da190b; }
            .layer-btn.deselect-all:hover { background: #da1a0b; }
            .met-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
                display: flex; gap: 4px; flex-wrap: wrap;
            }
            .met-btn {
                padding: 5px 10px; font-size: 10px; border: 2px solid #2196F3; background: white;
                color: #1976D2; cursor: pointer; border-radius: 6px; flex: 1; text-align: center;
                min-width: 60px; font-weight: 600; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .met-btn:hover { background: #2196F3; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .ruta-buttons-row {
                padding: 6px 8px; border-bottom: 1px solid #ddd;
                background: linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%);
                display: grid; grid-template-columns: repeat(auto-fit, minmax(45px, 1fr));
                gap: 4px; max-width: 100%;
            }
            .ruta-btn {
                padding: 5px 8px; font-size: 10px; border: 2px solid #9C27B0; background: white;
                color: #7B1FA2; cursor: pointer; border-radius: 6px; text-align: center;
                min-width: 45px; font-weight: 700; white-space: nowrap; overflow: hidden;
                text-overflow: ellipsis; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }
            .ruta-btn:hover { background: #9C27B0; color: white; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }
            .search-ruta-container {
                display: flex; gap: 5px; padding: 8px; background: #f0f8ff;
                border-bottom: 1px solid #b3d9ff; margin-bottom: 8px; align-items: center;
            }
            .search-ruta-input {
                flex: 1; padding: 6px 10px; border: 1px solid #ced4da; border-radius: 4px;
                font-size: 12px; outline: none;
            }
            .search-ruta-input:focus { border-color: #0066cc; box-shadow: 0 0 0 2px rgba(0, 102, 204, 0.1); }
            .search-ruta-btn {
                padding: 6px 15px; border: 1px solid #0066cc; border-radius: 4px; background: #0066cc;
                color: white; cursor: pointer; font-size: 12px; font-weight: 500;
                transition: all 0.2s; white-space: nowrap;
            }
            .search-ruta-btn:hover { background: #0052a3; border-color: #0052a3; }
            .search-ruta-clear {
                padding: 6px 12px; border: 1px solid #dc3545; border-radius: 4px; background: white;
                color: #dc3545; cursor: pointer; font-size: 12px; font-weight: 500; transition: all 0.2s;
            }
            .search-ruta-clear:hover { background: #dc3545; color: white; }
            .leaflet-control-layers-overlays { display: none; }
            </style>
            '''
            mapa_base.get_root().html.add_child(folium.Element(estilos_css))
            
            paneles_js = r'''
            <script>
            function crearPanelResumen() {
                const panel = document.createElement('div');
                panel.id = 'panel-resumen';
                panel.innerHTML = `<h3>📊 Resumen de Rutas</h3><div id="panel-resumen-content"></div>`;
                document.body.appendChild(panel);
                return panel;
            }
            function inicializarPaneles(mapInstance) {
                console.log('=== Inicializando paneles personalizados ===');
                if (document.getElementById('panel-resumen')) {
                    console.log('ℹ️ Paneles ya inicializados');
                    return true;
                }
                const panelResumen = crearPanelResumen();
                const map = mapInstance;
                if (!map) { console.error('❌ Instancia del mapa (mapInstance) es nula.'); return false; }
                let totalRutas = 0;
                const metRutas = {};
                map.eachLayer(function(layer) {
                    if (layer._url || !layer.options || !layer.options.pane) return;
                    const layerName = layer.options.attribution || '';
                    const rutaMatch = layerName.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+Ruta\s+(\d+)$/); // Modificado para ignorar advertencia
                    if (rutaMatch) {
                        const metNombre = rutaMatch[1]; // Captura el nombre limpio
                        totalRutas++;
                        if (!metRutas[metNombre]) metRutas[metNombre] = 0;
                        metRutas[metNombre]++;
                    }
                });
                const resumenHTML = [];
                Object.keys(metRutas).sort().forEach(function(metNombre) {
                    resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">${metNombre}:</span><span class="resumen-value">${metRutas[metNombre]} rutas</span></div>`);
                });
                resumenHTML.push(`<div class="resumen-item"><span class="resumen-label">Total:</span><span class="resumen-value">${totalRutas} rutas</span></div>`);
                document.getElementById('panel-resumen-content').innerHTML = resumenHTML.join('');
                console.log('✅ Paneles personalizados inicializados');
                return true;
            }
            let panelPollCount = 0;
            function tryInitPanels() {
                const mapElement = document.querySelector('.leaflet-container');
                let mapInstance = null;
                if (mapElement && mapElement._leaflet_map) { mapInstance = mapElement._leaflet_map; }
                if (typeof L !== 'undefined' && L.map && mapInstance) {
                    console.log('✅ (Poll) Leaflet (L) y MapInstance (_leaflet_map) listos. Iniciando paneles...');
                    inicializarPaneles(mapInstance);
                } else if (panelPollCount < 40) {
                    panelPollCount++;
                    console.log('Poll #' + panelPollCount + ': Esperando a L y _leaflet_map...');
                    setTimeout(tryInitPanels, 500);
                } else {
                    console.error('❌ No se pudo cargar la instancia del mapa (_leaflet_map) después de 20 segundos.');
                }
            }
            tryInitPanels();
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(paneles_js))
            
            filtros_js = r'''
            <script>
            function inicializarFiltros() {
                console.log('=== Inicializando filtros personalizados (Estilo Celaya) ===');
                const layerControl = document.querySelector('.leaflet-control-layers');
                if (!layerControl) { console.error('❌ No se encontró .leaflet-control-layers'); return false; }
                const layersList = layerControl.querySelector('.leaflet-control-layers-list');
                if (!layersList) { console.error('❌ No se encontró .leaflet-control-layers-list'); return false; }
                const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
                if (!overlaysDiv) { console.error('❌ No se encontró .leaflet-control-layers-overlays'); return false; }
                const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                console.log('✅ Total de checkboxes encontrados:', checkboxes.length);
                if (layersList.querySelector('.layer-control-buttons')) { console.log('ℹ️ Filtros ya inicializados'); return true; }
                
                console.log('📋 Listando todas las capas:');
                checkboxes.forEach(function(checkbox, index) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    console.log('   ' + (index + 1) + '. "' + label + '"');
                });
                
                const buttonsDiv = document.createElement('div');
                buttonsDiv.className = 'layer-control-buttons';
                const selectAllBtn = document.createElement('button');
                selectAllBtn.textContent = '✓ Todo';
                selectAllBtn.className = 'layer-btn select-all';
                selectAllBtn.title = 'Seleccionar todas las capas';
                const deselectAllBtn = document.createElement('button');
                deselectAllBtn.textContent = '✗ Nada';
                deselectAllBtn.className = 'layer-btn deselect-all';
                deselectAllBtn.title = 'Deseleccionar todas las capas';
                selectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (!cb.checked) cb.click(); }); };
                deselectAllBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); };
                buttonsDiv.appendChild(selectAllBtn);
                buttonsDiv.appendChild(deselectAllBtn);
                layersList.insertBefore(buttonsDiv, overlaysDiv);
                
                const searchContainer = document.createElement('div');
                searchContainer.className = 'search-ruta-container';
                const searchLabel = document.createElement('span');
                searchLabel.textContent = '🔍';
                searchLabel.style.fontSize = '14px';
                const searchInput = document.createElement('input');
                searchInput.type = 'text';
                searchInput.className = 'search-ruta-input';
                searchInput.placeholder = 'Buscar por Ruta ID (ej: 0, 1, 2...)';
                searchInput.title = 'Ingresa el número de ruta para filtrar';
                const searchBtn = document.createElement('button');
                searchBtn.textContent = 'Buscar';
                searchBtn.className = 'search-ruta-btn';
                searchBtn.title = 'Filtrar por Ruta ID';
                const clearBtn = document.createElement('button');
                clearBtn.textContent = 'Limpiar';
                clearBtn.className = 'search-ruta-clear';
                clearBtn.title = 'Limpiar búsqueda';
                const filtrarPorRutaId = function() {
                    const rutaId = searchInput.value.trim();
                    if (rutaId === '') { alert('Por favor ingresa un número de ruta'); return; }
                    console.log('🔍 Filtrando capas con Ruta ID:', rutaId);
                    checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                    setTimeout(function() {
                        let encontradas = 0;
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$'); // Busca el final
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                console.log('  ✓ Seleccionando:', label);
                                if (!checkbox.checked) { checkbox.click(); encontradas++; }
                            }
                        });
                        if (encontradas === 0) { console.warn('⚠️ No se encontraron capas con Ruta ID ' + rutaId); alert('No se encontraron capas con Ruta ID: ' + rutaId); }
                    }, 150);
                };
                searchBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); filtrarPorRutaId(); };
                searchInput.addEventListener('keypress', function(e) { if (e.key === 'Enter') { e.preventDefault(); filtrarPorRutaId(); } });
                clearBtn.onclick = function(e) { e.preventDefault(); e.stopPropagation(); searchInput.value = ''; checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); }); console.log('🧹 Búsqueda limpiada'); };
                searchContainer.appendChild(searchLabel);
                searchContainer.appendChild(searchInput);
                searchContainer.appendChild(searchBtn);
                searchContainer.appendChild(clearBtn);
                layersList.insertBefore(searchContainer, overlaysDiv);
                
                const metButtonsDiv = document.createElement('div');
                metButtonsDiv.className = 'met-buttons-row';
                const mets = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const metMatch = label.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/); // Modificado para incluir advertencia
                    if (metMatch) { mets.add(metMatch[1].trim()); } // Limpiar advertencia
                });
                console.log('🏢 METs disponibles:', Array.from(mets));
                Array.from(mets).sort().forEach(function(metNombre) {
                    const metBtn = document.createElement('button');
                    const nombreCorto = metNombre.replace('MET ', '').trim();
                    metBtn.textContent = nombreCorto;
                    metBtn.className = 'met-btn';
                    metBtn.title = 'Seleccionar solo capas de ' + metNombre;
                    metBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        console.log('🔍 Activando filtro para MET:', metNombre);
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                // Modificado para que siempre funcione
                                if (label.includes(metNombre + ' -')) {
                                    console.log('  ✓ Seleccionando:', label);
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas para ' + metNombre);
                        }, 150);
                    };
                    metButtonsDiv.appendChild(metBtn);
                });
                if (metButtonsDiv.children.length > 0) { layersList.insertBefore(metButtonsDiv, overlaysDiv); }
                
                const rutaButtonsDiv = document.createElement('div');
                rutaButtonsDiv.className = 'ruta-buttons-row';
                const rutas = new Set();
                checkboxes.forEach(function(checkbox) {
                    const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                    const rutaMatch = label.match(/- Ruta\s+(\d+)/);
                    if (rutaMatch) { rutas.add(parseInt(rutaMatch[1])); }
                });
                console.log('🚚 Rutas encontradas:', Array.from(rutas).sort(function(a, b) { return a - b; }));
                const rutasArray = Array.from(rutas).sort(function(a, b) { return a - b; });
                rutasArray.forEach(function(ruta) {
                    const rutaBtn = document.createElement('button');
                    rutaBtn.textContent = 'R' + ruta;
                    rutaBtn.className = 'ruta-btn';
                    rutaBtn.title = 'Seleccionar Ruta ' + ruta + ' de todos los METs';
                    rutaBtn.onclick = function(e) {
                        e.preventDefault(); e.stopPropagation();
                        const rutaId = ruta;
                        console.log('🔍 Activando filtro para Ruta', rutaId);
                        const rutaPattern = new RegExp('- Ruta\\s+' + rutaId + '$');
                        const metIconPattern = /- 🏠 MET$/;
                        const metNamePattern = /^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+(Ruta|🏠 MET)/;
                        const metsToShow = new Set();
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            if (rutaPattern.test(label)) {
                                const metMatch = label.match(metNamePattern);
                                if (metMatch) { metsToShow.add(metMatch[1].trim()); }
                            }
                        });
                        console.log('   ...Mostrando METs que tienen esta ruta:', Array.from(metsToShow));
                        checkboxes.forEach(function(cb) { if (cb.checked) cb.click(); });
                        setTimeout(function() {
                            let seleccionadas = 0;
                            checkboxes.forEach(function(checkbox) {
                                const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                if (rutaPattern.test(label)) {
                                    if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                } else if (metIconPattern.test(label)) {
                                    const metMatch = label.match(metNamePattern);
                                    if (metMatch && metsToShow.has(metMatch[1].trim())) {
                                        if (!checkbox.checked) { checkbox.click(); seleccionadas++; }
                                    }
                                }
                            });
                            console.log('✅ Seleccionadas ' + seleccionadas + ' capas (Rutas + METs) para Ruta ' + rutaId);
                        }, 150);
                    };
                    rutaButtonsDiv.appendChild(rutaBtn);
                });
                if (rutaButtonsDiv.children.length > 0) { layersList.insertBefore(rutaButtonsDiv, overlaysDiv); }
                
                console.log('=== Filtros inicializados correctamente ===');
                return true;
            }
            let filterPollCount = 0;
            function tryInitFilters() {
                const layerControlList = document.querySelector('.leaflet-control-layers-list');
                if (typeof L !== 'undefined' && layerControlList) {
                    console.log('✅ (Poll) Leaflet (L) y .leaflet-control-layers-list listos. Iniciando filtros...');
                    try { inicializarFiltros(); } catch (e) { console.error('❌ Error al ejecutar inicializarFiltros:', e); }
                } else if (filterPollCount < 40) {
                    filterPollCount++;
                    console.log('Poll Filtros #' + filterPollCount + ': Esperando a L y .leaflet-control-layers-list...');
                    setTimeout(tryInitFilters, 500);
                } else {
                    console.error('❌ No se pudo encontrar .leaflet-control-layers-list después de 20 segundos.');
                }
            }
            setTimeout(tryInitFilters, 500); 
            </script>
            '''
            mapa_base.get_root().html.add_child(folium.Element(filtros_js))
            
            titulo_html = f'''
            <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
                <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                    🗺️ ANÁLISIS DE CLUSTERS POR VOLUMEN
                </h2>
                <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                    📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas Generadas: <b>{len(summary_data)}</b> | 👥 Clientes Procesados: <b>{g_total_clientes_procesados}</b><br>
                    📦 Tamaño de Círculo = Volumen (m³/día) | 🎨 Color de Círculo = Ruta Asignada
                </p>
            </div>
            '''
            mapa_base.get_root().html.add_child(folium.Element(titulo_html))
            
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            map_filename = f'mapa_rutas_clusters_{timestamp}.html'
            g_map_path = os.path.join(output_dir, map_filename)
            mapa_base.save(g_map_path)
            
            print(f"\n{'='*70}")
            print(f"🗺️ Mapa guardado: {g_map_path}")
            
            if export_data and summary_data:
                excel_path = generate_consolidated_excel(export_data, summary_data, mets_seleccionados, timestamp, output_dir)
                if excel_path:
                    g_excel_paths = [excel_path]
                    print(f"📊 Excel guardado: {excel_path}")
            
            print(f"{'='*70}")
            print(f"✅ ANÁLISIS COMPLETADO")
            print(f"   • Total clientes procesados: {g_total_clientes_procesados}")
            print(f"   • Total rutas generadas: {len(summary_data)}")
            print(f"   • Archivos generados: {len(g_excel_paths) + 1}")
            print(f"{'='*70}\n")

        except Exception as e:
            import traceback
            print(f"❌ Error Inesperado en ejecución #{execution_count}:")
            traceback.print_exc()
        finally:
            # --- (NUEVO) RESTAURAR DF_CLIENTES ORIGINAL ---
            if 'df_clientes_original' in locals():
                df_clientes = df_clientes_original
                print("ℹ️ DataFrame de clientes restaurado a su estado original.")
            
            param_button.disabled = False # Desbloquear botón

# --- Registrar el handler ---
try:
    param_button.on_click(ejecutar_analisis, remove=True)
except:
    pass
param_button.on_click(ejecutar_analisis)

# === MOSTRAR UI (Modificada) ===
print("================== MÓDULO 2: Análisis de Rutas (Alfa 5.1) ==================")
display(widgets.VBox([
    widgets.Label("1. Cargar archivo Excel:", style={'font_weight': 'bold'}),
    upload_widget,
    output_upload,
    widgets.HTML("<hr>"),
    met_box,
    widgets.HTML("<hr>"),
    widgets.Label("2. Configurar parámetros de Ruta:", style={'font_weight': 'bold'}),
    widgets.HBox([min_clientes_cluster, max_clientes_cluster]),
    
    # --- (NUEVO) WIDGETS DE MUESTREO ---
    widgets.HBox([sample_size_per_met, run_full_analysis]),
    # --- FIN DE WIDGETS NUEVOS ---
    
    param_button,
    output_param
]))

Versión 5.1 Antes de meterle DBSCAN

In [ ]:
# =========================================================================
# === MÓDULO 1: BACKEND (CORRECCIÓN BUG NOTACIÓN CIENTÍFICA) ==============
# =========================================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, MultiPoint, Point
from sklearn.cluster import KMeans

# --- Configuración de Rutas ---
BASE_DIR = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan"
ICON_MET_PATH = os.path.join(BASE_DIR, "95_24.png")
OUTPUT_DIR = os.path.join(BASE_DIR, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- CLASE DE ESTADO ---
class CeylanState:
    def __init__(self):
        self.df_clientes = pd.DataFrame()
        self.df_cdr = pd.DataFrame()
        self.df_clientes_backup = pd.DataFrame()
        self.execution_count = 0
        self.met_checkboxes = []
        self.colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
        self.last_map_path = ""
        self.last_excel_paths = []
        self.total_processed = 0

state = CeylanState()

# =========================================================================
# === FUNCIONES HELPER & LÓGICA DE NEGOCIO ================================
# =========================================================================

def normalizar_id(valor):
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    return s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0: return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min: return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

def _clip_voronoi(vor, bounding_poly):
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty: return clipped_polys
    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points) or region_index == -1: continue
        region_indices = vor.regions[region_index]
        if not region_indices or -1 in region_indices: continue
        try:
            pts = [vor.vertices[v] for v in region_indices]
            poly = Polygon(pts)
            if not poly.is_valid: poly = poly.buffer(0)
            intersection = poly.intersection(bounding_poly)
            if not intersection.is_empty and intersection.geom_type in ['Polygon', 'MultiPolygon']:
                if intersection.geom_type == 'MultiPolygon': intersection = max(intersection.geoms, key=lambda p: p.area)
                clipped_polys[tuple(vor.points[i])] = [[y, x] for x, y in intersection.exterior.coords]
        except: pass
    return clipped_polys

# =========================================================================
# === ALGORITMOS DE CLUSTERING (ALFA 5.1) =================================
# =========================================================================

def segregar_por_frecuencia(df_clients, threshold=0.01):
    if 'Ruta_nueva' not in df_clients.columns: df_clients['Ruta_nueva'] = -1
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    return (df_no_met[df_no_met['entregas/día'] > threshold].copy(), 
            df_no_met[df_no_met['entregas/día'] <= threshold].copy())

def asignar_comodines_post_clustering(df_comodines, df_elegibles_asignados):
    if df_comodines.empty: return df_comodines
    coords_comodines = df_comodines[['U_longitud', 'U_latitud']].values
    coords_elegibles = df_elegibles_asignados[['U_longitud', 'U_latitud']].values
    dist_matrix = cdist(coords_comodines, coords_elegibles, metric='euclidean')
    idx_vecinos = np.argmin(dist_matrix, axis=1)
    df_comodines['Ruta_nueva'] = df_elegibles_asignados.iloc[idx_vecinos]['Ruta_nueva'].values
    df_comodines['es_comodin'] = True
    return df_comodines

def find_initial_seeds(coords, k, random_state=42):
    if k <= 0 or len(coords) == 0: return np.array([])
    try:
        kmeans = KMeans(n_clusters=min(k, len(coords)), random_state=random_state, n_init=10).fit(coords)
        return kmeans.cluster_centers_
    except:
        return coords[np.random.choice(len(coords), min(k, len(coords)), replace=False)]

def calcular_limites_dinamicos(dispersion, frecuencia, min_base, max_base):
    disp_norm = min(dispersion / 0.5, 1.0)
    frec_norm = min(frecuencia, 1.0)
    factor = disp_norm * (1.0 - frec_norm * 0.5)
    rango = max_base - min_base
    ajuste = int(rango * factor)
    return max(min_base, int(min_base * 0.5)), max_base - ajuste

def grow_clusters_from_seeds(df_elegibles, initial_seeds, max_size, min_size):
    k = len(initial_seeds)
    if k == 0: return df_elegibles, {}
    df = df_elegibles.copy()
    df['Ruta_nueva'] = -1
    df['es_comodin'] = False
    ruta_counts = {i: 0 for i in range(k)}
    
    df_anchors = df[df['entregas/día'] >= df['entregas/día'].quantile(0.8)].copy()
    if not df_anchors.empty:
        dists = distance_matrix(df_anchors[['U_longitud', 'U_latitud']].values, initial_seeds)
        for i, (idx, row) in enumerate(df_anchors.iterrows()):
            ordered_routes = np.argsort(dists[i])
            for r_idx in ordered_routes:
                if ruta_counts[r_idx] < max_size:
                    df.loc[idx, 'Ruta_nueva'] = r_idx
                    ruta_counts[r_idx] += 1
                    break
    
    df_rest = df[df['Ruta_nueva'] == -1]
    if not df_rest.empty:
        coords = df_rest[['U_longitud', 'U_latitud']].values
        dists = distance_matrix(coords, initial_seeds)
        for i in range(len(df_rest)):
            best_routes = np.argsort(dists[i])
            for r_idx in best_routes:
                rut_clients = df[df['Ruta_nueva'] == r_idx]
                dispersion = 0.1
                if len(rut_clients) > 1:
                     dispersion = np.mean(cdist(rut_clients[['U_longitud', 'U_latitud']].values, [[0,0]]))
                _, lim_max_dinamico = calcular_limites_dinamicos(dispersion, df_rest.iloc[i]['entregas/día'], min_size, max_size)
                if ruta_counts[r_idx] < lim_max_dinamico:
                    df.loc[df_rest.index[i], 'Ruta_nueva'] = r_idx
                    ruta_counts[r_idx] += 1
                    break
    return df, ruta_counts

def adjust_small_clusters(df_clustered, counts, min_size, max_size):
    rutas_peques = [r for r, c in counts.items() if c < min_size]
    rutas_validas = [r for r, c in counts.items() if c >= min_size]
    if not rutas_peques or not rutas_validas: return df_clustered, counts
    df = df_clustered.copy()
    for ruta_origen in rutas_peques:
        clientes_mover = df[df['Ruta_nueva'] == ruta_origen]
        for idx, row in clientes_mover.iterrows():
            pt = np.array([[row['U_longitud'], row['U_latitud']]])
            best_r, min_d = -1, float('inf')
            for ruta_dest in rutas_validas:
                clients_dest = df[df['Ruta_nueva'] == ruta_dest]
                if clients_dest.empty: continue
                centroid = clients_dest[['U_longitud', 'U_latitud']].mean().values
                d = np.linalg.norm(pt - centroid)
                if d < min_d and counts[ruta_dest] < max_size:
                    min_d, best_r = d, ruta_dest
            if best_r != -1:
                df.loc[idx, 'Ruta_nueva'] = best_r
                counts[best_r] += 1
                counts[ruta_origen] -= 1
    return df, counts

def calculate_cluster_polygons(df, method='voronoi'):
    polygons = {}
    rutas = [r for r in df['Ruta_nueva'].unique() if r >= 0]
    if len(rutas) < 3: method = 'convex_hull'
    centroides = []
    mapa_centroide_ruta = {}
    for r in rutas:
        pts = df[df['Ruta_nueva'] == r][['U_longitud', 'U_latitud']].values
        if len(pts) > 0:
            c = np.mean(pts, axis=0)
            centroides.append(c)
            mapa_centroide_ruta[tuple(c)] = r
    if method == 'voronoi' and len(centroides) >= 4:
        all_pts = df[df['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
        hull = MultiPoint(all_pts).convex_hull.buffer(0.001)
        vor = Voronoi(centroides)
        clipped = _clip_voronoi(vor, hull)
        for cent_tuple, poly_coords in clipped.items():
            if cent_tuple in mapa_centroide_ruta: polygons[mapa_centroide_ruta[cent_tuple]] = poly_coords
    for r in rutas:
        if r not in polygons:
            pts = df[df['Ruta_nueva'] == r][['U_latitud', 'U_longitud']].values
            if len(pts) >= 3:
                try: polygons[r] = pts[ConvexHull(pts).vertices].tolist()
                except: pass
    return polygons

def procesar_logica_met(met_id, n_muestra, analizar_todos, min_size, max_size):
    df_terr = state.df_clientes[state.df_clientes['MET_Asignado'] == met_id].copy()
    if df_terr.empty: return None, None, "Sin clientes"
    df_proc = df_terr.copy() if analizar_todos or len(df_terr) <= n_muestra else df_terr.sample(n=n_muestra).copy()
    
    # Seguridad de tipos (para evitar el error de str > float)
    df_proc['entregas/día'] = pd.to_numeric(df_proc['entregas/día'], errors='coerce').fillna(0)
    df_proc['m3/día'] = pd.to_numeric(df_proc['m3/día'], errors='coerce').fillna(0)
    
    coords = df_proc[['U_longitud', 'U_latitud']].values
    if len(coords) == 0: return None, None, "Sin coordenadas"
    
    k_init = max(1, len(df_proc) // max_size)
    seeds = find_initial_seeds(coords, k_init)
    
    df_eleg, df_como = segregar_por_frecuencia(df_proc)
    df_eleg, counts = grow_clusters_from_seeds(df_eleg, seeds, max_size, min_size)
    df_eleg, counts = adjust_small_clusters(df_eleg, counts, min_size, max_size)
    df_como = asignar_comodines_post_clustering(df_como, df_eleg)
    
    df_final = pd.concat([df_eleg, df_como], ignore_index=True)
    polys = calculate_cluster_polygons(df_final)
    return df_final, polys, "OK"

# =========================================================================
# === VISUALIZACIÓN Y RECURSOS ============================================
# =========================================================================

def get_full_css(): return ""
def get_full_js(): return ""
def inject_custom_assets(map_object): pass 

def generate_consolidated_excel(export_data, summary_data, output_dir, filename_suffix=None):
    if not export_data: return None
    if not filename_suffix: filename_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_dir, f'Resultados_Ceylan_{filename_suffix}.xlsx')
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        pd.DataFrame(summary_data).to_excel(writer, sheet_name='Resumen', index=False)
        pd.DataFrame(export_data).to_excel(writer, sheet_name='Detalle', index=False)
    return path

# --- HANDLER DE CARGA CORREGIDO (AQUÍ ESTÁ LA SOLUCIÓN) ---
def handle_upload(change, met_box, output_upload):
    with output_upload:
        clear_output()
        if not change['new']: return
        try:
            upl_file = change['owner'].value
            content = upl_file[0]['content'] if isinstance(upl_file, tuple) else list(upl_file.values())[0]['content']
            
            temp_path = "temp_upload.xlsx"
            with open(temp_path, 'wb') as f: f.write(content)
            xls = pd.read_excel(temp_path, sheet_name=None)
            state.df_clientes = xls[list(xls.keys())[0]]
            state.df_cdr = xls[list(xls.keys())[1]]
            
            # --- CORRECCIÓN DE LIMPIEZA ---
            # Método antiguo (Buggy): .astype(str).str.replace('-', '0') -> Rompía "9.24E-05"
            # Método nuevo (Seguro): pd.to_numeric con coerce
            
            # 1. Convertir a numérico, forzando errores (como '-') a NaN
            state.df_clientes['m3/día'] = pd.to_numeric(state.df_clientes['m3/día'], errors='coerce').fillna(0)
            state.df_clientes['entregas/día'] = pd.to_numeric(state.df_clientes['entregas/día'], errors='coerce').fillna(0)
            
            state.df_clientes_backup = state.df_clientes.copy()
            
            mets = sorted(state.df_cdr['CodMET'].unique())
            state.met_checkboxes = [widgets.Checkbox(description=str(m), value=False) for m in mets]
            met_box.children = [widgets.Label("Selecciona METs:")] + state.met_checkboxes
            
            print(f"✅ Archivo cargado. Datos numéricos saneados correctamente.")
            print(f"📊 Clientes: {len(state.df_clientes)}")
            os.remove(temp_path)
        except Exception as e:
            print(f"❌ Error al cargar: {e}")
            import traceback
            traceback.print_exc()

print("✅ MÓDULO 1 CARGADO: Backend Corregido (Fix Notación Científica).")

✅ MÓDULO 1 CARGADO: Backend Corregido (Fix Notación Científica).


In [ ]:
# =========================================================================
# === MÓDULO 2: EJECUCIÓN FINAL (ETIQUETAS RUTAS MEJORADAS + PANEL) =======
# =========================================================================

# --- Widgets ---
upl_widget = widgets.FileUpload(accept='.xlsx', description='Subir Excel')
met_box = widgets.VBox([widgets.Label('Esperando archivo...')])
out_upl = widgets.Output()

slider_min = widgets.IntSlider(value=20, min=5, max=50, description='Min Clientes')
slider_max = widgets.IntSlider(value=40, min=10, max=100, description='Max Clientes')
inp_sample = widgets.IntText(value=500, description='Muestra/MET')
chk_full = widgets.Checkbox(value=False, description='Analizar TODO')

btn_run = widgets.Button(description='🚀 Ejecutar Análisis', button_style='success', layout=widgets.Layout(width='100%'))
out_run = widgets.Output()

# Conectar carga
upl_widget.observe(lambda c: handle_upload(c, met_box, out_upl), names='value')

# --- LÓGICA PRINCIPAL ---
def click_ejecutar(b):
    state.execution_count += 1
    btn_run.disabled = True
    
    with out_run:
        clear_output()
        
        # 1. Validaciones
        sel_mets = [cb.description for cb in state.met_checkboxes if cb.value]
        if state.df_clientes.empty or not sel_mets:
            print("⚠️ Error: Carga un archivo y selecciona al menos un MET.")
            btn_run.disabled = False; return

        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f"🚀 INICIANDO ANÁLISIS - ID: {run_id}")
        
        # 2. Restaurar Backup
        state.df_clientes = state.df_clientes_backup.copy()
        
        # 3. Asignación de Territorios
        print("🗺️ Asignando territorios...")
        coords_cli = state.df_clientes[['U_latitud', 'U_longitud']].values
        coords_met = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)][['U_latitud', 'U_longitud']].values
        met_names = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)]['CodMET'].values
        dists = cdist(coords_cli, coords_met)
        state.df_clientes['MET_Asignado'] = met_names[np.argmin(dists, axis=1)]
        
        # 4. Procesamiento
        # --- ZOOM AUTOMÁTICO ---
        df_zoom = state.df_clientes[state.df_clientes['MET_Asignado'].isin(sel_mets)]
        if not df_zoom.empty:
            sw = df_zoom[['U_latitud', 'U_longitud']].min().values.tolist()
            ne = df_zoom[['U_latitud', 'U_longitud']].max().values.tolist()
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            mapa_base.fit_bounds([sw, ne])
        else:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')

        export_rows, summary_rows = [], []
        g_total_clientes = 0
        
        for met in sel_mets:
            print(f"🔨 Procesando {met}...", end=" ")
            
            # Datos MET
            met_info = state.df_cdr[state.df_cdr['CodMET'] == met].iloc[0]
            met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
            
            # Lógica M1
            df_res, polys, status = procesar_logica_met(
                met, inp_sample.value, chk_full.value, 
                slider_min.value, slider_max.value
            )
            
            if status != "OK": print(f"({status})"); continue
                
            rutas_ids = sorted([r for r in df_res['Ruta_nueva'].unique() if r >= 0])
            print(f"✅ {len(rutas_ids)} rutas.")
            g_total_clientes += len(df_res)
            
            # Rangos volumen
            df_normales = df_res[df_res.get('es_comodin', False) == False]
            if not df_normales.empty:
                vol_min_local = df_normales['m3/día'].min()
                vol_max_local = df_normales['m3/día'].max()
            else:
                vol_min_local, vol_max_local = 0, 1
            
            # Estadísticas MET
            volumen_total_met = df_res['m3/día'].sum()
            total_clientes_met = len(df_res)
            met_display_name = str(met).strip()
            
            # --- C. DIBUJAR ICONO DEL MET ---
            fg_met_home = folium.FeatureGroup(name=f"{met_display_name} - 🏠 MET", show=True, attribution=f"{met_display_name} - 🏠 MET")
            
            popup_met = f"""
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                  <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                      <div style="font-size: 22px; font-weight: bold;">🏠 {met}</div>
                  </div>
                  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                      <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                          <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                          <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                      </div>
                      <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                          <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                          <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                      </div>
                  </div>
                  <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                      <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                      <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_ids)}</div>
                  </div>
            </div>
            """
            
            if os.path.exists(ICON_MET_PATH):
                icon_obj = CustomIcon(ICON_MET_PATH, icon_size=(40, 40), icon_anchor=(20, 40))
                folium.Marker([met_lat, met_lon], icon=icon_obj, popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            else:
                folium.Marker([met_lat, met_lon], icon=folium.Icon(color='green', icon='home'), popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            fg_met_home.add_to(mapa_base)

            # --- D. DIBUJAR RUTAS Y CLIENTES ---
            for ruta_id in rutas_ids:
                ruta_df = df_res[df_res['Ruta_nueva'] == ruta_id]
                vol_ruta = ruta_df['m3/día'].sum()
                frec_ruta = ruta_df['entregas/día'].mean() * 100
                
                # CÁLCULOS PARA ETIQUETA
                n_total = len(ruta_df)
                n_comodines = len(ruta_df[ruta_df.get('es_comodin', False) == True])
                n_normales = n_total - n_comodines
                
                layer_name = f"{met_display_name} - Ruta {ruta_id}"
                fg_ruta = folium.FeatureGroup(name=layer_name, show=True, attribution=layer_name)
                color_ruta = state.colores[ruta_id % len(state.colores)]
                
                # --- ETIQUETA ESTILIZADA DE LA RUTA (Tooltip) ---
                ruta_tooltip_html = f"""
                <div style="font-family: 'Segoe UI', sans-serif; min-width: 160px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); border-radius: 6px; overflow: hidden; background: white;">
                    <div style="background-color: {color_ruta}; color: white; padding: 6px 10px; font-weight: bold; font-size: 13px;">
                        🚛 Ruta {ruta_id}
                    </div>
                    <div style="padding: 8px; font-size: 11px; color: #333; line-height: 1.5;">
                        <div style="border-bottom: 1px solid #eee; padding-bottom: 4px; margin-bottom: 4px;">
                            <div>📦 <b>Volumen:</b> {vol_ruta:.2f} m³</div>
                            <div>👥 <b>Clientes:</b> {n_total}</div>
                        </div>
                        <div style="display: flex; justify-content: space-between; background: #f8f9fa; padding: 4px; border-radius: 4px; font-weight: 600;">
                            <span style="color: #2E7D32;">✅ Nor: {n_normales}</span>
                            <span style="color: #C62828;">🔺 Com: {n_comodines}</span>
                        </div>
                    </div>
                </div>
                """

                # --- POLÍGONO ---
                if ruta_id in polys and polys[ruta_id]:
                    raw_poly = polys[ruta_id]
                    final_poly = raw_poly
                    # Corrección Antártida
                    if raw_poly and len(raw_poly) > 0:
                        p1 = raw_poly[0]
                        if p1[0] < p1[1]: # Si Lon < Lat (América), invertir a [Lat, Lon]
                            final_poly = [[c[1], c[0]] for c in raw_poly]
                    
                    folium.Polygon(
                        locations=final_poly,
                        color=color_ruta, weight=2, fill=True, fill_opacity=0.2,
                        tooltip=folium.Tooltip(ruta_tooltip_html, sticky=True), # <--- ETIQUETA NUEVA AQUÍ
                        className='polygon-shape'
                    ).add_to(fg_ruta)
                
                # Marcadores
                for _, row in ruta_df.iterrows():
                    es_comodin = row.get('es_comodin', False)
                    vol = row['m3/día']
                    frec = row['entregas/día'] * 100
                    
                    popup_html = f"""
                    <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                         <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                             {'🔺 COMODÍN' if es_comodin else '✅ NORMAL'} | {row['CodCli']}
                         </div>
                         <div style="font-size:14px; color:#333; margin-bottom:4px;">
                             {row.get('Nombre', 'N/A')}
                         </div>
                         <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                             📦 Volumen: <b>{vol:.3f} m³</b>
                         </div>
                         <div style="font-size:14px; color:{'#888' if es_comodin else '#1976D2'}; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                             📊 Frecuencia: <b>{frec:.1f}%</b> (entregas/día)
                         </div>
                         <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                             🗺️ Ruta (Orig): <b>{row.get('Ruta', 'N/A')}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                         </div>
                    </div>
                    """
                    
                    if es_comodin:
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], 
                            radius=4,
                            color='#D32F2F', fill=True, fill_color='#FFCDD2', fill_opacity=0.9,
                            popup=folium.Popup(popup_html, max_width=320)
                        ).add_to(fg_ruta)
                    else:
                        tamano_calc = obtener_tamano_marcador(vol, vol_min_local, vol_max_local)
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], 
                            radius=tamano_calc,
                            color='#333333', weight=1, fill=True,
                            fill_color=color_ruta, fill_opacity=0.85,
                            popup=folium.Popup(popup_html, max_width=320)
                        ).add_to(fg_ruta)
                    
                    export_rows.append(row.to_dict())
                
                fg_ruta.add_to(mapa_base)
                summary_rows.append({'MET': met, 'Ruta': ruta_id, 'Clientes': len(ruta_df)})

        # --- INYECCIÓN DE ESTILOS Y SCRIPTS (ESTÉTICA PREMIUM EXACTA) ---
        folium.LayerControl(collapsed=False).add_to(mapa_base)

        estilos_css = r'''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 600px; overflow-y: auto; min-width: 340px; padding: 12px; font-family: 'Segoe UI', Arial, sans-serif; background: #fff; }
        
        /* CLASE PARA OCULTAR POLÍGONOS */
        .hide-all-polygons .polygon-shape { stroke: none !important; fill: none !important; pointer-events: none !important; }

        /* VISUALIZACIÓN */
        .visual-control-container { display: flex; gap: 8px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 12px; }
        .visual-btn { flex: 1; padding: 8px; border: 1px solid #ccc; background: white; cursor: pointer; border-radius: 6px; font-weight: 700; font-size: 12px; color: #555; box-shadow: 0 2px 4px rgba(0,0,0,0.1); display: flex; align-items: center; justify-content: center; gap: 5px; }
        .visual-btn:hover { background-color: #f5f5f5; }
        .visual-btn.active { background-color: #e3f2fd; color: #1976D2; border-color: #2196F3; }

        /* 1. Botones TODO / NADA (Sólidos) */
        .layer-control-buttons { display: flex; gap: 8px; margin-bottom: 12px; }
        .layer-btn { flex: 1; padding: 8px 0; font-size: 13px; font-weight: bold; border: none; border-radius: 4px; cursor: pointer; color: white; box-shadow: 0 2px 4px rgba(0,0,0,0.2); }
        .select-all { background-color: #4CAF50; } 
        .select-all:hover { background-color: #43A047; }
        .deselect-all { background-color: #F44336; } 
        .deselect-all:hover { background-color: #E53935; }

        /* 2. BUSCADOR */
        .search-ruta-container { display: flex; gap: 6px; align-items: center; margin-bottom: 12px; padding: 4px 0; }
        .search-ruta-input { flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 4px; font-size: 13px; outline: none; }
        .search-ruta-input:focus { border-color: #0275d8; box-shadow: 0 0 0 2px rgba(2,117,216,0.1); }
        .search-ruta-btn { background: #0275d8; color: white; border: none; padding: 8px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }
        .search-ruta-clear { background: white; color: #dc3545; border: 1px solid #dc3545; padding: 8px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }

        /* 3. METS (Borde Azul Grueso) */
        .met-buttons-row { display: flex; gap: 8px; margin-bottom: 12px; }
        .met-btn { flex: 1; padding: 8px; font-size: 12px; font-weight: 700; color: #0275d8; background: white; border: 2px solid #0275d8; border-radius: 6px; cursor: pointer; transition: all 0.1s; }
        .met-btn:hover { background: #e3f2fd; }

        /* 4. RUTAS (Fondo Lila, Grid 6) */
        .ruta-buttons-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 6px; background: #E1BEE7; padding: 10px; border-radius: 8px; }
        .ruta-btn { padding: 8px 0; text-align: center; background: white; border: 2px solid #8E24AA; color: #8E24AA; border-radius: 6px; cursor: pointer; font-size: 11px; font-weight: 700; box-shadow: 0 1px 2px rgba(0,0,0,0.1); }
        .ruta-btn:hover { background: #F3E5F5; transform: scale(1.05); }
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))

        filtros_js = r'''
        <script>
        function inicializarFiltros() {
            const layerControl = document.querySelector('.leaflet-control-layers');
            if (!layerControl) return;
            const layersList = layerControl.querySelector('.leaflet-control-layers-list');
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
            
            if (layersList.querySelector('.ruta-buttons-row')) return;

            // 0. VISUALIZACIÓN
            const visualDiv = document.createElement('div');
            visualDiv.className = 'visual-control-container';
            visualDiv.innerHTML = `<button class="visual-btn" id="btn-toggle-poly"><span>🟦</span> <span>Ocultar Polígonos</span></button>`;
            const mapContainer = document.querySelector('.leaflet-container');
            const btnPoly = visualDiv.querySelector('#btn-toggle-poly');
            let polysHidden = false;
            btnPoly.onclick = (e) => {
                e.preventDefault(); e.stopPropagation();
                polysHidden = !polysHidden;
                if(polysHidden) {
                    mapContainer.classList.add('hide-all-polygons');
                    btnPoly.innerHTML = '<span>⬜</span> <span>Mostrar Polígonos</span>';
                    btnPoly.classList.add('active');
                } else {
                    mapContainer.classList.remove('hide-all-polygons');
                    btnPoly.innerHTML = '<span>🟦</span> <span>Ocultar Polígonos</span>';
                    btnPoly.classList.remove('active');
                }
            };
            layersList.insertBefore(visualDiv, overlaysDiv);

            // 1. TODO/NADA
            const buttonsDiv = document.createElement('div');
            buttonsDiv.className = 'layer-control-buttons';
            buttonsDiv.innerHTML = '<button class="layer-btn select-all" title="Seleccionar todas las capas">✓ Todo</button><button class="layer-btn deselect-all" title="Deseleccionar todas las capas">✗ Nada</button>';
            buttonsDiv.querySelector('.select-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            buttonsDiv.querySelector('.deselect-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); };
            layersList.insertBefore(buttonsDiv, overlaysDiv);

            // 2. BUSCADOR
            const searchDiv = document.createElement('div');
            searchDiv.className = 'search-ruta-container';
            searchDiv.innerHTML = `
                <span style="font-size: 16px; margin-right: 4px;">🔍</span>
                <input type="text" class="search-ruta-input" placeholder="Ruta ID..." title="Ingresa el número de ruta">
                <button class="search-ruta-btn">Buscar</button>
                <button class="search-ruta-clear">Limpiar</button>
            `;
            const input = searchDiv.querySelector('input');
            const doSearch = () => {
                const val = input.value.trim();
                if(!val) return;
                checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); 
                setTimeout(() => {
                    checkboxes.forEach(cb => {
                        const label = cb.parentElement.textContent;
                        const regex = new RegExp(`- Ruta ${val}$`); // Exact match
                        if(regex.test(label)) { if(!cb.checked) cb.click(); }
                    });
                }, 100);
            };
            const doClear = () => { input.value = ''; checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            searchDiv.querySelector('.search-ruta-btn').onclick = (e) => { e.preventDefault(); doSearch(); };
            searchDiv.querySelector('.search-ruta-clear').onclick = (e) => { e.preventDefault(); doClear(); };
            input.onkeypress = (e) => { if(e.key === 'Enter') { e.preventDefault(); doSearch(); } };
            layersList.insertBefore(searchDiv, overlaysDiv);

            // 3. METS
            const metDiv = document.createElement('div');
            metDiv.className = 'met-buttons-row';
            const mets = new Set();
            checkboxes.forEach(cb => {
                const match = cb.parentElement.textContent.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+/);
                if(match) mets.add(match[1].trim());
            });
            Array.from(mets).sort().forEach(met => {
                const btn = document.createElement('button');
                btn.className = 'met-btn';
                btn.textContent = met.replace('MET ', '');
                btn.onclick = (e) => {
                    e.preventDefault();
                    checkboxes.forEach(cb => { if(cb.checked) cb.click(); });
                    setTimeout(() => {
                        checkboxes.forEach(cb => {
                            if(cb.parentElement.textContent.includes(met + ' -')) if(!cb.checked) cb.click();
                        });
                    }, 100);
                };
                metDiv.appendChild(btn);
            });
            layersList.insertBefore(metDiv, overlaysDiv);

            // 4. RUTAS
            const rutaDiv = document.createElement('div');
            rutaDiv.className = 'ruta-buttons-row';
            const rutasSet = new Set();
            checkboxes.forEach(cb => {
                const label = cb.parentElement.textContent;
                const match = label.match(/- Ruta\s+(\d+)/);
                if(match) rutasSet.add(parseInt(match[1]));
            });
            Array.from(rutasSet).sort((a,b)=>a-b).forEach(rutaId => {
                const btn = document.createElement('button');
                btn.className = 'ruta-btn';
                btn.textContent = 'R' + rutaId;
                btn.onclick = (e) => {
                    e.preventDefault();
                    checkboxes.forEach(cb => { if(cb.checked) cb.click(); });
                    setTimeout(() => {
                        checkboxes.forEach(cb => {
                            const label = cb.parentElement.textContent;
                            const regex = new RegExp(`- Ruta ${rutaId}$`);
                            if(regex.test(label)) { if(!cb.checked) cb.click(); }
                        });
                    }, 100);
                };
                rutaDiv.appendChild(btn);
            });
            layersList.insertBefore(rutaDiv, overlaysDiv);
        }
        function tryInit() {
            if(document.querySelector('.leaflet-control-layers-list')) inicializarFiltros();
            else setTimeout(tryInit, 500);
        }
        setTimeout(tryInit, 1500);
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(filtros_js))
        
        # --- TITULO FLOTANTE ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 10px 20px; border-radius: 20px; box-shadow: 0 5px 15px rgba(0,0,0,0.3); border: 2px solid white; font-family: Arial;">
            <div style="font-size:14px; font-weight:bold; text-align:center;">🗺️ ANÁLISIS DE TERRITORIO</div>
            <div style="font-size:11px; margin-top:4px; opacity:0.9;">ID: {run_id} | Rutas: {len(summary_rows)} | Clientes: {g_total_clientes}</div>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))

        # --- Guardar ---
        map_path = os.path.join(OUTPUT_DIR, f"Mapa_Ceylan_{run_id}.html")
        mapa_base.save(map_path)
        xls_path = generate_consolidated_excel(export_rows, summary_rows, OUTPUT_DIR, filename_suffix=run_id)
        
        print(f"\n✅ FINALIZADO."); print(f"🗺️ Mapa: {os.path.basename(map_path)}"); print(f"📊 Excel: {os.path.basename(xls_path)}")
        btn_run.disabled = False

btn_run.on_click(click_ejecutar)

# --- Layout ---
print("================== CEYLAN ALFA 5.1 (UI FINAL + ETIQUETAS RUTAS) ==================")
display(widgets.VBox([
    widgets.HBox([widgets.Label("1. DATOS:"), upl_widget]), out_upl,
    met_box, widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("2. PARÁMETROS:"), slider_min, slider_max]),
    widgets.HBox([widgets.Label("3. MUESTREO:"), inp_sample, chk_full]),
    widgets.HTML("<br>"),
    btn_run, out_run
]))

================== CEYLAN ALFA 5.1 (UI FINAL + ETIQUETAS RUTAS) ==================


Versión 5.1 con DBSCAN 01/12/2025

In [ ]:
# =========================================================================
# === MÓDULO 1: BACKEND (CON CLASIFICACIÓN DE OUTLIERS) ===================
# =========================================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, MultiPoint, Point
from sklearn.cluster import KMeans, DBSCAN

# --- Configuración ---
BASE_DIR = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan"
ICON_MET_PATH = os.path.join(BASE_DIR, "95_24.png")
OUTPUT_DIR = os.path.join(BASE_DIR, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

class CeylanState:
    def __init__(self):
        self.df_clientes = pd.DataFrame()
        self.df_cdr = pd.DataFrame()
        self.df_clientes_backup = pd.DataFrame()
        self.execution_count = 0
        self.met_checkboxes = []
        self.colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
state = CeylanState()

# --- HELPERS ---
def normalizar_id(valor):
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    return s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0: return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min: return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

def _clip_voronoi(vor, bounding_poly):
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty: return clipped_polys
    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points) or region_index == -1: continue
        region_indices = vor.regions[region_index]
        if not region_indices or -1 in region_indices: continue
        try:
            pts = [vor.vertices[v] for v in region_indices]
            poly = Polygon(pts)
            if not poly.is_valid: poly = poly.buffer(0)
            intersection = poly.intersection(bounding_poly)
            if not intersection.is_empty and intersection.geom_type in ['Polygon', 'MultiPolygon']:
                if intersection.geom_type == 'MultiPolygon': intersection = max(intersection.geoms, key=lambda p: p.area)
                clipped_polys[tuple(vor.points[i])] = [[y, x] for x, y in intersection.exterior.coords]
        except: pass
    return clipped_polys

# --- CLUSTERING ---
def segregar_por_frecuencia(df_clients, threshold=0.01):
    if 'Ruta_nueva' not in df_clients.columns: df_clients['Ruta_nueva'] = -1
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    
    # Marcar banderas iniciales
    df_elegibles['es_comodin'] = False
    df_elegibles['es_outlier'] = False
    
    df_comodines['es_comodin'] = True
    df_comodines['es_outlier'] = False
    
    return df_elegibles, df_comodines

def filtrar_outliers_dbscan(df_core, eps_km=2.0, min_samples=3):
    if len(df_core) < min_samples:
        return df_core, pd.DataFrame(columns=df_core.columns)

    coords_rad = np.radians(df_core[['U_latitud', 'U_longitud']].values)
    eps_rad = eps_km / 6371.0
    
    db = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine').fit(coords_rad)
    mask_outliers = db.labels_ == -1
    
    df_outliers = df_core[mask_outliers].copy()
    df_clean = df_core[~mask_outliers].copy()
    
    if not df_outliers.empty:
        # AQUI ESTA EL CAMBIO: Los marcamos explícitamente como OUTLIER
        df_outliers['es_outlier'] = True
        df_outliers['es_comodin'] = False 
        
    return df_clean, df_outliers

def asignar_comodines_post_clustering(df_pendientes, df_elegibles_asignados):
    if df_pendientes.empty: return df_pendientes
    if df_elegibles_asignados.empty: return df_pendientes
        
    coords_pend = df_pendientes[['U_longitud', 'U_latitud']].values
    coords_eleg = df_elegibles_asignados[['U_longitud', 'U_latitud']].values
    
    dist_matrix = cdist(coords_pend, coords_eleg, metric='euclidean')
    idx_vecinos = np.argmin(dist_matrix, axis=1)
    df_pendientes['Ruta_nueva'] = df_elegibles_asignados.iloc[idx_vecinos]['Ruta_nueva'].values
    
    return df_pendientes

def find_initial_seeds(coords, k, random_state=42):
    if k <= 0 or len(coords) == 0: return np.array([])
    try:
        kmeans = KMeans(n_clusters=min(k, len(coords)), random_state=random_state, n_init=10).fit(coords)
        return kmeans.cluster_centers_
    except:
        return coords[np.random.choice(len(coords), min(k, len(coords)), replace=False)]

def calcular_limites_dinamicos(dispersion, frecuencia, min_base, max_base):
    disp_norm = min(dispersion / 0.5, 1.0)
    frec_norm = min(frecuencia, 1.0)
    factor = disp_norm * (1.0 - frec_norm * 0.5)
    rango = max_base - min_base
    ajuste = int(rango * factor)
    return max(min_base, int(min_base * 0.5)), max_base - ajuste

def grow_clusters_from_seeds(df_elegibles, initial_seeds, max_size, min_size):
    k = len(initial_seeds)
    if k == 0: return df_elegibles, {}
    df = df_elegibles.copy()
    df['Ruta_nueva'] = -1
    ruta_counts = {i: 0 for i in range(k)}
    
    df_anchors = df[df['entregas/día'] >= df['entregas/día'].quantile(0.8)].copy()
    if not df_anchors.empty:
        dists = distance_matrix(df_anchors[['U_longitud', 'U_latitud']].values, initial_seeds)
        for i, (idx, row) in enumerate(df_anchors.iterrows()):
            ordered_routes = np.argsort(dists[i])
            for r_idx in ordered_routes:
                if ruta_counts[r_idx] < max_size:
                    df.loc[idx, 'Ruta_nueva'] = r_idx
                    ruta_counts[r_idx] += 1
                    break
    
    df_rest = df[df['Ruta_nueva'] == -1]
    if not df_rest.empty:
        coords = df_rest[['U_longitud', 'U_latitud']].values
        dists = distance_matrix(coords, initial_seeds)
        for i in range(len(df_rest)):
            best_routes = np.argsort(dists[i])
            for r_idx in best_routes:
                rut_clients = df[df['Ruta_nueva'] == r_idx]
                dispersion = 0.1
                if len(rut_clients) > 1:
                     dispersion = np.mean(cdist(rut_clients[['U_longitud', 'U_latitud']].values, [[0,0]]))
                _, lim_max_dinamico = calcular_limites_dinamicos(dispersion, df_rest.iloc[i]['entregas/día'], min_size, max_size)
                if ruta_counts[r_idx] < lim_max_dinamico:
                    df.loc[df_rest.index[i], 'Ruta_nueva'] = r_idx
                    ruta_counts[r_idx] += 1
                    break
    return df, ruta_counts

def adjust_small_clusters(df_clustered, counts, min_size, max_size):
    rutas_peques = [r for r, c in counts.items() if c < min_size]
    rutas_validas = [r for r, c in counts.items() if c >= min_size]
    if not rutas_peques or not rutas_validas: return df_clustered, counts
    df = df_clustered.copy()
    for ruta_origen in rutas_peques:
        clientes_mover = df[df['Ruta_nueva'] == ruta_origen]
        for idx, row in clientes_mover.iterrows():
            pt = np.array([[row['U_longitud'], row['U_latitud']]])
            best_r, min_d = -1, float('inf')
            for ruta_dest in rutas_validas:
                clients_dest = df[df['Ruta_nueva'] == ruta_dest]
                if clients_dest.empty: continue
                centroid = clients_dest[['U_longitud', 'U_latitud']].mean().values
                d = np.linalg.norm(pt - centroid)
                if d < min_d and counts[ruta_dest] < max_size:
                    min_d, best_r = d, ruta_dest
            if best_r != -1:
                df.loc[idx, 'Ruta_nueva'] = best_r
                counts[best_r] += 1
                counts[ruta_origen] -= 1
    return df, counts

def calculate_cluster_polygons(df, method='voronoi'):
    polygons = {}
    rutas = [r for r in df['Ruta_nueva'].unique() if r >= 0]
    if len(rutas) < 3: method = 'convex_hull'
    centroides = []
    mapa_centroide_ruta = {}
    for r in rutas:
        pts = df[df['Ruta_nueva'] == r][['U_longitud', 'U_latitud']].values
        if len(pts) > 0:
            c = np.mean(pts, axis=0)
            centroides.append(c)
            mapa_centroide_ruta[tuple(c)] = r
    if method == 'voronoi' and len(centroides) >= 4:
        all_pts = df[df['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
        hull = MultiPoint(all_pts).convex_hull.buffer(0.001)
        vor = Voronoi(centroides)
        clipped = _clip_voronoi(vor, hull)
        for cent_tuple, poly_coords in clipped.items():
            if cent_tuple in mapa_centroide_ruta: polygons[mapa_centroide_ruta[cent_tuple]] = poly_coords
    for r in rutas:
        if r not in polygons:
            pts = df[df['Ruta_nueva'] == r][['U_latitud', 'U_longitud']].values
            if len(pts) >= 3:
                try: polygons[r] = pts[ConvexHull(pts).vertices].tolist()
                except: pass
    return polygons

def procesar_logica_met(met_id, n_muestra, analizar_todos, min_size, max_size):
    df_terr = state.df_clientes[state.df_clientes['MET_Asignado'] == met_id].copy()
    if df_terr.empty: return None, None, "Sin clientes"
    df_proc = df_terr.copy() if analizar_todos or len(df_terr) <= n_muestra else df_terr.sample(n=n_muestra).copy()
    
    df_proc['entregas/día'] = pd.to_numeric(df_proc['entregas/día'], errors='coerce').fillna(0)
    df_proc['m3/día'] = pd.to_numeric(df_proc['m3/día'], errors='coerce').fillna(0)
    
    coords = df_proc[['U_longitud', 'U_latitud']].values
    if len(coords) == 0: return None, None, "Sin coordenadas"
    
    k_init = max(1, len(df_proc) // max_size)
    seeds = find_initial_seeds(coords, k_init)
    
    # 1. Segregar (Normales vs Comodines)
    df_eleg, df_como = segregar_por_frecuencia(df_proc)
    
    # 2. DBSCAN: Sacar Outliers de la lista de Elegibles
    df_eleg, df_outliers = filtrar_outliers_dbscan(df_eleg, eps_km=3.0, min_samples=3)
    
    # 3. Clustering (Solo con el núcleo limpio)
    df_eleg, counts = grow_clusters_from_seeds(df_eleg, seeds, max_size, min_size)
    df_eleg, counts = adjust_small_clusters(df_eleg, counts, min_size, max_size)
    
    # 4. Asignar Comodines y Outliers (Juntos al final, pero con banderas distintas)
    df_pendientes = pd.concat([df_como, df_outliers], ignore_index=True)
    df_pendientes = asignar_comodines_post_clustering(df_pendientes, df_eleg)
    
    df_final = pd.concat([df_eleg, df_pendientes], ignore_index=True)
    polys = calculate_cluster_polygons(df_final)
    return df_final, polys, "OK"

# --- VISUALIZACIÓN VACÍA (Se llena en Módulo 2) ---
def get_full_css(): return ""
def get_full_js(): return ""
def inject_custom_assets(map_object): pass 
def generate_consolidated_excel(export_data, summary_data, output_dir, filename_suffix=None):
    if not export_data: return None
    if not filename_suffix: filename_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_dir, f'Resultados_Ceylan_{filename_suffix}.xlsx')
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        pd.DataFrame(summary_data).to_excel(writer, sheet_name='Resumen', index=False)
        pd.DataFrame(export_data).to_excel(writer, sheet_name='Detalle', index=False)
    return path

# --- HANDLER ---
def handle_upload(change, met_box, output_upload):
    with output_upload:
        clear_output()
        if not change['new']: return
        try:
            upl_file = change['owner'].value
            content = upl_file[0]['content'] if isinstance(upl_file, tuple) else list(upl_file.values())[0]['content']
            temp_path = "temp_upload.xlsx"
            with open(temp_path, 'wb') as f: f.write(content)
            xls = pd.read_excel(temp_path, sheet_name=None)
            state.df_clientes = xls[list(xls.keys())[0]]
            state.df_cdr = xls[list(xls.keys())[1]]
            
            # Limpieza Numérica
            state.df_clientes['m3/día'] = pd.to_numeric(state.df_clientes['m3/día'], errors='coerce').fillna(0)
            state.df_clientes['entregas/día'] = pd.to_numeric(state.df_clientes['entregas/día'], errors='coerce').fillna(0)
            
            # Anti-Anomalías
            mask_freq = state.df_clientes['entregas/día'] > 50
            if mask_freq.any(): state.df_clientes.loc[mask_freq, 'entregas/día'] = 0
            
            state.df_clientes_backup = state.df_clientes.copy()
            mets = sorted(state.df_cdr['CodMET'].unique())
            state.met_checkboxes = [widgets.Checkbox(description=str(m), value=False) for m in mets]
            met_box.children = [widgets.Label("Selecciona METs:")] + state.met_checkboxes
            print(f"✅ Cargado y Saneado. Clientes: {len(state.df_clientes)}")
            os.remove(temp_path)
        except Exception as e:
            print(f"❌ Error: {e}")

print("✅ MÓDULO 1: Backend con Outliers Explícitos.")

✅ MÓDULO 1: Backend con Outliers Explícitos.


In [ ]:
# =========================================================================
# === MÓDULO 2: EJECUCIÓN FINAL (ETIQUETAS RUTAS MEJORADAS + PANEL) =======
# =========================================================================

# --- Widgets ---
upl_widget = widgets.FileUpload(accept='.xlsx', description='Subir Excel')
met_box = widgets.VBox([widgets.Label('Esperando archivo...')])
out_upl = widgets.Output()

slider_min = widgets.IntSlider(value=20, min=5, max=50, description='Min Clientes')
slider_max = widgets.IntSlider(value=40, min=10, max=100, description='Max Clientes')
inp_sample = widgets.IntText(value=500, description='Muestra/MET')
chk_full = widgets.Checkbox(value=False, description='Analizar TODO')

btn_run = widgets.Button(description='🚀 Ejecutar Análisis', button_style='success', layout=widgets.Layout(width='100%'))
out_run = widgets.Output()

# Conectar carga
upl_widget.observe(lambda c: handle_upload(c, met_box, out_upl), names='value')

# --- LÓGICA PRINCIPAL ---
def click_ejecutar(b):
    state.execution_count += 1
    btn_run.disabled = True
    
    with out_run:
        clear_output()
        
        # 1. Validaciones
        sel_mets = [cb.description for cb in state.met_checkboxes if cb.value]
        if state.df_clientes.empty or not sel_mets:
            print("⚠️ Error: Carga un archivo y selecciona al menos un MET.")
            btn_run.disabled = False; return

        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f"🚀 INICIANDO ANÁLISIS - ID: {run_id}")
        
        # 2. Restaurar Backup
        state.df_clientes = state.df_clientes_backup.copy()
        
        # 3. Asignación de Territorios
        print("🗺️ Asignando territorios...")
        coords_cli = state.df_clientes[['U_latitud', 'U_longitud']].values
        coords_met = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)][['U_latitud', 'U_longitud']].values
        met_names = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)]['CodMET'].values
        dists = cdist(coords_cli, coords_met)
        state.df_clientes['MET_Asignado'] = met_names[np.argmin(dists, axis=1)]
        
        # 4. Procesamiento
        # --- ZOOM AUTOMÁTICO ---
        df_zoom = state.df_clientes[state.df_clientes['MET_Asignado'].isin(sel_mets)]
        if not df_zoom.empty:
            sw = df_zoom[['U_latitud', 'U_longitud']].min().values.tolist()
            ne = df_zoom[['U_latitud', 'U_longitud']].max().values.tolist()
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            mapa_base.fit_bounds([sw, ne])
        else:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')

        export_rows, summary_rows = [], []
        g_total_clientes = 0
        
        for met in sel_mets:
            print(f"🔨 Procesando {met}...", end=" ")
            
            # Datos MET
            met_info = state.df_cdr[state.df_cdr['CodMET'] == met].iloc[0]
            met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
            
            # Lógica M1
            df_res, polys, status = procesar_logica_met(
                met, inp_sample.value, chk_full.value, 
                slider_min.value, slider_max.value
            )
            
            if status != "OK": print(f"({status})"); continue
                
            rutas_ids = sorted([r for r in df_res['Ruta_nueva'].unique() if r >= 0])
            print(f"✅ {len(rutas_ids)} rutas.")
            g_total_clientes += len(df_res)
            
            # Rangos volumen
            df_normales = df_res[df_res.get('es_comodin', False) == False]
            if not df_normales.empty:
                vol_min_local = df_normales['m3/día'].min()
                vol_max_local = df_normales['m3/día'].max()
            else:
                vol_min_local, vol_max_local = 0, 1
            
            # Estadísticas MET
            volumen_total_met = df_res['m3/día'].sum()
            total_clientes_met = len(df_res)
            met_display_name = str(met).strip()
            
            # --- C. DIBUJAR ICONO DEL MET ---
            fg_met_home = folium.FeatureGroup(name=f"{met_display_name} - 🏠 MET", show=True, attribution=f"{met_display_name} - 🏠 MET")
            
            popup_met = f"""
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                  <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                      <div style="font-size: 22px; font-weight: bold;">🏠 {met}</div>
                  </div>
                  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                      <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                          <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                          <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                      </div>
                      <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                          <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                          <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                      </div>
                  </div>
                  <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                      <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                      <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_ids)}</div>
                  </div>
            </div>
            """
            
            if os.path.exists(ICON_MET_PATH):
                icon_obj = CustomIcon(ICON_MET_PATH, icon_size=(40, 40), icon_anchor=(20, 40))
                folium.Marker([met_lat, met_lon], icon=icon_obj, popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            else:
                folium.Marker([met_lat, met_lon], icon=folium.Icon(color='green', icon='home'), popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            fg_met_home.add_to(mapa_base)

            # --- D. DIBUJAR RUTAS Y CLIENTES ---
            for ruta_id in rutas_ids:
                ruta_df = df_res[df_res['Ruta_nueva'] == ruta_id]
                vol_ruta = ruta_df['m3/día'].sum()
                frec_ruta = ruta_df['entregas/día'].mean() * 100
                
                # CÁLCULOS PARA ETIQUETA
                n_total = len(ruta_df)
                n_comodines = len(ruta_df[ruta_df.get('es_comodin', False) == True])
                n_normales = n_total - n_comodines
                
                layer_name = f"{met_display_name} - Ruta {ruta_id}"
                fg_ruta = folium.FeatureGroup(name=layer_name, show=True, attribution=layer_name)
                color_ruta = state.colores[ruta_id % len(state.colores)]
                
                # --- ETIQUETA ESTILIZADA DE LA RUTA (Tooltip) ---
                ruta_tooltip_html = f"""
                <div style="font-family: 'Segoe UI', sans-serif; min-width: 160px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); border-radius: 6px; overflow: hidden; background: white;">
                    <div style="background-color: {color_ruta}; color: white; padding: 6px 10px; font-weight: bold; font-size: 13px;">
                        🚛 Ruta {ruta_id}
                    </div>
                    <div style="padding: 8px; font-size: 11px; color: #333; line-height: 1.5;">
                        <div style="border-bottom: 1px solid #eee; padding-bottom: 4px; margin-bottom: 4px;">
                            <div>📦 <b>Volumen:</b> {vol_ruta:.2f} m³</div>
                            <div>👥 <b>Clientes:</b> {n_total}</div>
                        </div>
                        <div style="display: flex; justify-content: space-between; background: #f8f9fa; padding: 4px; border-radius: 4px; font-weight: 600;">
                            <span style="color: #2E7D32;">✅ Nor: {n_normales}</span>
                            <span style="color: #C62828;">🔺 Com: {n_comodines}</span>
                        </div>
                    </div>
                </div>
                """

                # --- POLÍGONO ---
                if ruta_id in polys and polys[ruta_id]:
                    raw_poly = polys[ruta_id]
                    final_poly = raw_poly
                    # Corrección Antártida
                    if raw_poly and len(raw_poly) > 0:
                        p1 = raw_poly[0]
                        if p1[0] < p1[1]: # Si Lon < Lat (América), invertir a [Lat, Lon]
                            final_poly = [[c[1], c[0]] for c in raw_poly]
                    
                    folium.Polygon(
                        locations=final_poly,
                        color=color_ruta, weight=2, fill=True, fill_opacity=0.2,
                        tooltip=folium.Tooltip(ruta_tooltip_html, sticky=True), # <--- ETIQUETA NUEVA AQUÍ
                        className='polygon-shape'
                    ).add_to(fg_ruta)
                
                # Marcadores
                for _, row in ruta_df.iterrows():
                    es_comodin = row.get('es_comodin', False)
                    vol = row['m3/día']
                    frec = row['entregas/día'] * 100
                    
                    popup_html = f"""
                    <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_ruta}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                         <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                             {'🔺 COMODÍN' if es_comodin else '✅ NORMAL'} | {row['CodCli']}
                         </div>
                         <div style="font-size:14px; color:#333; margin-bottom:4px;">
                             {row.get('Nombre', 'N/A')}
                         </div>
                         <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                             📦 Volumen: <b>{vol:.3f} m³</b>
                         </div>
                         <div style="font-size:14px; color:{'#888' if es_comodin else '#1976D2'}; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                             📊 Frecuencia: <b>{frec:.1f}%</b> (entregas/día)
                         </div>
                         <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                             🗺️ Ruta (Orig): <b>{row.get('Ruta', 'N/A')}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                         </div>
                    </div>
                    """
                    
                    if es_comodin:
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], 
                            radius=4,
                            color='#D32F2F', fill=True, fill_color='#FFCDD2', fill_opacity=0.9,
                            popup=folium.Popup(popup_html, max_width=320)
                        ).add_to(fg_ruta)
                    else:
                        tamano_calc = obtener_tamano_marcador(vol, vol_min_local, vol_max_local)
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], 
                            radius=tamano_calc,
                            color='#333333', weight=1, fill=True,
                            fill_color=color_ruta, fill_opacity=0.85,
                            popup=folium.Popup(popup_html, max_width=320)
                        ).add_to(fg_ruta)
                    
                    export_rows.append(row.to_dict())
                
                fg_ruta.add_to(mapa_base)
                summary_rows.append({'MET': met, 'Ruta': ruta_id, 'Clientes': len(ruta_df)})

        # --- INYECCIÓN DE ESTILOS Y SCRIPTS (ESTÉTICA PREMIUM EXACTA) ---
        folium.LayerControl(collapsed=False).add_to(mapa_base)

        estilos_css = r'''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 600px; overflow-y: auto; min-width: 340px; padding: 12px; font-family: 'Segoe UI', Arial, sans-serif; background: #fff; }
        
        /* CLASE PARA OCULTAR POLÍGONOS */
        .hide-all-polygons .polygon-shape { stroke: none !important; fill: none !important; pointer-events: none !important; }

        /* VISUALIZACIÓN */
        .visual-control-container { display: flex; gap: 8px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 12px; }
        .visual-btn { flex: 1; padding: 8px; border: 1px solid #ccc; background: white; cursor: pointer; border-radius: 6px; font-weight: 700; font-size: 12px; color: #555; box-shadow: 0 2px 4px rgba(0,0,0,0.1); display: flex; align-items: center; justify-content: center; gap: 5px; }
        .visual-btn:hover { background-color: #f5f5f5; }
        .visual-btn.active { background-color: #e3f2fd; color: #1976D2; border-color: #2196F3; }

        /* 1. Botones TODO / NADA (Sólidos) */
        .layer-control-buttons { display: flex; gap: 8px; margin-bottom: 12px; }
        .layer-btn { flex: 1; padding: 8px 0; font-size: 13px; font-weight: bold; border: none; border-radius: 4px; cursor: pointer; color: white; box-shadow: 0 2px 4px rgba(0,0,0,0.2); }
        .select-all { background-color: #4CAF50; } 
        .select-all:hover { background-color: #43A047; }
        .deselect-all { background-color: #F44336; } 
        .deselect-all:hover { background-color: #E53935; }

        /* 2. BUSCADOR */
        .search-ruta-container { display: flex; gap: 6px; align-items: center; margin-bottom: 12px; padding: 4px 0; }
        .search-ruta-input { flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 4px; font-size: 13px; outline: none; }
        .search-ruta-input:focus { border-color: #0275d8; box-shadow: 0 0 0 2px rgba(2,117,216,0.1); }
        .search-ruta-btn { background: #0275d8; color: white; border: none; padding: 8px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }
        .search-ruta-clear { background: white; color: #dc3545; border: 1px solid #dc3545; padding: 8px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }

        /* 3. METS (Borde Azul Grueso) */
        .met-buttons-row { display: flex; gap: 8px; margin-bottom: 12px; }
        .met-btn { flex: 1; padding: 8px; font-size: 12px; font-weight: 700; color: #0275d8; background: white; border: 2px solid #0275d8; border-radius: 6px; cursor: pointer; transition: all 0.1s; }
        .met-btn:hover { background: #e3f2fd; }

        /* 4. RUTAS (Fondo Lila, Grid 6) */
        .ruta-buttons-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 6px; background: #E1BEE7; padding: 10px; border-radius: 8px; }
        .ruta-btn { padding: 8px 0; text-align: center; background: white; border: 2px solid #8E24AA; color: #8E24AA; border-radius: 6px; cursor: pointer; font-size: 11px; font-weight: 700; box-shadow: 0 1px 2px rgba(0,0,0,0.1); }
        .ruta-btn:hover { background: #F3E5F5; transform: scale(1.05); }
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))

        filtros_js = r'''
        <script>
        function inicializarFiltros() {
            const layerControl = document.querySelector('.leaflet-control-layers');
            if (!layerControl) return;
            const layersList = layerControl.querySelector('.leaflet-control-layers-list');
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
            
            if (layersList.querySelector('.ruta-buttons-row')) return;

            // 0. VISUALIZACIÓN
            const visualDiv = document.createElement('div');
            visualDiv.className = 'visual-control-container';
            visualDiv.innerHTML = `<button class="visual-btn" id="btn-toggle-poly"><span>🟦</span> <span>Ocultar Polígonos</span></button>`;
            const mapContainer = document.querySelector('.leaflet-container');
            const btnPoly = visualDiv.querySelector('#btn-toggle-poly');
            let polysHidden = false;
            btnPoly.onclick = (e) => {
                e.preventDefault(); e.stopPropagation();
                polysHidden = !polysHidden;
                if(polysHidden) {
                    mapContainer.classList.add('hide-all-polygons');
                    btnPoly.innerHTML = '<span>⬜</span> <span>Mostrar Polígonos</span>';
                    btnPoly.classList.add('active');
                } else {
                    mapContainer.classList.remove('hide-all-polygons');
                    btnPoly.innerHTML = '<span>🟦</span> <span>Ocultar Polígonos</span>';
                    btnPoly.classList.remove('active');
                }
            };
            layersList.insertBefore(visualDiv, overlaysDiv);

            // 1. TODO/NADA
            const buttonsDiv = document.createElement('div');
            buttonsDiv.className = 'layer-control-buttons';
            buttonsDiv.innerHTML = '<button class="layer-btn select-all" title="Seleccionar todas las capas">✓ Todo</button><button class="layer-btn deselect-all" title="Deseleccionar todas las capas">✗ Nada</button>';
            buttonsDiv.querySelector('.select-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            buttonsDiv.querySelector('.deselect-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); };
            layersList.insertBefore(buttonsDiv, overlaysDiv);

            // 2. BUSCADOR
            const searchDiv = document.createElement('div');
            searchDiv.className = 'search-ruta-container';
            searchDiv.innerHTML = `
                <span style="font-size: 16px; margin-right: 4px;">🔍</span>
                <input type="text" class="search-ruta-input" placeholder="Ruta ID..." title="Ingresa el número de ruta">
                <button class="search-ruta-btn">Buscar</button>
                <button class="search-ruta-clear">Limpiar</button>
            `;
            const input = searchDiv.querySelector('input');
            const doSearch = () => {
                const val = input.value.trim();
                if(!val) return;
                checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); 
                setTimeout(() => {
                    checkboxes.forEach(cb => {
                        const label = cb.parentElement.textContent;
                        const regex = new RegExp(`- Ruta ${val}$`); // Exact match
                        if(regex.test(label)) { if(!cb.checked) cb.click(); }
                    });
                }, 100);
            };
            const doClear = () => { input.value = ''; checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            searchDiv.querySelector('.search-ruta-btn').onclick = (e) => { e.preventDefault(); doSearch(); };
            searchDiv.querySelector('.search-ruta-clear').onclick = (e) => { e.preventDefault(); doClear(); };
            input.onkeypress = (e) => { if(e.key === 'Enter') { e.preventDefault(); doSearch(); } };
            layersList.insertBefore(searchDiv, overlaysDiv);

            // 3. METS
            const metDiv = document.createElement('div');
            metDiv.className = 'met-buttons-row';
            const mets = new Set();
            checkboxes.forEach(cb => {
                const match = cb.parentElement.textContent.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+/);
                if(match) mets.add(match[1].trim());
            });
            Array.from(mets).sort().forEach(met => {
                const btn = document.createElement('button');
                btn.className = 'met-btn';
                btn.textContent = met.replace('MET ', '');
                btn.onclick = (e) => {
                    e.preventDefault();
                    checkboxes.forEach(cb => { if(cb.checked) cb.click(); });
                    setTimeout(() => {
                        checkboxes.forEach(cb => {
                            if(cb.parentElement.textContent.includes(met + ' -')) if(!cb.checked) cb.click();
                        });
                    }, 100);
                };
                metDiv.appendChild(btn);
            });
            layersList.insertBefore(metDiv, overlaysDiv);

            // 4. RUTAS
            const rutaDiv = document.createElement('div');
            rutaDiv.className = 'ruta-buttons-row';
            const rutasSet = new Set();
            checkboxes.forEach(cb => {
                const label = cb.parentElement.textContent;
                const match = label.match(/- Ruta\s+(\d+)/);
                if(match) rutasSet.add(parseInt(match[1]));
            });
            Array.from(rutasSet).sort((a,b)=>a-b).forEach(rutaId => {
                const btn = document.createElement('button');
                btn.className = 'ruta-btn';
                btn.textContent = 'R' + rutaId;
                btn.onclick = (e) => {
                    e.preventDefault();
                    checkboxes.forEach(cb => { if(cb.checked) cb.click(); });
                    setTimeout(() => {
                        checkboxes.forEach(cb => {
                            const label = cb.parentElement.textContent;
                            const regex = new RegExp(`- Ruta ${rutaId}$`);
                            if(regex.test(label)) { if(!cb.checked) cb.click(); }
                        });
                    }, 100);
                };
                rutaDiv.appendChild(btn);
            });
            layersList.insertBefore(rutaDiv, overlaysDiv);
        }
        function tryInit() {
            if(document.querySelector('.leaflet-control-layers-list')) inicializarFiltros();
            else setTimeout(tryInit, 500);
        }
        setTimeout(tryInit, 1500);
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(filtros_js))
        
        # --- TITULO FLOTANTE ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 10px 20px; border-radius: 20px; box-shadow: 0 5px 15px rgba(0,0,0,0.3); border: 2px solid white; font-family: Arial;">
            <div style="font-size:14px; font-weight:bold; text-align:center;">🗺️ ANÁLISIS DE TERRITORIO</div>
            <div style="font-size:11px; margin-top:4px; opacity:0.9;">ID: {run_id} | Rutas: {len(summary_rows)} | Clientes: {g_total_clientes}</div>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))

        # --- Guardar ---
        map_path = os.path.join(OUTPUT_DIR, f"Mapa_Ceylan_{run_id}.html")
        mapa_base.save(map_path)
        xls_path = generate_consolidated_excel(export_rows, summary_rows, OUTPUT_DIR, filename_suffix=run_id)
        
        print(f"\n✅ FINALIZADO."); print(f"🗺️ Mapa: {os.path.basename(map_path)}"); print(f"📊 Excel: {os.path.basename(xls_path)}")
        btn_run.disabled = False

btn_run.on_click(click_ejecutar)

# --- Layout ---
print("================== CEYLAN ALFA 5.1 (UI FINAL + ETIQUETAS RUTAS) ==================")
display(widgets.VBox([
    widgets.HBox([widgets.Label("1. DATOS:"), upl_widget]), out_upl,
    met_box, widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("2. PARÁMETROS:"), slider_min, slider_max]),
    widgets.HBox([widgets.Label("3. MUESTREO:"), inp_sample, chk_full]),
    widgets.HTML("<br>"),
    btn_run, out_run
]))

================== CEYLAN ALFA 5.1 (UI FINAL + ETIQUETAS RUTAS) ==================


Versión 5.1 con DBSCAN y filtros de outliers y poligonos 01/12/2025

In [1]:
# =========================================================================
# === MÓDULO 1: BACKEND (CON CLASIFICACIÓN DE OUTLIERS) ===================
# =========================================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, MultiPoint, Point
from sklearn.cluster import KMeans, DBSCAN

# --- Configuración ---
BASE_DIR = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan"
ICON_MET_PATH = os.path.join(BASE_DIR, "95_24.png")
OUTPUT_DIR = os.path.join(BASE_DIR, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

class CeylanState:
    def __init__(self):
        self.df_clientes = pd.DataFrame()
        self.df_cdr = pd.DataFrame()
        self.df_clientes_backup = pd.DataFrame()
        self.execution_count = 0
        self.met_checkboxes = []
        self.colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
state = CeylanState()

# --- HELPERS ---
def normalizar_id(valor):
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    return s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0: return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min: return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

def _clip_voronoi(vor, bounding_poly):
    clipped_polys = {}
    if vor is None or bounding_poly is None or bounding_poly.is_empty: return clipped_polys
    for i, region_index in enumerate(vor.point_region):
        if i >= len(vor.points) or region_index == -1: continue
        region_indices = vor.regions[region_index]
        if not region_indices or -1 in region_indices: continue
        try:
            pts = [vor.vertices[v] for v in region_indices]
            poly = Polygon(pts)
            if not poly.is_valid: poly = poly.buffer(0)
            intersection = poly.intersection(bounding_poly)
            if not intersection.is_empty and intersection.geom_type in ['Polygon', 'MultiPolygon']:
                if intersection.geom_type == 'MultiPolygon': intersection = max(intersection.geoms, key=lambda p: p.area)
                clipped_polys[tuple(vor.points[i])] = [[y, x] for x, y in intersection.exterior.coords]
        except: pass
    return clipped_polys

# --- CLUSTERING ---
def segregar_por_frecuencia(df_clients, threshold=0.01):
    if 'Ruta_nueva' not in df_clients.columns: df_clients['Ruta_nueva'] = -1
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    
    # Marcar banderas iniciales
    df_elegibles['es_comodin'] = False
    df_elegibles['es_outlier'] = False
    
    df_comodines['es_comodin'] = True
    df_comodines['es_outlier'] = False
    
    return df_elegibles, df_comodines

def filtrar_outliers_dbscan(df_core, eps_km=2.0, min_samples=3):
    if len(df_core) < min_samples:
        return df_core, pd.DataFrame(columns=df_core.columns)

    coords_rad = np.radians(df_core[['U_latitud', 'U_longitud']].values)
    eps_rad = eps_km / 6371.0
    
    db = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine').fit(coords_rad)
    mask_outliers = db.labels_ == -1
    
    df_outliers = df_core[mask_outliers].copy()
    df_clean = df_core[~mask_outliers].copy()
    
    if not df_outliers.empty:
        # AQUI ESTA EL CAMBIO: Los marcamos explícitamente como OUTLIER
        df_outliers['es_outlier'] = True
        df_outliers['es_comodin'] = False 
        
    return df_clean, df_outliers

def asignar_comodines_post_clustering(df_pendientes, df_elegibles_asignados):
    if df_pendientes.empty: return df_pendientes
    if df_elegibles_asignados.empty: return df_pendientes
        
    coords_pend = df_pendientes[['U_longitud', 'U_latitud']].values
    coords_eleg = df_elegibles_asignados[['U_longitud', 'U_latitud']].values
    
    dist_matrix = cdist(coords_pend, coords_eleg, metric='euclidean')
    idx_vecinos = np.argmin(dist_matrix, axis=1)
    df_pendientes['Ruta_nueva'] = df_elegibles_asignados.iloc[idx_vecinos]['Ruta_nueva'].values
    
    return df_pendientes

def find_initial_seeds(coords, k, random_state=42):
    if k <= 0 or len(coords) == 0: return np.array([])
    try:
        kmeans = KMeans(n_clusters=min(k, len(coords)), random_state=random_state, n_init=10).fit(coords)
        return kmeans.cluster_centers_
    except:
        return coords[np.random.choice(len(coords), min(k, len(coords)), replace=False)]

def calcular_limites_dinamicos(dispersion, frecuencia, min_base, max_base):
    disp_norm = min(dispersion / 0.5, 1.0)
    frec_norm = min(frecuencia, 1.0)
    factor = disp_norm * (1.0 - frec_norm * 0.5)
    rango = max_base - min_base
    ajuste = int(rango * factor)
    return max(min_base, int(min_base * 0.5)), max_base - ajuste

def grow_clusters_from_seeds(df_elegibles, initial_seeds, max_size, min_size):
    k = len(initial_seeds)
    if k == 0: return df_elegibles, {}
    df = df_elegibles.copy()
    df['Ruta_nueva'] = -1
    ruta_counts = {i: 0 for i in range(k)}
    
    df_anchors = df[df['entregas/día'] >= df['entregas/día'].quantile(0.8)].copy()
    if not df_anchors.empty:
        dists = distance_matrix(df_anchors[['U_longitud', 'U_latitud']].values, initial_seeds)
        for i, (idx, row) in enumerate(df_anchors.iterrows()):
            ordered_routes = np.argsort(dists[i])
            for r_idx in ordered_routes:
                if ruta_counts[r_idx] < max_size:
                    df.loc[idx, 'Ruta_nueva'] = r_idx
                    ruta_counts[r_idx] += 1
                    break
    
    df_rest = df[df['Ruta_nueva'] == -1]
    if not df_rest.empty:
        coords = df_rest[['U_longitud', 'U_latitud']].values
        dists = distance_matrix(coords, initial_seeds)
        for i in range(len(df_rest)):
            best_routes = np.argsort(dists[i])
            for r_idx in best_routes:
                rut_clients = df[df['Ruta_nueva'] == r_idx]
                dispersion = 0.1
                if len(rut_clients) > 1:
                     dispersion = np.mean(cdist(rut_clients[['U_longitud', 'U_latitud']].values, [[0,0]]))
                _, lim_max_dinamico = calcular_limites_dinamicos(dispersion, df_rest.iloc[i]['entregas/día'], min_size, max_size)
                if ruta_counts[r_idx] < lim_max_dinamico:
                    df.loc[df_rest.index[i], 'Ruta_nueva'] = r_idx
                    ruta_counts[r_idx] += 1
                    break
    return df, ruta_counts

def adjust_small_clusters(df_clustered, counts, min_size, max_size):
    rutas_peques = [r for r, c in counts.items() if c < min_size]
    rutas_validas = [r for r, c in counts.items() if c >= min_size]
    if not rutas_peques or not rutas_validas: return df_clustered, counts
    df = df_clustered.copy()
    for ruta_origen in rutas_peques:
        clientes_mover = df[df['Ruta_nueva'] == ruta_origen]
        for idx, row in clientes_mover.iterrows():
            pt = np.array([[row['U_longitud'], row['U_latitud']]])
            best_r, min_d = -1, float('inf')
            for ruta_dest in rutas_validas:
                clients_dest = df[df['Ruta_nueva'] == ruta_dest]
                if clients_dest.empty: continue
                centroid = clients_dest[['U_longitud', 'U_latitud']].mean().values
                d = np.linalg.norm(pt - centroid)
                if d < min_d and counts[ruta_dest] < max_size:
                    min_d, best_r = d, ruta_dest
            if best_r != -1:
                df.loc[idx, 'Ruta_nueva'] = best_r
                counts[best_r] += 1
                counts[ruta_origen] -= 1
    return df, counts

def calculate_cluster_polygons(df, method='voronoi'):
    polygons = {}
    rutas = [r for r in df['Ruta_nueva'].unique() if r >= 0]
    if len(rutas) < 3: method = 'convex_hull'
    centroides = []
    mapa_centroide_ruta = {}
    for r in rutas:
        pts = df[df['Ruta_nueva'] == r][['U_longitud', 'U_latitud']].values
        if len(pts) > 0:
            c = np.mean(pts, axis=0)
            centroides.append(c)
            mapa_centroide_ruta[tuple(c)] = r
    if method == 'voronoi' and len(centroides) >= 4:
        all_pts = df[df['Ruta_nueva'] != -1][['U_longitud', 'U_latitud']].values
        hull = MultiPoint(all_pts).convex_hull.buffer(0.001)
        vor = Voronoi(centroides)
        clipped = _clip_voronoi(vor, hull)
        for cent_tuple, poly_coords in clipped.items():
            if cent_tuple in mapa_centroide_ruta: polygons[mapa_centroide_ruta[cent_tuple]] = poly_coords
    for r in rutas:
        if r not in polygons:
            pts = df[df['Ruta_nueva'] == r][['U_latitud', 'U_longitud']].values
            if len(pts) >= 3:
                try: polygons[r] = pts[ConvexHull(pts).vertices].tolist()
                except: pass
    return polygons

def procesar_logica_met(met_id, n_muestra, analizar_todos, min_size, max_size):
    df_terr = state.df_clientes[state.df_clientes['MET_Asignado'] == met_id].copy()
    if df_terr.empty: return None, None, "Sin clientes"
    df_proc = df_terr.copy() if analizar_todos or len(df_terr) <= n_muestra else df_terr.sample(n=n_muestra).copy()
    
    df_proc['entregas/día'] = pd.to_numeric(df_proc['entregas/día'], errors='coerce').fillna(0)
    df_proc['m3/día'] = pd.to_numeric(df_proc['m3/día'], errors='coerce').fillna(0)
    
    coords = df_proc[['U_longitud', 'U_latitud']].values
    if len(coords) == 0: return None, None, "Sin coordenadas"
    
    k_init = max(1, len(df_proc) // max_size)
    seeds = find_initial_seeds(coords, k_init)
    
    # 1. Segregar (Normales vs Comodines)
    df_eleg, df_como = segregar_por_frecuencia(df_proc)
    
    # 2. DBSCAN: Sacar Outliers de la lista de Elegibles
    df_eleg, df_outliers = filtrar_outliers_dbscan(df_eleg, eps_km=3.0, min_samples=3)
    
    # 3. Clustering (Solo con el núcleo limpio)
    df_eleg, counts = grow_clusters_from_seeds(df_eleg, seeds, max_size, min_size)
    df_eleg, counts = adjust_small_clusters(df_eleg, counts, min_size, max_size)
    
    # 4. Asignar Comodines y Outliers (Juntos al final, pero con banderas distintas)
    df_pendientes = pd.concat([df_como, df_outliers], ignore_index=True)
    df_pendientes = asignar_comodines_post_clustering(df_pendientes, df_eleg)
    
    df_final = pd.concat([df_eleg, df_pendientes], ignore_index=True)
    polys = calculate_cluster_polygons(df_final)
    return df_final, polys, "OK"

# --- VISUALIZACIÓN VACÍA (Se llena en Módulo 2) ---
def get_full_css(): return ""
def get_full_js(): return ""
def inject_custom_assets(map_object): pass 
def generate_consolidated_excel(export_data, summary_data, output_dir, filename_suffix=None):
    if not export_data: return None
    if not filename_suffix: filename_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_dir, f'Resultados_Ceylan_{filename_suffix}.xlsx')
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        pd.DataFrame(summary_data).to_excel(writer, sheet_name='Resumen', index=False)
        pd.DataFrame(export_data).to_excel(writer, sheet_name='Detalle', index=False)
    return path

# --- HANDLER ---
def handle_upload(change, met_box, output_upload):
    with output_upload:
        clear_output()
        if not change['new']: return
        try:
            upl_file = change['owner'].value
            content = upl_file[0]['content'] if isinstance(upl_file, tuple) else list(upl_file.values())[0]['content']
            temp_path = "temp_upload.xlsx"
            with open(temp_path, 'wb') as f: f.write(content)
            xls = pd.read_excel(temp_path, sheet_name=None)
            state.df_clientes = xls[list(xls.keys())[0]]
            state.df_cdr = xls[list(xls.keys())[1]]
            
            # Limpieza Numérica
            state.df_clientes['m3/día'] = pd.to_numeric(state.df_clientes['m3/día'], errors='coerce').fillna(0)
            state.df_clientes['entregas/día'] = pd.to_numeric(state.df_clientes['entregas/día'], errors='coerce').fillna(0)
            
            # Anti-Anomalías
            mask_freq = state.df_clientes['entregas/día'] > 50
            if mask_freq.any(): state.df_clientes.loc[mask_freq, 'entregas/día'] = 0
            
            state.df_clientes_backup = state.df_clientes.copy()
            mets = sorted(state.df_cdr['CodMET'].unique())
            state.met_checkboxes = [widgets.Checkbox(description=str(m), value=False) for m in mets]
            met_box.children = [widgets.Label("Selecciona METs:")] + state.met_checkboxes
            print(f"✅ Cargado y Saneado. Clientes: {len(state.df_clientes)}")
            os.remove(temp_path)
        except Exception as e:
            print(f"❌ Error: {e}")

print("✅ MÓDULO 1: Backend con Outliers Explícitos.")

✅ MÓDULO 1: Backend con Outliers Explícitos.


In [2]:
# =========================================================================
# === MÓDULO 2: EJECUCIÓN FINAL (CONTADORES CORREGIDOS + TOGGLES) =========
# =========================================================================

# --- Widgets ---
upl_widget = widgets.FileUpload(accept='.xlsx', description='Subir Excel')
met_box = widgets.VBox([widgets.Label('Esperando archivo...')])
out_upl = widgets.Output()

slider_min = widgets.IntSlider(value=20, min=5, max=50, description='Min Clientes')
slider_max = widgets.IntSlider(value=40, min=10, max=100, description='Max Clientes')
inp_sample = widgets.IntText(value=500, description='Muestra/MET')
chk_full = widgets.Checkbox(value=False, description='Analizar TODO')

btn_run = widgets.Button(description='🚀 Ejecutar Análisis', button_style='success', layout=widgets.Layout(width='100%'))
out_run = widgets.Output()

# Conectar carga
upl_widget.observe(lambda c: handle_upload(c, met_box, out_upl), names='value')

# --- LÓGICA PRINCIPAL ---
def click_ejecutar(b):
    state.execution_count += 1
    btn_run.disabled = True
    
    with out_run:
        clear_output()
        
        # 1. Validaciones
        sel_mets = [cb.description for cb in state.met_checkboxes if cb.value]
        if state.df_clientes.empty or not sel_mets:
            print("⚠️ Error: Carga un archivo y selecciona al menos un MET.")
            btn_run.disabled = False; return

        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f"🚀 INICIANDO ANÁLISIS - ID: {run_id}")
        
        # 2. Restaurar Backup
        state.df_clientes = state.df_clientes_backup.copy()
        
        # 3. Asignación de Territorios
        print("🗺️ Asignando territorios...")
        coords_cli = state.df_clientes[['U_latitud', 'U_longitud']].values
        coords_met = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)][['U_latitud', 'U_longitud']].values
        met_names = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)]['CodMET'].values
        dists = cdist(coords_cli, coords_met)
        state.df_clientes['MET_Asignado'] = met_names[np.argmin(dists, axis=1)]
        
        # 4. Procesamiento
        # --- ZOOM AUTOMÁTICO ---
        df_zoom = state.df_clientes[state.df_clientes['MET_Asignado'].isin(sel_mets)]
        if not df_zoom.empty:
            sw = df_zoom[['U_latitud', 'U_longitud']].min().values.tolist()
            ne = df_zoom[['U_latitud', 'U_longitud']].max().values.tolist()
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            mapa_base.fit_bounds([sw, ne])
        else:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')

        export_rows, summary_rows = [], []
        
        # --- INICIALIZACIÓN DE CONTADORES (ESTO FALTABA) ---
        g_total_clientes = 0
        g_total_normales = 0
        g_total_comodines = 0
        g_total_outliers = 0
        g_total_rutas = 0
        
        for met in sel_mets:
            print(f"🔨 Procesando {met}...", end=" ")
            
            # Datos MET
            met_info = state.df_cdr[state.df_cdr['CodMET'] == met].iloc[0]
            met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
            
            # Lógica M1
            df_res, polys, status = procesar_logica_met(
                met, inp_sample.value, chk_full.value, 
                slider_min.value, slider_max.value
            )
            
            if status != "OK": print(f"({status})"); continue
                
            rutas_ids = sorted([r for r in df_res['Ruta_nueva'].unique() if r >= 0])
            print(f"✅ {len(rutas_ids)} rutas.")
            
            # --- CÁLCULO DE CONTADORES (RESTAURADO) ---
            g_total_rutas += len(rutas_ids)
            g_total_clientes += len(df_res)
            g_total_outliers += len(df_res[df_res.get('es_outlier', False) == True])
            g_total_comodines += len(df_res[(df_res.get('es_comodin', False) == True) & (df_res.get('es_outlier', False) == False)])
            g_total_normales += len(df_res[(df_res.get('es_comodin', False) == False) & (df_res.get('es_outlier', False) == False)])
            
            # Rangos volumen
            df_normales = df_res[df_res.get('es_comodin', False) == False]
            if not df_normales.empty:
                vol_min_local = df_normales['m3/día'].min()
                vol_max_local = df_normales['m3/día'].max()
            else:
                vol_min_local, vol_max_local = 0, 1
            
            # Estadísticas MET
            volumen_total_met = df_res['m3/día'].sum()
            total_clientes_met = len(df_res)
            met_display_name = str(met).strip()
            
            # --- POPUP MET ---
            fg_met_home = folium.FeatureGroup(name=f"{met_display_name} - 🏠 MET", show=True, attribution=f"{met_display_name} - 🏠 MET")
            
            popup_met = f"""
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                  <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                      <div style="font-size: 22px; font-weight: bold;">🏠 {met}</div>
                  </div>
                  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                      <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                          <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                          <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                      </div>
                      <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                          <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                          <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                      </div>
                  </div>
                  <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                      <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                      <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_ids)}</div>
                  </div>
            </div>
            """
            
            if os.path.exists(ICON_MET_PATH):
                icon_obj = CustomIcon(ICON_MET_PATH, icon_size=(40, 40), icon_anchor=(20, 40))
                folium.Marker([met_lat, met_lon], icon=icon_obj, popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            else:
                folium.Marker([met_lat, met_lon], icon=folium.Icon(color='green', icon='home'), popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            fg_met_home.add_to(mapa_base)

            # --- RUTAS Y CLIENTES ---
            for ruta_id in rutas_ids:
                ruta_df = df_res[df_res['Ruta_nueva'] == ruta_id]
                vol_ruta = ruta_df['m3/día'].sum()
                frec_ruta = ruta_df['entregas/día'].mean() * 100
                
                layer_name = f"{met_display_name} - Ruta {ruta_id}"
                fg_ruta = folium.FeatureGroup(name=layer_name, show=True, attribution=layer_name)
                color_ruta = state.colores[ruta_id % len(state.colores)]
                
                # Polígono
                if ruta_id in polys and polys[ruta_id]:
                    raw_poly = polys[ruta_id]
                    final_poly = raw_poly
                    if raw_poly and len(raw_poly) > 0:
                        p1 = raw_poly[0]
                        if p1[0] < p1[1]: final_poly = [[c[1], c[0]] for c in raw_poly]
                    
                    folium.Polygon(
                        locations=final_poly,
                        color=color_ruta, weight=2, fill=True, fill_opacity=0.2,
                        tooltip=f"{layer_name} | Vol: {vol_ruta:.1f}m³",
                        className='polygon-shape'
                    ).add_to(fg_ruta)
                
                # Marcadores
                for _, row in ruta_df.iterrows():
                    es_comodin = row.get('es_comodin', False)
                    es_outlier = row.get('es_outlier', False) 
                    vol = row['m3/día']
                    frec = row['entregas/día'] * 100
                    
                    # Definir Tipo
                    if es_outlier:
                        tipo_txt, color_txt, borde = "💀 OUTLIER", "black", "3px solid black"
                    elif es_comodin:
                        tipo_txt, color_txt, borde = "🔺 COMODÍN", "#D32F2F", f"2px solid {color_ruta}"
                    else:
                        tipo_txt, color_txt, borde = "✅ NORMAL", color_ruta, f"2px solid {color_ruta}"

                    popup_html = f"""
                    <div style="background: #fff; border-radius: 12px; padding: 12px; border: {borde}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                         <div style="font-size:16px; color:{color_txt}; font-weight:bold; margin-bottom:8px;">
                             {tipo_txt} | {row['CodCli']}
                         </div>
                         <div style="font-size:14px; color:#333; margin-bottom:4px;">
                             {row.get('Nombre', 'N/A')}
                         </div>
                         <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                             📦 Volumen: <b>{vol:.3f} m³</b>
                         </div>
                         <div style="font-size:14px; color:#1976D2; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                             📊 Frecuencia: <b>{frec:.1f}%</b> (entregas/día)
                         </div>
                         <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                             🗺️ Ruta (Orig): <b>{row.get('Ruta', 'N/A')}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                         </div>
                    </div>
                    """
                    
                    if es_outlier:
                        folium.Marker(
                            [row['U_latitud'], row['U_longitud']],
                            icon=folium.Icon(color='black', icon='remove', prefix='glyphicon'), 
                            popup=folium.Popup(popup_html, max_width=320),
                            tooltip="Outlier Detectado"
                        ).add_to(fg_ruta)
                    elif es_comodin:
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], radius=4,
                            color='#D32F2F', fill=True, fill_color='#FFCDD2', fill_opacity=0.9,
                            popup=folium.Popup(popup_html, max_width=320)
                        ).add_to(fg_ruta)
                    else:
                        tamano_calc = obtener_tamano_marcador(vol, vol_min_local, vol_max_local)
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], 
                            radius=tamano_calc,
                            color='#333333', weight=1, fill=True,
                            fill_color=color_ruta, fill_opacity=0.85,
                            popup=folium.Popup(popup_html, max_width=320)
                        ).add_to(fg_ruta)
                    
                    export_rows.append(row.to_dict())
                
                fg_ruta.add_to(mapa_base)
                summary_rows.append({'MET': met, 'Ruta': ruta_id, 'Clientes': len(ruta_df)})

        # --- INYECCIÓN CSS/JS ---
        folium.LayerControl(collapsed=False).add_to(mapa_base)

        estilos_css = r'''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 600px; overflow-y: auto; width: 340px; padding: 12px; font-family: 'Segoe UI', Arial, sans-serif; background: #fff; }
        
        /* OCULTAR CLASES */
        .hide-all-polygons .polygon-shape { stroke: none !important; fill: none !important; pointer-events: none !important; }
        .hide-outliers .awesome-marker-icon-black { display: none !important; opacity: 0 !important; }
        .hide-outliers .awesome-marker-shadow { display: none !important; opacity: 0 !important; }

        /* VISUALIZACIÓN */
        .visual-control-container { display: flex; gap: 8px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 12px; }
        .visual-btn { flex: 1; padding: 8px; border: 1px solid #ccc; background: white; cursor: pointer; border-radius: 6px; font-weight: 700; font-size: 12px; color: #555; box-shadow: 0 2px 4px rgba(0,0,0,0.1); display: flex; align-items: center; justify-content: center; gap: 5px; }
        .visual-btn:hover { background-color: #f5f5f5; }
        .visual-btn.active { background-color: #e3f2fd; color: #1976D2; border-color: #2196F3; }

        /* ESTILOS EXACTOS */
        .layer-control-buttons { display: flex; gap: 8px; margin-bottom: 12px; }
        .layer-btn { flex: 1; padding: 8px 0; font-size: 13px; font-weight: bold; border: none; border-radius: 4px; cursor: pointer; color: white; box-shadow: 0 2px 4px rgba(0,0,0,0.2); }
        .select-all { background-color: #4CAF50; } .select-all:hover { background-color: #43A047; }
        .deselect-all { background-color: #F44336; } .deselect-all:hover { background-color: #E53935; }

        .search-ruta-container { display: flex; gap: 6px; align-items: center; margin-bottom: 12px; padding: 4px 0; }
        .search-ruta-input { flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 4px; font-size: 13px; outline: none; }
        .search-ruta-input:focus { border-color: #0275d8; box-shadow: 0 0 0 2px rgba(2,117,216,0.1); }
        .search-ruta-btn { background: #0275d8; color: white; border: none; padding: 8px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }
        .search-ruta-clear { background: white; color: #dc3545; border: 1px solid #dc3545; padding: 8px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }

        .met-buttons-row { display: flex; gap: 8px; margin-bottom: 12px; }
        .met-btn { flex: 1; padding: 8px; font-size: 12px; font-weight: 700; color: #0275d8; background: white; border: 2px solid #0275d8; border-radius: 6px; cursor: pointer; }
        .met-btn:hover { background: #e3f2fd; }

        .ruta-buttons-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 6px; background: #E1BEE7; padding: 10px; border-radius: 8px; }
        .ruta-btn { padding: 8px 0; text-align: center; background: white; border: 2px solid #8E24AA; color: #8E24AA; border-radius: 6px; cursor: pointer; font-size: 11px; font-weight: 700; box-shadow: 0 1px 2px rgba(0,0,0,0.1); }
        .ruta-btn:hover { background: #F3E5F5; transform: scale(1.05); }
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))

        filtros_js = r'''
        <script>
        function inicializarFiltros() {
            const layerControl = document.querySelector('.leaflet-control-layers');
            if (!layerControl) return;
            const layersList = layerControl.querySelector('.leaflet-control-layers-list');
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
            if (layersList.querySelector('.ruta-buttons-row')) return;

            // Visual
            const visualDiv = document.createElement('div'); visualDiv.className = 'visual-control-container';
            visualDiv.innerHTML = `
                <button class="visual-btn" id="btn-toggle-poly"><span>🟦</span> <span>Polígonos</span></button>
                <button class="visual-btn" id="btn-toggle-out"><span>💀</span> <span>Outliers</span></button>
            `;
            const mapContainer = document.querySelector('.leaflet-container');
            const btnPoly = visualDiv.querySelector('#btn-toggle-poly');
            const btnOut = visualDiv.querySelector('#btn-toggle-out');
            let polysHidden = false;
            let outsHidden = false;

            btnPoly.onclick = (e) => { e.preventDefault(); e.stopPropagation(); polysHidden = !polysHidden;
                if(polysHidden) { mapContainer.classList.add('hide-all-polygons'); btnPoly.classList.add('active'); }
                else { mapContainer.classList.remove('hide-all-polygons'); btnPoly.classList.remove('active'); }
            };
            btnOut.onclick = (e) => { e.preventDefault(); e.stopPropagation(); outsHidden = !outsHidden;
                if(outsHidden) { mapContainer.classList.add('hide-outliers'); btnOut.classList.add('active'); }
                else { mapContainer.classList.remove('hide-outliers'); btnOut.classList.remove('active'); }
            };
            layersList.insertBefore(visualDiv, overlaysDiv);

            // Todo/Nada
            const buttonsDiv = document.createElement('div'); buttonsDiv.className = 'layer-control-buttons';
            buttonsDiv.innerHTML = '<button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button>';
            buttonsDiv.querySelector('.select-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            buttonsDiv.querySelector('.deselect-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); };
            layersList.insertBefore(buttonsDiv, overlaysDiv);

            // Buscador
            const searchDiv = document.createElement('div'); searchDiv.className = 'search-ruta-container';
            searchDiv.innerHTML = `<span style="font-size:16px;margin-right:4px;">🔍</span><input type="text" class="search-ruta-input" placeholder="Ruta ID..."><button class="search-ruta-btn">Buscar</button><button class="search-ruta-clear">Limpiar</button>`;
            const input = searchDiv.querySelector('input');
            const doSearch = () => {
                const val = input.value.trim(); if(!val) return;
                checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); 
                setTimeout(() => { checkboxes.forEach(cb => { const label = cb.parentElement.textContent; const regex = new RegExp(`- Ruta ${val}$`); if(regex.test(label)) { if(!cb.checked) cb.click(); } }); }, 100);
            };
            const doClear = () => { input.value = ''; checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            searchDiv.querySelector('.search-ruta-btn').onclick = (e) => { e.preventDefault(); doSearch(); };
            searchDiv.querySelector('.search-ruta-clear').onclick = (e) => { e.preventDefault(); doClear(); };
            input.onkeypress = (e) => { if(e.key === 'Enter') { e.preventDefault(); doSearch(); } };
            layersList.insertBefore(searchDiv, overlaysDiv);

            // METs
            const metDiv = document.createElement('div'); metDiv.className = 'met-buttons-row';
            const mets = new Set();
            checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+/); if(match) mets.add(match[1].trim()); });
            Array.from(mets).sort().forEach(met => {
                const btn = document.createElement('button'); btn.className = 'met-btn'; btn.textContent = met.replace('MET ', '');
                btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { if(cb.parentElement.textContent.includes(met + ' -')) if(!cb.checked) cb.click(); }); }, 100); };
                metDiv.appendChild(btn);
            });
            layersList.insertBefore(metDiv, overlaysDiv);

            // Rutas
            const rutaDiv = document.createElement('div'); rutaDiv.className = 'ruta-buttons-row';
            const rutasSet = new Set();
            checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/- Ruta\s+(\d+)/); if(match) rutasSet.add(parseInt(match[1])); });
            Array.from(rutasSet).sort((a,b)=>a-b).forEach(rutaId => {
                const btn = document.createElement('button'); btn.className = 'ruta-btn'; btn.textContent = 'R' + rutaId;
                btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { const regex = new RegExp(`- Ruta ${rutaId}$`); if(regex.test(cb.parentElement.textContent)) { if(!cb.checked) cb.click(); } }); }, 100); };
                rutaDiv.appendChild(btn);
            });
            layersList.insertBefore(rutaDiv, overlaysDiv);
        }
        setTimeout(inicializarFiltros, 1500);
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(filtros_js))
        
        # --- TITLE DASHBOARD ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1565C0 0%, #42A5F5 100%); color: white; padding: 10px 20px; border-radius: 30px; box-shadow: 0 8px 20px rgba(0,0,0,0.3); border: 3px solid white; font-family: 'Segoe UI', Arial; text-align: center; min-width: 400px;">
            <div style="font-size:15px; font-weight:bold; margin-bottom:6px; letter-spacing:1px;">🗺️ DASHBOARD DE TERRITORIO</div>
            <div style="display: flex; justify-content: space-around; gap: 15px; background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">
                <div title="Total de Clientes">👥 <b>{g_total_clientes}</b></div>
                <div title="Clientes Normales" style="color: #C8E6C9;">✅ <b>{g_total_normales}</b></div>
                <div title="Comodines" style="color: #FFCDD2;">🔺 <b>{g_total_comodines}</b></div>
                <div title="Outliers Detectados" style="color: #212121; background:#fff; padding:0 5px; border-radius:10px;">💀 <b>{g_total_outliers}</b></div>
            </div>
            <div style="font-size:10px; margin-top:4px; opacity:0.8;">ID: {run_id} | Rutas: {g_total_rutas}</div>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))

        # --- Guardar ---
        map_path = os.path.join(OUTPUT_DIR, f"Mapa_Ceylan_{run_id}.html")
        mapa_base.save(map_path)
        xls_path = generate_consolidated_excel(export_rows, summary_rows, OUTPUT_DIR, filename_suffix=run_id)
        
        print(f"\n✅ FINALIZADO."); print(f"🗺️ Mapa: {os.path.basename(map_path)}"); print(f"📊 Excel: {os.path.basename(xls_path)}")
        btn_run.disabled = False

btn_run.on_click(click_ejecutar)

# --- Layout ---
print("================== CEYLAN ALFA 5.1 (FINAL CORREGIDO) ==================")
display(widgets.VBox([
    widgets.HBox([widgets.Label("1. DATOS:"), upl_widget]), out_upl,
    met_box, widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("2. PARÁMETROS:"), slider_min, slider_max]),
    widgets.HBox([widgets.Label("3. MUESTREO:"), inp_sample, chk_full]),
    widgets.HTML("<br>"),
    btn_run, out_run
]))

================== CEYLAN ALFA 5.1 (FINAL CORREGIDO) ==================


Vesion constreñida (compactas) con filtros: (Poligonos,Outliers,Ruta,MET,Todo/Nada), outliers, comodines, popups: (Ruta,Clientes,MET,Resumen), poligonos (vertices de clientes)

In [ ]:
# =========================================================================
# === MÓDULO 1: BACKEND (DBSCAN + K-MEANS CONSTREÑIDO + CONVEX HULL) ======
# =========================================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, MultiPoint, Point
from sklearn.cluster import DBSCAN, KMeans

# --- IMPORTANTE: Intentar importar la librería nueva ---
try:
    from k_means_constrained import KMeansConstrained
    HAS_KMEANS_CONST = True
except ImportError:
    HAS_KMEANS_CONST = False
    print("⚠️ ADVERTENCIA: 'k-means-constrained' no está instalado. Ejecuta '!pip install k-means-constrained'")

# --- Configuración ---
BASE_DIR = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan"
ICON_MET_PATH = os.path.join(BASE_DIR, "95_24.png")
OUTPUT_DIR = os.path.join(BASE_DIR, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

class CeylanState:
    def __init__(self):
        self.df_clientes = pd.DataFrame()
        self.df_cdr = pd.DataFrame()
        self.df_clientes_backup = pd.DataFrame()
        self.execution_count = 0
        self.met_checkboxes = []
        self.colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
state = CeylanState()

# --- HELPERS ---
def normalizar_id(valor):
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    return s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0: return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min: return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# --- CLUSTERING & LÓGICA ---

def segregar_por_frecuencia(df_clients, threshold=0.01):
    if 'Ruta_nueva' not in df_clients.columns: df_clients['Ruta_nueva'] = -1
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    
    # Separamos clientes frecuentes (Core) de los esporádicos (Comodines)
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    
    df_elegibles['es_comodin'] = False
    df_elegibles['es_outlier'] = False
    df_comodines['es_comodin'] = True
    df_comodines['es_outlier'] = False
    
    return df_elegibles, df_comodines

def filtrar_outliers_dbscan(df_core, eps_km=2.0, min_samples=3):
    if len(df_core) < min_samples: return df_core, pd.DataFrame(columns=df_core.columns)
    coords_rad = np.radians(df_core[['U_latitud', 'U_longitud']].values)
    
    db = DBSCAN(eps=eps_km/6371.0, min_samples=min_samples, metric='haversine').fit(coords_rad)
    
    mask = db.labels_ == -1
    df_out = df_core[mask].copy()
    df_cln = df_core[~mask].copy()
    
    if not df_out.empty:
        df_out['es_outlier'] = True
        df_out['es_comodin'] = False 
    return df_cln, df_out

def ejecutar_clustering_constrenido(df_elegibles, min_size, max_size):
    if df_elegibles.empty: return df_elegibles
    
    coords = df_elegibles[['U_latitud', 'U_longitud']].values
    n_samples = len(coords)
    
    target_avg = (min_size + max_size) / 2
    n_clusters = int(round(n_samples / target_avg))
    
    if n_clusters < 1: n_clusters = 1
    if n_clusters * max_size < n_samples: n_clusters = int(np.ceil(n_samples / max_size))
    if n_clusters * min_size > n_samples: n_clusters = int(np.floor(n_samples / min_size))
    if n_clusters < 1: n_clusters = 1

    print(f"   ⮑ Optimizando: {n_samples} clientes en {n_clusters} rutas (Obj: {min_size}-{max_size})...")

    if HAS_KMEANS_CONST:
        try:
            clf = KMeansConstrained(
                n_clusters=n_clusters,
                size_min=min_size,
                size_max=max_size,
                random_state=42,
                n_init=10
            )
            clf.fit(coords)
            df_elegibles['Ruta_nueva'] = clf.labels_
        except Exception as e:
            print(f"⚠️ Falló K-Means Constreñido ({e}). Usando K-Means simple.")
            kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
            df_elegibles['Ruta_nueva'] = kmeans.labels_
    else:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
        df_elegibles['Ruta_nueva'] = kmeans.labels_

    return df_elegibles

def asignar_comodines_post_clustering(df_pendientes, df_elegibles_asignados, max_km=5.0):
    if df_pendientes.empty: return df_pendientes
    if df_elegibles_asignados.empty: return df_pendientes
        
    coords_pend = df_pendientes[['U_longitud', 'U_latitud']].values
    rutas_info = df_elegibles_asignados.groupby('Ruta_nueva')[['U_longitud', 'U_latitud']].mean()
    coords_centroides = rutas_info.values
    rutas_ids = rutas_info.index.values
    
    dist_matrix = cdist(coords_pend, coords_centroides, metric='euclidean')
    
    min_dists = np.min(dist_matrix, axis=1)
    idx_closest_centroid = np.argmin(dist_matrix, axis=1)
    limit_deg = max_km / 111.0 
    
    rutas_asignadas = rutas_ids[idx_closest_centroid]
    rutas_finales = np.where(min_dists <= limit_deg, rutas_asignadas, -1)
    
    df_pendientes['Ruta_nueva'] = rutas_finales
    df_pendientes.loc[df_pendientes['Ruta_nueva'] == -1, 'es_lejano'] = True
    
    return df_pendientes

# =========================================================================
# === CAMBIO CLAVE AQUÍ: LÓGICA DE DIBUJO DE POLÍGONOS ====================
# =========================================================================
def calculate_cluster_polygons(df):
    """
    MODIFICADO: Genera polígonos usando ÚNICAMENTE Convex Hull.
    El polígono será la "liga elástica" que envuelve a los clientes extremos.
    """
    polygons = {}
    rutas = sorted([r for r in df['Ruta_nueva'].unique() if r >= 0])
    
    for r in rutas:
        # Extraemos coordenadas [Lat, Lon] para Folium
        # NOTA: ConvexHull funciona bien con Lat/Lon directo
        pts = df[df['Ruta_nueva'] == r][['U_latitud', 'U_longitud']].values
        
        # Necesitamos al menos 3 puntos para hacer un polígono (triángulo)
        if len(pts) >= 3:
            try:
                # 1. Calculamos la Envolvente Convexa
                hull = ConvexHull(pts)
                
                # 2. Obtenemos los vértices en orden (sentido antihorario)
                # hull.vertices nos da los índices de los puntos extremos
                vertices = pts[hull.vertices]
                
                # 3. Guardamos la lista de coordenadas para Folium
                polygons[r] = vertices.tolist()
            except Exception as e:
                pass # Si fallan las mates (puntos colineales), no dibujamos polígono
                
    return polygons
# =========================================================================

def procesar_logica_met(met_id, n_muestra, analizar_todos, min_size, max_size):
    df_terr = state.df_clientes[state.df_clientes['MET_Asignado'] == met_id].copy()
    if df_terr.empty: return None, None, "Sin clientes"
    
    df_proc = df_terr.copy() if analizar_todos or len(df_terr) <= n_muestra else df_terr.sample(n=n_muestra).copy()
    
    df_proc['entregas/día'] = pd.to_numeric(df_proc['entregas/día'], errors='coerce').fillna(0)
    df_proc['m3/día'] = pd.to_numeric(df_proc['m3/día'], errors='coerce').fillna(0)
    
    coords = df_proc[['U_longitud', 'U_latitud']].values
    if len(coords) == 0: return None, None, "Sin coordenadas"
    
    # 1. Separar Comodines
    df_eleg, df_como = segregar_por_frecuencia(df_proc, threshold=0.01)
    
    # 2. Detectar Outliers
    df_eleg, df_outliers = filtrar_outliers_dbscan(df_eleg, eps_km=3.0, min_samples=3)
    
    # 3. CLUSTERING (K-Means Constrained)
    df_eleg = ejecutar_clustering_constrenido(df_eleg, min_size, max_size)
    
    # 4. Asignar Pendientes
    df_pend = pd.concat([df_como, df_outliers], ignore_index=True)
    df_pend = asignar_comodines_post_clustering(df_pend, df_eleg, max_km=10.0)
    
    # 5. Unificación
    df_final = pd.concat([df_eleg, df_pend], ignore_index=True)
    
    # 6. Polígonos (AHORA USANDO SOLO CONVEX HULL)
    polys = calculate_cluster_polygons(df_final)
    
    return df_final, polys, "OK"

# --- MÓDULO DE EXPORTACIÓN ---
def generate_consolidated_excel(export_data, summary_data, output_dir, filename_suffix=None):
    if not export_data: return None
    if not filename_suffix: filename_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_dir, f'Resultados_Ceylan_{filename_suffix}.xlsx')
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        if summary_data: pd.DataFrame(summary_data).to_excel(writer, sheet_name='Resumen', index=False)
        if export_data: pd.DataFrame(export_data).to_excel(writer, sheet_name='Detalle', index=False)
    return path

# --- HANDLER DE CARGA ---
def handle_upload(c, mb, ou):
    with ou:
        clear_output()
        if not c['new']: return
        try:
            content = c['owner'].value[0]['content'] if isinstance(c['owner'].value, tuple) else list(c['owner'].value.values())[0]['content']
            with open("temp.xlsx", 'wb') as f: f.write(content)
            xls = pd.read_excel("temp.xlsx", sheet_name=None)
            state.df_clientes = xls[list(xls.keys())[0]]
            state.df_cdr = xls[list(xls.keys())[1]]
            
            state.df_clientes['m3/día'] = pd.to_numeric(state.df_clientes['m3/día'], errors='coerce').fillna(0)
            state.df_clientes['entregas/día'] = pd.to_numeric(state.df_clientes['entregas/día'], errors='coerce').fillna(0)
            
            mask_freq = state.df_clientes['entregas/día'] > 50
            if mask_freq.any(): state.df_clientes.loc[mask_freq, 'entregas/día'] = 0
            
            state.df_clientes_backup = state.df_clientes.copy()
            mets = sorted(state.df_cdr['CodMET'].unique())
            state.met_checkboxes = [widgets.Checkbox(description=str(m), value=False) for m in mets]
            mb.children = [widgets.Label("Selecciona METs:")] + state.met_checkboxes
            print(f"✅ Cargado. Clientes: {len(state.df_clientes)}")
            os.remove("temp.xlsx")
        except Exception as e: print(f"❌ Error: {e}")

print("✅ MÓDULO 1: ACTUALIZADO (POLÍGONOS AJUSTADOS A VÉRTICES / CONVEX HULL).")

In [ ]:
# =========================================================================
# === MÓDULO 2: EJECUCIÓN FINAL (CON ETIQUETAS + TARJETAS DE INFORMACIÓN) =
# =========================================================================

from folium.features import DivIcon

# --- Widgets ---
upl_widget = widgets.FileUpload(accept='.xlsx', description='Subir Excel')
met_box = widgets.VBox([widgets.Label('Esperando archivo...')])
out_upl = widgets.Output()

slider_min = widgets.IntSlider(value=20, min=5, max=50, description='Min Clientes')
slider_max = widgets.IntSlider(value=40, min=10, max=100, description='Max Clientes')
inp_sample = widgets.IntText(value=500, description='Muestra/MET')
chk_full = widgets.Checkbox(value=False, description='Analizar TODO')

btn_run = widgets.Button(description='🚀 Ejecutar Análisis', button_style='success', layout=widgets.Layout(width='100%'))
out_run = widgets.Output()

# Conectar carga
upl_widget.observe(lambda c: handle_upload(c, met_box, out_upl), names='value')

# --- LÓGICA PRINCIPAL ---
def click_ejecutar(b):
    state.execution_count += 1
    btn_run.disabled = True
    
    with out_run:
        clear_output()
        
        # 1. Validaciones
        sel_mets = [cb.description for cb in state.met_checkboxes if cb.value]
        if state.df_clientes.empty or not sel_mets:
            print("⚠️ Error: Carga un archivo y selecciona al menos un MET.")
            btn_run.disabled = False; return

        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f"🚀 INICIANDO ANÁLISIS - ID: {run_id}")
        
        # 2. Restaurar Backup
        state.df_clientes = state.df_clientes_backup.copy()
        
        # 3. Asignación de Territorios
        print("🗺️ Asignando territorios...")
        coords_cli = state.df_clientes[['U_latitud', 'U_longitud']].values
        coords_met = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)][['U_latitud', 'U_longitud']].values
        met_names = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)]['CodMET'].values
        dists = cdist(coords_cli, coords_met)
        state.df_clientes['MET_Asignado'] = met_names[np.argmin(dists, axis=1)]
        
        # 4. Procesamiento
        # --- ZOOM AUTOMÁTICO ---
        df_zoom = state.df_clientes[state.df_clientes['MET_Asignado'].isin(sel_mets)]
        if not df_zoom.empty:
            sw = df_zoom[['U_latitud', 'U_longitud']].min().values.tolist()
            ne = df_zoom[['U_latitud', 'U_longitud']].max().values.tolist()
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            mapa_base.fit_bounds([sw, ne])
        else:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')

        export_rows, summary_rows = [], []
        
        # --- INICIALIZACIÓN DE CONTADORES GLOBALES ---
        g_total_clientes = 0
        g_total_normales = 0
        g_total_comodines = 0
        g_total_outliers = 0
        g_total_rutas = 0
        
        for met in sel_mets:
            print(f"🔨 Procesando {met}...", end=" ")
            
            # Datos MET
            met_info = state.df_cdr[state.df_cdr['CodMET'] == met].iloc[0]
            met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
            
            # Lógica M1
            df_res, polys, status = procesar_logica_met(
                met, inp_sample.value, chk_full.value, 
                slider_min.value, slider_max.value
            )
            
            if status != "OK": print(f"({status})"); continue
                
            rutas_ids = sorted([r for r in df_res['Ruta_nueva'].unique() if r >= 0])
            print(f"✅ {len(rutas_ids)} rutas.")
            
            # --- CÁLCULO DE CONTADORES GLOBALES ---
            g_total_rutas += len(rutas_ids)
            g_total_clientes += len(df_res)
            g_total_outliers += len(df_res[df_res.get('es_outlier', False) == True])
            g_total_comodines += len(df_res[(df_res.get('es_comodin', False) == True) & (df_res.get('es_outlier', False) == False)])
            g_total_normales += len(df_res[(df_res.get('es_comodin', False) == False) & (df_res.get('es_outlier', False) == False)])
            
            # Rangos volumen
            df_normales = df_res[df_res.get('es_comodin', False) == False]
            if not df_normales.empty:
                vol_min_local = df_normales['m3/día'].min()
                vol_max_local = df_normales['m3/día'].max()
            else:
                vol_min_local, vol_max_local = 0, 1
            
            # Estadísticas MET
            volumen_total_met = df_res['m3/día'].sum()
            total_clientes_met = len(df_res)
            met_display_name = str(met).strip()
            
            # --- POPUP MET (HOME) ---
            fg_met_home = folium.FeatureGroup(name=f"{met_display_name} - 🏠 MET", show=True, attribution=f"{met_display_name} - 🏠 MET")
            
            popup_met = f"""
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                  <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                      <div style="font-size: 22px; font-weight: bold;">🏠 {met}</div>
                  </div>
                  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                      <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                          <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                          <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                      </div>
                      <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                          <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                          <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                      </div>
                  </div>
                  <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                      <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                      <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_ids)}</div>
                  </div>
            </div>
            """
            
            if os.path.exists(ICON_MET_PATH):
                icon_obj = CustomIcon(ICON_MET_PATH, icon_size=(40, 40), icon_anchor=(20, 40))
                folium.Marker([met_lat, met_lon], icon=icon_obj, popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            else:
                folium.Marker([met_lat, met_lon], icon=folium.Icon(color='green', icon='home'), popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            fg_met_home.add_to(mapa_base)

            # --- RUTAS Y CLIENTES ---
            for ruta_id in rutas_ids:
                ruta_df = df_res[df_res['Ruta_nueva'] == ruta_id]
                
                # --- CALCULAR ESTADÍSTICAS POR RUTA PARA LA TARJETA ---
                vol_ruta = ruta_df['m3/día'].sum()
                total_cli_ruta = len(ruta_df)
                
                # Desglose exacto: Normales, Comodines, Outliers
                n_normales_ruta = len(ruta_df[(ruta_df.get('es_comodin', False) == False) & (ruta_df.get('es_outlier', False) == False)])
                n_comodines_ruta = len(ruta_df[(ruta_df.get('es_comodin', False) == True) & (ruta_df.get('es_outlier', False) == False)])
                n_outliers_ruta = len(ruta_df[ruta_df.get('es_outlier', False) == True])
                
                layer_name = f"{met_display_name} - Ruta {ruta_id}"
                fg_ruta = folium.FeatureGroup(name=layer_name, show=True, attribution=layer_name)
                color_ruta = state.colores[ruta_id % len(state.colores)]
                
                # =========================================================
                # === DISEÑO DE TARJETA POPUP (AZUL Y BLANCA) =============
                # =========================================================
                html_popup_poligono = f"""
                <div style="font-family: 'Segoe UI', Arial, sans-serif; min-width: 220px; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.2); border: 1px solid #ccc;">
                    <div style="background-color: #1a237e; color: white; padding: 10px 12px; font-weight: bold; font-size: 15px; display: flex; align-items: center; justify-content: space-between;">
                        <span>🚛 Ruta {ruta_id}</span>
                        <span style="font-size: 12px; font-weight: normal; opacity: 0.8;">{met}</span>
                    </div>
                    
                    <div style="padding: 12px; background-color: white; color: #333;">
                        <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 6px;">
                            <span style="font-size: 16px;">📦</span>
                            <span style="font-weight: 600; font-size: 13px;">Volumen:</span>
                            <span style="font-size: 14px;">{vol_ruta:.2f} m³</span>
                        </div>
                        <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
                            <span style="font-size: 16px;">👥</span>
                            <span style="font-weight: 600; font-size: 13px;">Clientes:</span>
                            <span style="font-size: 14px;">{total_cli_ruta}</span>
                        </div>
                        
                        <div style="border-top: 1px solid #eee; margin: 8px 0;"></div>
                        
                        <div style="display: flex; justify-content: space-between; font-size: 12px; font-weight: bold; gap: 4px;">
                            <div title="Normales" style="color: #2E7D32; background: #E8F5E8; padding: 3px 6px; border-radius: 4px; flex: 1; text-align: center;">
                                ✅ Nor: {n_normales_ruta}
                            </div>
                            <div title="Comodines" style="color: #C62828; background: #FFEBEE; padding: 3px 6px; border-radius: 4px; flex: 1; text-align: center;">
                                ▲ Com: {n_comodines_ruta}
                            </div>
                        </div>
                        
                        {f'''<div style="margin-top: 5px; color: #fff; background: #333; padding: 2px 6px; border-radius: 4px; text-align: center; font-size: 11px;">
                                💀 Outliers: {n_outliers_ruta}
                             </div>''' if n_outliers_ruta > 0 else ''}
                    </div>
                </div>
                """
                # =========================================================

                # Polígono
                if ruta_id in polys and polys[ruta_id]:
                    raw_poly = polys[ruta_id]
                    final_poly = raw_poly
                    if raw_poly and len(raw_poly) > 0:
                        p1 = raw_poly[0]
                        if p1[0] < p1[1]: final_poly = [[c[1], c[0]] for c in raw_poly]
                    
                    folium.Polygon(
                        locations=final_poly,
                        color=color_ruta, weight=2, fill=True, fill_opacity=0.2,
                        # AQUÍ ESTÁ LA MAGIA: Popup HTML + Tooltip simple
                        popup=folium.Popup(html_popup_poligono, max_width=300),
                        tooltip=f"Ver detalles Ruta {ruta_id}",
                        className='polygon-shape'
                    ).add_to(fg_ruta)

                # =================================================================
                # === ETIQUETAS DE TEXTO DE RUTA (R5) =============================
                # =================================================================
                if not ruta_df.empty:
                    center_lat = ruta_df['U_latitud'].mean()
                    center_lon = ruta_df['U_longitud'].mean()
                    
                    icon_text = DivIcon(
                        icon_size=(150,36), icon_anchor=(75,18),
                        html=f'''<div style="font-size: 14pt; font-weight: 900; color: {color_ruta}; text-align: center; white-space: nowrap; text-shadow: -1px -1px 0 #fff, 1px -1px 0 #fff, -1px 1px 0 #fff, 1px 1px 0 #fff, 0px 0px 4px rgba(255,255,255,0.8); font-family: 'Segoe UI', Arial, sans-serif; pointer-events: none;">R{ruta_id}</div>'''
                    )
                    folium.Marker([center_lat, center_lon], icon=icon_text, zIndexOffset=1000).add_to(fg_ruta)
                # =================================================================
                
                # Marcadores de Clientes
                for _, row in ruta_df.iterrows():
                    es_comodin = row.get('es_comodin', False)
                    es_outlier = row.get('es_outlier', False) 
                    vol = row['m3/día']
                    frec = row['entregas/día'] * 100
                    
                    # Definir Tipo y Estilos
                    if es_outlier:
                        tipo_txt, color_txt, borde = "💀 OUTLIER", "black", "3px solid black"
                    elif es_comodin:
                        tipo_txt, color_txt, borde = "🔺 COMODÍN", "#D32F2F", f"2px solid {color_ruta}"
                    else:
                        tipo_txt, color_txt, borde = "✅ NORMAL", color_ruta, f"2px solid {color_ruta}"

                    # Popup Cliente Individual
                    popup_html_cli = f"""
                    <div style="background: #fff; border-radius: 12px; padding: 12px; border: {borde}; min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                          <div style="font-size:16px; color:{color_txt}; font-weight:bold; margin-bottom:8px;">
                              {tipo_txt} | {row['CodCli']}
                          </div>
                          <div style="font-size:14px; color:#333; margin-bottom:4px;">
                              {row.get('Nombre', 'N/A')}
                          </div>
                          <div style="font-size:14px; color:#FF6B00; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                              📦 Volumen: <b>{vol:.3f} m³</b>
                          </div>
                          <div style="font-size:14px; color:#1976D2; margin-bottom:8px; border-top: 1px solid #eee; padding-top: 8px;">
                              📊 Frecuencia: <b>{frec:.1f}%</b> (entregas/día)
                          </div>
                          <div style="font-size:12px; color:#666; border-top: 1px solid #eee; padding-top: 8px;">
                              🗺️ Ruta (Orig): <b>{row.get('Ruta', 'N/A')}</b> | 🚚 Ruta (Nueva): <b>{ruta_id}</b>
                          </div>
                    </div>
                    """
                    
                    if es_outlier:
                        folium.Marker(
                            [row['U_latitud'], row['U_longitud']],
                            icon=folium.Icon(color='black', icon='remove', prefix='glyphicon'), 
                            popup=folium.Popup(popup_html_cli, max_width=320),
                            tooltip="Outlier Detectado"
                        ).add_to(fg_ruta)
                    elif es_comodin:
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], radius=4,
                            color='#D32F2F', fill=True, fill_color='#FFCDD2', fill_opacity=0.9,
                            popup=folium.Popup(popup_html_cli, max_width=320)
                        ).add_to(fg_ruta)
                    else:
                        tamano_calc = obtener_tamano_marcador(vol, vol_min_local, vol_max_local)
                        folium.CircleMarker(
                            [row['U_latitud'], row['U_longitud']], 
                            radius=tamano_calc,
                            color='#333333', weight=1, fill=True,
                            fill_color=color_ruta, fill_opacity=0.85,
                            popup=folium.Popup(popup_html_cli, max_width=320)
                        ).add_to(fg_ruta)
                    
                    export_rows.append(row.to_dict())
                
                fg_ruta.add_to(mapa_base)
                summary_rows.append({'MET': met, 'Ruta': ruta_id, 'Clientes': len(ruta_df)})

        # --- INYECCIÓN CSS/JS ---
        folium.LayerControl(collapsed=False).add_to(mapa_base)

        estilos_css = r'''
        <style>
        .leaflet-control-layers-overlays { display: none !important; }
        .leaflet-control-layers-list { max-height: 600px; overflow-y: auto; width: 340px; padding: 12px; font-family: 'Segoe UI', Arial, sans-serif; background: #fff; }
        
        /* OCULTAR CLASES */
        .hide-all-polygons .polygon-shape { stroke: none !important; fill: none !important; pointer-events: none !important; }
        .hide-outliers .awesome-marker-icon-black { display: none !important; opacity: 0 !important; }
        .hide-outliers .awesome-marker-shadow { display: none !important; opacity: 0 !important; }

        /* VISUALIZACIÓN */
        .visual-control-container { display: flex; gap: 8px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 12px; }
        .visual-btn { flex: 1; padding: 8px; border: 1px solid #ccc; background: white; cursor: pointer; border-radius: 6px; font-weight: 700; font-size: 12px; color: #555; box-shadow: 0 2px 4px rgba(0,0,0,0.1); display: flex; align-items: center; justify-content: center; gap: 5px; }
        .visual-btn:hover { background-color: #f5f5f5; }
        .visual-btn.active { background-color: #e3f2fd; color: #1976D2; border-color: #2196F3; }

        /* ESTILOS EXACTOS */
        .layer-control-buttons { display: flex; gap: 8px; margin-bottom: 12px; }
        .layer-btn { flex: 1; padding: 8px 0; font-size: 13px; font-weight: bold; border: none; border-radius: 4px; cursor: pointer; color: white; box-shadow: 0 2px 4px rgba(0,0,0,0.2); }
        .select-all { background-color: #4CAF50; } .select-all:hover { background-color: #43A047; }
        .deselect-all { background-color: #F44336; } .deselect-all:hover { background-color: #E53935; }

        .search-ruta-container { display: flex; gap: 6px; align-items: center; margin-bottom: 12px; padding: 4px 0; }
        .search-ruta-input { flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 4px; font-size: 13px; outline: none; }
        .search-ruta-input:focus { border-color: #0275d8; box-shadow: 0 0 0 2px rgba(2,117,216,0.1); }
        .search-ruta-btn { background: #0275d8; color: white; border: none; padding: 8px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }
        .search-ruta-clear { background: white; color: #dc3545; border: 1px solid #dc3545; padding: 8px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; }

        .met-buttons-row { display: flex; gap: 8px; margin-bottom: 12px; }
        .met-btn { flex: 1; padding: 8px; font-size: 12px; font-weight: 700; color: #0275d8; background: white; border: 2px solid #0275d8; border-radius: 6px; cursor: pointer; }
        .met-btn:hover { background: #e3f2fd; }

        .ruta-buttons-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 6px; background: #E1BEE7; padding: 10px; border-radius: 8px; }
        .ruta-btn { padding: 8px 0; text-align: center; background: white; border: 2px solid #8E24AA; color: #8E24AA; border-radius: 6px; cursor: pointer; font-size: 11px; font-weight: 700; box-shadow: 0 1px 2px rgba(0,0,0,0.1); }
        .ruta-btn:hover { background: #F3E5F5; transform: scale(1.05); }
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))

        filtros_js = r'''
        <script>
        function inicializarFiltros() {
            const layerControl = document.querySelector('.leaflet-control-layers');
            if (!layerControl) return;
            const layersList = layerControl.querySelector('.leaflet-control-layers-list');
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
            if (layersList.querySelector('.ruta-buttons-row')) return;

            // Visual
            const visualDiv = document.createElement('div'); visualDiv.className = 'visual-control-container';
            visualDiv.innerHTML = `
                <button class="visual-btn" id="btn-toggle-poly"><span>🟦</span> <span>Polígonos</span></button>
                <button class="visual-btn" id="btn-toggle-out"><span>💀</span> <span>Outliers</span></button>
            `;
            const mapContainer = document.querySelector('.leaflet-container');
            const btnPoly = visualDiv.querySelector('#btn-toggle-poly');
            const btnOut = visualDiv.querySelector('#btn-toggle-out');
            let polysHidden = false;
            let outsHidden = false;

            btnPoly.onclick = (e) => { e.preventDefault(); e.stopPropagation(); polysHidden = !polysHidden;
                if(polysHidden) { mapContainer.classList.add('hide-all-polygons'); btnPoly.classList.add('active'); }
                else { mapContainer.classList.remove('hide-all-polygons'); btnPoly.classList.remove('active'); }
            };
            btnOut.onclick = (e) => { e.preventDefault(); e.stopPropagation(); outsHidden = !outsHidden;
                if(outsHidden) { mapContainer.classList.add('hide-outliers'); btnOut.classList.add('active'); }
                else { mapContainer.classList.remove('hide-outliers'); btnOut.classList.remove('active'); }
            };
            layersList.insertBefore(visualDiv, overlaysDiv);

            // Todo/Nada
            const buttonsDiv = document.createElement('div'); buttonsDiv.className = 'layer-control-buttons';
            buttonsDiv.innerHTML = '<button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button>';
            buttonsDiv.querySelector('.select-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            buttonsDiv.querySelector('.deselect-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); };
            layersList.insertBefore(buttonsDiv, overlaysDiv);

            // Buscador
            const searchDiv = document.createElement('div'); searchDiv.className = 'search-ruta-container';
            searchDiv.innerHTML = `<span style="font-size:16px;margin-right:4px;">🔍</span><input type="text" class="search-ruta-input" placeholder="Ruta ID..."><button class="search-ruta-btn">Buscar</button><button class="search-ruta-clear">Limpiar</button>`;
            const input = searchDiv.querySelector('input');
            const doSearch = () => {
                const val = input.value.trim(); if(!val) return;
                checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); 
                setTimeout(() => { checkboxes.forEach(cb => { const label = cb.parentElement.textContent; const regex = new RegExp(`- Ruta ${val}$`); if(regex.test(label)) { if(!cb.checked) cb.click(); } }); }, 100);
            };
            const doClear = () => { input.value = ''; checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); };
            searchDiv.querySelector('.search-ruta-btn').onclick = (e) => { e.preventDefault(); doSearch(); };
            searchDiv.querySelector('.search-ruta-clear').onclick = (e) => { e.preventDefault(); doClear(); };
            input.onkeypress = (e) => { if(e.key === 'Enter') { e.preventDefault(); doSearch(); } };
            layersList.insertBefore(searchDiv, overlaysDiv);

            // METs
            const metDiv = document.createElement('div'); metDiv.className = 'met-buttons-row';
            const mets = new Set();
            checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+/); if(match) mets.add(match[1].trim()); });
            Array.from(mets).sort().forEach(met => {
                const btn = document.createElement('button'); btn.className = 'met-btn'; btn.textContent = met.replace('MET ', '');
                btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { if(cb.parentElement.textContent.includes(met + ' -')) if(!cb.checked) cb.click(); }); }, 100); };
                metDiv.appendChild(btn);
            });
            layersList.insertBefore(metDiv, overlaysDiv);

            // Rutas
            const rutaDiv = document.createElement('div'); rutaDiv.className = 'ruta-buttons-row';
            const rutasSet = new Set();
            checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/- Ruta\s+(\d+)/); if(match) rutasSet.add(parseInt(match[1])); });
            Array.from(rutasSet).sort((a,b)=>a-b).forEach(rutaId => {
                const btn = document.createElement('button'); btn.className = 'ruta-btn'; btn.textContent = 'R' + rutaId;
                btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { const regex = new RegExp(`- Ruta ${rutaId}$`); if(regex.test(cb.parentElement.textContent)) { if(!cb.checked) cb.click(); } }); }, 100); };
                rutaDiv.appendChild(btn);
            });
            layersList.insertBefore(rutaDiv, overlaysDiv);
        }
        setTimeout(inicializarFiltros, 1500);
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(filtros_js))
        
        # --- TITLE DASHBOARD ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1565C0 0%, #42A5F5 100%); color: white; padding: 10px 20px; border-radius: 30px; box-shadow: 0 8px 20px rgba(0,0,0,0.3); border: 3px solid white; font-family: 'Segoe UI', Arial; text-align: center; min-width: 400px;">
            <div style="font-size:15px; font-weight:bold; margin-bottom:6px; letter-spacing:1px;">🗺️ DASHBOARD DE TERRITORIO</div>
            <div style="display: flex; justify-content: space-around; gap: 15px; background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">
                <div title="Total de Clientes">👥 <b>{g_total_clientes}</b></div>
                <div title="Clientes Normales" style="color: #C8E6C9;">✅ <b>{g_total_normales}</b></div>
                <div title="Comodines" style="color: #FFCDD2;">🔺 <b>{g_total_comodines}</b></div>
                <div title="Outliers Detectados" style="color: #212121; background:#fff; padding:0 5px; border-radius:10px;">💀 <b>{g_total_outliers}</b></div>
            </div>
            <div style="font-size:10px; margin-top:4px; opacity:0.8;">ID: {run_id} | Rutas: {g_total_rutas}</div>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))

        # --- Guardar ---
        map_path = os.path.join(OUTPUT_DIR, f"Mapa_Ceylan_{run_id}.html")
        mapa_base.save(map_path)
        xls_path = generate_consolidated_excel(export_rows, summary_rows, OUTPUT_DIR, filename_suffix=run_id)
        
        print(f"\n✅ FINALIZADO."); print(f"🗺️ Mapa: {os.path.basename(map_path)}"); print(f"📊 Excel: {os.path.basename(xls_path)}")
        btn_run.disabled = False

btn_run.on_click(click_ejecutar)

# --- Layout ---
print("================== CEYLAN ALFA 5.4 (FINAL: ETIQUETAS + TARJETAS) ==================")
display(widgets.VBox([
    widgets.HBox([widgets.Label("1. DATOS:"), upl_widget]), out_upl,
    met_box, widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("2. PARÁMETROS:"), slider_min, slider_max]),
    widgets.HBox([widgets.Label("3. MUESTREO:"), inp_sample, chk_full]),
    widgets.HTML("<br>"),
    btn_run, out_run
]))

Vesion constreñida (compactas) con filtros: (Poligonos,Outliers,Ruta,MET,Todo/Nada), outliers, comodines, popups: (Ruta,Clientes,MET,Resumen), poligonos (vertices de clientes); Excel con totales y volumen m3/entrega

In [ ]:
# =========================================================================
# === MÓDULO 1: BACKEND (DBSCAN + K-MEANS CONSTREÑIDO + CONVEX HULL) ======
# =========================================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import folium
from folium.features import CustomIcon
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import random
import numpy as np
import openpyxl
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, MultiPoint, Point
from sklearn.cluster import DBSCAN, KMeans

# --- IMPORTANTE: Intentar importar la librería nueva ---
try:
    from k_means_constrained import KMeansConstrained
    HAS_KMEANS_CONST = True
except ImportError:
    HAS_KMEANS_CONST = False
    print("⚠️ ADVERTENCIA: 'k-means-constrained' no está instalado. Ejecuta '!pip install k-means-constrained'")

# --- Configuración ---
BASE_DIR = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan"
ICON_MET_PATH = os.path.join(BASE_DIR, "95_24.png")
OUTPUT_DIR = os.path.join(BASE_DIR, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

class CeylanState:
    def __init__(self):
        self.df_clientes = pd.DataFrame()
        self.df_cdr = pd.DataFrame()
        self.df_clientes_backup = pd.DataFrame()
        self.execution_count = 0
        self.met_checkboxes = []
        self.colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
state = CeylanState()

# --- HELPERS ---
def normalizar_id(valor):
    s = str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('/', '').replace('\\', '')
    return s.replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')

def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0: return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min: return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# --- CLUSTERING & LÓGICA ---

def segregar_por_frecuencia(df_clients, threshold=0.01):
    if 'Ruta_nueva' not in df_clients.columns: df_clients['Ruta_nueva'] = -1
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    
    # Separamos clientes frecuentes (Core) de los esporádicos (Comodines)
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    
    df_elegibles['es_comodin'] = False
    df_elegibles['es_outlier'] = False
    df_comodines['es_comodin'] = True
    df_comodines['es_outlier'] = False
    
    return df_elegibles, df_comodines

def filtrar_outliers_dbscan(df_core, eps_km=2.0, min_samples=3):
    if len(df_core) < min_samples: return df_core, pd.DataFrame(columns=df_core.columns)
    coords_rad = np.radians(df_core[['U_latitud', 'U_longitud']].values)
    
    db = DBSCAN(eps=eps_km/6371.0, min_samples=min_samples, metric='haversine').fit(coords_rad)
    
    mask = db.labels_ == -1
    df_out = df_core[mask].copy()
    df_cln = df_core[~mask].copy()
    
    if not df_out.empty:
        df_out['es_outlier'] = True
        df_out['es_comodin'] = False 
    return df_cln, df_out

def ejecutar_clustering_constrenido(df_elegibles, min_size, max_size):
    if df_elegibles.empty: return df_elegibles
    
    coords = df_elegibles[['U_latitud', 'U_longitud']].values
    n_samples = len(coords)
    
    target_avg = (min_size + max_size) / 2
    n_clusters = int(round(n_samples / target_avg))
    
    if n_clusters < 1: n_clusters = 1
    if n_clusters * max_size < n_samples: n_clusters = int(np.ceil(n_samples / max_size))
    if n_clusters * min_size > n_samples: n_clusters = int(np.floor(n_samples / min_size))
    if n_clusters < 1: n_clusters = 1

    print(f"   ⮑ Optimizando: {n_samples} clientes en {n_clusters} rutas (Obj: {min_size}-{max_size})...")

    if HAS_KMEANS_CONST:
        try:
            clf = KMeansConstrained(
                n_clusters=n_clusters,
                size_min=min_size,
                size_max=max_size,
                random_state=42,
                n_init=10
            )
            clf.fit(coords)
            df_elegibles['Ruta_nueva'] = clf.labels_
        except Exception as e:
            print(f"⚠️ Falló K-Means Constreñido ({e}). Usando K-Means simple.")
            kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
            df_elegibles['Ruta_nueva'] = kmeans.labels_
    else:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
        df_elegibles['Ruta_nueva'] = kmeans.labels_

    return df_elegibles

def asignar_comodines_post_clustering(df_pendientes, df_elegibles_asignados, max_km=5.0):
    if df_pendientes.empty: return df_pendientes
    if df_elegibles_asignados.empty: return df_pendientes
        
    coords_pend = df_pendientes[['U_longitud', 'U_latitud']].values
    rutas_info = df_elegibles_asignados.groupby('Ruta_nueva')[['U_longitud', 'U_latitud']].mean()
    coords_centroides = rutas_info.values
    rutas_ids = rutas_info.index.values
    
    dist_matrix = cdist(coords_pend, coords_centroides, metric='euclidean')
    
    min_dists = np.min(dist_matrix, axis=1)
    idx_closest_centroid = np.argmin(dist_matrix, axis=1)
    limit_deg = max_km / 111.0 
    
    rutas_asignadas = rutas_ids[idx_closest_centroid]
    rutas_finales = np.where(min_dists <= limit_deg, rutas_asignadas, -1)
    
    df_pendientes['Ruta_nueva'] = rutas_finales
    df_pendientes.loc[df_pendientes['Ruta_nueva'] == -1, 'es_lejano'] = True
    
    return df_pendientes

# =========================================================================
# === CAMBIO CLAVE AQUÍ: LÓGICA DE DIBUJO DE POLÍGONOS ====================
# =========================================================================
def calculate_cluster_polygons(df):
    """
    MODIFICADO: Genera polígonos usando ÚNICAMENTE Convex Hull.
    El polígono será la "liga elástica" que envuelve a los clientes extremos.
    """
    polygons = {}
    rutas = sorted([r for r in df['Ruta_nueva'].unique() if r >= 0])
    
    for r in rutas:
        # Extraemos coordenadas [Lat, Lon] para Folium
        # NOTA: ConvexHull funciona bien con Lat/Lon directo
        pts = df[df['Ruta_nueva'] == r][['U_latitud', 'U_longitud']].values
        
        # Necesitamos al menos 3 puntos para hacer un polígono (triángulo)
        if len(pts) >= 3:
            try:
                # 1. Calculamos la Envolvente Convexa
                hull = ConvexHull(pts)
                
                # 2. Obtenemos los vértices en orden (sentido antihorario)
                # hull.vertices nos da los índices de los puntos extremos
                vertices = pts[hull.vertices]
                
                # 3. Guardamos la lista de coordenadas para Folium
                polygons[r] = vertices.tolist()
            except Exception as e:
                pass # Si fallan las mates (puntos colineales), no dibujamos polígono
                
    return polygons
# =========================================================================

def procesar_logica_met(met_id, n_muestra, analizar_todos, min_size, max_size):
    df_terr = state.df_clientes[state.df_clientes['MET_Asignado'] == met_id].copy()
    if df_terr.empty: return None, None, "Sin clientes"
    
    df_proc = df_terr.copy() if analizar_todos or len(df_terr) <= n_muestra else df_terr.sample(n=n_muestra).copy()
    
    df_proc['entregas/día'] = pd.to_numeric(df_proc['entregas/día'], errors='coerce').fillna(0)
    df_proc['m3/día'] = pd.to_numeric(df_proc['m3/día'], errors='coerce').fillna(0)
    
    coords = df_proc[['U_longitud', 'U_latitud']].values
    if len(coords) == 0: return None, None, "Sin coordenadas"
    
    # 1. Separar Comodines
    df_eleg, df_como = segregar_por_frecuencia(df_proc, threshold=0.01)
    
    # 2. Detectar Outliers
    df_eleg, df_outliers = filtrar_outliers_dbscan(df_eleg, eps_km=3.0, min_samples=3)
    
    # 3. CLUSTERING (K-Means Constrained)
    df_eleg = ejecutar_clustering_constrenido(df_eleg, min_size, max_size)
    
    # 4. Asignar Pendientes
    df_pend = pd.concat([df_como, df_outliers], ignore_index=True)
    df_pend = asignar_comodines_post_clustering(df_pend, df_eleg, max_km=10.0)
    
    # 5. Unificación
    df_final = pd.concat([df_eleg, df_pend], ignore_index=True)
    
    # 6. Polígonos (AHORA USANDO SOLO CONVEX HULL)
    polys = calculate_cluster_polygons(df_final)
    
    return df_final, polys, "OK"

# --- MÓDULO DE EXPORTACIÓN ---
def generate_consolidated_excel(export_data, summary_data, output_dir, filename_suffix=None):
    if not export_data: return None
    if not filename_suffix: filename_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_dir, f'Resultados_Ceylan_{filename_suffix}.xlsx')
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        if summary_data: pd.DataFrame(summary_data).to_excel(writer, sheet_name='Resumen', index=False)
        if export_data: pd.DataFrame(export_data).to_excel(writer, sheet_name='Detalle', index=False)
    return path

# --- HANDLER DE CARGA ---
def handle_upload(c, mb, ou):
    with ou:
        clear_output()
        if not c['new']: return
        try:
            content = c['owner'].value[0]['content'] if isinstance(c['owner'].value, tuple) else list(c['owner'].value.values())[0]['content']
            with open("temp.xlsx", 'wb') as f: f.write(content)
            xls = pd.read_excel("temp.xlsx", sheet_name=None)
            state.df_clientes = xls[list(xls.keys())[0]]
            state.df_cdr = xls[list(xls.keys())[1]]
            
            state.df_clientes['m3/día'] = pd.to_numeric(state.df_clientes['m3/día'], errors='coerce').fillna(0)
            state.df_clientes['entregas/día'] = pd.to_numeric(state.df_clientes['entregas/día'], errors='coerce').fillna(0)
            
            mask_freq = state.df_clientes['entregas/día'] > 50
            if mask_freq.any(): state.df_clientes.loc[mask_freq, 'entregas/día'] = 0
            
            state.df_clientes_backup = state.df_clientes.copy()
            mets = sorted(state.df_cdr['CodMET'].unique())
            state.met_checkboxes = [widgets.Checkbox(description=str(m), value=False) for m in mets]
            mb.children = [widgets.Label("Selecciona METs:")] + state.met_checkboxes
            print(f"✅ Cargado. Clientes: {len(state.df_clientes)}")
            os.remove("temp.xlsx")
        except Exception as e: print(f"❌ Error: {e}")

print("✅ MÓDULO 1: ACTUALIZADO (POLÍGONOS AJUSTADOS A VÉRTICES / CONVEX HULL).")

✅ MÓDULO 1: ACTUALIZADO (POLÍGONOS AJUSTADOS A VÉRTICES / CONVEX HULL).


In [ ]:
# =========================================================================
# === MÓDULO 2: EJECUCIÓN FINAL (CORREGIDO: ERROR PARAMETRO CLASS_NAME) ===
# =========================================================================

from folium.features import DivIcon

# --- Widgets ---
upl_widget = widgets.FileUpload(accept='.xlsx', description='Subir Excel')
met_box = widgets.VBox([widgets.Label('Esperando archivo...')])
out_upl = widgets.Output()

slider_min = widgets.IntSlider(value=20, min=5, max=50, description='Min Clientes')
slider_max = widgets.IntSlider(value=40, min=10, max=100, description='Max Clientes')
inp_sample = widgets.IntText(value=500, description='Muestra/MET')
chk_full = widgets.Checkbox(value=False, description='Analizar TODO')

btn_run = widgets.Button(description='🚀 Ejecutar Análisis', button_style='success', layout=widgets.Layout(width='100%'))
out_run = widgets.Output()

# Conectar carga
upl_widget.observe(lambda c: handle_upload(c, met_box, out_upl), names='value')

# --- LÓGICA PRINCIPAL ---
def click_ejecutar(b):
    state.execution_count += 1
    btn_run.disabled = True
    
    with out_run:
        clear_output()
        
        # 1. Validaciones
        sel_mets = [cb.description for cb in state.met_checkboxes if cb.value]
        if state.df_clientes.empty or not sel_mets:
            print("⚠️ Error: Carga un archivo y selecciona al menos un MET.")
            btn_run.disabled = False; return

        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f"🚀 INICIANDO ANÁLISIS - ID: {run_id}")
        
        # 2. Restaurar Backup
        state.df_clientes = state.df_clientes_backup.copy()

        # =====================================================================
        # === LIMPIEZA DE DECIMALES (COMAS POR PUNTOS) ========================
        # =====================================================================
        state.df_clientes.columns = state.df_clientes.columns.str.strip()
        cols_to_clean = ['m3/entrega', 'productos/entrega']
        
        for col in cols_to_clean:
            if col in state.df_clientes.columns:
                try:
                    state.df_clientes[col] = state.df_clientes[col].astype(str).str.replace(',', '.', regex=False)
                    state.df_clientes[col] = pd.to_numeric(state.df_clientes[col], errors='coerce').fillna(0)
                except Exception as e:
                    print(f"⚠️ Error limpiando columna {col}: {e}")
        # =====================================================================
        
        # 3. Asignación de Territorios
        print("🗺️ Asignando territorios...")
        coords_cli = state.df_clientes[['U_latitud', 'U_longitud']].values
        coords_met = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)][['U_latitud', 'U_longitud']].values
        met_names = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)]['CodMET'].values
        dists = cdist(coords_cli, coords_met)
        state.df_clientes['MET_Asignado'] = met_names[np.argmin(dists, axis=1)]
        
        # 4. Procesamiento
        # --- ZOOM AUTOMÁTICO ---
        df_zoom = state.df_clientes[state.df_clientes['MET_Asignado'].isin(sel_mets)]
        if not df_zoom.empty:
            sw = df_zoom[['U_latitud', 'U_longitud']].min().values.tolist()
            ne = df_zoom[['U_latitud', 'U_longitud']].max().values.tolist()
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            mapa_base.fit_bounds([sw, ne])
        else:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')

        export_rows, summary_rows = [], []
        
        # --- INICIALIZACIÓN DE CONTADORES GLOBALES ---
        g_total_clientes = 0
        g_total_normales = 0
        g_total_comodines = 0
        g_total_outliers = 0
        g_total_rutas = 0
        
        for met in sel_mets:
            print(f"🔨 Procesando {met}...", end=" ")
            
            # Datos MET
            met_info = state.df_cdr[state.df_cdr['CodMET'] == met].iloc[0]
            met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
            
            # Lógica M1
            df_res, polys, status = procesar_logica_met(
                met, inp_sample.value, chk_full.value, 
                slider_min.value, slider_max.value
            )
            
            if status != "OK": print(f"({status})"); continue
                
            rutas_ids = sorted([r for r in df_res['Ruta_nueva'].unique() if r >= 0])
            print(f"✅ {len(rutas_ids)} rutas.")
            
            # --- CÁLCULO DE CONTADORES GLOBALES ---
            g_total_rutas += len(rutas_ids)
            g_total_clientes += len(df_res)
            g_total_outliers += len(df_res[df_res.get('es_outlier', False) == True])
            g_total_comodines += len(df_res[(df_res.get('es_comodin', False) == True) & (df_res.get('es_outlier', False) == False)])
            g_total_normales += len(df_res[(df_res.get('es_comodin', False) == False) & (df_res.get('es_outlier', False) == False)])
            
            # Rangos volumen
            df_normales = df_res[df_res.get('es_comodin', False) == False]
            col_vol_visual = 'm3/entrega' if 'm3/entrega' in df_res.columns else 'm3/día'
            
            if not df_normales.empty:
                vol_min_local = df_normales[col_vol_visual].min()
                vol_max_local = df_normales[col_vol_visual].max()
            else:
                vol_min_local, vol_max_local = 0, 1
            
            # Estadísticas MET
            volumen_total_met = df_res[col_vol_visual].sum()
            total_clientes_met = len(df_res)
            met_display_name = str(met).strip()
            
            # --- POPUP MET (HOME) ---
            fg_met_home = folium.FeatureGroup(name=f"{met_display_name} - 🏠 MET", show=True, attribution=f"{met_display_name} - 🏠 MET")
            
            popup_met = f"""
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial, sans-serif;">
                  <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                      <div style="font-size: 22px; font-weight: bold;">🏠 {met}</div>
                  </div>
                  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                      <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                          <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN TOTAL</div>
                          <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_total_met:,.2f} m³</div>
                          <div style="font-size: 10px; color: #555;">(Suma m³/entrega)</div>
                      </div>
                      <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                          <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                          <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{total_clientes_met}</div>
                      </div>
                  </div>
                  <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                      <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 RUTAS GENERADAS</div>
                      <div style="font-size: 18px; font-weight: bold; color: #1B5E20;">{len(rutas_ids)}</div>
                  </div>
            </div>
            """
            
            if os.path.exists(ICON_MET_PATH):
                icon_obj = CustomIcon(ICON_MET_PATH, icon_size=(40, 40), icon_anchor=(20, 40))
                folium.Marker([met_lat, met_lon], icon=icon_obj, popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            else:
                folium.Marker([met_lat, met_lon], icon=folium.Icon(color='green', icon='home'), popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met_home)
            fg_met_home.add_to(mapa_base)

            # --- RUTAS Y CLIENTES ---
            for ruta_id in rutas_ids:
                ruta_df = df_res[df_res['Ruta_nueva'] == ruta_id]
                
                # TOTALES POR RUTA
                vol_ruta = ruta_df[col_vol_visual].sum()
                total_cli_ruta = len(ruta_df)
                
                n_normales_ruta = len(ruta_df[(ruta_df.get('es_comodin', False) == False) & (ruta_df.get('es_outlier', False) == False)])
                n_comodines_ruta = len(ruta_df[(ruta_df.get('es_comodin', False) == True) & (ruta_df.get('es_outlier', False) == False)])
                n_outliers_ruta = len(ruta_df[ruta_df.get('es_outlier', False) == True])
                
                sum_m3_dia_total = ruta_df['m3/día'].sum()
                col_prod_ent = 'productos/entrega'
                sum_prod_entrega_total = ruta_df[col_prod_ent].sum() if col_prod_ent in ruta_df.columns else 0

                layer_name = f"{met_display_name} - Ruta {ruta_id}"
                fg_ruta = folium.FeatureGroup(name=layer_name, show=True, attribution=layer_name)
                color_ruta = state.colores[ruta_id % len(state.colores)]
                
                # === TARJETA POPUP POLÍGONO (Ruta) ===
                html_popup_poligono = f"""
                <div style="font-family: 'Segoe UI', Arial, sans-serif; min-width: 220px; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.2); border: 1px solid #ccc;">
                    <div style="background-color: #1a237e; color: white; padding: 10px 12px; font-weight: bold; font-size: 15px; display: flex; align-items: center; justify-content: space-between;">
                        <span>🚛 Ruta {ruta_id}</span>
                        <span style="font-size: 12px; font-weight: normal; opacity: 0.8;">{met}</span>
                    </div>
                    <div style="padding: 12px; background-color: white; color: #333;">
                        <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 6px;">
                            <span style="font-size: 16px;">📦</span>
                            <span style="font-weight: 600; font-size: 13px;">Volumen:</span>
                            <span style="font-size: 14px;">{vol_ruta:.2f} m³</span>
                        </div>
                        <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
                            <span style="font-size: 16px;">👥</span>
                            <span style="font-weight: 600; font-size: 13px;">Clientes:</span>
                            <span style="font-size: 14px;">{total_cli_ruta}</span>
                        </div>
                        <div style="border-top: 1px solid #eee; margin: 8px 0;"></div>
                        <div style="display: flex; justify-content: space-between; font-size: 12px; font-weight: bold; gap: 4px;">
                            <div title="Normales" style="color: #2E7D32; background: #E8F5E8; padding: 3px 6px; border-radius: 4px; flex: 1; text-align: center;">
                                ✅ Nor: {n_normales_ruta}
                            </div>
                            <div title="Comodines" style="color: #C62828; background: #FFEBEE; padding: 3px 6px; border-radius: 4px; flex: 1; text-align: center;">
                                ▲ Com: {n_comodines_ruta}
                            </div>
                        </div>
                        {f'''<div style="margin-top: 5px; color: #fff; background: #333; padding: 2px 6px; border-radius: 4px; text-align: center; font-size: 11px;">
                                💀 Outliers: {n_outliers_ruta}
                             </div>''' if n_outliers_ruta > 0 else ''}
                    </div>
                </div>
                """

                # Polígono
                if ruta_id in polys and polys[ruta_id]:
                    raw_poly = polys[ruta_id]
                    final_poly = raw_poly
                    if raw_poly and len(raw_poly) > 0:
                        p1 = raw_poly[0]
                        if p1[0] < p1[1]: final_poly = [[c[1], c[0]] for c in raw_poly]
                    
                    folium.Polygon(
                        locations=final_poly,
                        color=color_ruta, weight=2, fill=True, fill_opacity=0.2,
                        popup=folium.Popup(html_popup_poligono, max_width=300),
                        tooltip=f"Ver detalles Ruta {ruta_id}",
                        className='polygon-shape'
                    ).add_to(fg_ruta)

                # =================================================================
                # === FIX: ETIQUETAS DE RUTA (class_name corregido) ===============
                # =================================================================
                if not ruta_df.empty:
                    center_lat = ruta_df['U_latitud'].mean()
                    center_lon = ruta_df['U_longitud'].mean()
                    
                    icon_text = DivIcon(
                        class_name='ruta-label-icon', # <--- ¡AHORA SÍ ES CORRECTO!
                        icon_size=(100,30), icon_anchor=(50,15),
                        html=f'''
                        <div style="
                            font-size: 14pt; 
                            font-weight: 900; 
                            color: {color_ruta};
                            opacity: 0.85; 
                            text-align: center; 
                            white-space: nowrap;
                            font-family: 'Segoe UI', Arial, sans-serif;
                            text-shadow: 
                                -2px -2px 0 #fff,  
                                 2px -2px 0 #fff,
                                -2px  2px 0 #fff,
                                 2px  2px 0 #fff;
                            pointer-events: none;
                        ">
                            R{ruta_id}
                        </div>
                        '''
                    )
                    folium.Marker([center_lat, center_lon], icon=icon_text, zIndexOffset=1000).add_to(fg_ruta)
                # =================================================================
                
                # Marcadores
                for _, row in ruta_df.iterrows():
                    es_comodin = row.get('es_comodin', False)
                    es_outlier = row.get('es_outlier', False) 
                    vol = row[col_vol_visual] 
                    frec = row['entregas/día'] * 100
                    
                    if es_outlier:
                        tipo_txt = "💀 OUTLIER"
                    elif es_comodin:
                        tipo_txt = "🔺 COMODÍN"
                    else:
                        tipo_txt = "✅ NORMAL"

                    popup_html_cli = f"""
                    <div style="background: #fff; border-radius: 12px; padding: 0; overflow: hidden; box-shadow: 0 4px 12px rgba(0,0,0,0.15); min-width: 280px; font-family: 'Segoe UI', Arial, sans-serif;">
                          <div style="background: #f0841f; border-left: 6px solid #FF3D00; padding: 12px; font-size: 16px; color: #000; font-weight: bold; border-bottom: 1px solid #FFE0B2;">
                              {tipo_txt} | <span style="font-weight:900;">{row['CodCli']}</span>
                          </div>
                          <div style="padding: 12px;">
                              <div style="font-size:14px; color:#333; margin-bottom:8px; font-weight: 500;">{row.get('Nombre', 'N/A')}</div>
                              <div style="font-size:14px; color:#BF360C; margin-bottom:6px; background: #FFF8E1; padding: 6px; border-radius: 6px; border: 1px solid #FFE0B2;">
                                  📦 Vol: <b>{vol:.3f} m³</b> (Ent.)
                              </div>
                              <div style="font-size:14px; color:#1565C0; margin-bottom:6px;">📊 Frec: <b>{frec:.1f}%</b> (ent/día)</div>
                              <div style="font-size:12px; color:#757575; border-top: 1px solid #eee; padding-top: 8px; margin-top: 8px;">📍 Ruta: <b>{ruta_id}</b></div>
                          </div>
                    </div>
                    """
                    
                    if es_outlier:
                        folium.Marker([row['U_latitud'], row['U_longitud']], icon=folium.Icon(color='black', icon='remove', prefix='glyphicon'), popup=folium.Popup(popup_html_cli, max_width=320)).add_to(fg_ruta)
                    elif es_comodin:
                        folium.CircleMarker([row['U_latitud'], row['U_longitud']], radius=4, color='#D32F2F', fill=True, fill_color='#FFCDD2', fill_opacity=0.9, popup=folium.Popup(popup_html_cli, max_width=320)).add_to(fg_ruta)
                    else:
                        tamano_calc = obtener_tamano_marcador(vol, vol_min_local, vol_max_local)
                        folium.CircleMarker([row['U_latitud'], row['U_longitud']], radius=tamano_calc, color='#333333', weight=1, fill=True, fill_color=color_ruta, fill_opacity=0.85, popup=folium.Popup(popup_html_cli, max_width=320)).add_to(fg_ruta)
                    
                    row_dict = row.to_dict()
                    row_dict['m3/día Total'] = sum_m3_dia_total
                    row_dict['m3/entrega Total'] = vol_ruta
                    row_dict['productos/entrega Total'] = sum_prod_entrega_total
                    row_dict['es_comodin Total'] = n_comodines_ruta
                    row_dict['es_outlier Total'] = n_outliers_ruta
                    export_rows.append(row_dict)
                
                fg_ruta.add_to(mapa_base)
                
                summary_rows.append({
                    'MET': met, 
                    'Ruta': ruta_id, 
                    'Clientes': len(ruta_df),
                    'm3/día Total': sum_m3_dia_total,
                    'm3/entrega Total': vol_ruta,
                    'productos/entrega Total': sum_prod_entrega_total,
                    'es_comodin Total': n_comodines_ruta,
                    'es_outlier Total': n_outliers_ruta
                })

        # --- INYECCIÓN CSS/JS CON FILTRO NUEVO ---
        folium.LayerControl(collapsed=False).add_to(mapa_base)
        
        estilos_css = r'''
        <style>
        .leaflet-control-layers-overlays { display: none !important; } 
        .leaflet-control-layers-list { max-height: 600px; overflow-y: auto; width: 340px; padding: 12px; font-family: 'Segoe UI', Arial, sans-serif; background: #fff; } 
        
        .visual-control-container { display: flex; gap: 8px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 12px; flex-wrap: wrap; } 
        .visual-btn { flex: 1; padding: 8px; border: 1px solid #ccc; background: white; cursor: pointer; border-radius: 6px; font-weight: 700; font-size: 11px; color: #555; white-space: nowrap; display: flex; align-items: center; justify-content: center; gap: 4px; } 
        .visual-btn.active { background-color: #e3f2fd; color: #1976D2; border-color: #2196F3; } 
        
        /* REGLAS DE OCULTAMIENTO */
        .hide-all-polygons .polygon-shape { stroke: none !important; fill: none !important; pointer-events: none !important; } 
        .hide-outliers .awesome-marker-icon-black { display: none !important; opacity: 0 !important; } 
        .hide-outliers .awesome-marker-shadow { display: none !important; opacity: 0 !important; } 
        
        /* NUEVA REGLA PARA OCULTAR ETIQUETAS */
        .hide-labels .ruta-label-icon { display: none !important; opacity: 0 !important; }

        .layer-control-buttons { display: flex; gap: 8px; margin-bottom: 12px; } 
        .layer-btn { flex: 1; padding: 8px 0; font-size: 13px; font-weight: bold; border: none; border-radius: 4px; cursor: pointer; color: white; box-shadow: 0 2px 4px rgba(0,0,0,0.2); } 
        .select-all { background-color: #4CAF50; } .select-all:hover { background-color: #43A047; } 
        .deselect-all { background-color: #F44336; } .deselect-all:hover { background-color: #E53935; } 
        
        .search-ruta-container { display: flex; gap: 6px; align-items: center; margin-bottom: 12px; padding: 4px 0; } 
        .search-ruta-input { flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 4px; font-size: 13px; outline: none; } 
        .search-ruta-input:focus { border-color: #0275d8; box-shadow: 0 0 0 2px rgba(2,117,216,0.1); } 
        .search-ruta-btn { background: #0275d8; color: white; border: none; padding: 8px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; } 
        .search-ruta-clear { background: white; color: #dc3545; border: 1px solid #dc3545; padding: 8px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; } 
        
        .met-buttons-row { display: flex; gap: 8px; margin-bottom: 12px; } 
        .met-btn { flex: 1; padding: 8px; font-size: 12px; font-weight: 700; color: #0275d8; background: white; border: 2px solid #0275d8; border-radius: 6px; cursor: pointer; } 
        .met-btn:hover { background: #e3f2fd; } 
        .ruta-buttons-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 6px; background: #E1BEE7; padding: 10px; border-radius: 8px; } 
        .ruta-btn { padding: 8px 0; text-align: center; background: white; border: 2px solid #8E24AA; color: #8E24AA; border-radius: 6px; cursor: pointer; font-size: 11px; font-weight: 700; box-shadow: 0 1px 2px rgba(0,0,0,0.1); } 
        .ruta-btn:hover { background: #F3E5F5; transform: scale(1.05); } 
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))

        filtros_js = r'''
        <script> 
        function inicializarFiltros() { 
            const layerControl = document.querySelector('.leaflet-control-layers'); 
            if (!layerControl) return; 
            const layersList = layerControl.querySelector('.leaflet-control-layers-list'); 
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays'); 
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]'); 
            if (layersList.querySelector('.ruta-buttons-row')) return; 
            
            // --- BOTONES VISUALES ---
            const visualDiv = document.createElement('div'); visualDiv.className = 'visual-control-container'; 
            visualDiv.innerHTML = `
                <button class="visual-btn" id="btn-toggle-poly"><span>🟦</span> <span>Polígonos</span></button>
                <button class="visual-btn" id="btn-toggle-out"><span>💀</span> <span>Outliers</span></button>
                <button class="visual-btn" id="btn-toggle-lbl"><span>🏷️</span> <span>Etiquetas</span></button>
            `; 
            const mapContainer = document.querySelector('.leaflet-container'); 
            const btnPoly = visualDiv.querySelector('#btn-toggle-poly'); 
            const btnOut = visualDiv.querySelector('#btn-toggle-out'); 
            const btnLbl = visualDiv.querySelector('#btn-toggle-lbl'); 
            
            let polysHidden = false; 
            let outsHidden = false; 
            let lblsHidden = false;

            btnPoly.onclick = (e) => { e.preventDefault(); e.stopPropagation(); polysHidden = !polysHidden; 
                if(polysHidden) { mapContainer.classList.add('hide-all-polygons'); btnPoly.classList.add('active'); } 
                else { mapContainer.classList.remove('hide-all-polygons'); btnPoly.classList.remove('active'); } 
            }; 
            btnOut.onclick = (e) => { e.preventDefault(); e.stopPropagation(); outsHidden = !outsHidden; 
                if(outsHidden) { mapContainer.classList.add('hide-outliers'); btnOut.classList.add('active'); } 
                else { mapContainer.classList.remove('hide-outliers'); btnOut.classList.remove('active'); } 
            }; 
            // NUEVO HANDLER PARA ETIQUETAS
            btnLbl.onclick = (e) => { e.preventDefault(); e.stopPropagation(); lblsHidden = !lblsHidden; 
                if(lblsHidden) { mapContainer.classList.add('hide-labels'); btnLbl.classList.add('active'); } 
                else { mapContainer.classList.remove('hide-labels'); btnLbl.classList.remove('active'); } 
            };

            layersList.insertBefore(visualDiv, overlaysDiv); 
            
            // --- RESTO DE LOS BOTONES (TODO/NADA, BUSCADOR, ETC) ---
            const buttonsDiv = document.createElement('div'); buttonsDiv.className = 'layer-control-buttons'; 
            buttonsDiv.innerHTML = '<button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button>'; 
            buttonsDiv.querySelector('.select-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); }; 
            buttonsDiv.querySelector('.deselect-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); }; 
            layersList.insertBefore(buttonsDiv, overlaysDiv); 
            
            const searchDiv = document.createElement('div'); searchDiv.className = 'search-ruta-container'; 
            searchDiv.innerHTML = `<span style="font-size:16px;margin-right:4px;">🔍</span><input type="text" class="search-ruta-input" placeholder="Ruta ID..."><button class="search-ruta-btn">Buscar</button><button class="search-ruta-clear">Limpiar</button>`; 
            const input = searchDiv.querySelector('input'); 
            const doSearch = () => { const val = input.value.trim(); if(!val) return; checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { const label = cb.parentElement.textContent; const regex = new RegExp(`- Ruta ${val}$`); if(regex.test(label)) { if(!cb.checked) cb.click(); } }); }, 100); }; 
            const doClear = () => { input.value = ''; checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); }; 
            searchDiv.querySelector('.search-ruta-btn').onclick = (e) => { e.preventDefault(); doSearch(); }; 
            searchDiv.querySelector('.search-ruta-clear').onclick = (e) => { e.preventDefault(); doClear(); }; 
            input.onkeypress = (e) => { if(e.key === 'Enter') { e.preventDefault(); doSearch(); } }; 
            layersList.insertBefore(searchDiv, overlaysDiv); 
            
            const metDiv = document.createElement('div'); metDiv.className = 'met-buttons-row'; 
            const mets = new Set(); checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+/); if(match) mets.add(match[1].trim()); }); 
            Array.from(mets).sort().forEach(met => { const btn = document.createElement('button'); btn.className = 'met-btn'; btn.textContent = met.replace('MET ', ''); btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { if(cb.parentElement.textContent.includes(met + ' -')) if(!cb.checked) cb.click(); }); }, 100); }; metDiv.appendChild(btn); }); 
            layersList.insertBefore(metDiv, overlaysDiv); 
            
            const rutaDiv = document.createElement('div'); rutaDiv.className = 'ruta-buttons-row'; 
            const rutasSet = new Set(); checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/- Ruta\s+(\d+)/); if(match) rutasSet.add(parseInt(match[1])); }); 
            Array.from(rutasSet).sort((a,b)=>a-b).forEach(rutaId => { const btn = document.createElement('button'); btn.className = 'ruta-btn'; btn.textContent = 'R' + rutaId; btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { const regex = new RegExp(`- Ruta ${rutaId}$`); if(regex.test(cb.parentElement.textContent)) { if(!cb.checked) cb.click(); } }); }, 100); }; rutaDiv.appendChild(btn); }); 
            layersList.insertBefore(rutaDiv, overlaysDiv); 
        } 
        setTimeout(inicializarFiltros, 1500); 
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(filtros_js))
        
        # --- TITLE DASHBOARD ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1565C0 0%, #42A5F5 100%); color: white; padding: 10px 20px; border-radius: 30px; box-shadow: 0 8px 20px rgba(0,0,0,0.3); border: 3px solid white; font-family: 'Segoe UI', Arial; text-align: center; min-width: 400px;">
            <div style="font-size:15px; font-weight:bold; margin-bottom:6px; letter-spacing:1px;">🗺️ DASHBOARD DE TERRITORIO</div>
            <div style="display: flex; justify-content: space-around; gap: 15px; background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">
                <div title="Total de Clientes">👥 <b>{g_total_clientes}</b></div>
                <div title="Clientes Normales" style="color: #C8E6C9;">✅ <b>{g_total_normales}</b></div>
                <div title="Comodines" style="color: #FFCDD2;">🔺 <b>{g_total_comodines}</b></div>
                <div title="Outliers Detectados" style="color: #212121; background:#fff; padding:0 5px; border-radius:10px;">💀 <b>{g_total_outliers}</b></div>
            </div>
            <div style="font-size:10px; margin-top:4px; opacity:0.8;">ID: {run_id} | Rutas: {g_total_rutas}</div>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))

        # --- Guardar ---
        map_path = os.path.join(OUTPUT_DIR, f"Mapa_Ceylan_{run_id}.html")
        mapa_base.save(map_path)
        xls_path = generate_consolidated_excel(export_rows, summary_rows, OUTPUT_DIR, filename_suffix=run_id)
        
        print(f"\n✅ FINALIZADO."); print(f"🗺️ Mapa: {os.path.basename(map_path)}"); print(f"📊 Excel: {os.path.basename(xls_path)}")
        btn_run.disabled = False

btn_run.on_click(click_ejecutar)

# --- Layout ---
print("================== CEYLAN ALFA 6.3 (FINAL: FILTRO ETIQUETAS) ==================")
display(widgets.VBox([
    widgets.HBox([widgets.Label("1. DATOS:"), upl_widget]), out_upl,
    met_box, widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("2. PARÁMETROS:"), slider_min, slider_max]),
    widgets.HBox([widgets.Label("3. MUESTREO:"), inp_sample, chk_full]),
    widgets.HTML("<br>"),
    btn_run, out_run
]))

================== CEYLAN ALFA 6.3 (FINAL: FILTRO ETIQUETAS) ==================


Versión definitiva del algoritmo de creación de rutas

In [ ]:
# =========================================================================
# === MÓDULO 1: BACKEND V8.3 (TORNEO RADIAL + EFICIENCIA + HULL) ==========
# =========================================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import numpy as np
from scipy.spatial import ConvexHull, Voronoi, distance_matrix
from scipy.spatial.distance import cdist
from shapely.geometry import Polygon, MultiPoint
from sklearn.cluster import DBSCAN, KMeans
import warnings

# Ignorar advertencias
warnings.filterwarnings('ignore')

# --- Intentar importar librería ---
try:
    from k_means_constrained import KMeansConstrained
    HAS_KMEANS_CONST = True
except ImportError:
    HAS_KMEANS_CONST = False
    print("⚠️ ADVERTENCIA: 'k-means-constrained' no está instalado. Ejecuta '!pip install k-means-constrained'")

class CeylanState:
    def __init__(self):
        self.df_clientes = pd.DataFrame()
        self.df_cdr = pd.DataFrame()
        self.df_clientes_backup = pd.DataFrame()
        self.execution_count = 0
        self.met_checkboxes = []
        self.colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
state = CeylanState()

# Configuración Rutas
BASE_DIR = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan"
ICON_MET_PATH = os.path.join(BASE_DIR, "95_24.png")
OUTPUT_DIR = os.path.join(BASE_DIR, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- HELPERS ---
def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=6, tamano_max=25):
    if vol_max == vol_min or volumen <= 0: return tamano_min
    log_vol = np.log1p(volumen)
    log_min = np.log1p(vol_min)
    log_max = np.log1p(vol_max)
    if log_max == log_min: return tamano_min
    proporcion = (log_vol - log_min) / (log_max - log_min)
    return int(tamano_min + proporcion * (tamano_max - tamano_min))

# --- CLUSTERING ---
def segregar_por_frecuencia(df_clients, threshold=0.01):
    if 'Ruta_nueva' not in df_clients.columns: df_clients['Ruta_nueva'] = -1
    df_no_met = df_clients[df_clients['Ruta_nueva'] == -1].copy()
    df_elegibles = df_no_met[df_no_met['entregas/día'] > threshold].copy()
    df_comodines = df_no_met[df_no_met['entregas/día'] <= threshold].copy()
    df_elegibles['es_comodin'] = False; df_elegibles['es_outlier'] = False
    df_comodines['es_comodin'] = True; df_comodines['es_outlier'] = False
    return df_elegibles, df_comodines

def filtrar_outliers_dbscan(df_core, eps_km=2.0, min_samples=3):
    if len(df_core) < min_samples: return df_core, pd.DataFrame(columns=df_core.columns)
    coords_rad = np.radians(df_core[['U_latitud', 'U_longitud']].values)
    db = DBSCAN(eps=eps_km/6371.0, min_samples=min_samples, metric='haversine').fit(coords_rad)
    mask = db.labels_ == -1
    df_out = df_core[mask].copy()
    df_cln = df_core[~mask].copy()
    if not df_out.empty:
        df_out['es_outlier'] = True; df_out['es_comodin'] = False 
    return df_cln, df_out

# =============================================================================
# === 🔴 CEREBRO: TORNEO RADIAL (Distancia al MET) + LLENADO AGRESIVO =========
# =============================================================================
def ejecutar_clustering_constrenido_radial(df_elegibles, min_size, max_size, met_coords):
    if df_elegibles.empty: return df_elegibles
    
    coords = df_elegibles[['U_latitud', 'U_longitud']].values
    n_samples = len(coords)
    
    # 1. Llenado Agresivo (90%) para minimizar rutas
    target_fill = max_size * 0.90 
    n_clusters = int(np.ceil(n_samples / target_fill))
    
    # Ajustes matemáticos
    if n_clusters * max_size < n_samples: n_clusters = int(np.ceil(n_samples / max_size))
    if n_clusters * min_size > n_samples: n_clusters = int(np.floor(n_samples / min_size))
    if n_clusters < 1: n_clusters = 1

    print(f"   ⮑ Torneo Radial: {n_samples} clientes -> {n_clusters} rutas (Meta: Cercanía MET)...")

    if not HAS_KMEANS_CONST:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
        df_elegibles['Ruta_nueva'] = kmeans.labels_
        return df_elegibles

    # 2. El Torneo (5 Rondas)
    best_labels = None
    best_score = float('inf')
    
    try:
        for i in range(5):
            clf = KMeansConstrained(
                n_clusters=n_clusters,
                size_min=min_size,
                size_max=max_size,
                random_state=42 + i, # Semilla variable
                n_init=10 
            )
            clf.fit(coords)
            
            # Juez: Distancia de los centros de ruta al MET
            centroides = clf.cluster_centers_
            dists = cdist(centroides, [met_coords], metric='euclidean')
            score = np.sum(dists)
            
            if score < best_score:
                best_score = score
                best_labels = clf.labels_
        
        df_elegibles['Ruta_nueva'] = best_labels
        print(f"      🏆 Ganador encontrado (Score: {best_score:.4f})")

    except Exception as e:
        print(f"⚠️ Error optimización ({e}). Usando fallback.")
        kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
        df_elegibles['Ruta_nueva'] = kmeans.labels_

    return df_elegibles

def asignar_comodines_post_clustering(df_pendientes, df_elegibles_asignados, max_km=5.0):
    if df_pendientes.empty or df_elegibles_asignados.empty: return df_pendientes
    coords_pend = df_pendientes[['U_longitud', 'U_latitud']].values
    rutas_info = df_elegibles_asignados.groupby('Ruta_nueva')[['U_longitud', 'U_latitud']].mean()
    coords_centroides = rutas_info.values
    rutas_ids = rutas_info.index.values
    
    dist_matrix = cdist(coords_pend, coords_centroides, metric='euclidean')
    min_dists = np.min(dist_matrix, axis=1)
    idx_closest = np.argmin(dist_matrix, axis=1)
    
    rutas_finales = np.where(min_dists <= (max_km/111.0), rutas_ids[idx_closest], -1)
    df_pendientes['Ruta_nueva'] = rutas_finales
    df_pendientes.loc[df_pendientes['Ruta_nueva'] == -1, 'es_lejano'] = True
    return df_pendientes

def calculate_cluster_polygons(df):
    polygons = {}
    rutas = sorted([r for r in df['Ruta_nueva'].unique() if r >= 0])
    for r in rutas:
        pts = df[df['Ruta_nueva'] == r][['U_latitud', 'U_longitud']].values
        if len(pts) >= 3:
            try:
                hull = ConvexHull(pts)
                polygons[r] = pts[hull.vertices].tolist()
            except: pass
    return polygons

def procesar_logica_met(met_id, n_muestra, analizar_todos, min_size, max_size):
    df_terr = state.df_clientes[state.df_clientes['MET_Asignado'] == met_id].copy()
    if df_terr.empty: return None, None, "Sin clientes"
    
    met_data = state.df_cdr[state.df_cdr['CodMET'] == met_id].iloc[0]
    met_coords = [met_data['U_latitud'], met_data['U_longitud']]
    
    df_proc = df_terr.copy() if analizar_todos or len(df_terr) <= n_muestra else df_terr.sample(n=n_muestra, random_state=42).copy()
    
    df_proc['entregas/día'] = pd.to_numeric(df_proc['entregas/día'], errors='coerce').fillna(0)
    df_proc['m3/día'] = pd.to_numeric(df_proc['m3/día'], errors='coerce').fillna(0)
    
    coords = df_proc[['U_longitud', 'U_latitud']].values
    if len(coords) == 0: return None, None, "Sin coordenadas"
    
    df_eleg, df_como = segregar_por_frecuencia(df_proc, threshold=0.01)
    df_eleg, df_outliers = filtrar_outliers_dbscan(df_eleg, eps_km=3.0, min_samples=3)
    
    # 🔴 LLAMADA AL CEREBRO RADIAL
    df_eleg = ejecutar_clustering_constrenido_radial(df_eleg, min_size, max_size, met_coords)
    
    df_pend = pd.concat([df_como, df_outliers], ignore_index=True)
    df_pend = asignar_comodines_post_clustering(df_pend, df_eleg, max_km=10.0)
    
    df_final = pd.concat([df_eleg, df_pend], ignore_index=True)
    polys = calculate_cluster_polygons(df_final)
    
    return df_final, polys, "OK"

def generate_consolidated_excel(export_data, summary_data, output_dir, filename_suffix=None):
    if not export_data: return None
    if not filename_suffix: filename_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_dir, f'Resultados_Ceylan_{filename_suffix}.xlsx')
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        if summary_data: pd.DataFrame(summary_data).to_excel(writer, sheet_name='Resumen', index=False)
        if export_data: pd.DataFrame(export_data).to_excel(writer, sheet_name='Detalle', index=False)
    return path

def handle_upload(c, mb, ou):
    with ou:
        clear_output()
        if not c['new']: return
        try:
            content = c['owner'].value[0]['content'] if isinstance(c['owner'].value, tuple) else list(c['owner'].value.values())[0]['content']
            with open("temp.xlsx", 'wb') as f: f.write(content)
            xls = pd.read_excel("temp.xlsx", sheet_name=None)
            state.df_clientes = xls[list(xls.keys())[0]]
            state.df_cdr = xls[list(xls.keys())[1]]
            
            state.df_clientes['m3/día'] = pd.to_numeric(state.df_clientes['m3/día'], errors='coerce').fillna(0)
            state.df_clientes['entregas/día'] = pd.to_numeric(state.df_clientes['entregas/día'], errors='coerce').fillna(0)
            mask_freq = state.df_clientes['entregas/día'] > 50
            if mask_freq.any(): state.df_clientes.loc[mask_freq, 'entregas/día'] = 0
            
            state.df_clientes_backup = state.df_clientes.copy()
            mets = sorted(state.df_cdr['CodMET'].unique())
            state.met_checkboxes = [widgets.Checkbox(description=str(m), value=False) for m in mets]
            mb.children = [widgets.Label("Selecciona METs:")] + state.met_checkboxes
            print(f"✅ Cargado. Clientes: {len(state.df_clientes)}")
            os.remove("temp.xlsx")
        except Exception as e: print(f"❌ Error: {e}")

print("✅ MÓDULO 1: ACTUALIZADO (TORNEO RADIAL + EFICIENCIA).")

✅ MÓDULO 1: ACTUALIZADO (TORNEO RADIAL + EFICIENCIA).


In [ ]:
# =========================================================================
# === MÓDULO 2: V7.5 (FILTROS CHIDOS + DASHBOARD + BARRA + OSRM) ==========
# =========================================================================

from folium.features import DivIcon
import requests
import math
import time

# --- Widgets ---
upl_widget = widgets.FileUpload(accept='.xlsx', description='Subir Excel')
met_box = widgets.VBox([widgets.Label('Esperando archivo...')])
out_upl = widgets.Output()

slider_min = widgets.IntSlider(value=20, min=5, max=50, description='Min Clientes')
slider_max = widgets.IntSlider(value=40, min=10, max=100, description='Max Clientes')
inp_sample = widgets.IntText(value=500, description='Muestra/MET')
chk_full = widgets.Checkbox(value=False, description='Analizar TODO')

# Barra de Progreso
prog_bar = widgets.IntProgress(
    value=0, min=0, max=100, description='Espera...',
    bar_style='info', style={'bar_color': '#007bff'},
    orientation='horizontal', layout=widgets.Layout(width='98%', display='none')
)
prog_label = widgets.Label(value="")

btn_run = widgets.Button(description='🚀 Ejecutar Análisis', button_style='success', layout=widgets.Layout(width='100%'))
out_run = widgets.Output()

# Conectar carga
upl_widget.observe(lambda c: handle_upload(c, met_box, out_upl), names='value')

# =============================================================================
# === CACHÉ Y MOTOR OSRM ======================================================
# =============================================================================
distancia_cache = {}

def obtener_distancia_carretera(lat1, lon1, lat2, lon2):
    key = f"{lat1:.4f},{lon1:.4f},{lat2:.4f},{lon2:.4f}"
    if key in distancia_cache: return distancia_cache[key]

    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=false"
    try:
        time.sleep(0.25) # Pausa seguridad API
        r = requests.get(url, timeout=2)
        if r.status_code == 200:
            data = r.json()
            if 'routes' in data and len(data['routes']) > 0:
                dist_km = data['routes'][0]['distance'] / 1000.0
                dur_min = data['routes'][0]['duration'] / 60.0
                res = (dist_km, dur_min, "Real (OSRM)")
                distancia_cache[key] = res
                return res
    except: pass
    
    # Fallback
    R = 6371 
    dLat = math.radians(lat2 - lat1); dLon = math.radians(lon2 - lon1)
    a = math.sin(dLat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dLon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    dist_est = (R * c) * 1.3
    time_est = (dist_est / 30.0) * 60
    res = (dist_est, time_est, "Estimada")
    distancia_cache[key] = res
    return res

# --- LÓGICA PRINCIPAL ---
def click_ejecutar(b):
    state.execution_count += 1
    btn_run.disabled = True
    prog_bar.layout.display = 'flex'
    prog_bar.value = 0
    prog_label.value = "Iniciando..."
    
    with out_run:
        clear_output()
        
        sel_mets = [cb.description for cb in state.met_checkboxes if cb.value]
        if state.df_clientes.empty or not sel_mets:
            print("⚠️ Error: Carga un archivo y selecciona al menos un MET.")
            btn_run.disabled = False; prog_bar.layout.display = 'none'; return

        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f"🚀 INICIANDO ANÁLISIS - ID: {run_id}")
        
        state.df_clientes = state.df_clientes_backup.copy()

        # Limpieza
        state.df_clientes.columns = state.df_clientes.columns.str.strip()
        for col in ['m3/entrega', 'productos/entrega']:
            if col in state.df_clientes.columns:
                try:
                    state.df_clientes[col] = state.df_clientes[col].astype(str).str.replace(',', '.', regex=False)
                    state.df_clientes[col] = pd.to_numeric(state.df_clientes[col], errors='coerce').fillna(0)
                except: pass
        
        print("🗺️ Asignando territorios...")
        prog_bar.description = "Territorios"
        coords_cli = state.df_clientes[['U_latitud', 'U_longitud']].values
        coords_met = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)][['U_latitud', 'U_longitud']].values
        met_names = state.df_cdr[state.df_cdr['CodMET'].isin(sel_mets)]['CodMET'].values
        state.df_clientes['MET_Asignado'] = met_names[np.argmin(cdist(coords_cli, coords_met), axis=1)]
        
        # Zoom
        df_zoom = state.df_clientes[state.df_clientes['MET_Asignado'].isin(sel_mets)]
        if not df_zoom.empty:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')
            mapa_base.fit_bounds([df_zoom[['U_latitud', 'U_longitud']].min().values.tolist(), df_zoom[['U_latitud', 'U_longitud']].max().values.tolist()])
        else:
            mapa_base = folium.Map(location=[23.63, -102.55], zoom_start=5, tiles='OpenStreetMap')

        export_rows, summary_rows = [], []
        g_total_clientes = g_total_normales = g_total_comodines = g_total_outliers = g_total_rutas = 0
        
        # --- BUCLE METS ---
        for met_idx, met in enumerate(sel_mets):
            print(f"🔨 ({met_idx+1}/{len(sel_mets)}) Procesando {met}...", end=" ")
            prog_bar.description = f"MET: {met}"
            
            met_info = state.df_cdr[state.df_cdr['CodMET'] == met].iloc[0]
            met_lat, met_lon = met_info['U_latitud'], met_info['U_longitud']
            
            df_res, polys, status = procesar_logica_met(met, inp_sample.value, chk_full.value, slider_min.value, slider_max.value)
            if status != "OK": print(f"({status})"); continue
                
            rutas_ids = sorted([r for r in df_res['Ruta_nueva'].unique() if r >= 0])
            print(f"✅ {len(rutas_ids)} rutas generadas.")
            
            g_total_rutas += len(rutas_ids)
            g_total_clientes += len(df_res)
            g_total_outliers += len(df_res[df_res.get('es_outlier', False) == True])
            g_total_comodines += len(df_res[(df_res.get('es_comodin', False) == True) & (df_res.get('es_outlier', False) == False)])
            g_total_normales += len(df_res[(df_res.get('es_comodin', False) == False) & (df_res.get('es_outlier', False) == False)])
            
            col_vol = 'm3/entrega' if 'm3/entrega' in df_res.columns else 'm3/día'
            df_norm = df_res[df_res.get('es_comodin', False) == False]
            vol_min, vol_max = (df_norm[col_vol].min(), df_norm[col_vol].max()) if not df_norm.empty else (0, 1)
            
            # Popup MET
            popup_met = f"""
            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; font-family: 'Segoe UI', Arial;">
                  <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center; font-weight: bold; font-size: 22px;">🏠 {met}</div>
                  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
                      <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #66BB6A;">
                          <div style="font-size: 12px; color: #2E7D32; font-weight: bold;">📦 VOL TOTAL</div>
                          <div style="font-size: 18px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{df_res[col_vol].sum():,.2f}</div>
                      </div>
                      <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #42A5F5;">
                          <div style="font-size: 12px; color: #1976D2; font-weight: bold;">👥 CLIENTES</div>
                          <div style="font-size: 18px; font-weight: bold; color: #0D47A1; margin: 4px 0;">{len(df_res)}</div>
                      </div>
                  </div>
            </div>
            """
            fg_met = folium.FeatureGroup(name=f"{str(met).strip()} - 🏠 MET", show=True)
            icon_obj = CustomIcon(ICON_MET_PATH, icon_size=(40, 40)) if os.path.exists(ICON_MET_PATH) else folium.Icon(color='green', icon='home')
            folium.Marker([met_lat, met_lon], icon=icon_obj, popup=folium.Popup(popup_met, max_width=400)).add_to(fg_met)
            fg_met.add_to(mapa_base)

            # Barra progreso rutas
            prog_bar.min = 0; prog_bar.max = len(rutas_ids); prog_bar.value = 0
            
            for i, r_id in enumerate(rutas_ids):
                prog_bar.value = i + 1; prog_label.value = f"Calc OSRM: Ruta {r_id} ({i+1}/{len(rutas_ids)})..."
                
                df_r = df_res[df_res['Ruta_nueva'] == r_id]
                vol_r = df_r[col_vol].sum()
                color = state.colores[r_id % len(state.colores)]
                fg_r = folium.FeatureGroup(name=f"{str(met).strip()} - Ruta {r_id}", show=True)
                
                # Distancia
                c_lat, c_lon = df_r['U_latitud'].mean(), df_r['U_longitud'].mean()
                dist_km, time_min, _ = obtener_distancia_carretera(met_lat, met_lon, c_lat, c_lon) if not df_r.empty else (0, 0, "")
                
                n_nor = len(df_r[(df_r.get('es_comodin',False)==False) & (df_r.get('es_outlier',False)==False)])
                n_com = len(df_r[(df_r.get('es_comodin',False)==True) & (df_r.get('es_outlier',False)==False)])
                n_out = len(df_r[df_r.get('es_outlier',False)==True])
                
                # POPUP RUTA
                html_poly = f"""
                <div style="font-family: 'Segoe UI', Arial; min-width: 240px; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.2); border: 1px solid #ccc;">
                    <div style="background-color: #1a237e; color: white; padding: 10px; font-weight: bold; display: flex; justify-content: space-between;">
                        <span>🚛 Ruta {r_id}</span><span style="font-size: 12px; opacity: 0.8;">{met}</span>
                    </div>
                    <div style="padding: 12px; background: white; color: #333;">
                        <div style="margin-bottom: 8px; background: #f0f0f0; padding: 8px; border-radius: 6px; display: flex; align-items: center; justify-content: space-between;">
                            <div style="display:flex; align-items:center; gap:4px;">
                                <span style="font-size: 14px;">🛣️</span><span style="font-size: 13px; font-weight: bold;">{dist_km:.1f} km</span>
                            </div>
                            <div style="display:flex; align-items:center; gap:4px; color: #555;">
                                <span style="font-size: 14px;">⏱️</span><span style="font-size: 13px; font-weight: bold;">{time_min:.0f} min</span>
                            </div>
                        </div>
                        <div style="margin-bottom: 6px;">📦 <b>{vol_r:.2f} m³</b> (Vol)</div>
                        <div style="margin-bottom: 10px;">👥 <b>{len(df_r)}</b> Clientes</div>
                        <div style="border-top: 1px solid #eee; padding-top: 8px; display: flex; gap: 4px; font-size: 12px; font-weight: bold;">
                            <div style="color: #2E7D32; background: #E8F5E8; padding: 3px 6px; border-radius: 4px; flex: 1; text-align: center;">✅ Nor: {n_nor}</div>
                            <div style="color: #C62828; background: #FFEBEE; padding: 3px 6px; border-radius: 4px; flex: 1; text-align: center;">▲ Com: {n_com}</div>
                        </div>
                        {f'<div style="margin-top:5px;background:#333;color:white;padding:2px;border-radius:4px;text-align:center;font-size:11px;">💀 Out: {n_out}</div>' if n_out>0 else ''}
                    </div>
                </div>
                """
                if r_id in polys and polys[r_id]:
                    p1 = polys[r_id][0]
                    locs = [[c[1], c[0]] for c in polys[r_id]] if p1[0] < p1[1] else polys[r_id]
                    folium.Polygon(locs, color=color, weight=2, fill=True, fill_opacity=0.2, popup=folium.Popup(html_poly, max_width=300), className='polygon-shape').add_to(fg_r)

                if not df_r.empty:
                    folium.Marker([c_lat, c_lon], icon=DivIcon(class_name='ruta-label-icon', icon_size=(100,30), icon_anchor=(50,15), html=f'<div style="font-size: 14pt; font-weight: 900; color: {color}; opacity: 0.85; text-align: center; white-space: nowrap; font-family: \'Segoe UI\'; text-shadow: -2px -2px 0 #fff, 2px -2px 0 #fff, -2px 2px 0 #fff, 2px 2px 0 #fff; pointer-events: none;">R{r_id}</div>'), zIndexOffset=1000).add_to(fg_r)

                for _, row in df_r.iterrows():
                    tipo, color_borde, bg_head = ("✅ NORMAL", color, "#f0841f")
                    if row.get('es_outlier'): tipo, color_borde, bg_head = ("💀 OUTLIER", "black", "black")
                    elif row.get('es_comodin'): tipo, color_borde, bg_head = ("🔺 COMODÍN", "#D32F2F", "#D32F2F")
                    
                    vol = row[col_vol] 
                    frec = row['entregas/día'] * 100
                    
                    pop_cli = f"""
                    <div style="background: #fff; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 12px rgba(0,0,0,0.15); min-width: 280px; font-family: 'Segoe UI';">
                        <div style="background: {bg_head}; border-left: 6px solid #FF3D00; padding: 12px; font-size: 16px; color: #fff; font-weight: bold;">{tipo} | {row['CodCli']}</div>
                        <div style="padding: 12px;">
                            <div style="font-size:14px; color:#333; margin-bottom:8px;">{row.get('Nombre', 'N/A')}</div>
                            <div style="font-size:14px; color:#BF360C; margin-bottom:6px; background: #FFF8E1; padding: 6px; border-radius: 6px; border: 1px solid #FFE0B2;">📦 Vol: <b>{vol:.3f} m³</b> (Ent.)</div>
                            <div style="font-size:14px; color:#1565C0; margin-bottom:6px;">📊 Frec: <b>{frec:.1f}%</b> (ent/día)</div>
                            <div style="font-size:12px; color:#757575; border-top: 1px solid #eee; padding-top: 8px;">📍 Ruta: <b>{r_id}</b></div>
                        </div>
                    </div>
                    """
                    
                    if row.get('es_outlier'): folium.Marker([row['U_latitud'], row['U_longitud']], icon=folium.Icon(color='black', icon='remove'), popup=folium.Popup(pop_cli, max_width=320)).add_to(fg_r)
                    elif row.get('es_comodin'): folium.CircleMarker([row['U_latitud'], row['U_longitud']], radius=4, color='#D32F2F', fill=True, fill_color='#FFCDD2', fill_opacity=0.9, popup=folium.Popup(pop_cli, max_width=320)).add_to(fg_r)
                    else: folium.CircleMarker([row['U_latitud'], row['U_longitud']], radius=obtener_tamano_marcador(row[col_vol], vol_min, vol_max), color='#333', weight=1, fill=True, fill_color=color, fill_opacity=0.85, popup=folium.Popup(pop_cli, max_width=320)).add_to(fg_r)
                    
                    row_dict = row.to_dict()
                    row_dict['Volumen Total por Ruta (m3/entrega)'] = vol_r
                    row_dict['Volumen Diario Total (m3/día)'] = df_r['m3/día'].sum()
                    row_dict['Productos por Entrega Total'] = df_r['productos/entrega'].sum() if 'productos/entrega' in df_r else 0
                    row_dict['Total Comodines en Ruta'] = n_com
                    row_dict['Total Outliers en Ruta'] = n_out
                    row_dict['Distancia Estimada al MET (km)'] = dist_km
                    row_dict['Tiempo Estimado al MET (min)'] = time_min
                    export_rows.append(row_dict)
                
                fg_r.add_to(mapa_base)
                summary_rows.append({'MET': met, 'Ruta': r_id, 'Clientes': len(df_r), 'Volumen Ruta (m3/entrega)': vol_r, 'Distancia (km)': dist_km, 'Tiempo Viaje (min)': time_min, 'Comodines': n_com, 'Outliers': n_out})

        # --- RESTAURACIÓN DEL TITLE DASHBOARD (ESTO FALTABA) ---
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #1565C0 0%, #42A5F5 100%); color: white; padding: 10px 20px; border-radius: 30px; box-shadow: 0 8px 20px rgba(0,0,0,0.3); border: 3px solid white; font-family: 'Segoe UI', Arial; text-align: center; min-width: 400px;">
            <div style="font-size:15px; font-weight:bold; margin-bottom:6px; letter-spacing:1px;">🗺️ DASHBOARD DE TERRITORIO</div>
            <div style="display: flex; justify-content: space-around; gap: 15px; background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">
                <div title="Total de Clientes">👥 <b>{g_total_clientes}</b></div>
                <div title="Clientes Normales" style="color: #C8E6C9;">✅ <b>{g_total_normales}</b></div>
                <div title="Comodines" style="color: #FFCDD2;">🔺 <b>{g_total_comodines}</b></div>
                <div title="Outliers Detectados" style="color: #212121; background:#fff; padding:0 5px; border-radius:10px;">💀 <b>{g_total_outliers}</b></div>
            </div>
            <div style="font-size:10px; margin-top:4px; opacity:0.8;">ID: {run_id} | Rutas: {g_total_rutas}</div>
        </div>
        '''
        mapa_base.get_root().html.add_child(folium.Element(titulo_html))

        # --- RESTAURACIÓN DE FILTROS CHIDOS ---
        folium.LayerControl(collapsed=False).add_to(mapa_base)
        
        estilos_css = r'''
        <style>
        .leaflet-control-layers-overlays { display: none !important; } 
        .leaflet-control-layers-list { max-height: 600px; overflow-y: auto; width: 340px; padding: 12px; font-family: 'Segoe UI', Arial, sans-serif; background: #fff; } 
        
        .visual-control-container { display: flex; gap: 8px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 12px; flex-wrap: wrap; } 
        .visual-btn { flex: 1; padding: 8px; border: 1px solid #ccc; background: white; cursor: pointer; border-radius: 6px; font-weight: 700; font-size: 11px; color: #555; white-space: nowrap; display: flex; align-items: center; justify-content: center; gap: 4px; } 
        .visual-btn.active { background-color: #e3f2fd; color: #1976D2; border-color: #2196F3; } 
        
        /* OCULTAMIENTO */
        .hide-all-polygons .polygon-shape { stroke: none !important; fill: none !important; pointer-events: none !important; } 
        .hide-outliers .awesome-marker-icon-black { display: none !important; opacity: 0 !important; } 
        .hide-outliers .awesome-marker-shadow { display: none !important; opacity: 0 !important; } 
        .hide-labels .ruta-label-icon { display: none !important; opacity: 0 !important; }

        .layer-control-buttons { display: flex; gap: 8px; margin-bottom: 12px; } 
        .layer-btn { flex: 1; padding: 8px 0; font-size: 13px; font-weight: bold; border: none; border-radius: 4px; cursor: pointer; color: white; box-shadow: 0 2px 4px rgba(0,0,0,0.2); } 
        .select-all { background-color: #4CAF50; } .select-all:hover { background-color: #43A047; } 
        .deselect-all { background-color: #F44336; } .deselect-all:hover { background-color: #E53935; } 
        
        .search-ruta-container { display: flex; gap: 6px; align-items: center; margin-bottom: 12px; padding: 4px 0; } 
        .search-ruta-input { flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 4px; font-size: 13px; outline: none; } 
        .search-ruta-input:focus { border-color: #0275d8; box-shadow: 0 0 0 2px rgba(2,117,216,0.1); } 
        .search-ruta-btn { background: #0275d8; color: white; border: none; padding: 8px 12px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; } 
        .search-ruta-clear { background: white; color: #dc3545; border: 1px solid #dc3545; padding: 8px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; font-weight: bold; } 
        
        .met-buttons-row { display: flex; gap: 8px; margin-bottom: 12px; } 
        .met-btn { flex: 1; padding: 8px; font-size: 12px; font-weight: 700; color: #0275d8; background: white; border: 2px solid #0275d8; border-radius: 6px; cursor: pointer; } 
        .met-btn:hover { background: #e3f2fd; } 
        .ruta-buttons-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 6px; background: #E1BEE7; padding: 10px; border-radius: 8px; } 
        .ruta-btn { padding: 8px 0; text-align: center; background: white; border: 2px solid #8E24AA; color: #8E24AA; border-radius: 6px; cursor: pointer; font-size: 11px; font-weight: 700; box-shadow: 0 1px 2px rgba(0,0,0,0.1); } 
        .ruta-btn:hover { background: #F3E5F5; transform: scale(1.05); } 
        </style>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css))

        filtros_js = r'''
        <script> 
        function inicializarFiltros() { 
            const layerControl = document.querySelector('.leaflet-control-layers'); 
            if (!layerControl) return; 
            const layersList = layerControl.querySelector('.leaflet-control-layers-list'); 
            const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays'); 
            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]'); 
            if (layersList.querySelector('.ruta-buttons-row')) return; 
            
            // --- BOTONES VISUALES ---
            const visualDiv = document.createElement('div'); visualDiv.className = 'visual-control-container'; 
            visualDiv.innerHTML = `
                <button class="visual-btn" id="btn-toggle-poly"><span>🟦</span> <span>Polígonos</span></button>
                <button class="visual-btn" id="btn-toggle-out"><span>💀</span> <span>Outliers</span></button>
                <button class="visual-btn" id="btn-toggle-lbl"><span>🏷️</span> <span>Etiquetas</span></button>
            `; 
            const mapContainer = document.querySelector('.leaflet-container'); 
            const btnPoly = visualDiv.querySelector('#btn-toggle-poly'); 
            const btnOut = visualDiv.querySelector('#btn-toggle-out'); 
            const btnLbl = visualDiv.querySelector('#btn-toggle-lbl'); 
            
            let polysHidden = false; 
            let outsHidden = false; 
            let lblsHidden = false;

            btnPoly.onclick = (e) => { e.preventDefault(); e.stopPropagation(); polysHidden = !polysHidden; 
                if(polysHidden) { mapContainer.classList.add('hide-all-polygons'); btnPoly.classList.add('active'); } 
                else { mapContainer.classList.remove('hide-all-polygons'); btnPoly.classList.remove('active'); } 
            }; 
            btnOut.onclick = (e) => { e.preventDefault(); e.stopPropagation(); outsHidden = !outsHidden; 
                if(outsHidden) { mapContainer.classList.add('hide-outliers'); btnOut.classList.add('active'); } 
                else { mapContainer.classList.remove('hide-outliers'); btnOut.classList.remove('active'); } 
            }; 
            btnLbl.onclick = (e) => { e.preventDefault(); e.stopPropagation(); lblsHidden = !lblsHidden; 
                if(lblsHidden) { mapContainer.classList.add('hide-labels'); btnLbl.classList.add('active'); } 
                else { mapContainer.classList.remove('hide-labels'); btnLbl.classList.remove('active'); } 
            };

            layersList.insertBefore(visualDiv, overlaysDiv); 
            
            // --- FUNCION SEGURA: APAGA TODO MENOS METS ---
            const safeUncheck = () => checkboxes.forEach(cb => {
                if (!cb.parentElement.textContent.includes('🏠 MET') && cb.checked) cb.click();
            });

            const buttonsDiv = document.createElement('div'); buttonsDiv.className = 'layer-control-buttons'; 
            buttonsDiv.innerHTML = '<button class="layer-btn select-all">✓ Todo</button><button class="layer-btn deselect-all">✗ Nada</button>'; 
            buttonsDiv.querySelector('.select-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(!cb.checked) cb.click(); }); }; 
            buttonsDiv.querySelector('.deselect-all').onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); }; 
            layersList.insertBefore(buttonsDiv, overlaysDiv); 
            
            const searchDiv = document.createElement('div'); searchDiv.className = 'search-ruta-container'; 
            searchDiv.innerHTML = `<span style="font-size:16px;margin-right:4px;">🔍</span><input type="text" class="search-ruta-input" placeholder="Ruta ID..."><button class="search-ruta-btn">Buscar</button><button class="search-ruta-clear">Limpiar</button>`; 
            const input = searchDiv.querySelector('input'); 
            
            const doSearch = () => { 
                const val = input.value.trim(); if(!val) return; 
                safeUncheck(); 
                setTimeout(() => checkboxes.forEach(cb => { 
                    if (new RegExp(`- Ruta ${val}$`).test(cb.parentElement.textContent) && !cb.checked) cb.click(); 
                }), 100); 
            }; 
            const doClear = () => { input.value = ''; safeUncheck(); }; 
            
            searchDiv.querySelector('.search-ruta-btn').onclick = (e) => { e.preventDefault(); doSearch(); }; 
            searchDiv.querySelector('.search-ruta-clear').onclick = (e) => { e.preventDefault(); doClear(); }; 
            input.onkeypress = (e) => { if(e.key === 'Enter') { e.preventDefault(); doSearch(); } }; 
            layersList.insertBefore(searchDiv, overlaysDiv); 
            
            const metDiv = document.createElement('div'); metDiv.className = 'met-buttons-row'; 
            const mets = new Set(); checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/^(?:⚠️ NO RENTABLE - )?(.+?)\s+-\s+/); if(match) mets.add(match[1].trim()); }); 
            Array.from(mets).sort().forEach(met => { const btn = document.createElement('button'); btn.className = 'met-btn'; btn.textContent = met.replace('MET ', ''); btn.onclick = (e) => { e.preventDefault(); checkboxes.forEach(cb => { if(cb.checked) cb.click(); }); setTimeout(() => { checkboxes.forEach(cb => { if(cb.parentElement.textContent.includes(met + ' -')) if(!cb.checked) cb.click(); }); }, 100); }; metDiv.appendChild(btn); }); 
            layersList.insertBefore(metDiv, overlaysDiv); 
            
            const rutaDiv = document.createElement('div'); rutaDiv.className = 'ruta-buttons-row'; 
            const rutasSet = new Set(); checkboxes.forEach(cb => { const match = cb.parentElement.textContent.match(/- Ruta\s+(\d+)/); if(match) rutasSet.add(parseInt(match[1])); }); 
            Array.from(rutasSet).sort((a,b)=>a-b).forEach(rutaId => { const btn = document.createElement('button'); btn.className = 'ruta-btn'; btn.textContent = 'R' + rutaId; btn.onclick = (e) => { e.preventDefault(); safeUncheck(); setTimeout(() => { checkboxes.forEach(cb => { const regex = new RegExp(`- Ruta ${rutaId}$`); if(regex.test(cb.parentElement.textContent)) { if(!cb.checked) cb.click(); } }); }, 100); }; rutaDiv.appendChild(btn); }); 
            layersList.insertBefore(rutaDiv, overlaysDiv); 
        } 
        setTimeout(inicializarFiltros, 1500); 
        </script>
        '''
        mapa_base.get_root().html.add_child(folium.Element(estilos_css + filtros_js))
        
        map_path = os.path.join(OUTPUT_DIR, f"Mapa_Ceylan_{run_id}.html")
        mapa_base.save(map_path)
        xls_path = generate_consolidated_excel(export_rows, summary_rows, OUTPUT_DIR, filename_suffix=run_id)
        
        prog_bar.value = prog_bar.max; prog_label.value = "¡Finalizado!"
        print(f"\n✅ FINALIZADO."); print(f"🗺️ Mapa: {os.path.basename(map_path)}"); print(f"📊 Excel: {os.path.basename(xls_path)}")
        btn_run.disabled = False

btn_run.on_click(click_ejecutar)

print("================== CEYLAN ALFA 7.5 (FINAL: FILTROS CHIDOS + DASHBOARD + BARRA) ==================")
display(widgets.VBox([
    widgets.HBox([widgets.Label("1. DATOS:"), upl_widget]), out_upl,
    met_box, widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("2. PARÁMETROS:"), slider_min, slider_max]),
    widgets.HBox([widgets.Label("3. MUESTREO:"), inp_sample, chk_full]),
    widgets.HTML("<br>"),
    widgets.HBox([prog_bar, prog_label]),
    btn_run, out_run
]))

================== CEYLAN ALFA 7.5 (FINAL: FILTROS CHIDOS + DASHBOARD + BARRA) ==================
